# Deteksi Fault 4 Sensor Kelembaban Tanah: EDM-Fuzzy vs JSD-Fuzzy + ANN-LM

**Notebook sidang interaktif.** Semua angka dibaca dari hasil penelitian yang sudah
diekspor; tidak ada model yang dilatih ulang di sini, jadi tiap sel selesai dalam
hitungan detik.

Cara pakai: **Run All** sekali, lalu ubah pilihan di panel kontrol (hari, sensor,
jenis fault, intensitas). Grafik dan tebakan model ikut berubah.

In [ ]:
# === Setup 1/3: pustaka angka ===
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd

In [ ]:
# === Setup 2/3: grafik dan widget ===
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, HTML, Markdown

In [ ]:
# === Setup 3/3: lokasi file, mode presentasi ===

# Lokal: repo penelitian. Kaggle: file dataset `sidang-bahan` (datar) disusun ulang
# ke struktur folder repo di /kaggle/working/bahan memakai symlink.
BERKAS = {"cache_interaktif.npz": "sidang/assets", "model_sidang.npz": "sidang/assets",
          "11_rq5_f1_berpasangan.csv": "hasil11/exports", "11_sensor_ringkas.csv": "hasil11/exports",
          "statistik_sel.csv": "sidang/qa/hasil_uji", "statistik_lipatan.csv": "sidang/qa/hasil_uji",
          "efek_purge_permutasi.csv": "sidang/qa/hasil_uji"}
ROOT = next((p for p in [Path(".."), Path(".")]
             if (p / "sidang" / "assets" / "cache_interaktif.npz").exists()), None)
if ROOT is None:
    import glob
    ROOT = Path("/kaggle/working/bahan")
    for nama, folder in BERKAS.items():
        asal = glob.glob(f"/kaggle/input/**/{nama}", recursive=True)
        if asal:
            (ROOT / folder).mkdir(parents=True, exist_ok=True)
            tujuan = ROOT / folder / nama
            if not tujuan.exists():
                tujuan.symlink_to(asal[0])
C = np.load(ROOT / "sidang" / "assets" / "cache_interaktif.npz")
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})

_gaya = w.Output()
def _mode(presentasi):
    with _gaya:
        _gaya.clear_output()
        if presentasi:
            display(HTML("<style>.catatan-internal{display:none!important}</style>"))
tombol_mode = w.ToggleButton(value=False, description="Mode presentasi", icon="eye-slash",
                             tooltip="Sembunyikan semua catatan internal")
tombol_mode.observe(lambda ch: _mode(ch["new"]), "value")
display(w.HBox([tombol_mode, _gaya]))
print("Folder kerja:", ROOT.resolve())

<details class="catatan-internal">
<summary><b>Catatan internal — cara memakai notebook ini</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

- **Mode presentasi**: tekan tombol di atas. Semua kotak "Catatan internal" hilang
  dari layar. Kalau tombol tidak bekerja (mis. di VS Code atau tampilan statis
  Kaggle), buka salinan `04_Sidang_Interaktif_presentasi.ipynb` yang memang tidak
  berisi catatan internal.
- **Alur bicara** (±15 menit): masalah → data → fault → entropi (EDM vs JSD) →
  ANN-LM → tebakan model → hasil → statistik → kelemahan → penutup.
- **Aturan menjawab**: angka hanya boleh disebut kalau tampil di notebook atau
  berstatus COCOK di `sidang/qa/01_klaim_bukti.csv`. Kalau ragu: "saya tunjukkan
  di notebook", lalu buka bagian yang relevan.

#### Tanya-jawab penguji

**T1. Bagaimana memakai notebook ini saat sidang?**
- *30 detik:* "Notebook ini memuat model yang sudah dilatih dan fitur yang
  sudah dihitung, jadi setiap sel selesai di bawah dua detik. Bapak/Ibu bisa
  memilih hari, sensor, jenis gangguan, dan intensitasnya, lalu melihat sinyal,
  kurva entropi, dan tebakan model berubah."
- *Penjelasan:* melatih ulang model butuh jam (NB11 di Kaggle
  ±10 jam). Saat presentasi kita tidak melatih; kita **memuat** bobot jaringan
  yang sudah tersimpan, lalu hanya melakukan perkalian matriks (forward pass).
  Itu seperti memakai kalkulator yang rumusnya sudah jadi.
- *Bukti:* `sidang/bukti_demo/04_Sidang_Interaktif.waktu.csv` (13 sel kode,
  total 9,7 s, sel terlama 1,7 s); `sidang/qa/test_notebook_sidang.py`
  (540 kombinasi panel tanpa galat, tiap panggilan < 1 s).

**T2. Apakah model di notebook ini sama dengan model penelitian?**
- *30 detik:* "Sama persis. Bobotnya diekspor dari model tersimpan N = 700 dan
  prediksinya dibandingkan dengan model asli: selisih maksimum nol."
- *Penjelasan:* model asli disimpan sebagai objek Python yang butuh
  scikit-learn (lambat diimpor, ±3–6 detik). Kita ambil angka-angkanya saja
  (bobot W, bias b, rata-rata dan skala normalisasi) ke file `.npz`, lalu
  menulis ulang forward pass dengan numpy. Supaya yakin tidak ada salah salin,
  keluaran kedua versi dibandingkan pada semua jendela uji.
- *Bukti:* `sidang/qa/buat_cache_interaktif.py` fungsi `ekspor_model`
  (assert selisih < 1e-9, terukur 0,00e+00); uji
  `test_bobot_numpy_sama_dengan_model_tersimpan`.

**T3. Kenapa catatan ini tidak tampil saat presentasi?**
- *30 detik:* "Catatan internal hanya untuk persiapan saya. Ada tombol mode
  presentasi dan ada salinan presentasi yang sel catatannya dibuang otomatis."
- *Bukti:* `sidang/qa/ekspor_notebook_sidang.py` (gagal kalau masih ada sel
  bertanda `catatan-internal`); `sidang/04_Sidang_Interaktif_presentasi.ipynb`.

#### Bukti

- Gerbang angka: `./venv/bin/python sidang/qa/cek_klaim.py` → `GERBANG KLAIM: LULUS`.
- Gerbang notebook: `sidang/qa/jalankan_notebook.py` (waktu tiap sel dicatat).

</details>

## 1. Masalahnya

Sensor kelembaban tanah di sistem irigasi bisa rusak diam-diam: nilainya bergeser,
melonjak, merayap naik, atau macet. Sistem yang percaya pada sensor rusak akan
menyiram terlalu banyak atau terlalu sedikit.

**Pertanyaan penelitian:** kalau sinyal 4 sensor dipotong per jendela waktu dan
tiap sensor diringkas dengan **entropi multiskala**, apakah mengganti ukuran
kemiripan **Euclidean (EDM-Fuzzy)** dengan **Jensen–Shannon Divergence
(JSD-Fuzzy)** membuat ANN-LM lebih baik mengenali (1) ada tidaknya fault,
(2) jenis/kombinasi fault, dan (3) sensor mana yang rusak?

<details class="catatan-internal">
<summary><b>Catatan internal — masalah dan kontribusi</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

Bayangkan empat termometer di satu ruangan. Kalau satu menunjukkan angka aneh,
kita curiga termometer itu rusak karena tiga lainnya "normal". Penelitian ini
melakukan hal yang sama untuk sensor kelembaban tanah, tetapi tidak membandingkan
angkanya langsung. Yang dibandingkan adalah **pola** sinyalnya: seberapa teratur
atau acak naik-turunnya. Ukuran keteraturan itu disebut **entropi**.

Entropi dihitung dengan cara membandingkan potongan-potongan kecil sinyal: "apakah
potongan ini mirip potongan itu?". Cara mengukur "mirip" itulah yang diganti:
- **EDM** (Euclidean): jarak garis lurus antar angka, seperti penggaris.
- **JSD**: tiap potongan diubah jadi histogram kecil, lalu dibandingkan bentuk
  histogramnya.

Kontribusinya: (1) JSD sebagai ukuran kemiripan di dalam fuzzy entropy multiskala,
(2) evaluasi berantai sampai identifikasi sensor, (3) pelaporan jujur kapan
keunggulannya signifikan dan kapan tidak. Rumusan kebaruan yang aman ada di
`sidang/qa/07_pustaka_dan_kebaruan.md`.

#### Tanya-jawab penguji

**T4. Penelitian ini tentang apa? (Q1)**
- *30 detik:* "Mendeteksi gangguan pada empat sensor kelembaban tanah dari
  pola sinyalnya. Sinyal dipotong menjadi jendela, tiap sensor diringkas dengan
  entropi multiskala, lalu jaringan saraf yang dilatih Levenberg–Marquardt
  menebak jenis gangguan dan sensor yang rusak. Yang saya uji: mengganti ukuran
  kemiripan Euclidean (EDM-Fuzzy) dengan Jensen–Shannon (JSD-Fuzzy)."
- *Penjelasan:* bayangkan empat termometer di satu kebun. Kalau satu
  mulai "melantur", angka rata-ratanya mungkin tetap wajar, tetapi *polanya*
  berubah: terlalu rata (macet), tiba-tiba melonjak (spike), atau pelan-pelan
  bergeser (drift). Entropi adalah angka yang mengukur seberapa "rumit" pola
  itu. Jaringan saraf belajar dari 60 angka entropi (4 sensor × 15 skala) untuk
  memutuskan "normal atau rusak, dan rusak jenis apa".
- *Bukti:* NB11 §0; `deteksi_fault_core.py` (pipeline lengkap).

**T5. Apa kebaruannya? (Q5)**
- *30 detik:* "Tempat JSD dipasang: sebagai jarak antar pasangan template di
  dalam fuzzy entropy. Penelusuran pustaka tidak menemukan karya yang sama;
  yang ada memakai jarak geometris di dalam entropi, atau memakai JSD pada level
  seluruh sinyal."
- *Penjelasan:* fuzzy entropy membandingkan setiap potongan pendek
  sinyal (template) dengan potongan lain. Biasanya "beda" diukur dengan
  penggaris (jarak Euclidean). Di sini setiap potongan diubah dulu menjadi
  histogram kecil, lalu dua histogram dibandingkan dengan JSD — ukuran beda dua
  sebaran peluang.
- *Bukti:* `07_pustaka_dan_kebaruan.md` (b) tabel prior-art dan log pencarian
  (f); sumber baseline Zhou dkk. 2021, IEEE TII, doi:10.1109/TII.2020.3009139.

**T6. EDM-Fuzzy itu buatan Anda?**
- *30 detik:* "Bukan. EDM-Fuzzy dari Zhou dkk. 2021. Saya memakainya sebagai
  pembanding dan mengganti hanya ukuran jaraknya."
- *Bukti:* `pustaka.bib` kunci `zhou2021edmfuzzy`.

**T7. Kenapa empat sensor, bukan satu, dan kenapa tidak dirata-rata? (Q2, Q13)**
- *30 detik:* "Rata-rata menyembunyikan sensor mana yang rusak dan menurunkan
  skor: F1 0,821 untuk empat kanal terpisah vs 0,748 setelah dirata-rata."
- *Penjelasan:* kalau sensor A naik 10 dan sensor B turun 10,
  rata-ratanya tidak berubah — gangguannya hilang dari pandangan.
- *Bukti:* NB02.

#### Bukti

- `Bahan_Sidang.md` Q1–Q5; `sidang/qa/07_pustaka_dan_kebaruan.md` (prior-art).

</details>

## 2. Data: 4 sensor, ±98 hari

Sensor mencatat tiap 30 detik; broker menyinkronkan keempatnya lalu merata-rata
per 5 menit. Grafik di bawah: rata-rata per jam, seluruh periode.

In [ ]:
jam = C["ringkas_per_jam"]
hari = np.arange(len(jam)) / 24
fig, ax = plt.subplots(figsize=(11, 3))
for j in range(4):
    ax.plot(hari, jam[:, j], lw=0.8, label=f"S{j+1}")
for h0 in C["hari"]:
    ax.axvspan(h0, h0 + int(C["win"]) * 300 / 86400, color="0.85", zorder=0)
ax.set_xlabel("hari sejak awal rekaman"); ax.set_ylabel("kelembaban (%)")
ax.set_title("Data lapangan (abu-abu = tiga jendela yang dipakai di panel interaktif)")
ax.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()
print(f"{int(C['n_sampel_5menit']):,} titik 5 menit = {int(C['n_sampel_5menit']) * 300 / 86400:.1f} hari")

<details class="catatan-internal">
<summary><b>Catatan internal — data, sinkronisasi, nilai nol S4</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

- **Broker** = penampung yang menerima data keempat sensor dan menyusunnya di satu
  sumbu waktu. Sensor tidak dirata-rata menjadi satu karena identitas tiap sensor
  diperlukan untuk menunjuk sensor yang rusak (notebook 02).
- **5 menit**: 10 titik 30 detik dirata-rata jadi satu. Derau berkurang, pola
  harian tetap.
- **Tidak ada baris dibuang karena nilainya aneh.** Nilai macet dan nol di S4
  dilaporkan, bukan dibuang, karena justru bisa gejala fault.
- Bagian grafik yang lurus sempurna di keempat sensor ada di file data asli,
  bukan hasil pengolahan.

#### Tanya-jawab penguji

**T8. Data dari mana, berapa banyak? (Q6, Q7)**
- *30 detik:* "Empat sensor kelembaban tanah, 14 September–21 Desember 2025,
  sampel tiap 30 detik, 281.721 baris. Saya rata-ratakan per 5 menit menjadi
  28.173 baris, 97,8 hari."
- *Penjelasan:* 30 detik → 5 menit artinya setiap 10 angka diganti
  rata-ratanya. Contoh: 40, 41, 40, 42, 41, 40, 41, 40, 42, 41 → 40,8. Derau
  kecil teredam, pola harian tetap.
- *Bukti:* NB11 §1–§2; `data_sensor.csv`.

**T9. Nilai nol di S4 diapakan? (Q9)**
- *30 detik:* "Tidak dibuang, dilabeli normal, dan dilaporkan sebagai batasan:
  11.371 sampel (4,0 %), tiga blok macet di 16–21 Desember plus 50 sampel
  pendek."
- *Penjelasan:* nol masih di rentang fisik 0–100 %, jadi tidak bisa
  disebut "salah ketik". Pembacaan macet di nol justru gejala gangguan; kalau
  dibuang, kita menyembunyikan kasus sulit.
- *Bukti:* NB11 §2b; Bahan Q9.

**T10. Missing value dan outlier? Sinkronisasi waktu? (Q8, Q14, Q15)**
- *30 detik:* "Tidak ada NaN dan tidak ada nilai di luar 0–100 %. Stempel waktu
  diubah ke UTC, diurutkan, duplikat dibuang, lalu dipaksa ke grid 5 menit;
  slot bolong nol."
- *Bukti:* NB11 §2 sel 7, §2b sel 8.

**T11. "Broker" itu apa? (Q12)**
- *30 detik:* "Komponen yang menyatukan empat aliran sensor menjadi satu tabel
  berwaktu sama tanpa melebur identitas sensornya; bukan message broker
  produksi."
- *Bukti:* NB02 sel 1–13; `deteksi_fault_core.sync_broker`.

#### Bukti

- Ledger: B-09a…g (nilai nol S4), R-ILU1…5 (mutu data), P-08 (segmen lurus).
- Notebook: `Ilustrasi_Data_Akuisisi_dan_Fault.ipynb` §1–§2, `07` §3b.

</details>

In [ ]:
# === Panel kontrol (dipakai bagian 3, 4, dan 6) ===
pil_hari = w.Dropdown(options=[(f"hari {int(h)}", i) for i, h in enumerate(C["hari"])], description="Jendela")
pil_sensor = w.Dropdown(options=[(f"S{s+1}", s) for s in range(4)], value=1, description="Sensor rusak")
pil_fault = w.Dropdown(options=[(f, i) for i, f in enumerate(C["fault"])], value=4, description="Fault")
pil_int = w.Dropdown(options=[(f"×{int(k)}" + (" (penelitian)" if k == 1 else ""), i)
                              for i, k in enumerate(C["intensitas"])], description="Intensitas")
PANEL = w.HBox([pil_hari, pil_sensor, pil_fault, pil_int])
KONTROL = dict(hi=pil_hari, s=pil_sensor, fi=pil_fault, ki=pil_int)

def hidupkan(fn, kontrol=None, panel=None):
    # Tampilkan hasil fn(**pilihan) sebagai output sel biasa (terlihat juga di tampilan
    # statis) lalu perbarui di tempat setiap kali pilihan panel berubah.
    kontrol = kontrol or KONTROL
    display(panel or PANEL)
    pegangan = []
    def render(*_):
        isi = fn(**{k: v.value for k, v in kontrol.items()})
        if not pegangan:
            pegangan.extend(display(x, display_id=True) for x in isi)
        else:
            for h, x in zip(pegangan, isi):
                h.update(x)
        plt.close("all")
    for v in kontrol.values():
        v.observe(render, "value")
    render()

display(PANEL)

## 3. Fault disuntikkan ke data asli

Data lapangan dianggap **bersih**; fault dibuat dengan simulator lalu disuntikkan
ke satu sensor. Empat jenis: **drift** (merayap naik pelan), **spike** (lonjakan
sesaat), **bias** (bergeser tetap), **hardware** (macet/nilai hilang); "semua" =
keempatnya sekaligus.

In [ ]:
_t_jam = np.arange(int(C["win"])) * 300 / 3600
FIG_F, AX_F = plt.subplots(1, 2, figsize=(11, 2.8), gridspec_kw={"width_ratios": [3, 1]})
L_asli, = AX_F[0].plot(_t_jam, _t_jam * 0, color="tab:blue", lw=1, label="asli")
L_rusak, = AX_F[0].plot(_t_jam, _t_jam * 0, color="tab:red", lw=0.9, alpha=0.8, label="diberi fault")
L_beda, = AX_F[1].plot(_t_jam, _t_jam * 0, color="tab:red", lw=0.8)
AX_F[0].set_xlabel("jam dalam jendela (N = 700 = 58 jam)"); AX_F[0].legend(fontsize=8)
AX_F[1].set_title("selisih (fault − asli)", fontsize=9); AX_F[1].set_xlabel("jam")
FIG_F.tight_layout(); plt.close(FIG_F)

def gambar_fault(hi, s, fi, ki):
    # satu figur dipakai ulang; hanya data garis yang diganti (respons cepat)
    bersih, rusak = C["sig_bersih"][hi][:, s], C["sig_fault"][fi, ki, s, hi]
    L_asli.set_ydata(bersih); L_rusak.set_ydata(rusak); L_beda.set_ydata(rusak - bersih)
    AX_F[0].set_ylabel(f"S{s+1} (%)")
    for ax in AX_F:
        ax.relim(); ax.autoscale_view()
    return [FIG_F, Markdown(f"Porsi sampel S{s+1} yang kena fault di jendela ini: "
                            f"**{C['porsi'][fi, ki, s, hi]:.0%}**")]

hidupkan(gambar_fault)

<details class="catatan-internal">
<summary><b>Catatan internal — simulator fault</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

- Fault **disimulasikan** karena data lapangan tidak punya label "rusak" yang
  pasti. Dengan simulasi kita tahu persis kapan dan di sensor mana fault terjadi,
  jadi jawaban benarnya diketahui.
- Rumus singkat (lihat `deteksi_fault_core.py` §3):
  drift = tambah laju × waktu, hanya di 3 episode 3 hari lalu di-reset;
  spike = lonjakan berkala setinggi 0,08 × SD; bias = + 0,08 poin tetap;
  hardware = ±3 % sampel diganti nilai dari waktu lain (macet) dan 5 % hilang
  (NaN, lalu diisi nilai sebelumnya). Parameter tertulis 8 %, tetapi sampel yang
  hilang menimpa sebagian sampel macet (kelemahan F4).
- Intensitas ×10 dan ×50 di panel hanya untuk **melihat**; model dilatih dengan ×1.
  Uji intensitas yang sebenarnya ada di notebook 13.
- Pertanyaan jebakan: "bias 0,08 poin itu kecil sekali?" — ya, dan itu sebabnya
  bias hampir tidak terdeteksi (lihat bagian 4 dan kelemahan B2).

#### Tanya-jawab penguji

**T12. Kenapa gangguannya disimulasikan? (Q10)**
- *30 detik:* "Data asli tidak berlabel rusak dan saya tidak boleh merusak
  sensor. Simulasi memberi label pasti dan bisa diulang dengan seed tetap.
  Batasnya: hasil berlaku untuk gangguan yang dimodelkan."
- *Penjelasan:* ini seperti latihan pemadam kebakaran dengan api
  buatan: kita tahu persis di mana apinya, jadi bisa menilai apakah petugas
  menemukannya.
- *Bukti:* NB11 §3; `02_kelemahan.md` B (validitas konstruk).

**T13. Tulis model tiap gangguan. (Q17, P17)**
- *30 detik:* "Drift: tambahan yang naik linear 1 poin/hari selama episode
  3 hari. Spike: lonjakan sesaat dengan peluang 1,5 %. Bias: geser tetap.
  Hardware: sebagian titik macet atau hilang."
- *Penjelasan:* sinyal 40, 40, 40, 40.
  Bias +2 → 42, 42, 42, 42. Spike di titik ke-3 → 40, 40, 47, 40.
  Drift → 40, 40,003, 40,007, 40,010 (1 poin/hari = 0,00347 poin per 5 menit).
  Macet → 40, 41, 41, 41 (nilai terakhir diulang).
- *Bukti:* `deteksi_fault_core.py` `simulate_drift_fault`,
  `simulate_spike_fault`, `simulate_bias_fault`, `simulate_hardware_fault`;
  `04_walkthrough_kode.md` baris 18.

**T14. Kenapa drift hanya 1 poin/hari? (Q11, Q42)**
- *30 detik:* "Agar tidak mustahil secara fisik: versi awal 57,6 poin/hari dan
  menembus 100 % dalam sehari. Rentang 0,003–0,17 poin/hari yang saya pakai
  sebagai pembanding adalah **asumsi skenario**; saya tidak menemukan sumber
  terbit yang mengukur laju drift sensor kelembaban tanah per hari."
- *Penjelasan:* dalam jendela N = 200 (16,7 jam) drift hanya
  menambah ±0,7 poin, lebih kecil dari ayunan harian sinyal. Karena itu drift
  memang paling sulit dideteksi, dan itu dilaporkan.
- *Bukti:* README "Koreksi simulator drift"; `07_pustaka_dan_kebaruan.md` (e).
- *Serangan lanjutan:* "Jadi skenario Anda tidak realistis?" → "Arahnya
  realistis (lambat), besarnya asumsi. Uji intensitas NB13 menunjukkan drift
  baru terdeteksi (AUC 0,80–0,85) pada total 60 poin."

**T15. Hardware malfunction 8 % macet dan 5 % hilang? (F4)**
- *30 detik:* "Total titik rusak 8 %: 5 % hilang (NaN) dan 3 % macet, karena
  titik hilang adalah bagian dari titik macet lalu ditimpa NaN. Deskripsi lama
  saya koreksi."
- *Bukti:* `deteksi_fault_core.py` `simulate_hardware_fault`;
  `02_kelemahan.md` F4.

**T16. Kenapa bias konstan tidak terdeteksi? (Q47)**
- *30 detik:* "Karena setiap template dikurangi rata-ratanya sebelum
  dibandingkan. Geser tetap ikut hilang. AUC bias sekitar 0,50 dari 0,08 sampai
  5 poin."
- *Penjelasan:* template [42, 43] dikurangi rata-rata 42,5 menjadi
  [−0,5, 0,5]; template tanpa bias [40, 41] juga menjadi [−0,5, 0,5]. Entropi
  melihat keduanya identik. Panel fault "bias" di notebook menunjukkan kurva
  entropi hampir tidak bergerak.
- *Bukti:* NB12 `12_sweep_fault_param.csv`; `embed_matrix_centered`.

**T17. Kapan jendela dilabel rusak? (Q22, P18)**
- *30 detik:* "Kalau porsi titik rusak di jendela > 1 %; jendela di bawah itu
  dibuang supaya model tidak belajar label yang salah."
- *Bukti:* `FAULT_RATIO_THR`; `11_akuntansi_window.csv`.

#### Bukti

- Ledger: R13-* (uji intensitas), B-11 (laju drift).
- `sidang/qa/02_kelemahan.md` B1–B3, F4–F5; notebook 13 dan 14.

</details>

## 4. Entropi multiskala: EDM-Fuzzy vs JSD-Fuzzy

Tiap sensor diringkas menjadi **15 angka** (satu per skala τ = 1..15): seberapa
tidak teratur polanya ketika sinyal dilihat makin kasar. Empat sensor × 15 = **60
fitur** per jendela. Garis putus-putus = setelah fault.

In [ ]:
T_SK = C["fit_fault"].shape[-1]; _tau = np.arange(1, T_SK + 1)
FIG_E, AX_E = plt.subplots(1, 2, figsize=(11, 2.8), sharex=True)
L_E = [(ax.plot(_tau, _tau * 0.0, "o-", ms=3, label="asli")[0],
        ax.plot(_tau, _tau * 0.0, "s--", ms=3, label="fault")[0]) for ax in AX_E]
for ax in AX_E:
    ax.set_xlabel("skala τ"); ax.legend(fontsize=8)
AX_E[0].set_ylabel("entropi"); FIG_E.tight_layout(); plt.close(FIG_E)

def gambar_entropi(hi, s, fi, ki):
    beda = []
    for mi, ax in enumerate(AX_E):
        asli = C["fit_bersih"][mi, hi, s * T_SK:(s + 1) * T_SK]
        rusak = C["fit_fault"][mi, fi, ki, s, hi]
        L_E[mi][0].set_ydata(asli); L_E[mi][1].set_ydata(rusak)
        ax.set_title(f"{C['metode'][mi]} — S{s+1}", fontsize=10)
        ax.relim(); ax.autoscale_view()
        beda.append(np.abs(rusak - asli).mean())
    return [FIG_E, Markdown("Rata-rata perubahan entropi akibat fault: " +
                            " · ".join(f"{C['metode'][mi]} **{beda[mi]:.3f}**" for mi in range(2)))]

hidupkan(gambar_entropi)

<details class="catatan-internal">
<summary><b>Catatan internal — fuzzy entropy, EDM, JSD</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

Langkah fuzzy entropy untuk satu sensor pada satu skala (contoh angka lengkap di
`sidang/qa/03_algoritma_dengan_tangan.md`):
1. **Coarse-graining komposit**: rata-rata tiap τ titik berurutan (skala τ),
   diulang untuk τ titik awal yang berbeda lalu hasil entropinya dirata-rata.
2. **Embedding**: potong jadi vektor pendek panjang m = 2 (dan m + 1 = 3), kurangi
   rata-rata tiap vektor (*centering*). Akibatnya pergeseran tetap (bias) hilang.
3. **Kemiripan**: bandingkan tiap vektor dengan 128 vektor acuan acak, lalu
   μ = 1 / (1 + d / r).
   - EDM: d = jarak Euclidean antar vektor, r = 0,2 × SD sinyal.
   - JSD: tiap vektor diubah jadi histogram lunak 8 bin (kernel Gauss di rentang
     ±3 SD), d = √JSD antar histogram (jarak Jensen–Shannon), r = 0,2 × √ln 2.
4. **Entropi** = ln(rata-rata kemiripan m ÷ rata-rata kemiripan m+1).
   Sinyal teratur → pola panjang tetap mirip → entropi kecil.

Kenapa JSD bisa lebih peka: ia membandingkan **bentuk sebaran** nilai di dalam
potongan, bukan jarak titik ke titik, sehingga lonjakan dan macet mengubah bentuk
histogram walau jaraknya kecil. Harganya: ekstraksi JSD lebih lambat.

#### Tanya-jawab penguji

**T18. Apa itu entropi di sini? (Q3, Q23, P9)**
- *30 detik:* "Fuzzy entropy mengukur seberapa sering pola pendek yang mirip
  tetap mirip ketika diperpanjang satu titik. Sinyal teratur: entropi kecil.
  Sinyal acak: entropi besar."
- *Penjelasan:* rumusnya FE = ln(φ_m / φ_{m+1}). φ_m adalah rata-rata
  "kemiripan" antar semua pasangan template sepanjang m = 2. Kalau pasangan yang
  mirip di panjang 2 tetap mirip di panjang 3, φ hampir sama dan ln-nya dekat 0.
  Contoh angka pada deret x = [3, 5, 4, 6, 5, 7, 6, 4]: r = 0,2 · sd =
  0,261861; jarak Euclidean template [−1, 1] dan [0,5, −0,5] = 2,121320,
  keanggotaan μ = 1/(1 + 2,121320/0,261861) = 0,109879.
- *Bukti:* `03_algoritma_dengan_tangan.md` (b), dicek otomatis oleh
  `cek_algoritma_tangan.py`; Chen dkk. 2007 (`chen2007fuzzyen`).

**T19. Kenapa multiskala τ = 1..15 dan 60 fitur? (Q24, Q28)**
- *30 detik:* "Gangguan berbeda tampak di skala berbeda: spike di skala halus,
  drift di skala kasar. 15 skala × 4 sensor = 60 fitur."
- *Penjelasan:* skala τ = 2 artinya rata-ratakan tiap dua titik:
  [3, 5, 4, 6] → [4, 5]. Komposit: hitung juga dengan geseran awal (mulai dari
  titik kedua) lalu rata-ratakan hasilnya, supaya tidak bergantung pada titik
  awal.
- *Bukti:* `coarse_grain_cmse`; Costa dkk. 2002, Wu dkk. 2013.

**T20. Di mana tepatnya EDM dan JSD berbeda? (Q4, Q26, Q27, P10)**
- *30 detik:* "Hanya di jarak d antar template. EDM: jarak Euclidean. JSD:
  tiap template diubah jadi histogram lunak 8 bin, jarak = akar JSD. Embedding,
  centroid shifting, fungsi keanggotaan, dan rumus entropi identik."
- *Penjelasan:* EDM seperti membandingkan dua foto piksel demi
  piksel. JSD seperti membandingkan dua foto dari histogram warnanya saja.
  Contoh JSD dari hitungan tangan: d = √0,011811 = 0,108678, ambang
  r_JSD = 0,2·√ln2 = 0,166511, sehingga μ = 1/(1 + 0,108678/0,166511) = 0,605078.
- *Bukti:* `deteksi_fault_core.py` `phi_euclidean`, `vectors_to_distributions`,
  `pairwise_jsd`, `phi_jsd`; `03_algoritma_dengan_tangan.md` (c).

**T21. Kenapa r = 0,2·SD dan r_JSD = 0,2·√ln2? (Q25)**
- *30 detik:* "0,2·SD adalah toleransi lazim dari literatur sample/fuzzy
  entropy, diskalakan dengan simpangan baku supaya adil untuk amplitudo
  berbeda. JSD (log natural) terbatas di [0, ln 2], jadi akar JSD maksimum √ln2;
  ambangnya 20 % dari maksimum itu."
- *Bukti:* `JSD_R_RATIO`, `jsd_fuzzy_entropy_1d`; Richman & Moorman 2000.

**T22. Sifat JSD yang penting? (P11)**
- *30 detik:* "Simetris, selalu berhingga, terbatas [0, ln 2], dan akarnya
  metrik. KL tidak simetris dan bisa tak hingga kalau ada bin nol."
- *Bukti:* Lin 1991; Endres & Schindelin 2003.

**T23. Apa kelemahan konseptual JSD-Fuzzy? (F7)**
- *30 detik:* "Histogram tidak peduli urutan. Template naik [−1, 1] dan turun
  [1, −1] menghasilkan histogram sama, jadi JSD = 0 persis, padahal Euclidean
  memberi 2,83. JSD-Fuzzy mengukur kemiripan sebaran nilai, bukan bentuk pola.
  Akibat lain: nilainya bisa negatif pada deret pendek."
- *Penjelasan:* kata "TAS" dan "SAT" punya huruf yang sama. Kalau
  hanya menghitung huruf, keduanya dianggap kembar.
- *Bukti:* `02_kelemahan.md` F7 (dicek 2026-11-20 dengan
  `vectors_to_distributions` + `pairwise_jsd`); `07_pustaka_dan_kebaruan.md`
  catatan jujur.
- *Serangan lanjutan:* "Kalau buta urutan, kenapa JSD menang?" → "Gangguan yang
  diuji mengubah sebaran amplitudo di dalam template (spike, macet); itu yang
  ditangkap JSD. Untuk gangguan yang hanya membalik arah, JSD tidak peka. Ini
  saya tulis sebagai batasan."

**T24. Kenapa n_ref = 128 titik acuan? (P14)**
- *30 detik:* "Semua pasangan berbiaya kuadrat; 128 acuan acak dengan seed
  tetap memberi aproksimasi yang sama untuk kedua metode."
- *Bukti:* `_pick_ref`.

**T25. Kenapa JSD lebih lambat? (P15)**
- *30 detik:* "Tiap template harus diubah menjadi histogram dan JSD butuh
  logaritma per bin; Euclidean cukup selisih kuadrat. Biaya ada di ekstraksi
  fitur; inferensi per jendela setara."
- *Bukti:* NB14 `hasil14` (1,7–6,8× pada ekstraksi fitur).

#### Bukti

- Kode: `deteksi_fault_core.py` `edm_fuzzy_entropy_1d`, `jsd_fuzzy_entropy_1d`, `pairwise_jsd`.
- Pemeriksa contoh tangan: `sidang/qa/cek_algoritma_tangan.py`.
- Ledger H14-42*, H14-45c/d (biaya ekstraksi).

</details>

## 5. ANN dilatih dengan Levenberg–Marquardt (LM)

60 fitur → jaringan saraf kecil (1–2 lapisan tersembunyi) → kelas. Bobot dicari
dengan LM: langkah **(JᵀJ + λ·diag(JᵀJ))·Δ = −Jᵀr**. λ kecil ≈ Gauss–Newton
(cepat dekat minimum), λ besar ≈ turun gradien (hati-hati jauh dari minimum).
Coba geser λ pada contoh mini di bawah (satu neuron, dua bobot).

In [ ]:
rng_toy = np.random.default_rng(0)
x_toy = np.linspace(0, 4, 30); y_toy = 2.0 * np.tanh(0.8 * x_toy) + rng_toy.normal(0, 0.05, 30)

def lm_mini(lam0, adaptif, iters=25):
    th = np.array([0.2, 3.0]); lam = lam0; hist = []
    for _ in range(iters):
        a, b = th; z = np.tanh(b * x_toy); r = a * z - y_toy
        J = np.column_stack([z, a * x_toy * (1 - z**2)])
        A = J.T @ J; g = J.T @ r
        step = np.linalg.solve(A + lam * np.diag(np.diag(A)) + 1e-12 * np.eye(2), -g)
        th_baru = th + step
        r_baru = th_baru[0] * np.tanh(th_baru[1] * x_toy) - y_toy
        if (r_baru**2).sum() < (r**2).sum():
            th = th_baru; lam = lam / 3 if adaptif else lam
        else:
            lam = lam * 2 if adaptif else lam
        hist.append(((th[0] * np.tanh(th[1] * x_toy) - y_toy) ** 2).mean())
    return th, hist

def gambar_lm(log_lambda, adaptif):
    th, hist = lm_mini(10.0 ** log_lambda, adaptif)
    fig, axs = plt.subplots(1, 2, figsize=(11, 2.6))
    axs[0].semilogy(hist, "o-", ms=3); axs[0].set_xlabel("iterasi"); axs[0].set_ylabel("MSE")
    axs[0].set_title(f"λ awal = {10.0 ** log_lambda:g}, {'adaptif' if adaptif else 'tetap'}", fontsize=10)
    axs[1].plot(x_toy, y_toy, "k.", label="data")
    axs[1].plot(x_toy, th[0] * np.tanh(th[1] * x_toy), "r-", label="hasil fit"); axs[1].legend(fontsize=8)
    plt.tight_layout()
    return [fig]

K_LM = dict(log_lambda=w.FloatSlider(value=-2, min=-4, max=3, step=0.5, description="log10 λ"),
            adaptif=w.Checkbox(value=True, description="λ adaptif (seperti penelitian)"))
hidupkan(gambar_lm, K_LM, w.HBox(list(K_LM.values())))

<details class="catatan-internal">
<summary><b>Catatan internal — ANN-LM dan cara evaluasi</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

- **Kenapa LM**: untuk jaringan kecil dan data ratusan sampel, LM memakai
  informasi lengkung (JᵀJ) sehingga butuh jauh lebih sedikit iterasi daripada
  Adam/gradien biasa. Notebook 09 membandingkan LM, Bayesian regularization, GA,
  dan Adam: LM ≈ BR, keduanya jauh di atas GA dan Adam.
- **Implementasi sendiri** dengan Jacobian analitik (turunan dihitung rumus, bukan
  beda-hingga), aturan λ Nielsen. Klasifikasi dilakukan dengan target one-hot +
  MSE, lalu softmax untuk peluang.
- **Evaluasi**: 5 lipatan *StratifiedGroupKFold*; grup = blok waktu, supaya jendela
  dari waktu yang sama tidak ada di data latih dan uji sekaligus. Arsitektur dipilih
  di dalam lipatan latih saja (nested CV).
- Contoh mini di atas: λ tetap kecil kadang melompat terlalu jauh, λ tetap besar
  lambat; λ adaptif menggabungkan keduanya.

#### Tanya-jawab penguji

**T26. ANN-nya seperti apa? (Q29)**
- *30 detik:* "Feed-forward, 60 masukan, satu atau dua lapisan tersembunyi
  dari kisi 8 arsitektur, aktivasi relu/tanh, keluaran softmax. Arsitektur
  dipilih di dalam cross-validation."
- *Bukti:* NB11 §10; `models/meta.json`.

**T27. Tulis rumus update LM dan arti λ. (P1, P2, P3)**
- *30 detik:* "(JᵀJ + λ·diag(JᵀJ))·Δ = −Jᵀr. λ kecil: mendekati Gauss–Newton,
  cepat dekat minimum. λ besar: langkah kecil searah gradien, aman. Galat naik:
  langkah ditolak, λ dikali 2. Galat turun: diterima, λ dikecilkan menurut
  rasio ρ (aturan Nielsen)."
- *Penjelasan:* LM seperti menuruni bukit dalam kabut. Kalau
  langkah besar membuat Anda naik (galat bertambah), Anda mundur dan mencoba
  langkah lebih kecil. Di contoh mainan: iterasi 1 ditolak (ρ = −0,58),
  λ 0,001 → 0,002; pada iterasi 9 galat turun dari 1,453047 ke 0,194302.
- *Bukti:* `03_algoritma_dengan_tangan.md` (a); `lm_fit`; Marquardt 1963,
  Hagan & Menhaj 1994, Madsen dkk. 2004.

**T28. Bagaimana Jacobian analitik dihitung dan dibuktikan? (Q31, P4)**
- *30 detik:* "Backpropagation per keluaran. Dibuktikan dengan membandingkan
  terhadap Jacobian numerik beda hingga."
- *Penjelasan:* turunan numerik: geser satu bobot sedikit (ε), lihat
  berapa keluaran berubah, bagi ε. Kalau rumus analitik dan cara numerik
  memberi angka sama sampai ±10⁻¹⁰, rumusnya benar.
- *Bukti:* NB11 §9a; `03` (d); `cek_algoritma_tangan.py` (Jacobian tangan vs
  kode, selisih 5,6e-17).

**T29. LM itu untuk regresi; klasifikasinya bagaimana? (P5)**
- *30 detik:* "Target one-hot, kuadrat terkecil pada keluaran linear, softmax
  saat prediksi — cara yang lazim di `trainlm` MATLAB. Kelemahannya
  probabilitasnya tidak terkalibrasi."

**T30. Kenapa LM, bukan Adam? (Q30, P7)**
- *30 detik:* "LM metode pada paper acuan dan cocok untuk jaringan kecil. Di
  arsip NB09 LM 0,433, BR 0,429, GA 0,306, Adam 0,268; LM dan BR tidak berbeda,
  keduanya jauh di atas GA dan Adam."
- *Bukti:* NB09 §10.

**T31. Kenapa jaringan dibatasi kecil? (P6)**
- *30 detik:* "JᵀJ berukuran P×P dan harus diselesaikan tiap iterasi, biayanya
  O(P³). Karena itu parameter dibatasi sekitar 5 ribu."

**T32. Jelaskan nested grouped CV. (Q33, P20)**
- *30 detik:* "Luar: StratifiedGroupKFold 5 lipatan, grup = blok waktu. Dalam:
  2 lipatan untuk memilih arsitektur dengan macro-F1. Model terpilih dilatih
  ulang di data latih luar dan dinilai di data uji luar."
- *Penjelasan:* jendela bertetangga berbagi 50 % isi. Kalau satu di
  latih dan tetangganya di uji, model "sudah pernah melihat" separuh soal ujian.
  Mengelompokkan per blok waktu memisahkan soal latihan dan soal ujian.
- *Bukti:* NB11 §10; Roberts dkk. 2017, Bergmeir & Benítez 2012.

**T33. Kenapa macro-F1? (P23)**
- *30 detik:* "Kelas tidak seimbang; macro-F1 memberi bobot sama pada tiap
  kelas, sehingga kelas sulit tidak tertutup kelas mudah."
- *Penjelasan:* 90 jendela normal dan 10 rusak. Model yang selalu
  menjawab "normal" akurasinya 90 %, tetapi F1 kelas rusak 0. Macro-F1 =
  rata-rata (F1 normal ≈ 0,95, F1 rusak 0) ≈ 0,47 — angka itu jujur.

**T34. Bobot awal Xavier? (A5, F2)**
- *30 detik:* "Jujur: urutan `lm_pack`/`lm_unpack` berbeda, sehingga sebagian
  bobot lapisan keluaran mulai dari nol. Konsisten saat latih dan prediksi,
  sama untuk kedua metode. Teks skripsi tidak menyebut 'Xavier dengan bias
  nol'."
- *Bukti:* `02_kelemahan.md` A5, F2.

#### Bukti

- Kode: `deteksi_fault_core.py` `lm_fit`, `lm_residual_jacobian`, `inner_search_best`.
- Ledger R09-* (perbandingan pelatih), B-29 (grid 8 arsitektur).
- `sidang/qa/04_walkthrough_kode.md`, `05_bank_pertanyaan.md` §1.

</details>

## 6. Tebakan model tersimpan untuk jendela pilihan

Model N = 700 yang disimpan (`models/`) dipakai langsung: **S1** menjawab "normal
atau rusak", model skenario sesuai jumlah fault menjawab jenisnya, dan model
sensor-ID menunjuk sensor yang rusak (skor > 0,5).

In [ ]:
# Bobot 10 model N = 700 dari models/ (diekspor ke numpy oleh sidang/qa/buat_cache_interaktif.py,
# prediksinya sudah dicek sama dengan model asli; selisih < 1e-9).
MZ = np.load(ROOT / "sidang" / "assets" / "model_sidang.npz")
META = json.loads(str(MZ["_meta"]))

def maju(kunci, X, sensor=False):
    X = np.array(X, float, copy=True)
    nan = np.isnan(X)
    if nan.any():
        X[nan] = np.broadcast_to(MZ[kunci + "|fill"], X.shape)[nan]
    a = (X - MZ[kunci + "|mean"]) / MZ[kunci + "|scale"]
    n, act = int(MZ[kunci + "|n"]), str(MZ[kunci + "|act"])
    for i in range(n):
        a = a @ MZ[f"{kunci}|W{i}"] + MZ[f"{kunci}|b{i}"]
        if i < n - 1:
            a = np.maximum(a, 0.0) if act == "relu" else np.tanh(a)
    if sensor:
        return a
    e = np.exp(a - a.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

MODELS = {}
for k in sorted({"|".join(x.split("|")[:2]) for x in MZ.files if not x.startswith("_")}):
    sc, met = k.split("|")
    MODELS[(sc, met)] = {"class_names": [str(c) for c in MZ[k + "|kelas"]],
                         "detektor": lambda X, k=k: maju(k + "|det", X),
                         "sensor_id": lambda X, k=k: maju(k + "|sen", X, sensor=True),
                         "cv_f1_nb11": float(MZ[k + "|cv_f1_nb11"])}
print(f"{len(MODELS)} model dimuat | dibuat {META['dibuat']} | commit {META['commit']} | N = {META['window']}")

In [ ]:
def tebak(hi, s, fi, ki):
    T = C["fit_fault"].shape[-1]
    sken = "S5" if C["fault"][fi] == "semua" else "S2"
    baris, sens = [], []
    for mi, met in enumerate(C["metode"]):
        f_asli = C["fit_bersih"][mi, hi].copy()
        f_rusak = f_asli.copy(); f_rusak[s * T:(s + 1) * T] = C["fit_fault"][mi, fi, ki, s, hi]
        X2 = np.vstack([f_asli, f_rusak])
        for sc in ["S1", sken]:
            b = MODELS[(sc, met)]; P = b["detektor"](X2)
            for wi, nama in enumerate(["asli", "diberi fault"]):
                k = int(P[wi].argmax())
                baris.append({"Jendela": nama, "Model": sc, "Metode": met,
                              "Tebakan": b["class_names"][k], "Yakin": f"{P[wi, k]:.0%}"})
        skor = MODELS[(sken, met)]["sensor_id"](f_rusak[None])[0]
        sens.append({"Metode": met, **{f"S{j+1}": round(float(v), 2) for j, v in enumerate(skor)},
                     "Ditebak rusak": ", ".join(f"S{j+1}" for j, v in enumerate(skor) if v > 0.5) or "-",
                     "Benar": f"S{s+1}"})
    benar_s = "drift+spike+bias+HW" if sken == "S5" else str(C["fault"][fi])
    return [Markdown(f"**Jawaban benar:** jendela asli = *fault free*; jendela fault = *faulty* (S1) "
                     f"dan *{benar_s}* ({sken}). Intensitas latih = ×1."),
            pd.DataFrame(baris), Markdown("**Sensor mana yang rusak** (skor > 0,5 = rusak):"),
            pd.DataFrame(sens)]

hidupkan(tebak)

<details class="catatan-internal">
<summary><b>Catatan internal — membaca tebakan model</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

- Satu contoh bisa benar bisa salah. Peluang dekat 50 % artinya model ragu. Yang
  adil adalah angka CV (bagian 7), bukan satu contoh.
- Model dilatih pada **semua** jendela skenario lalu disimpan; akurasi pada data
  latih bukan ukuran generalisasi. Angka resmi = F1 CV notebook 11.
- Model hanya mengenal intensitas ×1. Pada ×10/×50 tebakan bisa aneh karena
  polanya di luar data latih — itu jawaban jujur kalau ditanya.
- Demo cadangan dengan banyak jendela acak: `sidang/03_Demo_Model_Tersimpan.ipynb`
  Langkah 6.

#### Tanya-jawab penguji

**T35. Bagaimana membaca tebakan di panel?**
- *30 detik:* "Tabel pertama: tebakan model N = 700 (S1 normal/rusak dan S2
  atau S5 jenis gangguan) untuk jendela asli dan jendela yang diberi gangguan,
  EDM dan JSD berdampingan, dengan tingkat keyakinan. Tabel kedua: skor tiap
  sensor; skor > 0,5 berarti sensor itu ditebak rusak."
- *Penjelasan:* softmax mengubah skor mentah menjadi peluang yang
  jumlahnya 1. Skor [2, 0] → e²/(e² + e⁰) = 7,39/8,39 = 0,88 untuk kelas
  pertama.
- *Bukti:* `maju` di `buat_cache_interaktif.py`; sel prediksi notebook.

**T36. Kenapa tebakan kadang salah di panel?**
- *30 detik:* "Karena F1 validasi silang skenario ini memang belum sempurna,
  dan panel memakai satu jendela, bukan rata-rata. Contoh salah lebih jujur
  daripada demo yang dipilih-pilih."
- *Bukti:* `|cv_f1_nb11` di `model_sidang.npz`; `11_rq4_performa_ann.csv`.

**T37. Bisa dipakai real-time? (Q43, P32)**
- *30 detik:* "Belum. Satu keputusan N = 700 butuh 58 jam data; cocok untuk
  pemantauan harian. Untuk notifikasi cepat ada NB15 (per sampel, tanpa
  entropi) dengan aturan persistensi."
- *Bukti:* NB15, NB15b.

#### Bukti

- `models/meta.json` (arsitektur, cv_f1_nb11), `sidang/PANDUAN_DEMO.md`.
- Ledger P-04…P-06 (bukti run demo `sidang/bukti_demo/`).

</details>

## 7. Hasil utama (notebook 11, LM asli, 15 sel = 5 skenario × 3 panjang jendela)

Nilai = F1 makro rata-rata 5 lipatan CV. Hijau = JSD lebih tinggi.

In [ ]:
f1 = pd.read_csv(ROOT / "hasil11" / "exports" / "11_rq5_f1_berpasangan.csv")
f1 = f1[pd.to_numeric(f1["N_window"], errors="coerce").notna()].copy()
f1["N"] = f1["N_window"].astype(float).astype(int)
tabel = f1.pivot(index="Scenario", columns="N", values="Difference")
display(Markdown("**Selisih F1 (JSD − EDM)** per sel:"))
display(tabel.style.format("{:+.3f}").background_gradient(cmap="RdYlGn", vmin=-0.1, vmax=0.1))
r = f1[["EDM-Fuzzy F1", "JSD-Fuzzy F1"]].astype(float)
print(f"JSD menang {int((f1['Difference'].astype(float) > 0).sum())} dari {len(f1)} sel | "
      f"rata-rata F1 EDM {r['EDM-Fuzzy F1'].mean():.3f} vs JSD {r['JSD-Fuzzy F1'].mean():.3f}")
sid = pd.read_csv(ROOT / "hasil11" / "exports" / "11_sensor_ringkas.csv")
display(Markdown("**Identifikasi sensor** (dirantai dari prediksi out-of-fold):")); display(sid)

<details class="catatan-internal">
<summary><b>Catatan internal — cara membaca hasil</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

- **F1 makro**: rata-rata F1 semua kelas, sehingga kelas kecil dihitung sama
  penting. Akurasi bisa menipu kalau satu kelas dominan.
- S1 (normal vs rusak) dan S5 (empat fault sekaligus) paling mudah; S2–S4 sulit
  karena kelasnya banyak dan beberapa fault (bias, drift kecil) hampir tak terlihat.
- Kalimat aman: "JSD-Fuzzy unggul **konsisten** (menang di sebagian besar sel),
  tetapi pada protokol utama (NB11) keunggulan gabungannya **belum signifikan**;
  dengan fault bentuk episode (NB14) keunggulannya signifikan."

#### Tanya-jawab penguji

**T38. Hasil utamanya? (Q35, Q36)**
- *30 detik:* "JSD-Fuzzy lebih tinggi di 11 dari 15 sel, rata-rata selisih
  F1 +0,028. Per ukuran jendela: N = 200 menang 3/5, N = 700 5/5,
  N = 1000 3/5."
- *Penjelasan:* satu sel = satu skenario (S1–S5) × satu ukuran
  jendela (200/700/1000). Di tiap sel kedua metode dilatih dengan protokol yang
  sama; kita catat selisih F1-nya.
- *Bukti:* NB11 §10; `11_rq4_selisih.csv`; `statistik_sel.csv` baris NB11.

**T39. Hasil identifikasi sensor? (Q37, F1)**
- *30 detik:* "JSD 0,675 vs EDM 0,634, menang 9 dari 15. Catatan jujur: tahap
  ini memakai pembagian acak 70/30, jadi angka absolutnya optimistis; selisih
  kedua metode tetap adil karena pembagiannya sama."
- *Bukti:* NB11 §12; `11_sensor_ringkas.csv`; `02_kelemahan.md` F1.

**T40. JSD lebih tinggi, EDM lebih apa?**
- *30 detik:* "EDM lebih stabil (56 dari 63 pasangan). Keduanya saya
  tampilkan."
- *Bukti:* Bahan "Kalimat siap pakai" 3.

**T41. NB10, NB11, NB14 — kenapa ada tiga hasil?**
- *30 detik:* "NB10 dan NB11 protokol sama, berbeda anggaran sesi; NB14
  memakai simulator fault 'sedang' dengan posisi acak ber-seed. Arah ketiganya
  sama: JSD lebih tinggi."
- *Bukti:* `statistik_sel.csv` (NB10 11/15, NB11 11/15, NB14 12/15).

#### Bukti

- Ledger R11-01…R11-18, H14-01… (NB14).
- Notebook 11 §10–§12, notebook 14; `hasil11/Ringkasan_untuk_laporan.md`.

</details>

## 8. Seberapa kuat buktinya?

Uji berpasangan per notebook, ditambah ukuran efek (Cohen dz), interval
kepercayaan bootstrap 95 % untuk selisih rata-rata, dan koreksi Holm untuk tiga
panjang jendela.

In [ ]:
st = pd.read_csv(ROOT / "sidang" / "qa" / "hasil_uji" / "statistik_sel.csv")
kol = ["notebook", "N", "menang_JSD", "n_sel", "rerata_selisih", "p_t", "p_wilcoxon",
       "p_t_holm_3N", "cohen_dz", "ci95_bawah", "ci95_atas"]
display(st[kol].style.format({c: "{:.3f}" for c in kol[4:]}, na_rep="—"))
p_lip = ROOT / "sidang" / "qa" / "hasil_uji" / "statistik_lipatan.csv"
if p_lip.exists():
    display(Markdown("**Tingkat lipatan, N = 700** (arsitektur tetap; t terkoreksi Nadeau–Bengio):"))
    display(pd.read_csv(p_lip).round(3))

<details class="catatan-internal">
<summary><b>Catatan internal — statistik dan cara membela angka p</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

- **Uji-t berpasangan**: tiap sel punya dua angka (EDM, JSD) dari data dan lipatan
  yang sama, jadi yang diuji adalah selisihnya.
- **Wilcoxon** tidak mengandaikan selisih berdistribusi normal. Dengan 5 pasangan,
  p terkecil yang mungkin 0,0625, jadi Wilcoxon per-N tidak akan pernah < 0,05.
- **Holm**: tiga uji (N = 200/700/1000) sekaligus memperbesar peluang "kebetulan
  signifikan"; Holm menaikkan p untuk mengoreksinya.
- **Cohen dz** = rata-rata selisih ÷ simpangan baku selisih: ukuran efek yang
  tidak tergantung jumlah sel. ±0,2 kecil, ±0,5 sedang, ±0,8 besar.
- **Bootstrap CI**: ambil ulang 15 sel secara acak 10.000 kali; kalau rentang
  tidak memuat 0, selisihnya konsisten positif. Dengan hanya 15 sel, CI ini
  cenderung sempit; laporkan berdampingan dengan p, bukan pengganti.
- **Nadeau–Bengio**: lipatan CV berbagi data latih, sehingga uji-t biasa terlalu
  optimistis; koreksinya memperbesar variansi dengan faktor (1/k + n_uji/n_latih).

#### Tanya-jawab penguji

**T42. Apakah hasilnya signifikan? (Q39, Q41)**
- *30 detik:* "NB11 gabungan belum: uji-t p = 0,076, Wilcoxon 0,107. Ukuran
  efek sedang (d_z = 0,49), dan interval bootstrap 95 % selisih F1 [+0,002;
  +0,058] tidak memuat nol. Jadi saya menyebutnya konsisten lebih tinggi, bukan
  terbukti. NB14 gabungan signifikan: p = 0,0006, d_z = 1,13, CI [+0,043;
  +0,108]."
- *Penjelasan:* p menjawab "kalau sebenarnya tidak ada beda,
  seberapa sering kita melihat beda sebesar ini karena kebetulan?". p = 0,076
  berarti sekitar 7,6 %, di atas batas 5 %. Bootstrap: ambil ulang 15 selisih
  secara acak dengan pengembalian 10.000 kali, hitung rata-rata tiap kali, lalu
  lihat 2,5 % dan 97,5 % terbawah/teratas.
- *Bukti:* `sidang/qa/hasil_uji/statistik_sel.csv`; `statistik_tambahan.py`.
- *Serangan lanjutan:* "CI tidak memuat nol tapi p > 0,05 — mana yang benar?"
  → "Keduanya pendekatan berbeda pada n = 15: bootstrap persentil cenderung
  sempit pada sampel kecil, uji-t lebih konservatif. Saya ambil yang
  konservatif untuk klaim."

**T43. N = 700 signifikan, kan? (Q41)**
- *30 detik:* "Tanpa koreksi p = 0,027; setelah koreksi Holm untuk tiga ukuran
  jendela p = 0,081. Jadi tidak saya klaim signifikan setelah koreksi."
- *Penjelasan:* menguji tiga hal sekaligus menaikkan peluang satu
  'beruntung'. Holm: urutkan p, kalikan yang terkecil dengan 3, berikutnya
  dengan 2, dan seterusnya (dengan aturan tidak boleh turun).
- *Bukti:* `statistik_sel.csv` kolom `p_t_holm_3N`; Holm 1979.

**T44. Uji-t pada lipatan CV itu bias, kan? (P22, C2)**
- *30 detik:* "Ya, lipatan berbagi data latih sehingga varians diremehkan.
  Saya menambahkan uji-t terkoreksi Nadeau–Bengio. Uji utama saya per sel
  (skenario × N), bukan per lipatan."
- *Penjelasan:* koreksi Nadeau–Bengio memperbesar varians dengan
  faktor (1/k + n_uji/n_latih); hasilnya |t| selalu lebih kecil.
- *Bukti:* `statistik_tambahan.py` `t_nadeau_bengio`, keluaran per lipatan
  `hasil_uji/statistik_lipatan.csv` (dari run purge N = 700); Nadeau & Bengio
  2003; Dietterich 1998; uji `test_nadeau_bengio_*`.

**T45. Kenapa Wilcoxon per N tidak pernah signifikan? (F6)**
- *30 detik:* "Dengan lima pasangan, p eksak dua sisi terkecil 2/2⁵ = 0,0625.
  Jadi secara matematis tidak mungkin < 0,05."

**T46. Ukuran efek? (P25)**
- *30 detik:* "d_z NB11 gabungan 0,49 (sedang), NB14 1,13 (besar)."
- *Bukti:* `statistik_sel.csv` kolom `cohen_dz`; Cohen 1988.

#### Bukti

- `sidang/qa/statistik_tambahan.py` + `test_statistik_tambahan.py` (6 uji, bersumber pustaka).
- `sidang/qa/02_kelemahan.md` C1–C6, F6; ledger R11-03…05.

</details>

## 9. Kelemahan yang kami ketahui (dan apa yang sudah diukur)

In [ ]:
p_ef = ROOT / "sidang" / "qa" / "hasil_uji" / "efek_purge_permutasi.csv"
if p_ef.exists():
    ef = pd.read_csv(p_ef)
    display(Markdown("**Purge** = buang jendela latih yang bertumpang-tindih waktu dengan jendela uji. "
                     "**Permutasi** = label latih diacak; F1 harus turun ke tingkat tebakan."))
    display(ef.round(3))
else:
    print("Hasil uji purge/permutasi belum tersedia.")

| Jenis | Kelemahan | Dampak |
|---|---|---|
| Internal | Jendela bersebelahan berbagi setengah sampel di batas grup CV | skor absolut sedikit optimistis; berlaku sama untuk EDM dan JSD |
| Konstruk | Fault disimulasikan, bukan kerusakan nyata | hasil berlaku untuk bentuk fault yang dimodelkan |
| Konstruk | *Centering* menghapus bias tetap | bias tidak terdeteksi oleh kedua metode |
| Statistik | 15 sel tidak saling bebas; banyak uji | p gabungan dibaca hati-hati, dilengkapi Holm/dz/CI |
| Eksternal | Satu lokasi, 4 sensor, ±98 hari | perlu validasi di lokasi/sensor lain |

<details class="catatan-internal">
<summary><b>Catatan internal — menjawab serangan validitas</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

Pola jawaban tiap kelemahan: **akui → dampak → mitigasi**, dua kalimat. Contoh
untuk kebocoran CV: "Pengelompokan per blok waktu mencegah jendela yang sama bocor,
tetapi jendela di batas blok masih berbagi setengah sampelnya, jadi skor
absolutnya sedikit optimistis. Karena berlaku sama untuk EDM dan JSD, selisihnya
tetap sah, dan kami mengukurnya: lihat tabel purge di atas."

#### Tanya-jawab penguji

**T47. Apakah masih ada kebocoran data? (A1, P21)**
- *30 detik:* "Ada sedikit di batas grup: jendela terakhir satu blok bisa
  tumpang-tindih dengan jendela pertama blok tetangga. Sama untuk kedua metode.
  Saya ukur dengan purge: jendela latih yang tumpang-tindih dengan jendela uji
  dibuang, lalu model N = 700 dilatih ulang."
- *Penjelasan:* purge = membuang soal latihan yang isinya beririsan
  dengan soal ujian. Kalau skor turun banyak setelah purge, berarti skor lama
  terbantu kebocoran.
- *Hasil:* purge mengubah F1 −0,072 sampai +0,012 (7 dari 10 sel turun);
  JSD ≥ EDM tetap di 5 dari 5 skenario. Purge juga mengurangi data latih ±20 %
  (S1: 126 → 100 jendela), jadi penurunan skor tidak seluruhnya akibat
  kebocoran.
- *Bukti:* `sidang/qa/uji_purge_permutasi_N700.py`,
  `hasil_uji/efek_purge_permutasi.csv`, `hasil_uji/statistik_lipatan.csv`;
  `02_kelemahan.md` A1.

**T48. Bagaimana tahu model tidak belajar artefak? (kontrol negatif)**
- *30 detik:* "Label saya acak lalu model dilatih ulang dengan protokol sama.
  Kalau pipeline jujur, F1-nya turun ke tingkat tebak-tebakan."
- *Penjelasan:* untuk 2 kelas, tebakan acak memberi F1 sekitar 0,5.
  Kalau model dengan label acak masih jauh di atas itu, ada yang bocor.
- *Hasil:* F1 dengan label acak 0,446–0,508 (S1 dan S5, kedua metode),
  setara tebakan 0,5; model asli 0,64–0,83.
- *Bukti:* `hasil_uji/efek_purge_permutasi.csv` kolom `F1_permutasi`.

**T49. Validitas eksternal? (D)**
- *30 detik:* "Satu lokasi, satu musim, empat sensor, gangguan disimulasikan.
  Hasil berlaku untuk gangguan yang dimodelkan; langkah berikutnya data lapangan
  yang benar-benar rusak."
- *Bukti:* `02_kelemahan.md` D.

**T50. Data asli atau buatan? (B6, P16)**
- *30 detik:* "Gangguan disuntikkan secara simulasi ke `data_sensor.csv`."
  Asal-usul `data_sensor.csv` **belum dipastikan**: baris awalnya tampak hasil
  interpolasi ke grid 30 detik. Jawab hanya setelah dikonfirmasi ke pembimbing;
  jangan mengarang.
- *Bukti:* `02_kelemahan.md` B6.

**T51. p = 0,0006 di NB14 — apakah karena gangguan diperbesar sampai JSD menang?
(P24, A6)**
- *30 detik:* "Simulator NB14 ditetapkan sebelum membandingkan metode dan sama
  untuk keduanya. Tetap signifikan setelah koreksi ganda."
- *Bukti:* `02_kelemahan.md` A6, C3.

#### Bukti

- `sidang/qa/02_kelemahan.md` (daftar lengkap A1–F7 dengan file:baris).
- `sidang/qa/uji_purge_permutasi_N700.py` → `hasil_uji/purge_permutasi_N700.csv`.

</details>

## 10. Kesimpulan

1. Entropi multiskala 4 sensor + ANN-LM dapat mendeteksi fault dan menunjuk sensor
   yang rusak, paling baik untuk hardware/macet dan kombinasi banyak fault.
2. Mengganti Euclidean dengan **Jensen–Shannon Divergence** memberi keunggulan yang
   konsisten; kekuatan statistiknya bergantung pada bentuk fault (lihat bagian 8).
3. Bias tetap dan drift kecil tetap sulit: keterbatasan ini berasal dari
   *centering* di dalam entropi, bukan dari classifier.

<details class="catatan-internal">
<summary><b>Catatan internal — kalimat penutup dan langkah lanjut</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

Tiga kalimat siap pakai ada di `Bahan_Sidang.md` bagian "Kalimat siap pakai".
Langkah lanjut yang masuk akal: validasi di sensor/lokasi lain, fitur tanpa
centering untuk bias, jendela lebih pendek untuk pemantauan dekat real-time,
dan CV dengan purge sebagai protokol baku.

#### Tanya-jawab penguji

**T52. Kalimat penutup.**
- "Saya mengganti ukuran kemiripan di dalam fuzzy entropy dari Euclidean
  menjadi Jensen–Shannon dan mengujinya sampai identifikasi sensor."
- "JSD-Fuzzy lebih tinggi di 11 dari 15 sel (+0,028); secara gabungan NB11
  belum signifikan (p = 0,076), sehingga saya menyebutnya konsisten, bukan
  terbukti. Pada simulator NB14 selisihnya signifikan (p = 0,0006)."
- "Batasan yang saya laporkan: gangguan disimulasikan, JSD buta urutan,
  sedikit kebocoran di batas grup, dan bias konstan tidak terdeteksi oleh
  representasi terpusat."

**T53. Langkah berikutnya? (Q44)**
- Data lapangan yang benar-benar rusak; jendela adaptif untuk deteksi lebih
  cepat; fitur level untuk bias; GroupShuffleSplit pada tahap sensor-ID.

**T54. Kalau ditanya angka yang tidak ada di layar? (Q45)**
- "Saya cek di `01_klaim_bukti.csv` atau notebook sumbernya." Jangan mengarang
  angka baru.

#### Bukti

- `Bahan_Sidang.md` Q44; `sidang/qa/06_mock_sidang.md` (latihan).

</details>

<div class="catatan-internal">

## Lampiran internal — catatan persiapan lengkap

Semua bahan persiapan sidang, satu blok per file, salinan apa adanya dari repo saat
notebook dibangun. Hanya ada di salinan internal.

</div>

<details class="catatan-internal">
<summary><b>Catatan internal — percobaan data hipotetik tanpa entropi (NB15)</b> (klik untuk buka; tidak tampil saat presentasi)</summary>

#### Penjelasan

NB15 terpisah dari perbandingan EDM vs JSD: fitur lag per sampel, tanpa entropi,
untuk sistem notifikasi. Penguji bisa bertanya tentang urutan penyisipan gangguan,
jumlah neuron output, dan ruang grid search.

#### Tanya-jawab penguji

Bagian ini untuk percobaan data hipotetik tanpa entropi (NB15,
`build_15_data_hipotetik_ann_lm_notebook.py`). Belum dipakai di notebook
interaktif.

**T55. Gangguan disisipkan bersamaan atau bertahap?**
- *30 detik:* "Ada dua arti. Antar skenario bertahap: S1 satu fault, S2 tiap
  fault tunggal, S3 dua fault, S4 tiga, S5 keempatnya. Ada 16 kondisi dan
  masing-masing punya salinan datanya sendiri. Di dalam satu kondisi kombinasi,
  semua fault disisipkan pada interval waktu yang sama, yaitu tiga episode
  tiga hari. Secara waktu fault-nya bersamaan; secara kode diterapkan
  berurutan pada sinyal yang sama."
- *Penjelasan:* kondisi `spike+bias` diproses begini. Ambil
  sinyal bersih. Kalikan 1,52 % titik di dalam episode dengan 1,3 (spike).
  Tambahkan +5 poin ke seluruh episode (bias). Terakhir potong nilainya ke
  0–100 %. Label tiap sampel berupa bitmask: drift 1, spike 2, bias 4,
  hardware 8. Titik yang kena spike dan bias berlabel 6.
- *Detail yang harus bisa dijelaskan:*
  - fault tunggal: spike dan hardware tersebar di seluruh deret; drift dan
    bias selalu di dalam episode;
  - kombinasi: semua komponen dibatasi ke episode yang sama
    (`simulate_multiple_faults`, `within = _episode_mask`);
  - urutan kode S5: spike, bias, hardware, lalu drift. Nilai macet disalin dari
    deret yang sudah diberi spike dan bias. Drift ditambahkan terakhir, dan
    titik NaN tetap NaN;
  - versi 4 sensor: fault masuk ke 1, 2, 3, atau 4 sensor, bergiliran antar
    kondisi; sensor dipilih acak dengan seed tetap. Episode sama untuk semua
    sensor. Posisi spike dan hardware berbeda per sensor karena seed-nya
    berbeda. Di S1 jenis fault diundi per sensor;
  - model per kondisi dilatih terpisah per kondisi. Model gabungan dilatih
    sekaligus pada 16 kondisi ditambah data normal.
- *Bukti:* `build_14_lengkap_fault_baru_notebook.py` `NEW_SIM`
  (`_episode_mask`, `simulate_multiple_faults`, `simulate_choose_one`);
  `build_15_…py` `CONDITIONS` dan §3 `GEN_4S` / `GEN_AVG`.

**T56. Berapa neuron output ANN-LM?**
- *30 detik:* "Dua neuron output linear, untuk normal dan fault, dengan target
  one-hot. Saat prediksi, softmax mengubahnya jadi peluang. Peluang fault
  adalah unit kedua, lalu alarm menyala kalau peluang itu di atas ambang yang
  disetel di data validasi. Program pertama (Des 2025) memakai satu output
  sigmoid."
- *Penjelasan:* masukan berisi 13 fitur per sampel: lag1–lag10,
  selisih dengan sampel sebelumnya, penanda NaN, dan jumlah NaN dalam
  10 sampel terakhir. Contoh arsitektur (16, 8):
  13 → 16 → 8 → 2, aktivasi tanh. Jumlah parameternya
  13·16 + 16 + 16·8 + 8 + 8·2 + 2 = 378.
- *Bukti:* `build_15_…py` `fit_lm_model` (`lm_fit(..., 2, activation="tanh")`,
  `np.eye(2)[y]`), `FEATURE_NAMES`, `best_threshold` (ambang 0,05–0,95, langkah
  0,01).

**T57. Grid search-nya mencari di ruang apa?**
- *30 detik:* "Hanya arsitektur lapisan tersembunyi, lima kandidat: (6), (12),
  (6,3), (12,6), (16,8). Yang lain tetap: tanh, 60 iterasi LM, maksimum
  120 ribu baris latih, seed tetap. Kriteria pemilihannya F1 validasi; data
  uji tidak dipakai. Grid ini hanya untuk model gabungan. Model per kondisi
  memakai dua arsitektur program pertama, (6) dan (6,3), dan keduanya
  dilaporkan."
- *Hasil (log Kaggle v3):* F1 validasi rata-rata 0,702–0,709, terpilih (16,8),
  F1 uji 0,753. Pada 4 sensor F1 validasi 0,547–0,560, terpilih (12,6),
  F1 uji 0,640.
- *Jujur:* selisih antar kandidat ≤ 0,013, dan semua berhenti di batas
  60 iterasi (`max_iter`). Pemilihan arsitektur praktis tidak berpengaruh.
  Di versi rata-rata, kandidat terpilih justru F1 ujinya terendah (0,753 vs
  0,754–0,765). Itu tanda pemilihan memang tidak mengintip data uji.
- *Bukti:* `build_15_…py` `GABUNGAN_GRID`, `LM_ITERS`, `MAX_TRAIN_ROWS`;
  `hasil15/data-hipotetik-berlabel-ann-lm.log` baris `grid …`; ekspor
  `15_grid_search_gabungan.csv`; `15_model_card.json` kunci `grid_search`.

#### Bukti

- `build_15_data_hipotetik_ann_lm_notebook.py`, `build_14_lengkap_fault_baru_notebook.py` (`NEW_SIM`).
- `hasil15/data-hipotetik-berlabel-ann-lm.log`.

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Bahan sidang Q1–Q48 dan kalimat siap pakai</b> (`Bahan_Sidang.md`)</summary>

# Bahan Sidang — tanya-jawab dan peta logika

Dokumen ini dipakai saat latihan tanya-jawab. Setiap jawaban menunjuk
notebook/sel supaya bisa langsung dibuka kalau penguji minta bukti.
Rujukan lengkap ada di `sidang/MAPPING.md`.

**Aturan menjawab:** angka tanpa sumber jangan disebut. Kalau tidak yakin,
jawab "saya cek di notebook 11 §10" lalu buka. Jangan mengklaim "lebih baik
secara signifikan" kecuali menyebut p < 0,05.

---

## A. Gambaran besar

**Q1. Penelitian ini tentang apa?**
Mendeteksi kerusakan sensor kelembaban tanah dari pola sinyalnya, lalu
menentukan sensor mana yang rusak dan jenis kerusakannya. Caranya: potong
sinyal jadi jendela waktu, ukur entropi multiskala tiap sensor, beri ke ANN.
Yang diuji: mengganti ukuran kemiripan Euclidean (EDM-Fuzzy) dengan
Jensen–Shannon Divergence (JSD-Fuzzy). Rujukan: `11` §0 sel 1.

**Q2. Kenapa empat sensor, bukan satu?**
Karena satu sensor tidak bisa memberi tahu apakah nilai aneh itu gangguan
lokal atau pola normal. Empat kanal yang disinkronkan mempertahankan identitas
masing-masing, sehingga kerusakan bisa dilokalisasi per sensor. Rujukan: `02`
dan `11` §2.

**Q3. Kenapa entropi, bukan rata-rata/standar deviasi?**
Rata-rata dan standar deviasi hanya menangkap besar dan sebaran nilai.
Entropi menangkap **kerumitan pola** — sinyal yang macet atau berulang punya
entropi rendah walau rata-ratanya normal. Itu yang membuat fault seperti
*stuck* dan *drift* lebih terlihat. Rujukan: `11` §5.

**Q4. Apa beda EDM-Fuzzy dan JSD-Fuzzy?**
Keduanya mengukur "seberapa mirip dua potongan sinyal". EDM memakai jarak
garis lurus antar angka. JSD mengubah tiap potongan menjadi sebaran nilai
(histogram kecil), lalu membandingkan bentuk sebarannya. Rujukan: `11` §5
sel 15–17 dan gambar `11_similarity_euclidean_vs_jsd.png`.

**Q5. Apa kontribusi/kebaruannya?**
Tiga hal: (1) JSD-Fuzzy sebagai pengganti similarity di dalam entropi fuzzy,
(2) evaluasi berantai sampai identifikasi sensor dan jenis fault, (3) laporan
jujur bahwa keunggulannya konsisten tetapi belum kuat secara statistik
gabungan. Rujukan: `11` §13.

## B. Data

**Q6. Data dari mana dan berapa lama?**
Empat sensor kelembaban tanah, 14 September–21 Desember 2025, sampel tiap
30 detik: 281.721 baris. Rujukan: `Ilustrasi` §1 dan `11` §1.

**Q7. Kenapa 30 detik diubah jadi 5 menit?**
Rata-rata per 5 menit meredam derau dan memperkecil jumlah sampel tanpa
membuang informasi harian. Hasilnya 28.173 baris (97,8 hari). Rujukan: `11`
§2 sel 7.

**Q8. Missing value dan outlier dibuang?**
Tidak. Di data ini tidak ada NaN dan tidak ada nilai di luar 0–100 %.
Pembacaan macet dan pencilan **dilaporkan, tidak dibuang**, karena justru bisa
jadi gejala fault. Yang dibuang hanya cacat struktural (stempel waktu ganda),
dan jumlahnya nol. Rujukan: `11` §2b sel 8.

**Q9. Nilai 0 di sensor S4 diapakan?**
Tetap dipakai dan dilabeli normal. Totalnya 11.371 sampel 30 detik (4,0 %),
berupa tiga blok macet 45 jam, 37 jam, dan 12 jam (semuanya 16–21 Des 2025),
plus 50 sampel pendek (49 di 16–21 Des dan 1 pada 8 Des 2025 pukul 14.06). Setelah agregasi 5 menit: 1.135 titik nol. Nilainya
masih di dalam rentang fisik 0–100 %, jadi tidak dibuang; ini justru gejala
pembacaan macet. Dampaknya 10 dari 280 window N=200 (3,6 %) mayoritas nol
(N=700: 2/79; N=1000: 1/55). Dicatat sebagai batasan. Rujukan: `11` §2b dan
`data_sensor.csv`.

**Q10. Kenapa fault-nya simulasi, bukan data rusak asli?**
Karena data asli tidak punya label "rusak" dan tidak boleh merusak sensor
sungguhan. Simulasi memberi label yang pasti dan bisa diulang dengan seed
tetap. Batasnya diakui: hasil berlaku untuk fault yang dimodelkan, bukan semua
kerusakan fisik. Rujukan: `11` §3.

**Q11. Kenapa drift-nya cuma 1 poin per hari?**
Agar setara drift sensor nyata (0,003–0,17 poin/hari), bukan rampa besar yang
mustahil. Drift dibuat per episode 3 hari lalu di-reset, jadi totalnya kecil
dan nilai tetap 0–100 %. Konsekuensinya drift memang sulit dideteksi — itu
dilaporkan apa adanya. Rujukan: `11` §3 sel 10.

## C. Broker dan sinkronisasi

**Q12. "Broker" di sini maksudnya apa?**
Komponen yang mengumpulkan empat aliran sensor menjadi satu tabel dan
menyelaraskan waktunya, tanpa melebur identitas sensor. Ini menyiapkan data
untuk pemodelan, bukan message broker produksi. Rujukan: `02` sel 1–13.

**Q13. Kenapa empat sensor tidak dirata-rata jadi satu?**
Karena rata-rata menyembunyikan sensor mana yang rusak. Hasilnya juga lebih
rendah: F1 0,821 (4 kanal) vs 0,748 (dirata-rata); akurasi 0,829 vs 0,726.
Rujukan: `02`.

**Q14. Bagaimana waktu disinkronkan?**
Stempel waktu diubah ke UTC, diurutkan, duplikat dibuang, lalu semua kanal
dipaksa ke grid 5 menit. Slot yang bolong ditambal interpolasi waktu.
Rujukan: `11` §2 sel 7.

**Q15. Kalau timestamp bolong banyak?**
Di dataset ini tidak ada slot bolong (0 isian). Mekanismenya tetap ada:
interpolasi waktu lalu median sebagai jaring pengaman. Rujukan: `11` §2b.

**Q16. Apa yang dibuktikan notebook 02?**
Bahwa menggabungkan empat kanal lebih baik daripada merata-ratakan, terutama
untuk spike dan hardware malfunction yang waktunya berbeda antar sensor.
Rujukan: `02` sel 14.

## D. Fault dan window

**Q17. Jenis fault-nya apa saja?**
Empat dasar: drift (geser pelan), spike (loncatan sesaat), bias (geser tetap),
hardware malfunction (macet/hilang). Kombinasi dua, tiga, dan empat dipakai
juga, total 16 kondisi fault + normal = 17 kelas. Rujukan: `11` §3.

**Q18. Bagaimana memilih sensor yang kena fault?**
Injeksi bersifat sensor-selective: satu sampai empat sensor dipilih acak,
sisanya dibiarkan bersih. Jadi model tidak bisa menebak "semua rusak".
Rujukan: `11` §3 dan `04` §6.

**Q19. Kenapa 17 kelas, bukan 5?**
17 kelas (normal + 16 kondisi fault) adalah eksperimen terpisah di notebook
`01`/`04`, bukan S5. Ladder S1–S5 di notebook `11` justru memecah tugas itu:
S1 2 kelas, S2 5, S3 7 (6 kombinasi 2 fault), S4 5 (4 kombinasi 3 fault),
S5 2 (kombinasi 4 fault). Identifikasi "sensor mana yang rusak" juga terpisah
(RQ6, `11` §12), bukan S4. Rujukan: `11` §7.

**Q20. Window N = 200 itu berapa lama?**
200 sampel × 5 menit = 1.000 menit ≈ 16,7 jam (0,69 hari). Setara N = 2000
pada laju 30 detik, konfigurasi terbaik di notebook 07. Rujukan: `11` §4.

**Q21. Kenapa stride 50 %?**
Agar jumlah window cukup banyak tanpa kehilangan peristiwa di batas. Karena
window bertetangga tumpang-tindih, cross-validation harus dikelompokkan
menurut blok waktu. Rujukan: `11` §4 dan §10.

**Q22. Kenapa window dengan fault ≤ 1 % dibuang?**
Window seperti itu isinya praktis normal; melabelinya "fault" akan mengajari
model pola yang salah. Pada N = 200, 1.992 dari 36.120 window dibuang.
Rujukan: `11_akuntansi_window.csv`.

## E. Entropi

**Q23. Entropi yang dipakai entropi apa?**
Entropi sampel/fuzzy pada sinyal yang di-embed (dimensi m = 2), dihitung pada
beberapa skala waktu. "Fuzzy" berarti kemiripan diberi bobot, bukan ya/tidak.
Rujukan: `11` §5.

**Q24. Kenapa multiskala τ = 1..15?**
Fault berbeda muncul pada kekasaran berbeda: spike terlihat di skala halus,
drift di skala kasar. 15 skala × 4 sensor = 60 fitur. Rujukan: `11` §6.

**Q25. r = 0,2 × simpangan baku itu apa?**
Toleransi "dianggap mirip" pada entropi fuzzy. Dikalikan simpangan baku supaya
adil untuk sinyal dengan amplitudo berbeda. Nilainya sama untuk EDM dan JSD,
jadi perbandingannya setara. Rujukan: `11` §5.

**Q26. EDM-Fuzzy menghitung apa?**
Jarak Euclidean antar vektor yang di-embed, lalu dipetakan menjadi bobot
kemiripan fuzzy. Semakin kecil jarak, semakin mirip. Rujukan: `11` §5.

**Q27. JSD-Fuzzy mengubah apa?**
Sebelum dibandingkan, tiap vektor diubah menjadi sebaran (8 bin, rentang
±3 simpangan baku, kernel halus). Kemiripan dihitung dari bentuk sebaran,
bukan selisih angka. Rujukan: `11` §5 sel 15–17.

**Q28. Kenapa 60 fitur, bukan 4?**
Karena tiap sensor menyumbang 15 nilai (satu per skala), lalu keempatnya
digabung. Identitas sensor dipertahankan supaya bisa dilacak. Rujukan: `11` §6.

## F. Model dan evaluasi

**Q29. ANN-nya seperti apa?**
Jaringan feed-forward: 60 masukan, satu atau dua lapisan tersembunyi dari
kisi 8 arsitektur — (16,), (32,), (64,), (16,8), (32,8), (32,16), (32,32),
(64,16) — aktivasi relu/tanh, keluaran softmax. Arsitektur
dipilih di dalam cross-validation, bukan setelah melihat data uji. Rujukan:
`11` §10.

**Q30. Kenapa Levenberg–Marquardt, bukan Adam?**
LM adalah metode pelatihan pada paper acuan dan cocok untuk jaringan kecil.
Edisi sidang hanya menampilkan ANN-LM; perbandingan dengan BR, GA, dan Adam
ada di arsip notebook `09` dan tidak ditampilkan agar protokolnya tidak
tercampur. Rujukan: `11` §10.

**Q31. Apa itu Jacobian analitik?**
Turunan error terhadap semua bobot dihitung langsung dengan aturan rantai,
bukan didekati beda hingga. Hasilnya lebih presisi dan cepat. Kebenarannya
diperiksa dengan membandingkan terhadap Jacobian numerik (§9a) — lolos.
Rujukan: `11` §9a dan `09` §9.

**Q32. Kenapa hanya ANN-LM yang ditampilkan?**
Karena fokus penelitian adalah perbandingan metode entropi (EDM vs JSD), bukan
perbandingan classifier. Pelatih lain dan XGBoost sudah diuji di arsip `09`
dan `06`, tetapi tidak dimunculkan supaya klaim tetap satu protokol: ANN-LM
asli dengan Jacobian analitik. Rujukan: `11` §10.

**Q33. Kenapa cross-validation harus dikelompokkan?**
Window bertetangga tumpang-tindih 50 %. Kalau diacak biasa, potongan sinyal
yang sama bisa muncul di data latih dan uji — skor jadi terlalu optimistis.
GroupKFold per blok waktu mencegah kebocoran itu. Rujukan: `11` §10.

**Q34. Apakah program penelitian masih satu alur?**
Ya. Notebook `11` tetap satu program end-to-end: muat data → broker → injeksi
fault → segmentasi → entropi → ANN-LM → evaluasi → sensor-ID. Notebook sidang
adalah lapisan penyaji: ia membaca CSV ekspor dan memeriksa angkanya dengan
`assert`, tanpa melatih ulang. Rujukan: `11` §2–§13.

## G. Hasil

**Q35. Hasil utamanya apa?**
JSD-Fuzzy menang di 11 dari 15 sel (5 skenario × 3 ukuran window), rata-rata
selisih F1 +0,028. Identifikasi sensor juga lebih baik: 0,675 vs 0,634
(menang 9/15). Rujukan: `11` §10 dan §12.

**Q36. Berapa skenario yang dimenangkan?**
11 dari 15 sel. Kalau dihitung per ukuran window: N = 200 menang 3/5,
N = 700 menang 5/5, N = 1000 menang 3/5. Rujukan: `11_rq4_selisih.csv`.

**Q37. Skenario mana yang paling mudah dikenali sensornya?**
S5 (F1 0,841), lalu S4 (0,735), S3 (0,635), S2 (0,622), S1 (0,543) — angka
JSD-Fuzzy per skenario. Rujukan: `11_sensor_ringkas.csv`.

**Q38. Apa bedanya notebook sidang dengan notebook `11` selain drift?**
`11` menghitung semuanya (RQ1–RQ6, CV entropi, separabilitas, statistik,
confusion matrix, sensor-ID); notebook sidang hanya menampilkan performa utama
dan sensor-ID. Sidang memakai bahasa sederhana + aset gambar, sedangkan `11`
memuat kode pipeline lengkap. Angka keduanya sama karena sama-sama dari `11`
versi v3 (setelah koreksi drift). Rujukan: `11` §10–§12.

**Q39. Apakah hasilnya signifikan?** *(tidak ditampilkan di notebook)*
Tidak secara gabungan: pooled p = 0,076 (Wilcoxon 0,107). Hanya N = 700 yang
signifikan (p = 0,027). Karena itu klaim yang benar: **konsisten lebih tinggi**,
bukan "terbukti signifikan". Rujukan: `11` §11 (lampiran privat).

**Q40. Kalau begitu, kenapa tetap dipakai?** *(tidak ditampilkan di notebook)*
Karena arahnya konsisten (11/15), keunggulan muncul di skenario sulit, dan
biayanya sebanding. Nilainya sebagai bukti pendukung, bukan klaim kemenangan
mutlak. Rujukan: `11` §13.

## H. Batasan dan tindak lanjut

**Q41. Kenapa N = 700 signifikan tetapi gabungan tidak?** *(tidak ditampilkan di notebook)*
Karena arah efek berbeda antar ukuran window (ada sel yang negatif di N = 1000).
Menggabungkan tiga kondisi menambah varians, sehingga rasio efek/derau tidak
naik. Uji per ukuran window lebih tepat dibaca satu per satu. Rujukan:
`11_rq5_ttest.csv` + `11` §11.

**Q42. Kenapa drift sulit dideteksi?**
Karena drift hanya 1 poin/hari; dalam window 16,7 jam pergeserannya di bawah
ayunan alami sinyal. Deteksi drift lebih bergantung pada pola entropi jangka
panjang daripada selisih nilai. Rujukan: `11` §3 dan `11` §10.

**Q43. Apakah bisa dipakai real-time?**
Belum. Satu keputusan butuh window 16,7 jam, jadi cocok untuk pemantauan
harian, bukan alarm detik. Untuk real-time perlu window lebih pendek dan
pelatihan ulang. Rujukan: `11` §4 dan `11_rq4_biaya_komputasi.csv`.

**Q44. Apa langkah berikutnya?**
(1) Uji pada data lapangan yang benar-benar rusak, (2) window adaptif untuk
deteksi lebih cepat, (3) uji jumlah sensor lebih banyak, (4) kalibrasi ambang
keputusan untuk operasi irigasi. Rujukan: `11` §13 dan `08` §12.

**Q45. Kalau penguji menanyakan angka yang tidak ada di notebook?**
Angka pendukung (statistik, pelatih lain, XGBoost, jenis fault, two-stage)
tetap tersedia di `Bahan_Sidang.md` dan arsip kernel penelitian, tetapi
sengaja tidak dimunculkan di notebook sidang supaya klaim tetap satu
protokol: ANN-LM. Sebutkan sumbernya, jangan mengarang angka baru.

**Q46. Mengapa dari uji sensitivitas hanya hardware malfunction yang terdeteksi?**
Karena hardware merusak distribusi secara kasar: dengan parameter bawaan
(0,08; 0,05) AUC sudah 0,897, dan pada (0,3; 0,2) TPR 0,908 dengan FPR 0,083.
Fault halus tetap di bawah ambang: drift 20 poin/hari hanya TPR 0,667
(FPR 0,183), spike 1,5 SD TPR 0,854 (FPR 0,154). Rujukan: `12`,
`12_nilai_representatif.csv`.

**Q47. Mengapa bias konstan tidak terdeteksi sama sekali?**
Bukan karena nilainya kecil, tetapi karena representasi. *Centroid shifting*
mengurangi rata-rata tiap vektor sebelum entropi dihitung, sehingga offset tetap
hilang. AUC bias tetap sekitar 0,50 dari 0,08 sampai 5 poin. Untuk mendeteksi
bias perlu fitur level (rata-rata window) atau tanpa centroid shifting; itu di
luar lingkup notebook 12. Rujukan: `12_sweep_fault_param.csv`.

**Q48. Setelan soft binning mana yang disarankan?**
Yang benar-benar diuji: `bins=4`, `h_ratio=1,0`, `range_sd=3,0`,
`r_ratio=0,2`; F1 rata-rata validasi N700/S1 = 0,805 (N700 0,766; S1 0,844).
Di N200, `bins=4`, `h_ratio=0,5` sedikit lebih tinggi (F1 0,735), tetapi belum
divalidasi N700. Jangan mengarang nilai interpolasi. Rujukan: `12`,
`12_rekomendasi.csv`.

---

## Kalimat siap pakai (hafalkan tiga ini)

1. "Saya mengganti ukuran kemiripan di dalam entropi fuzzy dari jarak
   Euclidean menjadi Jensen–Shannon Divergence, lalu menguji dampaknya sampai
   identifikasi sensor."
2. "JSD-Fuzzy menang di 11 dari 15 sel dengan selisih rata-rata +0,028;
   secara gabungan belum signifikan (p = 0,076), jadi saya menyebutnya
   konsisten, bukan terbukti."
3. "Yang saya laporkan apa adanya: JSD lebih tinggi, tetapi EDM lebih stabil
   (56 dari 63 pasangan). Keduanya saya tampilkan."

## Peta cepat kalau penguji menunjuk layar

| Penguji menunjuk | Buka |
|---|---|
| Grafik 4 kanal | `11` §2 atau `ILU_empat_kanal_bersih.png` |
| Contoh sinyal kena fault | `11` §3b, `ILU_grid_fault_vs_asli_S1.png` |
| Tabel 17 kelas | `04` §15, `05` §13 |
| Grafik EDM vs JSD | `11` §5, `11_similarity_euclidean_vs_jsd.png` |
| Tabel F1 utama | `11` §10, `11_rq4_performa_ann.csv` |
| Identifikasi sensor | `11` §12, `11_sensor_ringkas.csv` |
| Uji statistik | `11` §11 (privat) |
| ANN vs XGBoost | `06` §36–41 |
| Empat pelatih | `09` §10 |

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Tanya-jawab lengkap tiga lapis T1–T57</b> (`sidang/qa/08_tanya_jawab_lengkap.md`)</summary>

# 08 — Tanya-jawab lengkap, tiga lapis

Satu dokumen untuk latihan dan untuk catatan internal notebook
`sidang/04_Sidang_Interaktif.ipynb`. Tiap bagian diawali penanda
`<!-- (bagian): nama -->`; builder notebook menyalin isi bagian itu ke sel
catatan internal yang sesuai, jadi **ubah di sini, lalu bangun ulang notebook**
(`./venv/bin/python build_sidang_interaktif_notebook.py`).

Tiap pertanyaan punya tiga lapis:

- **30 detik** — kalimat yang diucapkan di depan penguji.
- **Penjelasan** — dari nol, dengan contoh angka kecil
  yang bisa dihitung di kertas.
- **Bukti** — tempat membuka angka/kode di layar (notebook §, file:baris,
  CSV, pustaka). Angka hanya boleh disebut kalau berstatus COCOK di
  `01_klaim_bukti.csv` atau dibaca dari CSV di `hasil_uji/`.

Sumber yang digabung: `Bahan_Sidang.md` (Q1–Q48), `05_bank_pertanyaan.md`
(P1–P33), `02_kelemahan.md` (A–F), `03_algoritma_dengan_tangan.md`,
`07_pustaka_dan_kebaruan.md`, `hasil_uji/statistik_sel.csv`. Nomor asal
ditulis di akhir judul, misalnya (Q1, P9).

---

<!-- (bagian): pembuka -->

**T1. Bagaimana memakai notebook ini saat sidang?**
- *30 detik:* "Notebook ini memuat model yang sudah dilatih dan fitur yang
  sudah dihitung, jadi setiap sel selesai di bawah dua detik. Bapak/Ibu bisa
  memilih hari, sensor, jenis gangguan, dan intensitasnya, lalu melihat sinyal,
  kurva entropi, dan tebakan model berubah."
- *Penjelasan:* melatih ulang model butuh jam (NB11 di Kaggle
  ±10 jam). Saat presentasi kita tidak melatih; kita **memuat** bobot jaringan
  yang sudah tersimpan, lalu hanya melakukan perkalian matriks (forward pass).
  Itu seperti memakai kalkulator yang rumusnya sudah jadi.
- *Bukti:* `sidang/bukti_demo/04_Sidang_Interaktif.waktu.csv` (13 sel kode,
  total 9,7 s, sel terlama 1,7 s); `sidang/qa/test_notebook_sidang.py`
  (540 kombinasi panel tanpa galat, tiap panggilan < 1 s).

**T2. Apakah model di notebook ini sama dengan model penelitian?**
- *30 detik:* "Sama persis. Bobotnya diekspor dari model tersimpan N = 700 dan
  prediksinya dibandingkan dengan model asli: selisih maksimum nol."
- *Penjelasan:* model asli disimpan sebagai objek Python yang butuh
  scikit-learn (lambat diimpor, ±3–6 detik). Kita ambil angka-angkanya saja
  (bobot W, bias b, rata-rata dan skala normalisasi) ke file `.npz`, lalu
  menulis ulang forward pass dengan numpy. Supaya yakin tidak ada salah salin,
  keluaran kedua versi dibandingkan pada semua jendela uji.
- *Bukti:* `sidang/qa/buat_cache_interaktif.py` fungsi `ekspor_model`
  (assert selisih < 1e-9, terukur 0,00e+00); uji
  `test_bobot_numpy_sama_dengan_model_tersimpan`.

**T3. Kenapa catatan ini tidak tampil saat presentasi?**
- *30 detik:* "Catatan internal hanya untuk persiapan saya. Ada tombol mode
  presentasi dan ada salinan presentasi yang sel catatannya dibuang otomatis."
- *Bukti:* `sidang/qa/ekspor_notebook_sidang.py` (gagal kalau masih ada sel
  bertanda `catatan-internal`); `sidang/04_Sidang_Interaktif_presentasi.ipynb`.

<!-- (bagian): masalah -->

**T4. Penelitian ini tentang apa? (Q1)**
- *30 detik:* "Mendeteksi gangguan pada empat sensor kelembaban tanah dari
  pola sinyalnya. Sinyal dipotong menjadi jendela, tiap sensor diringkas dengan
  entropi multiskala, lalu jaringan saraf yang dilatih Levenberg–Marquardt
  menebak jenis gangguan dan sensor yang rusak. Yang saya uji: mengganti ukuran
  kemiripan Euclidean (EDM-Fuzzy) dengan Jensen–Shannon (JSD-Fuzzy)."
- *Penjelasan:* bayangkan empat termometer di satu kebun. Kalau satu
  mulai "melantur", angka rata-ratanya mungkin tetap wajar, tetapi *polanya*
  berubah: terlalu rata (macet), tiba-tiba melonjak (spike), atau pelan-pelan
  bergeser (drift). Entropi adalah angka yang mengukur seberapa "rumit" pola
  itu. Jaringan saraf belajar dari 60 angka entropi (4 sensor × 15 skala) untuk
  memutuskan "normal atau rusak, dan rusak jenis apa".
- *Bukti:* NB11 §0; `deteksi_fault_core.py` (pipeline lengkap).

**T5. Apa kebaruannya? (Q5)**
- *30 detik:* "Tempat JSD dipasang: sebagai jarak antar pasangan template di
  dalam fuzzy entropy. Penelusuran pustaka tidak menemukan karya yang sama;
  yang ada memakai jarak geometris di dalam entropi, atau memakai JSD pada level
  seluruh sinyal."
- *Penjelasan:* fuzzy entropy membandingkan setiap potongan pendek
  sinyal (template) dengan potongan lain. Biasanya "beda" diukur dengan
  penggaris (jarak Euclidean). Di sini setiap potongan diubah dulu menjadi
  histogram kecil, lalu dua histogram dibandingkan dengan JSD — ukuran beda dua
  sebaran peluang.
- *Bukti:* `07_pustaka_dan_kebaruan.md` (b) tabel prior-art dan log pencarian
  (f); sumber baseline Zhou dkk. 2021, IEEE TII, doi:10.1109/TII.2020.3009139.

**T6. EDM-Fuzzy itu buatan Anda?**
- *30 detik:* "Bukan. EDM-Fuzzy dari Zhou dkk. 2021. Saya memakainya sebagai
  pembanding dan mengganti hanya ukuran jaraknya."
- *Bukti:* `pustaka.bib` kunci `zhou2021edmfuzzy`.

**T7. Kenapa empat sensor, bukan satu, dan kenapa tidak dirata-rata? (Q2, Q13)**
- *30 detik:* "Rata-rata menyembunyikan sensor mana yang rusak dan menurunkan
  skor: F1 0,821 untuk empat kanal terpisah vs 0,748 setelah dirata-rata."
- *Penjelasan:* kalau sensor A naik 10 dan sensor B turun 10,
  rata-ratanya tidak berubah — gangguannya hilang dari pandangan.
- *Bukti:* NB02.

<!-- (bagian): data -->

**T8. Data dari mana, berapa banyak? (Q6, Q7)**
- *30 detik:* "Empat sensor kelembaban tanah, 14 September–21 Desember 2025,
  sampel tiap 30 detik, 281.721 baris. Saya rata-ratakan per 5 menit menjadi
  28.173 baris, 97,8 hari."
- *Penjelasan:* 30 detik → 5 menit artinya setiap 10 angka diganti
  rata-ratanya. Contoh: 40, 41, 40, 42, 41, 40, 41, 40, 42, 41 → 40,8. Derau
  kecil teredam, pola harian tetap.
- *Bukti:* NB11 §1–§2; `data_sensor.csv`.

**T9. Nilai nol di S4 diapakan? (Q9)**
- *30 detik:* "Tidak dibuang, dilabeli normal, dan dilaporkan sebagai batasan:
  11.371 sampel (4,0 %), tiga blok macet di 16–21 Desember plus 50 sampel
  pendek."
- *Penjelasan:* nol masih di rentang fisik 0–100 %, jadi tidak bisa
  disebut "salah ketik". Pembacaan macet di nol justru gejala gangguan; kalau
  dibuang, kita menyembunyikan kasus sulit.
- *Bukti:* NB11 §2b; Bahan Q9.

**T10. Missing value dan outlier? Sinkronisasi waktu? (Q8, Q14, Q15)**
- *30 detik:* "Tidak ada NaN dan tidak ada nilai di luar 0–100 %. Stempel waktu
  diubah ke UTC, diurutkan, duplikat dibuang, lalu dipaksa ke grid 5 menit;
  slot bolong nol."
- *Bukti:* NB11 §2 sel 7, §2b sel 8.

**T11. "Broker" itu apa? (Q12)**
- *30 detik:* "Komponen yang menyatukan empat aliran sensor menjadi satu tabel
  berwaktu sama tanpa melebur identitas sensornya; bukan message broker
  produksi."
- *Bukti:* NB02 sel 1–13; `deteksi_fault_core.sync_broker`.

<!-- (bagian): fault -->

**T12. Kenapa gangguannya disimulasikan? (Q10)**
- *30 detik:* "Data asli tidak berlabel rusak dan saya tidak boleh merusak
  sensor. Simulasi memberi label pasti dan bisa diulang dengan seed tetap.
  Batasnya: hasil berlaku untuk gangguan yang dimodelkan."
- *Penjelasan:* ini seperti latihan pemadam kebakaran dengan api
  buatan: kita tahu persis di mana apinya, jadi bisa menilai apakah petugas
  menemukannya.
- *Bukti:* NB11 §3; `02_kelemahan.md` B (validitas konstruk).

**T13. Tulis model tiap gangguan. (Q17, P17)**
- *30 detik:* "Drift: tambahan yang naik linear 1 poin/hari selama episode
  3 hari. Spike: lonjakan sesaat dengan peluang 1,5 %. Bias: geser tetap.
  Hardware: sebagian titik macet atau hilang."
- *Penjelasan:* sinyal 40, 40, 40, 40.
  Bias +2 → 42, 42, 42, 42. Spike di titik ke-3 → 40, 40, 47, 40.
  Drift → 40, 40,003, 40,007, 40,010 (1 poin/hari = 0,00347 poin per 5 menit).
  Macet → 40, 41, 41, 41 (nilai terakhir diulang).
- *Bukti:* `deteksi_fault_core.py` `simulate_drift_fault`,
  `simulate_spike_fault`, `simulate_bias_fault`, `simulate_hardware_fault`;
  `04_walkthrough_kode.md` baris 18.

**T14. Kenapa drift hanya 1 poin/hari? (Q11, Q42)**
- *30 detik:* "Agar tidak mustahil secara fisik: versi awal 57,6 poin/hari dan
  menembus 100 % dalam sehari. Rentang 0,003–0,17 poin/hari yang saya pakai
  sebagai pembanding adalah **asumsi skenario**; saya tidak menemukan sumber
  terbit yang mengukur laju drift sensor kelembaban tanah per hari."
- *Penjelasan:* dalam jendela N = 200 (16,7 jam) drift hanya
  menambah ±0,7 poin, lebih kecil dari ayunan harian sinyal. Karena itu drift
  memang paling sulit dideteksi, dan itu dilaporkan.
- *Bukti:* README "Koreksi simulator drift"; `07_pustaka_dan_kebaruan.md` (e).
- *Serangan lanjutan:* "Jadi skenario Anda tidak realistis?" → "Arahnya
  realistis (lambat), besarnya asumsi. Uji intensitas NB13 menunjukkan drift
  baru terdeteksi (AUC 0,80–0,85) pada total 60 poin."

**T15. Hardware malfunction 8 % macet dan 5 % hilang? (F4)**
- *30 detik:* "Total titik rusak 8 %: 5 % hilang (NaN) dan 3 % macet, karena
  titik hilang adalah bagian dari titik macet lalu ditimpa NaN. Deskripsi lama
  saya koreksi."
- *Bukti:* `deteksi_fault_core.py` `simulate_hardware_fault`;
  `02_kelemahan.md` F4.

**T16. Kenapa bias konstan tidak terdeteksi? (Q47)**
- *30 detik:* "Karena setiap template dikurangi rata-ratanya sebelum
  dibandingkan. Geser tetap ikut hilang. AUC bias sekitar 0,50 dari 0,08 sampai
  5 poin."
- *Penjelasan:* template [42, 43] dikurangi rata-rata 42,5 menjadi
  [−0,5, 0,5]; template tanpa bias [40, 41] juga menjadi [−0,5, 0,5]. Entropi
  melihat keduanya identik. Panel fault "bias" di notebook menunjukkan kurva
  entropi hampir tidak bergerak.
- *Bukti:* NB12 `12_sweep_fault_param.csv`; `embed_matrix_centered`.

**T17. Kapan jendela dilabel rusak? (Q22, P18)**
- *30 detik:* "Kalau porsi titik rusak di jendela > 1 %; jendela di bawah itu
  dibuang supaya model tidak belajar label yang salah."
- *Bukti:* `FAULT_RATIO_THR`; `11_akuntansi_window.csv`.

<!-- (bagian): entropi -->

**T18. Apa itu entropi di sini? (Q3, Q23, P9)**
- *30 detik:* "Fuzzy entropy mengukur seberapa sering pola pendek yang mirip
  tetap mirip ketika diperpanjang satu titik. Sinyal teratur: entropi kecil.
  Sinyal acak: entropi besar."
- *Penjelasan:* rumusnya FE = ln(φ_m / φ_{m+1}). φ_m adalah rata-rata
  "kemiripan" antar semua pasangan template sepanjang m = 2. Kalau pasangan yang
  mirip di panjang 2 tetap mirip di panjang 3, φ hampir sama dan ln-nya dekat 0.
  Contoh angka pada deret x = [3, 5, 4, 6, 5, 7, 6, 4]: r = 0,2 · sd =
  0,261861; jarak Euclidean template [−1, 1] dan [0,5, −0,5] = 2,121320,
  keanggotaan μ = 1/(1 + 2,121320/0,261861) = 0,109879.
- *Bukti:* `03_algoritma_dengan_tangan.md` (b), dicek otomatis oleh
  `cek_algoritma_tangan.py`; Chen dkk. 2007 (`chen2007fuzzyen`).

**T19. Kenapa multiskala τ = 1..15 dan 60 fitur? (Q24, Q28)**
- *30 detik:* "Gangguan berbeda tampak di skala berbeda: spike di skala halus,
  drift di skala kasar. 15 skala × 4 sensor = 60 fitur."
- *Penjelasan:* skala τ = 2 artinya rata-ratakan tiap dua titik:
  [3, 5, 4, 6] → [4, 5]. Komposit: hitung juga dengan geseran awal (mulai dari
  titik kedua) lalu rata-ratakan hasilnya, supaya tidak bergantung pada titik
  awal.
- *Bukti:* `coarse_grain_cmse`; Costa dkk. 2002, Wu dkk. 2013.

**T20. Di mana tepatnya EDM dan JSD berbeda? (Q4, Q26, Q27, P10)**
- *30 detik:* "Hanya di jarak d antar template. EDM: jarak Euclidean. JSD:
  tiap template diubah jadi histogram lunak 8 bin, jarak = akar JSD. Embedding,
  centroid shifting, fungsi keanggotaan, dan rumus entropi identik."
- *Penjelasan:* EDM seperti membandingkan dua foto piksel demi
  piksel. JSD seperti membandingkan dua foto dari histogram warnanya saja.
  Contoh JSD dari hitungan tangan: d = √0,011811 = 0,108678, ambang
  r_JSD = 0,2·√ln2 = 0,166511, sehingga μ = 1/(1 + 0,108678/0,166511) = 0,605078.
- *Bukti:* `deteksi_fault_core.py` `phi_euclidean`, `vectors_to_distributions`,
  `pairwise_jsd`, `phi_jsd`; `03_algoritma_dengan_tangan.md` (c).

**T21. Kenapa r = 0,2·SD dan r_JSD = 0,2·√ln2? (Q25)**
- *30 detik:* "0,2·SD adalah toleransi lazim dari literatur sample/fuzzy
  entropy, diskalakan dengan simpangan baku supaya adil untuk amplitudo
  berbeda. JSD (log natural) terbatas di [0, ln 2], jadi akar JSD maksimum √ln2;
  ambangnya 20 % dari maksimum itu."
- *Bukti:* `JSD_R_RATIO`, `jsd_fuzzy_entropy_1d`; Richman & Moorman 2000.

**T22. Sifat JSD yang penting? (P11)**
- *30 detik:* "Simetris, selalu berhingga, terbatas [0, ln 2], dan akarnya
  metrik. KL tidak simetris dan bisa tak hingga kalau ada bin nol."
- *Bukti:* Lin 1991; Endres & Schindelin 2003.

**T23. Apa kelemahan konseptual JSD-Fuzzy? (F7)**
- *30 detik:* "Histogram tidak peduli urutan. Template naik [−1, 1] dan turun
  [1, −1] menghasilkan histogram sama, jadi JSD = 0 persis, padahal Euclidean
  memberi 2,83. JSD-Fuzzy mengukur kemiripan sebaran nilai, bukan bentuk pola.
  Akibat lain: nilainya bisa negatif pada deret pendek."
- *Penjelasan:* kata "TAS" dan "SAT" punya huruf yang sama. Kalau
  hanya menghitung huruf, keduanya dianggap kembar.
- *Bukti:* `02_kelemahan.md` F7 (dicek 2026-11-20 dengan
  `vectors_to_distributions` + `pairwise_jsd`); `07_pustaka_dan_kebaruan.md`
  catatan jujur.
- *Serangan lanjutan:* "Kalau buta urutan, kenapa JSD menang?" → "Gangguan yang
  diuji mengubah sebaran amplitudo di dalam template (spike, macet); itu yang
  ditangkap JSD. Untuk gangguan yang hanya membalik arah, JSD tidak peka. Ini
  saya tulis sebagai batasan."

**T24. Kenapa n_ref = 128 titik acuan? (P14)**
- *30 detik:* "Semua pasangan berbiaya kuadrat; 128 acuan acak dengan seed
  tetap memberi aproksimasi yang sama untuk kedua metode."
- *Bukti:* `_pick_ref`.

**T25. Kenapa JSD lebih lambat? (P15)**
- *30 detik:* "Tiap template harus diubah menjadi histogram dan JSD butuh
  logaritma per bin; Euclidean cukup selisih kuadrat. Biaya ada di ekstraksi
  fitur; inferensi per jendela setara."
- *Bukti:* NB14 `hasil14` (1,7–6,8× pada ekstraksi fitur).

<!-- (bagian): model -->

**T26. ANN-nya seperti apa? (Q29)**
- *30 detik:* "Feed-forward, 60 masukan, satu atau dua lapisan tersembunyi
  dari kisi 8 arsitektur, aktivasi relu/tanh, keluaran softmax. Arsitektur
  dipilih di dalam cross-validation."
- *Bukti:* NB11 §10; `models/meta.json`.

**T27. Tulis rumus update LM dan arti λ. (P1, P2, P3)**
- *30 detik:* "(JᵀJ + λ·diag(JᵀJ))·Δ = −Jᵀr. λ kecil: mendekati Gauss–Newton,
  cepat dekat minimum. λ besar: langkah kecil searah gradien, aman. Galat naik:
  langkah ditolak, λ dikali 2. Galat turun: diterima, λ dikecilkan menurut
  rasio ρ (aturan Nielsen)."
- *Penjelasan:* LM seperti menuruni bukit dalam kabut. Kalau
  langkah besar membuat Anda naik (galat bertambah), Anda mundur dan mencoba
  langkah lebih kecil. Di contoh mainan: iterasi 1 ditolak (ρ = −0,58),
  λ 0,001 → 0,002; pada iterasi 9 galat turun dari 1,453047 ke 0,194302.
- *Bukti:* `03_algoritma_dengan_tangan.md` (a); `lm_fit`; Marquardt 1963,
  Hagan & Menhaj 1994, Madsen dkk. 2004.

**T28. Bagaimana Jacobian analitik dihitung dan dibuktikan? (Q31, P4)**
- *30 detik:* "Backpropagation per keluaran. Dibuktikan dengan membandingkan
  terhadap Jacobian numerik beda hingga."
- *Penjelasan:* turunan numerik: geser satu bobot sedikit (ε), lihat
  berapa keluaran berubah, bagi ε. Kalau rumus analitik dan cara numerik
  memberi angka sama sampai ±10⁻¹⁰, rumusnya benar.
- *Bukti:* NB11 §9a; `03` (d); `cek_algoritma_tangan.py` (Jacobian tangan vs
  kode, selisih 5,6e-17).

**T29. LM itu untuk regresi; klasifikasinya bagaimana? (P5)**
- *30 detik:* "Target one-hot, kuadrat terkecil pada keluaran linear, softmax
  saat prediksi — cara yang lazim di `trainlm` MATLAB. Kelemahannya
  probabilitasnya tidak terkalibrasi."

**T30. Kenapa LM, bukan Adam? (Q30, P7)**
- *30 detik:* "LM metode pada paper acuan dan cocok untuk jaringan kecil. Di
  arsip NB09 LM 0,433, BR 0,429, GA 0,306, Adam 0,268; LM dan BR tidak berbeda,
  keduanya jauh di atas GA dan Adam."
- *Bukti:* NB09 §10.

**T31. Kenapa jaringan dibatasi kecil? (P6)**
- *30 detik:* "JᵀJ berukuran P×P dan harus diselesaikan tiap iterasi, biayanya
  O(P³). Karena itu parameter dibatasi sekitar 5 ribu."

**T32. Jelaskan nested grouped CV. (Q33, P20)**
- *30 detik:* "Luar: StratifiedGroupKFold 5 lipatan, grup = blok waktu. Dalam:
  2 lipatan untuk memilih arsitektur dengan macro-F1. Model terpilih dilatih
  ulang di data latih luar dan dinilai di data uji luar."
- *Penjelasan:* jendela bertetangga berbagi 50 % isi. Kalau satu di
  latih dan tetangganya di uji, model "sudah pernah melihat" separuh soal ujian.
  Mengelompokkan per blok waktu memisahkan soal latihan dan soal ujian.
- *Bukti:* NB11 §10; Roberts dkk. 2017, Bergmeir & Benítez 2012.

**T33. Kenapa macro-F1? (P23)**
- *30 detik:* "Kelas tidak seimbang; macro-F1 memberi bobot sama pada tiap
  kelas, sehingga kelas sulit tidak tertutup kelas mudah."
- *Penjelasan:* 90 jendela normal dan 10 rusak. Model yang selalu
  menjawab "normal" akurasinya 90 %, tetapi F1 kelas rusak 0. Macro-F1 =
  rata-rata (F1 normal ≈ 0,95, F1 rusak 0) ≈ 0,47 — angka itu jujur.

**T34. Bobot awal Xavier? (A5, F2)**
- *30 detik:* "Jujur: urutan `lm_pack`/`lm_unpack` berbeda, sehingga sebagian
  bobot lapisan keluaran mulai dari nol. Konsisten saat latih dan prediksi,
  sama untuk kedua metode. Teks skripsi tidak menyebut 'Xavier dengan bias
  nol'."
- *Bukti:* `02_kelemahan.md` A5, F2.

<!-- (bagian): prediksi -->

**T35. Bagaimana membaca tebakan di panel?**
- *30 detik:* "Tabel pertama: tebakan model N = 700 (S1 normal/rusak dan S2
  atau S5 jenis gangguan) untuk jendela asli dan jendela yang diberi gangguan,
  EDM dan JSD berdampingan, dengan tingkat keyakinan. Tabel kedua: skor tiap
  sensor; skor > 0,5 berarti sensor itu ditebak rusak."
- *Penjelasan:* softmax mengubah skor mentah menjadi peluang yang
  jumlahnya 1. Skor [2, 0] → e²/(e² + e⁰) = 7,39/8,39 = 0,88 untuk kelas
  pertama.
- *Bukti:* `maju` di `buat_cache_interaktif.py`; sel prediksi notebook.

**T36. Kenapa tebakan kadang salah di panel?**
- *30 detik:* "Karena F1 validasi silang skenario ini memang belum sempurna,
  dan panel memakai satu jendela, bukan rata-rata. Contoh salah lebih jujur
  daripada demo yang dipilih-pilih."
- *Bukti:* `|cv_f1_nb11` di `model_sidang.npz`; `11_rq4_performa_ann.csv`.

**T37. Bisa dipakai real-time? (Q43, P32)**
- *30 detik:* "Belum. Satu keputusan N = 700 butuh 58 jam data; cocok untuk
  pemantauan harian. Untuk notifikasi cepat ada NB15 (per sampel, tanpa
  entropi) dengan aturan persistensi."
- *Bukti:* NB15, NB15b.

<!-- (bagian): hasil -->

**T38. Hasil utamanya? (Q35, Q36)**
- *30 detik:* "JSD-Fuzzy lebih tinggi di 11 dari 15 sel, rata-rata selisih
  F1 +0,028. Per ukuran jendela: N = 200 menang 3/5, N = 700 5/5,
  N = 1000 3/5."
- *Penjelasan:* satu sel = satu skenario (S1–S5) × satu ukuran
  jendela (200/700/1000). Di tiap sel kedua metode dilatih dengan protokol yang
  sama; kita catat selisih F1-nya.
- *Bukti:* NB11 §10; `11_rq4_selisih.csv`; `statistik_sel.csv` baris NB11.

**T39. Hasil identifikasi sensor? (Q37, F1)**
- *30 detik:* "JSD 0,675 vs EDM 0,634, menang 9 dari 15. Catatan jujur: tahap
  ini memakai pembagian acak 70/30, jadi angka absolutnya optimistis; selisih
  kedua metode tetap adil karena pembagiannya sama."
- *Bukti:* NB11 §12; `11_sensor_ringkas.csv`; `02_kelemahan.md` F1.

**T40. JSD lebih tinggi, EDM lebih apa?**
- *30 detik:* "EDM lebih stabil (56 dari 63 pasangan). Keduanya saya
  tampilkan."
- *Bukti:* Bahan "Kalimat siap pakai" 3.

**T41. NB10, NB11, NB14 — kenapa ada tiga hasil?**
- *30 detik:* "NB10 dan NB11 protokol sama, berbeda anggaran sesi; NB14
  memakai simulator fault 'sedang' dengan posisi acak ber-seed. Arah ketiganya
  sama: JSD lebih tinggi."
- *Bukti:* `statistik_sel.csv` (NB10 11/15, NB11 11/15, NB14 12/15).

<!-- (bagian): statistik -->

**T42. Apakah hasilnya signifikan? (Q39, Q41)**
- *30 detik:* "NB11 gabungan belum: uji-t p = 0,076, Wilcoxon 0,107. Ukuran
  efek sedang (d_z = 0,49), dan interval bootstrap 95 % selisih F1 [+0,002;
  +0,058] tidak memuat nol. Jadi saya menyebutnya konsisten lebih tinggi, bukan
  terbukti. NB14 gabungan signifikan: p = 0,0006, d_z = 1,13, CI [+0,043;
  +0,108]."
- *Penjelasan:* p menjawab "kalau sebenarnya tidak ada beda,
  seberapa sering kita melihat beda sebesar ini karena kebetulan?". p = 0,076
  berarti sekitar 7,6 %, di atas batas 5 %. Bootstrap: ambil ulang 15 selisih
  secara acak dengan pengembalian 10.000 kali, hitung rata-rata tiap kali, lalu
  lihat 2,5 % dan 97,5 % terbawah/teratas.
- *Bukti:* `sidang/qa/hasil_uji/statistik_sel.csv`; `statistik_tambahan.py`.
- *Serangan lanjutan:* "CI tidak memuat nol tapi p > 0,05 — mana yang benar?"
  → "Keduanya pendekatan berbeda pada n = 15: bootstrap persentil cenderung
  sempit pada sampel kecil, uji-t lebih konservatif. Saya ambil yang
  konservatif untuk klaim."

**T43. N = 700 signifikan, kan? (Q41)**
- *30 detik:* "Tanpa koreksi p = 0,027; setelah koreksi Holm untuk tiga ukuran
  jendela p = 0,081. Jadi tidak saya klaim signifikan setelah koreksi."
- *Penjelasan:* menguji tiga hal sekaligus menaikkan peluang satu
  'beruntung'. Holm: urutkan p, kalikan yang terkecil dengan 3, berikutnya
  dengan 2, dan seterusnya (dengan aturan tidak boleh turun).
- *Bukti:* `statistik_sel.csv` kolom `p_t_holm_3N`; Holm 1979.

**T44. Uji-t pada lipatan CV itu bias, kan? (P22, C2)**
- *30 detik:* "Ya, lipatan berbagi data latih sehingga varians diremehkan.
  Saya menambahkan uji-t terkoreksi Nadeau–Bengio. Uji utama saya per sel
  (skenario × N), bukan per lipatan."
- *Penjelasan:* koreksi Nadeau–Bengio memperbesar varians dengan
  faktor (1/k + n_uji/n_latih); hasilnya |t| selalu lebih kecil.
- *Bukti:* `statistik_tambahan.py` `t_nadeau_bengio`, keluaran per lipatan
  `hasil_uji/statistik_lipatan.csv` (dari run purge N = 700); Nadeau & Bengio
  2003; Dietterich 1998; uji `test_nadeau_bengio_*`.

**T45. Kenapa Wilcoxon per N tidak pernah signifikan? (F6)**
- *30 detik:* "Dengan lima pasangan, p eksak dua sisi terkecil 2/2⁵ = 0,0625.
  Jadi secara matematis tidak mungkin < 0,05."

**T46. Ukuran efek? (P25)**
- *30 detik:* "d_z NB11 gabungan 0,49 (sedang), NB14 1,13 (besar)."
- *Bukti:* `statistik_sel.csv` kolom `cohen_dz`; Cohen 1988.

<!-- (bagian): kelemahan -->

**T47. Apakah masih ada kebocoran data? (A1, P21)**
- *30 detik:* "Ada sedikit di batas grup: jendela terakhir satu blok bisa
  tumpang-tindih dengan jendela pertama blok tetangga. Sama untuk kedua metode.
  Saya ukur dengan purge: jendela latih yang tumpang-tindih dengan jendela uji
  dibuang, lalu model N = 700 dilatih ulang."
- *Penjelasan:* purge = membuang soal latihan yang isinya beririsan
  dengan soal ujian. Kalau skor turun banyak setelah purge, berarti skor lama
  terbantu kebocoran.
- *Hasil:* purge mengubah F1 −0,072 sampai +0,012 (7 dari 10 sel turun);
  JSD ≥ EDM tetap di 5 dari 5 skenario. Purge juga mengurangi data latih ±20 %
  (S1: 126 → 100 jendela), jadi penurunan skor tidak seluruhnya akibat
  kebocoran.
- *Bukti:* `sidang/qa/uji_purge_permutasi_N700.py`,
  `hasil_uji/efek_purge_permutasi.csv`, `hasil_uji/statistik_lipatan.csv`;
  `02_kelemahan.md` A1.

**T48. Bagaimana tahu model tidak belajar artefak? (kontrol negatif)**
- *30 detik:* "Label saya acak lalu model dilatih ulang dengan protokol sama.
  Kalau pipeline jujur, F1-nya turun ke tingkat tebak-tebakan."
- *Penjelasan:* untuk 2 kelas, tebakan acak memberi F1 sekitar 0,5.
  Kalau model dengan label acak masih jauh di atas itu, ada yang bocor.
- *Hasil:* F1 dengan label acak 0,446–0,508 (S1 dan S5, kedua metode),
  setara tebakan 0,5; model asli 0,64–0,83.
- *Bukti:* `hasil_uji/efek_purge_permutasi.csv` kolom `F1_permutasi`.

**T49. Validitas eksternal? (D)**
- *30 detik:* "Satu lokasi, satu musim, empat sensor, gangguan disimulasikan.
  Hasil berlaku untuk gangguan yang dimodelkan; langkah berikutnya data lapangan
  yang benar-benar rusak."
- *Bukti:* `02_kelemahan.md` D.

**T50. Data asli atau buatan? (B6, P16)**
- *30 detik:* "Gangguan disuntikkan secara simulasi ke `data_sensor.csv`."
  Asal-usul `data_sensor.csv` **belum dipastikan**: baris awalnya tampak hasil
  interpolasi ke grid 30 detik. Jawab hanya setelah dikonfirmasi ke pembimbing;
  jangan mengarang.
- *Bukti:* `02_kelemahan.md` B6.

**T51. p = 0,0006 di NB14 — apakah karena gangguan diperbesar sampai JSD menang?
(P24, A6)**
- *30 detik:* "Simulator NB14 ditetapkan sebelum membandingkan metode dan sama
  untuk keduanya. Tetap signifikan setelah koreksi ganda."
- *Bukti:* `02_kelemahan.md` A6, C3.

<!-- (bagian): penutup -->

**T52. Kalimat penutup.**
- "Saya mengganti ukuran kemiripan di dalam fuzzy entropy dari Euclidean
  menjadi Jensen–Shannon dan mengujinya sampai identifikasi sensor."
- "JSD-Fuzzy lebih tinggi di 11 dari 15 sel (+0,028); secara gabungan NB11
  belum signifikan (p = 0,076), sehingga saya menyebutnya konsisten, bukan
  terbukti. Pada simulator NB14 selisihnya signifikan (p = 0,0006)."
- "Batasan yang saya laporkan: gangguan disimulasikan, JSD buta urutan,
  sedikit kebocoran di batas grup, dan bias konstan tidak terdeteksi oleh
  representasi terpusat."

**T53. Langkah berikutnya? (Q44)**
- Data lapangan yang benar-benar rusak; jendela adaptif untuk deteksi lebih
  cepat; fitur level untuk bias; GroupShuffleSplit pada tahap sensor-ID.

**T54. Kalau ditanya angka yang tidak ada di layar? (Q45)**
- "Saya cek di `01_klaim_bukti.csv` atau notebook sumbernya." Jangan mengarang
  angka baru.

<!-- (bagian): data-hipotetik -->

Bagian ini untuk percobaan data hipotetik tanpa entropi (NB15,
`build_15_data_hipotetik_ann_lm_notebook.py`). Belum dipakai di notebook
interaktif.

**T55. Gangguan disisipkan bersamaan atau bertahap?**
- *30 detik:* "Ada dua arti. Antar skenario bertahap: S1 satu fault, S2 tiap
  fault tunggal, S3 dua fault, S4 tiga, S5 keempatnya. Ada 16 kondisi dan
  masing-masing punya salinan datanya sendiri. Di dalam satu kondisi kombinasi,
  semua fault disisipkan pada interval waktu yang sama, yaitu tiga episode
  tiga hari. Secara waktu fault-nya bersamaan; secara kode diterapkan
  berurutan pada sinyal yang sama."
- *Penjelasan:* kondisi `spike+bias` diproses begini. Ambil
  sinyal bersih. Kalikan 1,52 % titik di dalam episode dengan 1,3 (spike).
  Tambahkan +5 poin ke seluruh episode (bias). Terakhir potong nilainya ke
  0–100 %. Label tiap sampel berupa bitmask: drift 1, spike 2, bias 4,
  hardware 8. Titik yang kena spike dan bias berlabel 6.
- *Detail yang harus bisa dijelaskan:*
  - fault tunggal: spike dan hardware tersebar di seluruh deret; drift dan
    bias selalu di dalam episode;
  - kombinasi: semua komponen dibatasi ke episode yang sama
    (`simulate_multiple_faults`, `within = _episode_mask`);
  - urutan kode S5: spike, bias, hardware, lalu drift. Nilai macet disalin dari
    deret yang sudah diberi spike dan bias. Drift ditambahkan terakhir, dan
    titik NaN tetap NaN;
  - versi 4 sensor: fault masuk ke 1, 2, 3, atau 4 sensor, bergiliran antar
    kondisi; sensor dipilih acak dengan seed tetap. Episode sama untuk semua
    sensor. Posisi spike dan hardware berbeda per sensor karena seed-nya
    berbeda. Di S1 jenis fault diundi per sensor;
  - model per kondisi dilatih terpisah per kondisi. Model gabungan dilatih
    sekaligus pada 16 kondisi ditambah data normal.
- *Bukti:* `build_14_lengkap_fault_baru_notebook.py` `NEW_SIM`
  (`_episode_mask`, `simulate_multiple_faults`, `simulate_choose_one`);
  `build_15_…py` `CONDITIONS` dan §3 `GEN_4S` / `GEN_AVG`.

**T56. Berapa neuron output ANN-LM?**
- *30 detik:* "Dua neuron output linear, untuk normal dan fault, dengan target
  one-hot. Saat prediksi, softmax mengubahnya jadi peluang. Peluang fault
  adalah unit kedua, lalu alarm menyala kalau peluang itu di atas ambang yang
  disetel di data validasi. Program pertama (Des 2025) memakai satu output
  sigmoid."
- *Penjelasan:* masukan berisi 13 fitur per sampel: lag1–lag10,
  selisih dengan sampel sebelumnya, penanda NaN, dan jumlah NaN dalam
  10 sampel terakhir. Contoh arsitektur (16, 8):
  13 → 16 → 8 → 2, aktivasi tanh. Jumlah parameternya
  13·16 + 16 + 16·8 + 8 + 8·2 + 2 = 378.
- *Bukti:* `build_15_…py` `fit_lm_model` (`lm_fit(..., 2, activation="tanh")`,
  `np.eye(2)[y]`), `FEATURE_NAMES`, `best_threshold` (ambang 0,05–0,95, langkah
  0,01).

**T57. Grid search-nya mencari di ruang apa?**
- *30 detik:* "Hanya arsitektur lapisan tersembunyi, lima kandidat: (6), (12),
  (6,3), (12,6), (16,8). Yang lain tetap: tanh, 60 iterasi LM, maksimum
  120 ribu baris latih, seed tetap. Kriteria pemilihannya F1 validasi; data
  uji tidak dipakai. Grid ini hanya untuk model gabungan. Model per kondisi
  memakai dua arsitektur program pertama, (6) dan (6,3), dan keduanya
  dilaporkan."
- *Hasil (log Kaggle v3):* F1 validasi rata-rata 0,702–0,709, terpilih (16,8),
  F1 uji 0,753. Pada 4 sensor F1 validasi 0,547–0,560, terpilih (12,6),
  F1 uji 0,640.
- *Jujur:* selisih antar kandidat ≤ 0,013, dan semua berhenti di batas
  60 iterasi (`max_iter`). Pemilihan arsitektur praktis tidak berpengaruh.
  Di versi rata-rata, kandidat terpilih justru F1 ujinya terendah (0,753 vs
  0,754–0,765). Itu tanda pemilihan memang tidak mengintip data uji.
- *Bukti:* `build_15_…py` `GABUNGAN_GRID`, `LM_ITERS`, `MAX_TRAIN_ROWS`;
  `hasil15/data-hipotetik-berlabel-ann-lm.log` baris `grid …`; ekspor
  `15_grid_search_gabungan.csv`; `15_model_card.json` kunci `grid_search`.

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Bank pertanyaan menyerang P1–P33</b> (`sidang/qa/05_bank_pertanyaan.md`)</summary>

# 05 — Bank pertanyaan penguji (versi menyerang)

Pelengkap `Bahan_Sidang.md` (Q1–Q48), bukan pengganti. Di sini isinya
pertanyaan lanjutan yang muncul **setelah** jawaban pertama: "kenapa",
"buktikan", "bagaimana kalau". Tiap jawaban dibuat untuk ±30 detik dan
menunjuk bukti. Angka hanya boleh disebut kalau berstatus COCOK di
`01_klaim_bukti.csv`.

Kode level: **[L1]** konsep, **[L2]** algoritma/rumus, **[L3]** baris kode.

---

## 1. Levenberg–Marquardt

**P1 [L2]. Tulis rumus update LM.**
(JᵀJ + λ·diag(JᵀJ))·Δ = −Jᵀr, lalu θ ← θ + Δ kalau galat turun. J = turunan
residual terhadap semua bobot, r = keluaran − target. Contoh angka:
`03_algoritma_dengan_tangan.md` bagian (a).

**P2 [L2]. Apa arti λ? Kapan naik, kapan turun?**
λ kecil → langkah mendekati Gauss–Newton (cepat dekat minimum); λ besar →
langkah kecil searah gradien turun (aman jauh dari minimum). Galat naik →
langkah ditolak, λ dinaikkan (×2). Galat turun → diterima, λ dikecilkan
menurut rasio gain ρ (aturan Nielsen: λ·max(1/3, 1−(2ρ−1)³)).
Kode: `deteksi_fault_core.py` `lm_fit` (baris ±511–568).

**P3 [L2]. Kenapa diag(JᵀJ), bukan identitas?**
Itu versi Marquardt (1963): redaman diskalakan per parameter sehingga bobot
dengan kelengkungan besar tidak mendominasi. Identitas adalah versi Levenberg
(1944).

**P4 [L3]. Bagaimana Jacobian analitik dihitung?**
Backpropagation per keluaran: delta dimulai dari identitas (keluaran linear),
dimundurkan lewat tiap lapisan dengan turunan tanh (1 − h²) atau relu; blok
dW = aktivasi sebelumnya ⊗ delta. Kebenarannya dibuktikan dengan
membandingkan terhadap Jacobian numerik (selisih relatif < 1e-3; di NB15
1,3e-5 dan 4,3e-9). Kode: `lm_residual_jacobian`.

**P5 [L2]. LM itu untuk regresi kuadrat terkecil; klasifikasinya bagaimana?**
Target dibuat one-hot dan keluaran linear dilatih meminimalkan jumlah kuadrat
residual; probabilitas kelas didapat dari softmax keluaran saat prediksi.
Ini cara "LM untuk klasifikasi" yang lazim di toolbox MATLAB (`trainlm` +
target one-hot). Kelemahannya: bukan cross-entropy, jadi probabilitas tidak
terkalibrasi — karena itu ambang dipilih di validasi.

**P6 [L1]. Kenapa LM tidak dipakai untuk jaringan besar?**
JᵀJ berukuran P×P dan harus diselesaikan tiap iterasi (O(P³)), J berukuran
(m·kelas)×P. Karena itu arsitektur dibatasi ±5 ribu parameter
(`build_11…:1600–1609`).

**P7 [L1]. Bukti LM lebih cocok daripada Adam?**
Notebook 09: F1 rata-rata LM 0,433, BR 0,429, GA 0,306, Adam 0,268; LM vs BR
tidak beda (p = 0,81), LM/BR vs GA dan Adam signifikan. (Cek status di 01.)

**P8 [L3]. Apa yang terjadi di `lm_pack` / `lm_unpack`?**
Lihat `02_kelemahan.md` A5 dan `04_walkthrough_kode.md` "Cacat yang
diketahui". Jawab jujur: konsisten saat latih/prediksi, bobot awal tidak
persis Xavier, dampaknya sama untuk kedua metode.

## 2. Entropi fuzzy, EDM, JSD

**P9 [L2]. Tulis rumus fuzzy entropy.**
FuzzyEn(m, r) = ln φ_m(r) − ln φ_{m+1}(r), φ_m = rata-rata kemiripan antar
pasangan vektor embedding berdimensi m (setelah dikurangi rata-ratanya).
Multiskala: sinyal di-coarse-grain dulu pada skala τ. Hitungan tangan:
`03` bagian (b).

**P10 [L2]. Di mana tepatnya EDM dan JSD berbeda dalam rumus?**
Hanya di d(i,j): EDM = jarak Euclidean antar vektor; JSD = Jensen–Shannon
divergence antara sebaran halus kedua vektor. Fungsi keanggotaan
μ = 1/(1 + d/r), embedding, centroid shifting, dan rumus entropi identik.

**P11 [L2]. Sifat JSD yang relevan?**
Simetris, selalu berhingga, terbatas di [0, ln 2] (atau [0,1] basis 2), dan
akar JSD adalah metrik. Berbeda dengan KL yang tidak simetris dan bisa tak
hingga kalau ada bin nol.

**P12 [L2]. Bagaimana vektor 2 angka jadi sebaran?**
Soft binning: tiap nilai disebar ke 8 pusat bin dengan kernel halus lebar h,
rentang ±3 SD, lalu dinormalisasi. Lihat `02` B5 dan NB12 (bins=4,
h_ratio=1,0 disarankan).

**P13 [L2]. Kenapa μ = 1/(1 + d/r), bukan exp(−dⁿ/r)?**
Lihat `02` B4. Monoton turun, bernilai 1 saat d = 0, sama untuk kedua metode.

**P14 [L1]. Kenapa n_ref = 128 titik acuan?**
Menghitung semua pasangan O(n²) terlalu mahal untuk ribuan window ×15 skala ×
4 sensor; kemiripan dirata-rata terhadap 128 vektor acuan yang dipilih dengan
seed tetap. Ini aproksimasi yang sama untuk kedua metode.

**P15 [L1]. Kenapa JSD 1,7–6× lebih lambat?**
Tiap vektor harus diubah jadi sebaran dan JSD butuh log per bin, sedangkan
Euclidean cukup selisih kuadrat. Inferensi per window tetap setara (±0,09 ms)
karena biaya ada di ekstraksi fitur. (Cek angka di 01.)

## 3. Data, fault, label

**P16 [L1]. Data asli atau buatan?** — `02` B6. **Wajib dipastikan dulu.**

**P17 [L2]. Tulis model matematis tiap fault.**
NB11: drift x + α·k di episode (α = 1 poin/hari dalam sampel), spike periodik
+0,08·SD, bias +0,08, hardware: 8 % nilai diganti nilai acak lain (macet),
5 % NaN. NB14: bias +5 selama episode, drift naik linear 0→5 per episode,
spike x·(1+0,30) acak 1,52 %. Kode: `simulate_*_fault`.

**P18 [L3]. Window kapan dilabel fault?**
NB11: porsi sampel fault > 1 % (`FAULT_RATIO_THR`); window di bawah itu dibuang.
NB14: semua komponen kombinasi harus ada (spike ≥ 1 titik, lainnya > 1 %).

**P19 [L1]. Bagaimana sensor-ID dirantai dari prediksi?**
Lihat `04_walkthrough_kode.md` (`LMSensorID`, §12 NB11): prediksi kelas
out-of-fold → model per sensor → F1 per sensor dan "semua-4-benar".

## 4. Evaluasi dan statistik

**P20 [L2]. Jelaskan nested grouped CV Anda.**
Luar: StratifiedGroupKFold 5 lipatan, grup = blok waktu. Dalam: 2 lipatan
untuk memilih arsitektur/aktivasi dengan macro-F1. Model terpilih dilatih
ulang di lipatan luar latih, dinilai di lipatan luar uji.

**P21 [L2]. Apakah masih ada kebocoran?** — `02` A1 (jujur: ya, di batas grup;
sama untuk kedua metode).

**P22 [L2]. Kenapa uji-t berpasangan? Asumsinya?**
Pasangan = sel yang sama (skenario × N) dengan dua metode. Asumsi: selisih
berdistribusi normal — dengan n = 5 tidak bisa dicek, karena itu Wilcoxon
dilaporkan juga.

**P23 [L2]. Kenapa macro-F1?**
Kelas tidak seimbang setelah pembuangan window; macro-F1 memberi bobot sama
pada tiap kelas sehingga kelas sulit (kombinasi) tidak tertutup kelas mudah.

**P24 [L2]. p = 0,0006 di NB14 — apakah karena fault diperbesar sampai JSD
menang?** — `02` A6 + C3 (tetap signifikan setelah Bonferroni).

**P25 [L1]. Ukuran efek?** NB14 Cohen d = 1,13 (cek di 01). NB11 selisih F1
rata-rata +0,028.

## 5. Implementasi dan reprodusibilitas

**P26 [L3]. Bagaimana memastikan hasil bisa diulang?**
Seed tetap di semua langkah acak, kernel Kaggle tercatat versinya, builder
`build_*.py` menghasilkan notebook dari teks yang sama, log Kaggle disimpan di
`hasil*/`. Klaim dicek ulang otomatis oleh `cek_klaim.py`.

**P27 [L3]. Kenapa model disimpan numpy, bukan pickle?**
Pickle sklearn dari Kaggle (1.6.1) gagal dibuka di laptop (1.9.1).
`SavedLM` / `.npz` hanya menyimpan angka (pengisi NaN, mean, skala, bobot) dan
prediksinya identik (selisih 0).

**P28 [L3]. Kenapa `sliding_window_view` perlu di-transpose?**
Hasilnya berbentuk (n_window, kanal, panjang); entropi harus dihitung per
kanal sepanjang waktu, jadi diubah ke (n_window, panjang, kanal). Tanpa itu
entropi dihitung atas potongan sepanjang 4. Kode: `make_windows`.

**P29 [L1]. Berapa biaya komputasinya?** NB11 wall 610 menit, ±27,5 jam-CPU;
NB14 478 menit. (Cek di 01.)

## 6. Pertanyaan "bagaimana kalau"

**P30.** Bagaimana kalau sensor kelima ditambahkan? → fitur bertambah 15, model
dilatih ulang; arsitektur dan metode tidak berubah.
**P31.** Bagaimana kalau hanya satu sensor? → tidak ada pembanding antar sensor;
lokalisasi hilang (notebook 02).
**P32.** Bagaimana dipakai di lapangan? → model tersimpan (NB16 / NB15 `.npz`),
satu keputusan per window 16,7 jam; untuk notifikasi cepat pakai NB15 (per
sampel, tanpa entropi) dengan aturan persistensi.
**P33.** Bagaimana kalau penguji minta demo langsung? → `sidang/PANDUAN_DEMO.md`.

---

## Latihan

1. Tutup kolom jawaban, jawab lisan ≤ 30 detik, buka bukti di layar ≤ 20 detik.
2. Untuk P1, P4, P9, P10: tulis rumus di kertas tanpa melihat.
3. Untuk P8, P21, P24: latih kalimat "akui → dampak → mitigasi" sampai lancar.

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Kelemahan dan ancaman validitas</b> (`sidang/qa/02_kelemahan.md`)</summary>

# 02 — Daftar kelemahan (disusun sebelum penguji menemukannya)

Disusun dari kode dan hasil mentah, dikelompokkan menurut jenis ancaman
validitas. Tiap butir: **apa kelemahannya**, **bukti** (file:baris), dan
**jawaban 2 kalimat** (akui → dampak → mitigasi). Angka yang disebut harus
sudah berstatus COCOK di `01_klaim_bukti.csv`.

Status: **[DIKETAHUI]** sudah ditulis di dokumen penelitian; **[BARU]**
ditemukan saat menyusun bahan ini dan belum ada di laporan.

---

## A. Validitas internal (apakah perbandingannya adil dan bebas bocor?)

**A1. Kebocoran antar grup CV yang bersebelahan [BARU]**
- Bukti: `build_11_annlm_real_notebook.py:601` `groups = starts // WIN`,
  stride = WIN/2 (`:557`). Window yang mulai di WIN/2 (grup 0) mencakup
  [WIN/2, 1,5·WIN] dan tumpang-tindih 50 % dengan window yang mulai di WIN
  (grup 1). Grup tetangga bisa jatuh ke fold berbeda.
- Jawaban: "Pengelompokan per blok waktu mencegah window yang sama persis bocor,
  tetapi window di batas blok masih berbagi setengah sampelnya dengan blok
  sebelah, jadi skor absolutnya sedikit optimistis. Kebocoran ini berlaku sama
  untuk EDM dan JSD karena keduanya memakai window, grup, dan fold yang
  identik, sehingga **selisih** keduanya tetap sah; perbaikannya adalah memberi
  celah (purge) satu window di setiap batas fold."
- **Diukur (2026-11-20)**, `uji_purge_permutasi_N700.py` → `hasil_uji/efek_purge_permutasi.csv`
  (N = 700, S1–S5, 5 lipatan, arsitektur tetap dari `models/meta.json`,
  jendela latih yang tumpang-tindih dengan jendela uji dibuang):
  - Purge mengubah F1 antara −0,072 (S1 EDM) dan +0,012 (S5 JSD); 7 dari 10
    sel turun. Purge juga membuang ±20 % data latih (S1: 126 → 100 jendela),
    jadi penurunan itu batas atas efek kebocoran, bukan efek murninya.
  - Arah perbandingan tidak berubah: setelah purge JSD ≥ EDM di 5 dari 5
    skenario (tanpa purge 4 dari 5; S3 EDM sedikit lebih tinggi). Per skenario
    hanya S1 purge yang signifikan setelah koreksi Nadeau–Bengio dan Holm
    (p = 0,039); lainnya p_NB ≥ 0,41.
  - Kontrol label acak (S1, S5): F1 0,446–0,508, setara tebakan 0,5 → pipeline
    tidak belajar dari artefak.
- Jawaban dengan angka: "Saya ukur dengan purge: F1 turun paling banyak 0,07,
  dan JSD tetap tidak kalah dari EDM di kelima skenario. Dengan label acak F1
  jatuh ke 0,5, tingkat tebakan."

**A2. Satu sinyal bersih dipakai berulang di semua kondisi**
- Bukti: `segment_and_label` (`build_11…:556–595`) menyuntik fault ke salinan
  data yang sama untuk tiap kondisi; grup = blok waktu yang sama lintas kondisi.
- Jawaban: "Semua kelas berasal dari latar sinyal yang sama, jadi model tidak
  bisa membedakan kelas dari 'hari yang berbeda'; ini justru memaksa model
  belajar dari pola fault. Karena grup CV dihitung dari posisi waktu, blok
  yang sama di semua kondisi selalu masuk fold yang sama."

**A3. Anggaran waktu latih memotong sebagian fold**
- Bukti: `LM_SEARCH_BUDGET_S = 40`, `LM_REFIT_BUDGET_S = 150`
  (`build_11…:1613–1614`); `hasil14/14_draf_laporan.md` "beberapa fold berhenti
  karena batas waktu latih".
- Jawaban: "LM dibatasi waktu per fit supaya 15 sel muat di Kaggle; batasnya
  sama untuk kedua metode dan dimensi fiturnya sama (60), jadi tidak ada metode
  yang diberi anggaran lebih. Status henti setiap fold tercatat di log, sehingga
  bisa ditunjukkan berapa yang terpotong."
- Siapkan: jumlah fold `budget(MAX_TRAIN_S)` per metode dari log (lihat
  `01_klaim_bukti.csv`).

**A4. Pencarian arsitektur memakai iterasi pendek**
- Bukti: `LM_ITERS_SEARCH = 6`, `LM_ITERS_REFIT = 40` (`build_11…:1610–1611`).
- Jawaban: "LM konvergen cepat, jadi 6 iterasi cukup untuk mengurutkan kandidat;
  model akhir dilatih ulang 40 iterasi. Prosedurnya identik untuk kedua metode."

**A5. Bobot awal tersusun lain dari maksudnya (`lm_pack` vs `lm_unpack`) [BARU]**
- Bukti: `deteksi_fault_core.py:452–462`. `lm_pack` menyusun semua W lalu semua
  b; `lm_unpack` membaca W1, b1, W2, b2. Latih dan prediksi sama-sama lewat
  `lm_unpack`, jadi konsisten; yang berubah hanya bobot awal.
- Jawaban: "Ada ketidakcocokan urutan saat inisialisasi, sehingga sebagian bobot
  awal tidak persis mengikuti Xavier; pelatihan dan prediksi tetap konsisten
  karena memakai urutan yang sama. Karena EDM dan JSD memakai fungsi yang sama
  dengan seed yang sama, dampaknya tidak memihak satu metode; perbaikannya satu
  baris dan hasil uji ulang bisa dilampirkan."
- Detail angka: `04_walkthrough_kode.md` bagian "Cacat yang diketahui".

**A6. Parameter fault NB14 dipilih setelah melihat NB13 [DIKETAHUI, sebagian]**
- Bukti: `hasil14/14_draf_laporan.md` §Catatan — tingkat "sedang" diambil dari
  notebook 13, dan tab "Perubahan program" di dokumen usulan tidak tersedia.
- Risiko: penguji bisa bertanya apakah tingkat fault dipilih supaya JSD menang.
- Jawaban: "Tingkat 'sedang' adalah titik tengah kisi intensitas di notebook 13,
  ditetapkan sebelum notebook 14 dijalankan dan tidak diubah setelahnya. Hasil
  utama skripsi tetap notebook 11; notebook 14 saya laporkan sebagai uji
  tambahan, lengkap dengan sel di mana EDM menang (N=700 S4, S5; N=1000 S3)."

## B. Validitas konstruk (apakah yang diukur memang yang dimaksud?)

**B1. Fault simulasi, bukan kerusakan nyata [DIKETAHUI]** — Q10 `Bahan_Sidang.md`.
Jawaban siap: label pasti dan bisa diulang; berlaku untuk fault yang dimodelkan.

**B2. Entropi buta terhadap bias konstan karena centroid shifting [DIKETAHUI]**
- Bukti: `embed_matrix_centered` (`deteksi_fault_core.py:274`); NB12/13: AUC
  bias ≈ 0,50 berapa pun besarnya.
- Jawaban: "Ini keterbatasan representasi, bukan besarnya fault: rata-rata tiap
  vektor dikurangkan sebelum entropi dihitung, jadi offset hilang. Saran saya
  menambah fitur level (rata-rata window atau selisih antar sensor)."

**B3. Drift kecil tidak terlihat [DIKETAHUI]** — NB13: AUC ≤ 0,70 untuk total
2–9 poin; NB15 (tanpa entropi) drift tunggal F1 0,49–0,71. Jawaban: drift
realistis berada di bawah ayunan alami sinyal dalam satu window.

**B4. Bentuk fungsi keanggotaan berbeda dari rumus buku [perlu dijelaskan]**
- Bukti: `phi_euclidean` / `phi_jsd`: `mu = 1/(1 + d/r)` (`build_11…:706,756`),
  bukan `exp(-d^n/r)` seperti fuzzy entropy Chen dkk.
- Jawaban: "Saya memakai fungsi keanggotaan rasional yang turun monoton, sama
  untuk EDM dan JSD, sehingga satu-satunya perbedaan antar metode tetap pada
  ukuran jarak. Penurunannya ada di `03_algoritma_dengan_tangan.md`."

**B5. JSD dihitung dari vektor pendek (m = 2) yang diubah jadi sebaran 8 bin**
- Bukti: `vectors_to_distributions` (`deteksi_fault_core.py:317`), `JSD_BINS=8`.
- Pertanyaan yang mungkin muncul: "Sebaran dari dua angka itu bermakna?"
- Jawaban: "Dua angka tidak cukup untuk histogram biasa, karena itu dipakai soft
  binning dengan kernel halus selebar `h`, sehingga tiap vektor menjadi sebaran
  halus atas 8 pusat bin. Sensitivitas jumlah bin dan lebar kernel diuji di
  notebook 12 (rekomendasi `bins=4`, `h_ratio=1,0`)."

**B6. Asal-usul data [PERLU DIPASTIKAN SEBELUM SIDANG]**
- Pengamatan: baris awal `data_sensor.csv` bernilai pecahan yang naik rata
  (`id` 1,0 → 1,875 → 2,75), tanda hasil interpolasi ke grid 30 detik.
  `tabel_sensor4_generated.csv` bernama "generated" dan disebut Bu Luh "data
  buatan".
- Penguji bisa bertanya "data ini asli atau buatan?". Jawabannya harus pasti:
  dari mana sensor diambil, siapa yang menginterpolasi, dan apa yang
  "generated" pada tabel kedua. **Tanyakan ke Bu Luh**; jangan mengarang.

## C. Validitas kesimpulan statistik

**C1. Pooled NB11 tidak signifikan (p = 0,076; Wilcoxon 0,107) [DIKETAHUI]**
— klaim yang dipakai "konsisten", bukan "terbukti". Q39–Q41 `Bahan_Sidang.md`.

**C2. Uji gabungan memakai 15 sel yang tidak saling bebas [BARU]**
- Lima skenario yang sama muncul di tiga ukuran window, jadi 15 pasangan bukan
  15 pengamatan independen (pseudo-replikasi). Uji per N hanya punya n = 5
  (df = 4) — dayanya rendah.
- Jawaban: "Uji gabungan saya baca sebagai indikasi arah, bukan bukti kuat,
  karena sel-selnya berbagi skenario; uji per ukuran window yang lebih tepat
  hanya punya lima pasangan sehingga dayanya rendah. Karena itu saya
  menekankan konsistensi arah dan ukuran efek, bukan p-value."

**C3. Banyak uji tanpa koreksi perbandingan berganda [BARU]**
- NB11: 3 uji per N + 1 gabungan; NB14 menambah 4 lagi.
- Jawaban: "Dengan koreksi Bonferroni untuk 4 uji (α = 0,0125), hanya NB14
  gabungan p = 0,0006 yang tetap signifikan; N=700 NB11 (p = 0,027) tidak.
  Itu sejalan dengan klaim saya yang sudah berhati-hati."

**C4. Variasi antar seed tidak dilaporkan**
- Setiap sel satu seed; tidak ada selang kepercayaan antar ulangan.
- Jawaban: "Variasi yang saya laporkan adalah antar fold CV; ulangan multi-seed
  belum dilakukan karena satu run penuh butuh ±8–10 jam Kaggle. Ini saya
  masukkan sebagai saran."

**C5. EDM lebih stabil (56/63 pasangan) [DIKETAHUI]** — kalimat siap pakai no. 3.

**C6. NB10 dan NB11 memberi p berbeda untuk pipeline yang sama [DIKETAHUI]**
— README: beda protokol CV dan anggaran komputasi. Jawaban: keduanya dilaporkan
apa adanya; perbedaan itu sendiri menunjukkan hasil gabungan belum stabil.

## D. Validitas eksternal

**D1.** Satu lokasi, 4 sensor, ±98 hari; sensor kapasitif murah.
**D2.** Window 16,7 jam → pemantauan harian, bukan alarm real-time (Q43).
**D3.** S2–S4 tetap sulit (F1 0,21–0,53 di NB14) — model belum siap memilah
kombinasi fault.
**D4.** FPR kelas normal ±0,13 (NB13) — penyiraman/pengeringan alami menaikkan
entropi. Saran: konteks penyiraman/cuaca.

## E. Notebook 15 (data hipotetik, tanpa entropi)

**E1.** Drift tunggal F1 0,49 (4 sensor) / 0,71 (rata-rata).
**E2.** Pembagian data berbeda dari program pertama (blok harian 3:1:1, bukan
60/20/20 berurutan) — alasannya: bagian uji 60/20/20 tidak memuat episode.
**E3.** FPR sensor yang tidak disuntik fault ±6 %.
**E4.** S1 kebetulan memilih jenis fault berbeda di dua versi (seed berbeda).
**E5.** Fitur hanya lag; tidak memakai sensor tetangga — karena itu bias/drift
lemah. Bukan bagian klaim utama skripsi.

---

## Prioritas sebelum sidang

1. B6 — pastikan asal-usul data (tanya Bu Luh).
2. A1 dan A5 — siapkan jawaban; jika sempat, uji ulang satu sel dengan purge
   dan dengan `lm_pack` diperbaiki untuk menunjukkan dampaknya kecil.
3. C2/C3 — hafalkan kalimat Bonferroni.

---

## F. Temuan tambahan dari pemeriksaan kode baris per baris [BARU]

Semua dibuktikan oleh `cek_cacat_kode.py` (jalankan:
`./venv/bin/python sidang/qa/cek_cacat_kode.py`). Detail di
`04_walkthrough_kode.md` bagian "Cacat yang diketahui".

**F1. Tahap identifikasi sensor memakai split acak, bukan grouped**
- Bukti: `build_11_annlm_real_notebook.py:2065`
  `train_test_split(..., test_size=0.3, random_state=RANDOM_SEED)`.
- Dampak: window tumpang-tindih bisa masuk latih dan uji sekaligus → F1 sensor-ID
  (0,675 vs 0,634; NB14 0,693 vs 0,791) optimistis. Berlaku sama untuk EDM dan
  JSD.
- Jawaban: "Tahap klasifikasi memakai grouped CV, tetapi tahap identifikasi
  sensor memakai pembagian acak 70/30, sehingga angka F1 sensornya lebih
  optimistis daripada tahap pertama. Perbandingan EDM–JSD tetap memakai
  pembagian yang sama; perbaikannya mengganti split itu dengan GroupShuffleSplit
  berdasarkan blok waktu."

**F2. Bobot awal: lapisan terakhir mulai dari nol (lanjutan A5)**
- `lm_pack`/`lm_unpack` berbeda urutan → untuk jaringan 2 lapisan tersembunyi,
  100 % bobot lapisan keluaran mulai dari nol, sehingga pada iterasi pertama
  kolom Jacobian lapisan-lapisan awal bernilai nol; 1 lapisan dengan 2 keluaran:
  50 %. Hanya W1 yang benar-benar Xavier.
- Semua jalur (latih, prediksi, `SavedLM`, `LMSensorID`, `.npz` NB15) memakai
  urutan `lm_unpack` → konsisten, prediksi model tersimpan identik.
- Teks skripsi **jangan** menulis "inisialisasi Xavier dengan bias nol".
- Jawaban: lihat A5 — sama untuk kedua metode; perbaikan satu baris; perlu run
  ulang untuk angka baru.

**F3. Langkah LM yang ditolak tetap menghabiskan jatah iterasi**
- `lm_fit`: tiap penolakan λ×2 dan iterasi tetap terhitung; pencarian arsitektur
  hanya 6 iterasi (`LM_ITERS_SEARCH`). Pada contoh mini dibutuhkan 7 penolakan
  sebelum langkah pertama diterima, jadi pada pencarian sebagian kandidat bisa
  hampir tidak bergerak dari bobot awal (risiko; belum diukur di data asli —
  cek `lm_status`/`iters` di log).
- Jawaban: "Pemilihan arsitektur memakai iterasi pendek; kandidat yang belum
  bergerak dinilai rendah sehingga pemilihan cenderung ke arsitektur yang
  cepat konvergen. Prosedur ini identik untuk kedua metode."

**F4. Simulator NB11: hardware 'macet' efektif 3 %, bukan 8 %**
- `simulate_hardware_fault` (`deteksi_fault_core.py:135–146`): titik 'hilang'
  (rv < 0,05) adalah subset titik 'macet' (rv < 0,08) lalu ditimpa NaN.
  Total titik rusak tetap 8 %: 5 % NaN + 3 % macet.
- Jawaban: "Total proporsi titik rusak tetap 8 %, tetapi komposisinya 5 % hilang
  dan 3 % macet, bukan 8 % + 5 %. Deskripsi di laporan saya perbaiki."

**F5. Simulator NB11: drift, spike, bias tidak memakai seed**
- Posisi spike identik untuk semua seed dan semua sensor (periodik tiap
  1/p sampel). NB14 sudah memakai posisi acak ber-seed.
- Jawaban: "Pada NB11 spike sengaja periodik sehingga posisinya sama di semua
  sensor; variasi antar sensor hanya dari subset sensor yang disuntik. NB14
  memperbaikinya dengan posisi acak."

**F6. Wilcoxon dengan 5 pasangan tidak mungkin p < 0,05**
- Uji eksak dua sisi dengan n = 5: p minimum = 2/2⁵ = 0,0625. Jadi "Wilcoxon
  tidak signifikan" per ukuran window bukan bukti tidak ada efek.
- Jawaban: "Per ukuran window Wilcoxon hanya punya lima pasangan, sehingga
  secara matematis p terkecilnya 0,0625; karena itu saya membaca uji-t dan
  ukuran efek, dan menyebut hasilnya deskriptif."

**F7. Penyimpangan dari rumus buku (harus bisa dijelaskan, bukan bug)**
- Jarak Euclidean, bukan Chebyshev (Chen 2007).
- Fungsi keanggotaan 1/(1+d/r), bukan exp(−dⁿ/r).
- Coarse-graining komposit (Wu 2013), bukan satu offset (Costa).
- Titik acuan acak n_ref = 128, set berbeda untuk m dan m+1.
- JSD soft-histogram mengabaikan urutan nilai di dalam vektor (pola naik dan
  turun dianggap sama) → entropi JSD-Fuzzy bisa bernilai negatif
  (contoh: −0,115 pada deret 8 titik).
  Bukti kode: `vectors_to_distributions` menjumlahkan kernel atas elemen vektor
  (`K.sum(axis=1)`), jadi template [−1, 1] dan [1, −1] memberi JSD = 0 persis
  (dicek 2026-11-20), sedangkan Euclidean memberi d = 2,83. JSD-Fuzzy mengukur
  kemiripan *sebaran nilai*, bukan *bentuk pola*; tulis ini di bab metode.
- Argumen `r_ratio` pada `jsd_fuzzy_entropy_1d` tidak dipakai.
- Rujukan rumus dan contoh angka: `03_algoritma_dengan_tangan.md`.

## Prioritas sebelum sidang (diperbarui)

1. B6 — asal-usul data (tanya Bu Luh).
2. F1 dan A1 — kebocoran: siapkan jawaban; idealnya uji ulang satu sel
   dengan GroupShuffleSplit + purge.
3. F2/A5 — hapus klaim "Xavier dengan bias nol" dari teks; siapkan jawaban.
4. F4 — perbaiki deskripsi hardware di laporan (5 % hilang + 3 % macet).
5. C2/C3/F6 — hafalkan kalimat statistik.

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Algoritma dengan tangan (LM, EDM, JSD)</b> (`sidang/qa/03_algoritma_dengan_tangan.md`)</summary>

# 03 — Algoritma dengan tangan (angka nyata dari kode)

Dokumen ini menghitung **dengan tangan** empat hal yang paling sering diminta
penguji: (a) satu iterasi Levenberg–Marquardt, (b) fuzzy entropy dengan jarak
Euclidean (EDM-Fuzzy), (c) fuzzy entropy dengan kemiripan Jensen–Shannon
(JSD-Fuzzy), dan (d) self-check Jacobian. Semua angka **bukan karangan**: tiap
angka dihitung ulang oleh `sidang/qa/cek_algoritma_tangan.py` lewat dua jalur
(rumus ditulis terbuka dengan loop skalar, dan fungsi asli di
`deteksi_fault_core.py`), lalu dicocokkan dengan toleransi $10^{-9}$. Keluaran
lengkap skrip ditempel di bagian akhir.

Cara menjalankan (dari akar repo):

```
nice -n 19 ./venv/bin/python sidang/qa/cek_algoritma_tangan.py
```

Angka di tabel dibulatkan 6 desimal supaya terbaca; angka kunci ditulis 12
desimal dan itulah yang dicek skrip.

---

## (a) Satu iterasi Levenberg–Marquardt

### a.1 Rumus dari paper dan rumus di kode

**Masalah.** Jaringan dilatih dengan meminimalkan jumlah kuadrat residual

$$E(\theta) = \tfrac12\, r(\theta)^\top r(\theta),\qquad r = \operatorname{vec}(O(\theta) - Y),$$

dengan $O$ keluaran jaringan dan $Y$ target one-hot (kode: `lm_residuals`,
`deteksi_fault_core.py:477-479`; $r$ diurutkan baris-per-baris:
$r = [r_{1,1}, r_{1,2}, r_{2,1}, \dots]$).

**Levenberg (1944)** menambahkan $\lambda I$ ke persamaan Gauss–Newton.
**Marquardt (1963)** mengganti $I$ dengan diagonal $J^\top J$ supaya langkah
tidak tergantung skala tiap parameter:

$$\big(J^\top J + \lambda\,\operatorname{diag}(J^\top J)\big)\,\Delta = -J^\top r .$$

Kode: `deteksi_fault_core.py:536-539`
(`d = np.diag(A)`, dijepit minimal $10^{-12}$, lalu
`np.linalg.solve(A + lam * np.diag(d), -g)`). Penerapan LM ke jaringan saraf
dengan Jacobian dari backpropagation mengikuti **Hagan & Menhaj (1994)**.

**Rasio gain dan aturan Nielsen** (Nielsen 1999; Madsen, Nielsen & Tingleff
2004, "Methods for Non-Linear Least Squares Problems", Alg. 3.16):

$$\rho = \frac{E(\theta) - E(\theta+\Delta)}{L(0) - L(\Delta)},\qquad
L(0)-L(\Delta) = \tfrac12\,\Delta^\top\big(\lambda D\Delta - g\big),\quad g = J^\top r,\; D=\operatorname{diag}(d).$$

Jika $\rho>0$: terima, $\lambda \leftarrow \lambda\cdot\max\!\big(\tfrac13,\,1-(2\rho-1)^3\big)$, $\nu\leftarrow 2$.
Jika $\rho\le 0$: tolak, $\lambda \leftarrow \lambda\,\nu$, $\nu \leftarrow 2\nu$.

Kode: `deteksi_fault_core.py:546-563`.

**Perbedaan kode vs buku (jujur):**

1. Uji terima di kode adalah $E_\text{baru} < E$ (baris 552), bukan $\rho>0$.
   Keduanya **setara** di sini, karena penyebut selalu positif:
   dari $(A+\lambda D)\Delta = -g$ diperoleh
   $\lambda D\Delta - g = A\Delta + 2\lambda D\Delta$, sehingga
   $L(0)-L(\Delta) = \tfrac12(\Delta^\top A\Delta + 2\lambda\Delta^\top D\Delta) > 0$.
   Cabang `else lam * 2.0` di baris 560 (saat $\rho\le 0$ tetapi galat turun)
   praktis tidak pernah terjadi.
2. Saat ditolak, kode **selalu** mengalikan $\lambda$ dengan 2 (baris 562).
   Nielsen menggandakan $\nu$ tiap penolakan beruntun (×2, ×4, ×8, …). Akibatnya
   penolakan beruntun lebih lama keluar dari daerah $\lambda$ kecil — terlihat di
   contoh di bawah: 7 penolakan berturut-turut (Nielsen asli butuh 3–4).
3. Skala $\operatorname{diag}(J^\top J)$ (Marquardt) dipadukan dengan aturan
   $\lambda$ Nielsen yang aslinya dirumuskan untuk $\mu I$. Ini kombinasi yang
   lazim (lihat Transtrum & Sethna 2012), tetapi bukan resep satu paper.
4. Pengaman tambahan: $d_j \ge 10^{-12}$ (kolom Jacobian nol), $\lambda \in
   [10^{-12}, 10^{12}]$, $\lambda\times10$ bila `solve` gagal/hasil tak hingga,
   berhenti bila $\lVert g\rVert < 10^{-10}$, bila penurunan relatif
   $\le 10^{-12}$, bila iterasi habis, atau bila waktu latih > `max_train_s`.
5. **Iterasi yang ditolak tetap dihitung sebagai iterasi** (loop `for k in
   range(iters)`), dan Jacobian dihitung ulang walau $\theta$ tidak berubah.
   Dengan `LM_ITERS_SEARCH = 6` saat grid search, model bisa saja selesai tanpa
   satu langkah pun diterima (contoh di bawah butuh 8 iterasi sampai langkah
   pertama diterima).
6. Lapisan keluaran **linear** dan dilatih dengan kuadrat galat terhadap one-hot
   (gaya Hagan–Menhaj). Softmax **hanya** dipasang saat prediksi
   (`lm_predict_proba`, baris 570-575) — tidak ikut dilatih, jadi "probabilitas"
   tidak terkalibrasi, tetapi argmax-nya sama dengan argmax keluaran linear.

### a.2 Data mainan

Jaringan 2 masukan – 2 tersembunyi (tanh) – 2 keluaran (linear), 3 sampel:

$$X=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix},\qquad
Y=\begin{bmatrix}1&0\\0&1\\1&0\end{bmatrix}\;(\text{kelas }0,1,0).$$

Bobot awal **persis** seperti `lm_fit(X, Y, 2, [2], 2, seed=0)`:
`lm_init_layers` (Xavier/Glorot 2010, $U(-\ell,\ell)$, $\ell=\sqrt{6/(n_\text{in}+n_\text{out})}=\sqrt{6/4}=1{,}224745$)
memberi

$$W_1^\text{niat}=\begin{bmatrix}0.335486&-0.563905\\-1.124381&-1.184261\end{bmatrix},\;
W_2^\text{niat}=\begin{bmatrix}0.767352&1.011041\\0.261203&0.562149\end{bmatrix},\; b_1=b_2=0.$$

`lm_pack` menyusun $\theta_0$ = [semua W, lalu semua b] =
`[0.335486, -0.563905, -1.124381, -1.184261, 0.767352, 1.011041, 0.261203, 0.562149, 0, 0, 0, 0]`.
`lm_unpack` membaca berselang $W_1, b_1, W_2, b_2$, sehingga jaringan yang
**sungguh dipakai** adalah

$$W_1=\begin{bmatrix}0.335486&-0.563905\\-1.124381&-1.184261\end{bmatrix},\;
b_1=[0.767352,\;1.011041],\;
W_2=\begin{bmatrix}0.261203&0.562149\\0&0\end{bmatrix},\;
b_2=[0,\;0].$$

Ini cacat `lm_pack`/`lm_unpack` (dibahas lengkap di `04_walkthrough_kode.md`,
bagian "Cacat yang diketahui"). Di contoh ini akibatnya terlihat jelas: baris
kedua $W_2$ nol, jadi unit tersembunyi ke-2 tidak berpengaruh ke keluaran pada
awal latihan.

Urutan parameter (kolom Jacobian) = urutan `lm_unpack`:
$[W_{1,00}, W_{1,01}, W_{1,10}, W_{1,11}, b_{1,0}, b_{1,1}, W_{2,00}, W_{2,01}, W_{2,10}, W_{2,11}, b_{2,0}, b_{2,1}]$.

### a.3 Maju (forward)

$z_1 = XW_1 + b_1$, $h=\tanh(z_1)$, $o = hW_2 + b_2$. Contoh sampel 1
($x=(1,0)$): $z_1 = (0.335486+0.767352,\; -0.563905+1.011041) = (1.102838,\;0.447135)$,
$h = (\tanh 1.102838, \tanh 0.447135) = (0.801516, 0.419542)$,
$o_1 = 0.801516\cdot0.261203 + 0.419542\cdot 0 = 0.209359$,
$o_2 = 0.801516\cdot0.562149 = 0.450572$.

| sampel | $z_1$ | $h=\tanh z_1$ | $o$ | $r = o - y$ |
|---|---|---|---|---|
| 1 | (1.102838, 0.447135) | (0.801516, 0.419542) | (0.209359, 0.450572) | (−0.790641, 0.450572) |
| 2 | (−0.357028, −0.173220) | (−0.342594, −0.171508) | (−0.089487, −0.192589) | (−0.089487, −1.192589) |
| 3 | (−0.021542, −0.737125) | (−0.021539, −0.627405) | (−0.005626, −0.012108) | (−1.005626, −0.012108) |

$$E_0 = \tfrac12\sum r^2 = \mathbf{1.634917719156}.$$

### a.4 Jacobian analitik lewat backpropagation

Karena keluaran linear, turunan residual $r_{ic}$ (sampel $i$, keluaran $c$):

$$\frac{\partial r_{ic}}{\partial b_{2,q}} = [q=c],\quad
\frac{\partial r_{ic}}{\partial W_{2,pq}} = h_{ip}[q=c],\quad
\frac{\partial r_{ic}}{\partial b_{1,q}} = W_{2,qc}\,(1-h_{iq}^2),\quad
\frac{\partial r_{ic}}{\partial W_{1,pq}} = x_{ip}\,W_{2,qc}\,(1-h_{iq}^2).$$

Di kode (`lm_residual_jacobian`, baris 482-508) ini dikerjakan sekaligus untuk
semua keluaran: $\delta$ awal = identitas $I_{n_\text{out}}$ per sampel (baris
487, karena $\partial o_c/\partial o_q = [q=c]$), blok $W$ =
$a_\text{prev}\otimes\delta$ (`einsum "ip,icq->icpq"`, baris 493), blok $b$ =
$\delta$, lalu $\delta \leftarrow (\delta W^\top)\odot(1-h^2)$ (baris 497-500;
untuk ReLU turunannya $[h>0]$).

Contoh satu angka: $J_{(1,1),W_{1,00}} = x_{10}\,W_{2,00}(1-h_{10}^2) =
1\cdot0.261203\cdot(1-0.801516^2) = 0.261203\cdot0.357572 = 0.093399$.

$J$ ($6\times12$, baris = $r_{11}, r_{12}, r_{21}, r_{22}, r_{31}, r_{32}$):

| | W1_00 | W1_01 | W1_10 | W1_11 | b1_0 | b1_1 | W2_00 | W2_01 | W2_10 | W2_11 | b2_0 | b2_1 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| r11 | 0.093399 | 0 | 0 | 0 | 0.093399 | 0 | 0.801516 | 0 | 0.419542 | 0 | 1 | 0 |
| r12 | 0.201009 | 0 | 0 | 0 | 0.201009 | 0 | 0 | 0.801516 | 0 | 0.419542 | 0 | 1 |
| r21 | 0 | 0 | 0.230546 | 0 | 0.230546 | 0 | −0.342594 | 0 | −0.171508 | 0 | 1 | 0 |
| r22 | 0 | 0 | 0.496170 | 0 | 0.496170 | 0 | 0 | −0.342594 | 0 | −0.171508 | 0 | 1 |
| r31 | 0.261082 | 0 | 0.261082 | 0 | 0.261082 | 0 | −0.021539 | 0 | −0.627405 | 0 | 1 | 0 |
| r32 | 0.561889 | 0 | 0.561889 | 0 | 0.561889 | 0 | 0 | −0.021539 | 0 | −0.627405 | 0 | 1 |

Kolom $W_{1,01}, W_{1,11}, b_{1,1}$ nol karena $W_{2,1\cdot}=0$ (akibat cacat
pack/unpack). Itu juga contoh nyata **mengapa** kode menjepit
$d_j=\max(d_j,10^{-12})$: tanpa itu baris/kolom nol membuat
$A+\lambda D$ singular.

### a.5 Iterasi 1 (ditolak)

$g = J^\top r$:
`[-0.252630, 0, -0.881712, 0, -0.864988, 0, -0.581394, 0.769975, 0.314576, 0.401169, -1.885754, -0.754125]`,
$\lVert g\rVert = \mathbf{2.627734635861}$.

$A = J^\top J$ (lengkap di keluaran skrip), diagonalnya
$d = $ `[0.433011, 1e-12, 0.683218, 1e-12, 0.732346, 1e-12, 0.760263, 0.760263, 0.599067, 0.599067, 3, 3]`.

Dengan $\lambda = 10^{-3}$ (`lam0`), selesaikan $(A + \lambda D)\Delta=-g$:
$\Delta$ = `[-0.781785, 0, 0.618051, 0, 0.597494, 0, 1.534301, -1.319635, -1.285292, 0.862243, 0.116816, 0.282593]`
(sisa persamaan $\le 2{,}2\times10^{-16}$).

Residual percobaan $r(\theta_0+\Delta)$ =
`[-0.120300, 0.095003, 1.586013, -1.392115, 0.624098, -0.554071]`,
$E_\text{baru} = \mathbf{2.586706041095} > E_0$.

$L(0)-L(\Delta) = \tfrac12\Delta^\top(\lambda D\Delta - g) = 1.634907423098$,
$\rho = (1.634917719156 - 2.586706041095)/1.634907423098 = \mathbf{-0.582166493645}$.

**Ditolak**: $\theta$ tetap, $\lambda \leftarrow 2\lambda = 0.002$. Langkah
Gauss–Newton hampir murni ($\lambda$ kecil) terlalu jauh untuk masalah
nonlinear ini; 6 residual vs 12 parameter juga berarti $J^\top J$ singular
(rank $\le 6$), hanya $\lambda D$ yang membuatnya dapat dibalik.

### a.6 Iterasi 2–9

| it | $\lambda$ dipakai | $E$ | $E$ percobaan | $L(0)-L(\Delta)$ | $\rho$ | terima | $\lambda$ baru |
|---|---|---|---|---|---|---|---|
| 1 | 0.001 | 1.634917719 | 2.586706041 | 1.634907423 | −0.582166 | tidak | 0.002 |
| 2 | 0.002 | 1.634917719 | 2.570539124 | 1.634876917 | −0.572289 | tidak | 0.004 |
| 3 | 0.004 | 1.634917719 | 2.538810320 | 1.634757500 | −0.552922 | tidak | 0.008 |
| 4 | 0.008 | 1.634917719 | 2.477672606 | 1.634299728 | −0.515667 | tidak | 0.016 |
| 5 | 0.016 | 1.634917719 | 2.363942083 | 1.632613779 | −0.446538 | tidak | 0.032 |
| 6 | 0.032 | 1.634917719 | 2.165689337 | 1.626843925 | −0.326258 | tidak | 0.064 |
| 7 | 0.064 | 1.634917719 | 1.856812411 | 1.609412045 | −0.137873 | tidak | 0.128 |
| 8 | 0.128 | 1.634917719 | 1.453046600 | 1.565691880 | +0.116160 | ya | 0.185909510 |
| 9 | 0.185910 | 1.453046600 | 0.194302037 | 1.381334784 | +0.911252 | ya | 0.082462782 |

Iterasi 8 diterima tetapi $\rho$ kecil: faktor Nielsen
$1-(2\cdot0.116160-1)^3 = 1-(-0.767680)^3 = 1.452418 > 1$, jadi $\lambda$
justru **naik** ($0.128\times1.452418 = 0.185910$) — model "kurang percaya"
pada pendekatan kuadratnya.

**Iterasi 9 rinci (diterima, $\rho$ besar):**
$\lVert g\rVert = 4.094388793528$;
$\Delta$ = `[0.280035, -0.557554, -0.171412, 0.682996, -0.107852, 0.304873, -0.111151, 0.208397, -0.026669, -0.189804, -0.062157, 0.127196]`;
$E = 1.453046600031 \to E_\text{baru} = \mathbf{0.194302037423}$;
$L(0)-L(\Delta) = \mathbf{1.381334783791}$;
$\rho = (1.453046600031-0.194302037423)/1.381334783791 = \mathbf{0.911252346194}$;
faktor $=\max(1/3,\,1-(2\cdot0.911252-1)^3) = \mathbf{0.443564086033}$;
$\lambda_\text{baru} = 0.185909509851\times0.443564086033 = \mathbf{0.082462781822}$.

Sesudah iterasi 9, `predict_proba` (softmax keluaran linear) =
`[[0.698230, 0.301770], [0.418494, 0.581506], [0.693228, 0.306772]]` → kelas
`[0, 1, 0]` = target.

Skrip mencocokkan $\theta$ sesudah tiap iterasi dengan `lm_fit(..., iters=k)`
dan $\lambda$ dengan `hist["lam"]` untuk $k=1..9$: selisih 0.

---

## (b) Fuzzy entropy dengan jarak Euclidean (EDM-Fuzzy)

### b.1 Rumus paper

**Costa et al. (2002, 2005) — multiskala (coarse-graining):**

$$y^{(\tau)}_j = \frac1\tau\sum_{i=(j-1)\tau+1}^{j\tau} x_i,\qquad j = 1..\lfloor N/\tau\rfloor,$$

dengan toleransi $r$ **tetap** dari deret asli untuk semua skala.

**Composite MSE (Wu et al. 2013)** — dipakai kode: untuk skala $\tau$ dibuat
$\tau$ deret bergeser $k=0..\tau-1$:
$y^{(\tau,k)}_j = \frac1\tau\sum_{i=0}^{\tau-1} x_{k+(j-1)\tau+i}$
(`coarse_grain_cmse`, baris 267-271), dan entropi skala $\tau$ = rata-rata
entropi tiap $k$ (baris 311-313).

**Chen et al. (2007), FuzzyEn:**

1. vektor $X^m_i = [u_i,\dots,u_{i+m-1}] - \bar u_i$ (dikurangi rata-ratanya sendiri), $i=1..N-m$;
2. jarak $d^m_{ij} = \max_k |X^m_i(k) - X^m_j(k)|$ (Chebyshev);
3. keanggotaan $D^m_{ij} = \exp\!\big(-(d^m_{ij})^n/r\big)$;
4. $\phi^m = \frac1{N-m}\sum_i \frac1{N-m-1}\sum_{j\ne i} D^m_{ij}$;
5. $\mathrm{FuzzyEn}(m,n,r) = \ln\phi^m - \ln\phi^{m+1}$.

### b.2 Rumus di kode dan perbedaannya

`edm_fuzzy_entropy_1d` (baris 300-314) + `phi_euclidean` (286-297):

$$d_{ij} = \lVert X_i - X_j\rVert_2,\qquad \mu_{ij} = \frac{1}{1 + d_{ij}/r},\qquad
\phi^m = \frac1{|R|}\sum_{i\in R}\frac1{N_m-1}\sum_{j\ne i}\mu_{ij},$$
$$\mathrm{FE}_\tau = \frac1{|K|}\sum_{k\in K}\ln\frac{\phi^m_k}{\phi^{m+1}_k},\qquad r = 0.2\cdot\operatorname{sd}(x,\text{ddof}=1).$$

| Hal | Chen 2007 / Costa | Kode | Baris |
|---|---|---|---|
| Jarak | Chebyshev (maks) | **Euclidean** (inilah "EDM") | 292-294 |
| Keanggotaan | $\exp(-d^n/r)$ | **$1/(1+d/r)$** (rasional, ekor lebih panjang, tak pernah 0) | 295 |
| Jumlah vektor | $N-m$ untuk $m$ **dan** $m+1$ | semua: $N-m+1$ untuk $m$, $N-m$ untuk $m+1$ | 277 |
| Titik acuan $i$ | semua | **subsampel acak `n_ref=128`** bila $N_m>128$; $j$ tetap semua | 281-283, 290-291 |
| Acuan $m$ vs $m+1$ | sama | **seed berbeda** (`seed+11s+k` vs `seed+17s+k`) | 309-310 |
| Coarse-graining | Costa, $k=0$ | composite, rata-rata $\ln$ rasio atas $k$ (CMSE, bukan RCMSE) | 304-313 |
| $r$ | tetap dari deret asli | sama (dihitung sekali dari jendela) | 302 |
| Panjang minimum | – | offset dilewati bila $\lvert y\rvert < m+2$; $\phi=$NaN bila vektor $<3$ | 307, 288 |

Diagonal $\mu_{ii}$ di-nol-kan (baris 296) supaya pasangan diri sendiri tidak
ikut dihitung — setara $j\ne i$ di Chen.

### b.3 Data mainan

$x = [3, 5, 4, 6, 5, 7, 6, 4]$ ($N=8$), $\bar x = 5$,
$\operatorname{sd} = \sqrt{12/7} = 1.309307341416$,
$r = 0.2\cdot\operatorname{sd} = \mathbf{0.261861468283}$.

### b.4 Skala $\tau = 1$

Coarse-graining $\tau=1$ = deret itu sendiri.

**Embedding $m=2$, dipusatkan** ($[a,b] - \frac{a+b}2$):

| $i$ | mentah | terpusat |
|---|---|---|
| 1 | [3, 5] | [−1, 1] |
| 2 | [5, 4] | [0.5, −0.5] |
| 3 | [4, 6] | [−1, 1] |
| 4 | [6, 5] | [0.5, −0.5] |
| 5 | [5, 7] | [−1, 1] |
| 6 | [7, 6] | [0.5, −0.5] |
| 7 | [6, 4] | [1, −1] |

Hanya ada tiga pola, jadi jaraknya hanya:
$\lVert[-1,1]-[0.5,-0.5]\rVert = \sqrt{1.5^2+1.5^2} = 2.121320$;
$\lVert[-1,1]-[1,-1]\rVert = \sqrt{8} = 2.828427$;
$\lVert[0.5,-0.5]-[1,-1]\rVert = \sqrt{0.5} = 0.707107$; pola sama: 0.

Keanggotaan $\mu = 1/(1+d/r)$:
$d=2.121320 \Rightarrow 1/(1+8.100925) = 0.109879$;
$d=2.828427 \Rightarrow 0.084737$;
$d=0.707107 \Rightarrow 0.270248$; $d=0 \Rightarrow 1$.

Matriks $\mu$ ($7\times7$, diagonal 0):

| | 1 | 2 | 3 | 4 | 5 | 6 | 7 | rata-rata baris $\sum_{j\ne i}\mu_{ij}/6$ |
|---|---|---|---|---|---|---|---|---|
| 1 | 0 | 0.109879 | 1 | 0.109879 | 1 | 0.109879 | 0.084737 | 0.402396 |
| 2 | 0.109879 | 0 | 0.109879 | 1 | 0.109879 | 1 | 0.270248 | 0.433314 |
| 3 | 1 | 0.109879 | 0 | 0.109879 | 1 | 0.109879 | 0.084737 | 0.402396 |
| 4 | 0.109879 | 1 | 0.109879 | 0 | 0.109879 | 1 | 0.270248 | 0.433314 |
| 5 | 1 | 0.109879 | 1 | 0.109879 | 0 | 0.109879 | 0.084737 | 0.402396 |
| 6 | 0.109879 | 1 | 0.109879 | 1 | 0.109879 | 0 | 0.270248 | 0.433314 |
| 7 | 0.084737 | 0.270248 | 0.084737 | 0.270248 | 0.084737 | 0.270248 | 0 | 0.177492 |

$\phi^2 = (3\cdot0.402396 + 3\cdot0.433314 + 0.177492)/7 = \mathbf{0.383517345294}$.

**Embedding $m+1=3$, dipusatkan** (6 vektor):
$[-1,1,0]$ (i=1,3,5), $[0,-1,1]$ (i=2,4), $[1.333333, 0.333333, -1.666667]$ (i=6, dari [7,6,4]).
Jarak: $\lVert[-1,1,0]-[0,-1,1]\rVert=\sqrt6=2.449490$;
$\lVert[-1,1,0]-v_6\rVert = 2.943920$; $\lVert[0,-1,1]-v_6\rVert = 3.265986$.
$\mu$: 0.096580; 0.081684; 0.074227.
Rata-rata baris ($/5$): 0.454969 (i=1,3,5), 0.272793 (i=2,4), 0.078701 (i=6).

$\phi^3 = (3\cdot0.454969+2\cdot0.272793+0.078701)/6 = \mathbf{0.331532298319}$.

$$\mathrm{FE}_{\tau=1} = \ln\frac{0.383517345294}{0.331532298319} = \mathbf{0.145659613004}.$$

### b.5 Skala $\tau = 2$

- $k=0$: $y = [\tfrac{3+5}2, \tfrac{4+6}2, \tfrac{5+7}2, \tfrac{6+4}2] = [4, 5, 6, 5]$ (panjang 4 $\ge m+2$, dipakai).
  Vektor $m=2$ terpusat: $[-0.5,0.5], [-0.5,0.5], [0.5,-0.5]$; jarak $0, 1.414214$;
  $\mu$: $1$ dan $1/(1+1.414214/0.261861)=0.156235$;
  rata-rata baris $0.578117, 0.578117, 0.156235$;
  $\phi^2 = \mathbf{0.437489954604}$.
  Vektor $m=3$: hanya **2** vektor → `phi_euclidean` mengembalikan NaN (baris 288-289, butuh $\ge3$).
- $k=1$: $y = [4.5, 5.5, 6.5]$ (panjang 3 $< 4$) → **dilewati** (baris 307).

Tak ada $k$ yang sah → $\mathrm{FE}_{\tau=2} = $ NaN. Kode memberi
`edm_fuzzy_entropy_1d(x, [1,2]) = [0.145660, nan]`. NaN ini nanti diisi
median kolom oleh `sanitize` (baris 368-372; contoh skrip:
`[[0.3,1],[0.5,NaN],[0.1,3]] -> [[0.3,1],[0.5,2],[0.1,3]]`).

**Syarat panjang.** Skala $\tau$ sah bila $\lfloor (n-k)/\tau\rfloor \ge m+3 = 5$
untuk paling tidak satu $k$, yaitu $n \ge 5\tau$. Untuk $\tau=15$ perlu
$n\ge75$; jendela penelitian $n = 200/700/1000$ aman — NaN hanya muncul di contoh
mainan ini.

**`n_ref`.** Di contoh mainan jumlah vektor $<128$ sehingga semua dipakai.
Untuk jendela $N=200$, skala 1 punya 199 vektor, jadi hanya 128 titik acuan acak
yang dihitung barisnya (biaya $O(128\cdot N)$, bukan $O(N^2)$). Contoh di skrip:
39 vektor, $\phi$ penuh 0.331980 vs $\phi$ dengan `n_ref=10` 0.339608 —
perkiraan acak, bergantung `seed`.

---

## (c) Fuzzy entropy dengan kemiripan Jensen–Shannon (JSD-Fuzzy)

### c.1 Rumus

**Lin (1991):** untuk distribusi $P,Q$ dengan $M=\tfrac12(P+Q)$,

$$\mathrm{JS}(P,Q) = H(M) - \tfrac12H(P) - \tfrac12H(Q) = \tfrac12\mathrm{KL}(P\Vert M) + \tfrac12\mathrm{KL}(Q\Vert M),$$

dengan logaritma natural $0 \le \mathrm{JS} \le \ln 2$. $\sqrt{\mathrm{JS}}$
adalah metrik (Endres & Schindelin 2003).

**Di kode** (rancangan penelitian ini, bukan rumus satu paper):

1. `jsd_fuzzy_entropy_1d` (346-365): pusat bin
   $c_b = \text{linspace}(-3\,\mathrm{sd}, 3\,\mathrm{sd}, 8)$, lebar kernel
   $h = c_2-c_1 = 6\,\mathrm{sd}/7$, ambang $r_\text{JSD} = 0.2\sqrt{\ln2}$
   (20 % dari jarak JS maksimum $\sqrt{\ln 2}$).
2. `vectors_to_distributions` (317-321): tiap vektor terpusat $v$ (panjang $m$)
   diubah jadi histogram lunak
   $K_b = \sum_{t=1}^{m}\exp\!\big(-\tfrac12((v_t-c_b)/h)^2\big)$,
   $P_b = K_b/\sum_{b'}K_{b'}$.
3. `pairwise_jsd` (324-331): JS dengan $\ln$, $\epsilon=10^{-12}$ di dalam log.
4. `phi_jsd` (334-343): $d_{ij} = \sqrt{\mathrm{JS}(P_i,P_j)}$,
   $\mu_{ij} = 1/(1+d_{ij}/r_\text{JSD})$, lalu $\phi$ sama seperti EDM
   (diagonal 0, subsampel `n_ref`).
5. Entropi $= \ln(\phi^m/\phi^{m+1})$, coarse-graining dan syarat panjang sama
   dengan EDM.

**Perbedaan penting terhadap EDM / Chen:**

- Histogram **membuang urutan** nilai di dalam vektor: $[-a, a]$ (naik) dan
  $[a,-a]$ (turun) punya distribusi identik → JS $=0$, $\mu=1$. Skrip:
  JSD$([-0.5,0.5],[0.5,-0.5]) = 0$ sedangkan Euclid $=1.414214$.
- $\phi^{m+1}$ **bisa lebih besar** dari $\phi^m$, sehingga "entropi" JSD bisa
  **negatif** (terjadi di contoh ini). Itu bukan galat hitung; artinya
  histogram 3 titik lebih seragam antarvektor daripada histogram 2 titik.
- Pusat bin bergantung sd **jendela**, jadi skala histogram ikut menyesuaikan
  tiap jendela (sama seperti $r$ di EDM).
- Argumen `r_ratio` pada `jsd_fuzzy_entropy_1d` **tidak dipakai** (parameter mati).

### c.2 Hitungan

Deret sama: $\mathrm{sd}=1.309307341416$.
Pusat bin $c$ = `[-3.927922, -2.805659, -1.683395, -0.561132, 0.561132, 1.683395, 2.805659, 3.927922]`,
$h = \mathbf{1.122263435499}$, $r_\text{JSD} = 0.2\sqrt{0.693147} = \mathbf{0.166510922232}$.

**Vektor → distribusi ($m=2$).** Contoh $v_1=[-1,1]$, bin ke-4 ($c=-0.561132$):
$\exp(-\tfrac12((-1+0.561132)/1.122263)^2) + \exp(-\tfrac12((1+0.561132)/1.122263)^2)
= \exp(-0.076463) + \exp(-0.967519) = 0.926388 + 0.380025 = 1.306412$.

| pola | $K$ (8 bin) | $\sum K$ | $P$ |
|---|---|---|---|
| $[\mp1,\pm1]$ (i=1,3,5,7) | 0.033329, 0.277260, 0.888117, 1.306412, 1.306412, 0.888117, 0.277260, 0.033329 | 5.010236 | 0.006652, 0.055339, 0.177261, 0.260749, 0.260749, 0.177261, 0.055339, 0.006652 |
| $[\pm0.5,\mp0.5]$ (i=2,4,6) | 0.009837, 0.134248, 0.724213, 1.638054, 1.638054, 0.724213, 0.134248, 0.009837 | 5.012704 | 0.001962, 0.026782, 0.144475, 0.326781, 0.326781, 0.144475, 0.026782, 0.001962 |

Vektor 7 ($[1,-1]$) punya $P$ sama dengan vektor 1 — urutan hilang.

**JSD satu pasangan ($P_1$, $P_2$).**
$M = \tfrac12(P_1+P_2)$ = `[0.004307, 0.041060, 0.160868, 0.293765, 0.293765, 0.160868, 0.041060, 0.004307]`;
$\mathrm{KL}(P_1\Vert M) = 0.011040055742$, $\mathrm{KL}(P_2\Vert M) = 0.012581848500$;
$\mathrm{JS} = \tfrac12(0.011040055742+0.012581848500) = \mathbf{0.011810952121}$
(bentuk Lin $H(M)-\tfrac12(H(P_1)+H(P_2))$ memberi angka sama).
$d = \sqrt{0.011810952} = 0.108678$, $\mu = 1/(1+0.108678/0.166511) = 0.605078$.

Matriks $\mu$ ($m=2$): pasangan pola sama → 1, beda pola → 0.605078.
Rata-rata baris: 0.802539 (i=1,3,5,7; $=(3\cdot1+3\cdot0.605078)/6$),
0.736719 (i=2,4,6; $=(2\cdot1+4\cdot0.605078)/6$).

$\phi^2 = (4\cdot0.802539+3\cdot0.736719)/7 = \mathbf{0.774330360139}$.

**$m+1=3$.** Vektor $[-1,1,0]$ dan $[0,-1,1]$ berisi nilai yang sama
$\{-1,0,1\}$ → distribusi identik
`[0.004725, 0.042731, 0.161342, 0.291203, 0.291203, 0.161342, 0.042731, 0.004725]`;
$v_6=[1.333,0.333,-1.667]$ →
`[0.017603, 0.082431, 0.163345, 0.211073, 0.254277, 0.193102, 0.068174, 0.009994]`.
JS$(\cdot, v_6) = 0.011685$, $d=0.108097$, $\mu=0.606358$.
Rata-rata baris: 0.921272 (i=1..5; $=(4+0.606358)/5$), 0.606358 (i=6).

$\phi^3 = (5\cdot0.921272+0.606358)/6 = \mathbf{0.868786036155}$.

$$\mathrm{JSDFE}_{\tau=1} = \ln\frac{0.774330360139}{0.868786036155} = \mathbf{-0.115098272073}.$$

$\tau=2$: sama seperti EDM, $\phi^{m+1}$ NaN → `jsd_fuzzy_entropy_1d(x,[1,2]) = [-0.115098, nan]`.

**Bandingkan:** EDM melihat 3 pola berbeda pada $m=2$ (naik besar, turun kecil,
turun besar) dan memberi entropi positif 0.1457; JSD hanya melihat 2 pola
(amplitudo 1 dan 0.5, arah diabaikan), dan pada $m=3$ hampir semua vektor
dianggap kembar, sehingga entropinya negatif.

---

## (d) Self-check Jacobian: analitik vs numerik

**Ide.** Turunan analitik (backprop) mudah salah indeks. Cek dengan selisih
hingga terpusat:

$$J^\text{num}_{:,p} = \frac{r(\theta + \varepsilon e_p) - r(\theta - \varepsilon e_p)}{2\varepsilon},\quad \varepsilon=10^{-6},$$

galat pemotongan $O(\varepsilon^2)\approx10^{-12}$, galat pembulatan
$\approx 10^{-16}/\varepsilon = 10^{-10}$. Jadi selisih $\sim10^{-10}$ berarti
**cocok**.

Notebook 11 melakukan hal yang sama di sel §10a-check
(`build_11_annlm_real_notebook.py:1585-1599`) pada jaringan 60-[16,8]-3 dengan
selisih **maju** ($\varepsilon=10^{-6}$, mode `"numeric"`, baris 1469-1477) dan
ambang galat relatif $10^{-3}$; notebook 15 mengulanginya untuk [6] dan [6,3]
(`build_15_data_hipotetik_ann_lm_notebook.py:468-479`).

Hasil skrip pada jaringan mainan:

| aktivasi | titik $\theta$ | maks $\lvert J^\text{analitik}-J^\text{num}\rvert$ |
|---|---|---|
| tanh | awal | 8.1e-11 |
| relu | awal | 3.4e-11 |
| tanh | sesudah iterasi 9 | 1.2e-10 |
| relu | sesudah iterasi 9 | 9.6e-11 |

Di $\theta$ sesudah iterasi 9 semua kolom $W_1$ terisi (tak ada baris $W_2$ nol
lagi), jadi cek ini menguji semua jalur backprop, termasuk turunan
$(1-h^2)$ unit tersembunyi kedua.

---

## Daftar pustaka yang dirujuk

- K. Levenberg (1944), Quart. Appl. Math. 2:164-168.
- D. W. Marquardt (1963), J. SIAM 11(2):431-441 — skala $\operatorname{diag}(J^\top J)$.
- H. B. Nielsen (1999), "Damping parameter in Marquardt's method", IMM-REP-1999-05, DTU;
  K. Madsen, H. B. Nielsen, O. Tingleff (2004), "Methods for Non-Linear Least Squares Problems", DTU.
- M. T. Hagan, M. B. Menhaj (1994), IEEE Trans. Neural Networks 5(6):989-993.
- X. Glorot, Y. Bengio (2010), AISTATS — inisialisasi Xavier.
- W. Chen, Z. Wang, H. Xie, W. Yu (2007), IEEE TNSRE 15(2):266-272 — FuzzyEn.
- M. Costa, A. L. Goldberger, C.-K. Peng (2002), PRL 89:068102; (2005) PRE 71:021906 — MSE.
- S.-D. Wu et al. (2013), Entropy 15(3):1069-1084 — composite MSE.
- J. Lin (1991), IEEE Trans. Inf. Theory 37(1):145-151 — Jensen–Shannon divergence.
- D. M. Endres, J. E. Schindelin (2003), IEEE Trans. Inf. Theory 49(7):1858-1860 — $\sqrt{\mathrm{JS}}$ metrik.
- M. K. Transtrum, J. P. Sethna (2012), arXiv:1201.5885 — variasi skala/damping LM.

---

## Keluaran `cek_algoritma_tangan.py` (dijalankan 2026-11-20, numpy 2.5.3)

```
==============================================================================
(a) SATU ITERASI LEVENBERG-MARQUARDT, jaringan 2-[2]-2, 3 sampel
==============================================================================
Xavier W1 (template) =
[[ 0.335486 -0.563905]
 [-1.124381 -1.184261]]
Xavier W2 (template) =
[[0.767352 1.011041]
 [0.261203 0.562149]]
theta0 = lm_pack(template) =
[ 0.335486 -0.563905 -1.124381 -1.184261  0.767352  1.011041  0.261203  0.562149  0.        0.        0.        0.      ]
W1 (hasil lm_unpack) =
[[ 0.335486 -0.563905]
 [-1.124381 -1.184261]]
b1 (hasil lm_unpack) =
[0.767352 1.011041]
W2 (hasil lm_unpack) =
[[0.261203 0.562149]
 [0.       0.      ]]
b2 (hasil lm_unpack) =
[0. 0.]
z1 = X W1 + b1 =
[[ 1.102838  0.447135]
 [-0.357028 -0.17322 ]
 [-0.021542 -0.737125]]
h = tanh(z1) =
[[ 0.801516  0.419542]
 [-0.342594 -0.171508]
 [-0.021539 -0.627405]]
keluaran o = h W2 + b2 =
[[ 0.209359  0.450572]
 [-0.089487 -0.192589]
 [-0.005626 -0.012108]]
residual r = (o - Y).ravel() =
[-0.790641  0.450572 -0.089487 -1.192589 -1.005626 -0.012108]
  [OK] residual tangan vs lm_residuals                maks selisih = 0.0e+00
E0 = 0.5 * sum(r^2) = 1.634917719156
  [OK] E0 vs konstanta .md                            maks selisih = 2.8e-13
J (6 x 12), kolom: W1_00 W1_01 W1_10 W1_11 b1_0 b1_1 W2_00 W2_01 W2_10 W2_11 b2_0 b2_1 =
[[ 0.093399  0.        0.        0.        0.093399  0.        0.801516  0.        0.419542  0.        1.        0.      ]
 [ 0.201009  0.        0.        0.        0.201009  0.        0.        0.801516  0.        0.419542  0.        1.      ]
 [ 0.        0.        0.230546  0.        0.230546  0.       -0.342594 -0.       -0.171508 -0.        1.        0.      ]
 [ 0.        0.        0.49617   0.        0.49617   0.       -0.       -0.342594 -0.       -0.171508  0.        1.      ]
 [ 0.261082  0.        0.261082  0.        0.261082  0.       -0.021539 -0.       -0.627405 -0.        1.        0.      ]
 [ 0.561889  0.        0.561889  0.        0.561889  0.       -0.       -0.021539 -0.       -0.627405  0.        1.      ]]
  [OK] Jacobian tangan vs lm_residual_jacobian        maks selisih = 5.6e-17
--- iterasi 1 (rinci), lam = 0.001000000000 ---
   g = J^T r =
[-0.25263   0.       -0.881712  0.       -0.864988  0.       -0.581394  0.769975  0.314576  0.401169 -1.885754 -0.754125]
   ||g|| = 2.627734635861
   A = J^T J =
[[ 0.433011  0.        0.383883  0.        0.433011  0.        0.069237  0.149009 -0.12462  -0.2682    0.354481  0.762897]
 [ 0.        0.        0.        0.        0.        0.        0.        0.        0.        0.        0.        0.      ]
 [ 0.383883  0.        0.683218  0.        0.683218  0.       -0.084607 -0.182087 -0.203345 -0.437629  0.491628  1.058058]
 [ 0.        0.        0.        0.        0.        0.        0.        0.        0.        0.        0.        0.      ]
 [ 0.433011  0.        0.683218  0.        0.732346  0.       -0.009746 -0.020975 -0.16416  -0.353297  0.585027  1.259067]
 [ 0.        0.        0.        0.        0.        0.        0.        0.        0.        0.        0.        0.      ]
 [ 0.069237  0.       -0.084607  0.       -0.009746  0.        0.760263  0.        0.408541  0.        0.437384  0.      ]
 [ 0.149009  0.       -0.182087  0.       -0.020975  0.        0.        0.760263  0.        0.408541  0.        0.437384]
 [-0.12462   0.       -0.203345  0.       -0.16416   0.        0.408541  0.        0.599067  0.       -0.379372  0.      ]
 [-0.2682    0.       -0.437629  0.       -0.353297  0.        0.        0.408541  0.        0.599067  0.       -0.379372]
 [ 0.354481  0.        0.491628  0.        0.585027  0.        0.437384  0.       -0.379372  0.        3.        0.      ]
 [ 0.762897  0.        1.058058  0.        1.259067  0.        0.        0.437384  0.       -0.379372  0.        3.      ]]
   d = diag(A), dijepit >= 1e-12 =
[0.433011 0.       0.683218 0.       0.732346 0.       0.760263 0.760263 0.599067 0.599067 3.       3.      ]
   langkah Delta: (A + lam*diag(d)) Delta = -g =
[-0.781785 -0.        0.618051  0.        0.597494  0.        1.534301 -1.319635 -1.285292  0.862243  0.116816  0.282593]
   cek: maks |(A + lam D) Delta + g| = 2.22e-16
   residual percobaan r(theta + Delta) =
[-0.1203    0.095003  1.586013 -1.392115  0.624098 -0.554071]
   E = 1.634917719156, E_baru = 2.586706041095
   L(0)-L(Delta) = 0.5 Delta^T (lam d*Delta - g) = 1.634907423098
   rho = (E - E_baru)/(L(0)-L(Delta)) = -0.582166493645
   DITOLAK (E_baru >= E): theta tetap, lam dikali 2
  [OK] theta sesudah iterasi 1 vs lm_fit(iters=1)     maks selisih = 0.0e+00
  [OK] lam sesudah iterasi 1 vs hist['lam'][1]        maks selisih = 0.0e+00
  [OK] theta sesudah iterasi 2 vs lm_fit(iters=2)     maks selisih = 0.0e+00
  [OK] lam sesudah iterasi 2 vs hist['lam'][2]        maks selisih = 0.0e+00
  [OK] theta sesudah iterasi 3 vs lm_fit(iters=3)     maks selisih = 0.0e+00
  [OK] lam sesudah iterasi 3 vs hist['lam'][3]        maks selisih = 0.0e+00
  [OK] theta sesudah iterasi 4 vs lm_fit(iters=4)     maks selisih = 0.0e+00
  [OK] lam sesudah iterasi 4 vs hist['lam'][4]        maks selisih = 0.0e+00
  [OK] theta sesudah iterasi 5 vs lm_fit(iters=5)     maks selisih = 0.0e+00
  [OK] lam sesudah iterasi 5 vs hist['lam'][5]        maks selisih = 0.0e+00
  [OK] theta sesudah iterasi 6 vs lm_fit(iters=6)     maks selisih = 0.0e+00
  [OK] lam sesudah iterasi 6 vs hist['lam'][6]        maks selisih = 0.0e+00
  [OK] theta sesudah iterasi 7 vs lm_fit(iters=7)     maks selisih = 0.0e+00
  [OK] lam sesudah iterasi 7 vs hist['lam'][7]        maks selisih = 0.0e+00
  [OK] theta sesudah iterasi 8 vs lm_fit(iters=8)     maks selisih = 0.0e+00
  [OK] lam sesudah iterasi 8 vs hist['lam'][8]        maks selisih = 0.0e+00
--- iterasi 9 (rinci), lam = 0.185909509851 ---
   g = J^T r =
[ 0.368378 -0.16316   1.696101 -1.689594  1.644899 -1.615544  0.753384 -0.820597 -0.412135  0.428428  1.349901 -1.462166]
   ||g|| = 4.094388793528
   A = J^T J =
[[ 1.532505 -1.012648  1.244351 -0.61227   1.532505 -1.012648  0.856635 -0.288356 -0.449861  0.15143   1.565965 -0.527127]
 [-1.012648  0.91181  -0.61227   0.320315 -1.012648  0.91181  -0.712122  0.457536  0.027279 -0.017527 -1.1232    0.721653]
 [ 1.244351 -0.61227   2.003512 -1.37776   2.003512 -1.37776   0.959255 -0.322899 -0.804929  0.270951  1.882985 -0.63384 ]
 [-0.61227   0.320315 -1.37776   1.141006 -1.37776   1.141006 -0.66826   0.429355  0.429459 -0.275926 -1.23832   0.795617]
 [ 1.532505 -1.012648  2.003512 -1.37776   2.291665 -1.778137  1.356487 -0.456613 -0.591487  0.199103  2.391734 -0.805093]
 [-1.012648  0.91181  -1.37776   1.141006 -1.778137  1.732501 -1.173473  0.753953  0.157997 -0.101513 -1.885365  1.211341]
 [ 0.856635 -0.712122  0.959255 -0.66826   1.356487 -1.173473  1.164883  0.       -0.048871  0.        1.820658  0.      ]
 [-0.288356  0.457536 -0.322899  0.429355 -0.456613  0.753953  0.        1.164883  0.       -0.048871  0.        1.820658]
 [-0.449861  0.027279 -0.804929  0.429459 -0.591487  0.157997 -0.048871  0.        0.599067  0.       -0.379372  0.      ]
 [ 0.15143  -0.017527  0.270951 -0.275926  0.199103 -0.101513  0.       -0.048871  0.        0.599067  0.       -0.379372]
 [ 1.565965 -1.1232    1.882985 -1.23832   2.391734 -1.885365  1.820658  0.       -0.379372  0.        3.        0.      ]
 [-0.527127  0.721653 -0.63384   0.795617 -0.805093  1.211341  0.        1.820658  0.       -0.379372  0.        3.      ]]
   d = diag(A), dijepit >= 1e-12 =
[1.532505 0.91181  2.003512 1.141006 2.291665 1.732501 1.164883 1.164883 0.599067 0.599067 3.       3.      ]
   langkah Delta: (A + lam*diag(d)) Delta = -g =
[ 0.280035 -0.557554 -0.171412  0.682996 -0.107852  0.304873 -0.111151  0.208397 -0.026669 -0.189804 -0.062157  0.127196]
   cek: maks |(A + lam D) Delta + g| = 4.44e-16
   residual percobaan r(theta + Delta) =
[ 0.009152  0.170271  0.09392  -0.577119 -0.075423  0.109324]
   E = 1.453046600031, E_baru = 0.194302037423
   L(0)-L(Delta) = 0.5 Delta^T (lam d*Delta - g) = 1.381334783791
   rho = (E - E_baru)/(L(0)-L(Delta)) = 0.911252346194
   DITERIMA. faktor Nielsen = max(1/3, 1-(2rho-1)^3) = 0.443564086033
  [OK] theta sesudah iterasi 9 vs lm_fit(iters=9)     maks selisih = 0.0e+00
ringkas iterasi 1..9:
 it   lam_dipakai        E            E_percobaan      L(0)-L(D)        rho          terima  lam_baru
  1  0.001000  1.634917719  2.586706041  1.634907423  -0.582166494  tidak  0.002000000
  2  0.002000  1.634917719  2.570539124  1.634876917  -0.572288589  tidak  0.004000000
  3  0.004000  1.634917719  2.538810320  1.634757500  -0.552921519  tidak  0.008000000
  4  0.008000  1.634917719  2.477672606  1.634299728  -0.515667274  tidak  0.016000000
  5  0.016000  1.634917719  2.363942083  1.632613779  -0.446538167  tidak  0.032000000
  6  0.032000  1.634917719  2.165689337  1.626843925  -0.326258475  tidak  0.064000000
  7  0.064000  1.634917719  1.856812411  1.609412045  -0.137873140  tidak  0.128000000
  8  0.128000  1.634917719  1.453046600  1.565691880  +0.116160224  ya     0.185909510
  9  0.185910  1.453046600  0.194302037  1.381334784  +0.911252346  ya     0.082462782
  [OK] E0 vs konstanta .md                            maks selisih = 2.8e-13
  [OK] ||g|| it1 vs konstanta .md                     maks selisih = 4.6e-13
  [OK] E percobaan it1 vs konstanta .md               maks selisih = 2.3e-13
  [OK] rho it1 vs konstanta .md                       maks selisih = 2.3e-13
  [OK] E percobaan it9 vs konstanta .md               maks selisih = 2.8e-13
  [OK] denom it9 vs konstanta .md                     maks selisih = 2.2e-13
  [OK] rho it9 vs konstanta .md                       maks selisih = 4.2e-13
  [OK] faktor it9 vs konstanta .md                    maks selisih = 6.4e-14
  [OK] lam baru it9 vs konstanta .md                  maks selisih = 1.2e-13
softmax(keluaran) sesudah langkah diterima pertama (predict_proba) =
[[0.69823  0.30177 ]
 [0.418494 0.581506]
 [0.693228 0.306772]]
prediksi kelas = [0 1 0] (target [0 1 0])

==============================================================================
(b) EDM-FUZZY: deret 8 titik, m = 2, r = 0.2*sd
==============================================================================
x = [3. 5. 4. 6. 5. 7. 6. 4.]
mean = 5.0, sd(ddof=1) = 1.309307341416, r = 0.2*sd = 0.261861468283
  [OK] r vs konstanta .md                             maks selisih = 1.9e-13
  [OK] coarse-grain tau=1 k=0                         maks selisih = 0.0e+00
tau=1, k=0: y = [3. 5. 4. 6. 5. 7. 6. 4.]  (panjang 8)
  [OK] embed m=2 tau=1 k=0                            maks selisih = 0.0e+00
  [OK] embed m=3 tau=1 k=0                            maks selisih = 0.0e+00
   vektor m=2 terpusat (tau=1) =
[[-1.   1. ]
 [ 0.5 -0.5]
 [-1.   1. ]
 [ 0.5 -0.5]
 [-1.   1. ]
 [ 0.5 -0.5]
 [ 1.  -1. ]]
   vektor m=3 terpusat (tau=1) =
[[-1.        1.        0.      ]
 [ 0.       -1.        1.      ]
 [-1.        1.        0.      ]
 [ 0.       -1.        1.      ]
 [-1.        1.        0.      ]
 [ 1.333333  0.333333 -1.666667]]
   jarak Euclid m=2 (tau=1) =
[[0.       2.12132  0.       2.12132  0.       2.12132  2.828427]
 [2.12132  0.       2.12132  0.       2.12132  0.       0.707107]
 [0.       2.12132  0.       2.12132  0.       2.12132  2.828427]
 [2.12132  0.       2.12132  0.       2.12132  0.       0.707107]
 [0.       2.12132  0.       2.12132  0.       2.12132  2.828427]
 [2.12132  0.       2.12132  0.       2.12132  0.       0.707107]
 [2.828427 0.707107 2.828427 0.707107 2.828427 0.707107 0.      ]]
   keanggotaan mu = 1/(1+d/r), diagonal 0 (tau=1) =
[[0.       0.109879 1.       0.109879 1.       0.109879 0.084737]
 [0.109879 0.       0.109879 1.       0.109879 1.       0.270248]
 [1.       0.109879 0.       0.109879 1.       0.109879 0.084737]
 [0.109879 1.       0.109879 0.       0.109879 1.       0.270248]
 [1.       0.109879 1.       0.109879 0.       0.109879 0.084737]
 [0.109879 1.       0.109879 1.       0.109879 0.       0.270248]
 [0.084737 0.270248 0.084737 0.270248 0.084737 0.270248 0.      ]]
   rata-rata baris m=2 = [0.402396 0.433314 0.402396 0.433314 0.402396 0.433314 0.177492]
   jarak Euclid m=3 (tau=1) =
[[0.       2.44949  0.       2.44949  0.       2.94392 ]
 [2.44949  0.       2.44949  0.       2.44949  3.265986]
 [0.       2.44949  0.       2.44949  0.       2.94392 ]
 [2.44949  0.       2.44949  0.       2.44949  3.265986]
 [0.       2.44949  0.       2.44949  0.       2.94392 ]
 [2.94392  3.265986 2.94392  3.265986 2.94392  0.      ]]
   keanggotaan m=3 (tau=1) =
[[0.       0.09658  1.       0.09658  1.       0.081684]
 [0.09658  0.       0.09658  1.       0.09658  0.074227]
 [1.       0.09658  0.       0.09658  1.       0.081684]
 [0.09658  1.       0.09658  0.       0.09658  0.074227]
 [1.       0.09658  1.       0.09658  0.       0.081684]
 [0.081684 0.074227 0.081684 0.074227 0.081684 0.      ]]
   rata-rata baris m=3 = [0.454969 0.272793 0.454969 0.272793 0.454969 0.078701]
  [OK] phi_m tau=1 k=0                                maks selisih = 5.6e-17
  [OK] phi_m1 tau=1 k=0                               maks selisih = 0.0e+00
   phi_m = 0.383517345294, phi_m+1 = 0.33153229831940895
   ln(phi_m/phi_m+1) = 0.145659613004
  [OK] phi_m tau=1 vs konstanta .md                   maks selisih = 3.9e-13
  [OK] phi_m1 tau=1 vs konstanta .md                  maks selisih = 4.1e-13
  [OK] coarse-grain tau=2 k=0                         maks selisih = 0.0e+00
tau=2, k=0: y = [4. 5. 6. 5.]  (panjang 4)
  [OK] embed m=2 tau=2 k=0                            maks selisih = 0.0e+00
  [OK] embed m=3 tau=2 k=0                            maks selisih = 0.0e+00
   vektor m=2 terpusat (tau=2) =
[[-0.5  0.5]
 [-0.5  0.5]
 [ 0.5 -0.5]]
   vektor m=3 terpusat (tau=2) =
[[-1.        0.        1.      ]
 [-0.333333  0.666667 -0.333333]]
   jarak Euclid m=2 (tau=2) =
[[0.       0.       1.414214]
 [0.       0.       1.414214]
 [1.414214 1.414214 0.      ]]
   keanggotaan mu = 1/(1+d/r), diagonal 0 (tau=2) =
[[0.       1.       0.156235]
 [1.       0.       0.156235]
 [0.156235 0.156235 0.      ]]
   rata-rata baris m=2 = [0.578117 0.578117 0.156235]
  [OK] phi_m tau=2 k=0                                maks selisih = 0.0e+00
  [OK] phi_m1 tau=2 k=0                               maks selisih = 0.0e+00
   phi_m = 0.437489954604, phi_m+1 = nan
   phi_m+1 = NaN (hanya 2 vektor) -> tidak dihitung
  [OK] phi_m tau=2 vs konstanta .md                   maks selisih = 4.1e-14
  [OK] coarse-grain tau=2 k=1                         maks selisih = 0.0e+00
tau=2, k=1: y = [4.5 5.5 6.5]  (panjang 3)
   panjang < m+2 = 4 -> offset ini DILEWATI oleh kode
edm_fuzzy_entropy_1d(x, skala [1,2]) = [0.14566     nan]
  [OK] FE tau=1,2 tangan vs edm_fuzzy_entropy_1d      maks selisih = 1.9e-16
  [OK] FE tau=1 vs konstanta .md                      maks selisih = 1.6e-13
sanitize(contoh matriks fitur dengan NaN) =
[[0.3 1. ]
 [0.5 2. ]
 [0.1 3. ]]
  [OK] sanitize isi NaN dengan median kolom           maks selisih = 0.0e+00
n_ref: 39 vektor, phi penuh = 0.331980, phi dengan n_ref=10 = 0.339608 (perkiraan acak)

==============================================================================
(c) JSD-FUZZY: deret yang sama, m = 2, 8 bin, h = jarak antar pusat
==============================================================================
pusat bin c = [-3.927922 -2.805659 -1.683395 -0.561132  0.561132  1.683395  2.805659  3.927922]
h = 1.122263435499, r_jsd = 0.2*sqrt(ln 2) = 0.166510922232
  [OK] pusat bin vs np.linspace                       maks selisih = 8.9e-16
  [OK] h vs konstanta .md                             maks selisih = 3.9e-13
  [OK] r_jsd vs konstanta .md                         maks selisih = 4.6e-13
   kernel mentah K (m=2), baris = vektor =
[[0.033329 0.27726  0.888117 1.306412 1.306412 0.888117 0.27726  0.033329]
 [0.009837 0.134248 0.724213 1.638054 1.638054 0.724213 0.134248 0.009837]
 [0.033329 0.27726  0.888117 1.306412 1.306412 0.888117 0.27726  0.033329]
 [0.009837 0.134248 0.724213 1.638054 1.638054 0.724213 0.134248 0.009837]
 [0.033329 0.27726  0.888117 1.306412 1.306412 0.888117 0.27726  0.033329]
 [0.009837 0.134248 0.724213 1.638054 1.638054 0.724213 0.134248 0.009837]
 [0.033329 0.27726  0.888117 1.306412 1.306412 0.888117 0.27726  0.033329]]
   distribusi P (m=2), tiap baris jumlah 1 =
[[0.006652 0.055339 0.177261 0.260749 0.260749 0.177261 0.055339 0.006652]
 [0.001962 0.026782 0.144475 0.326781 0.326781 0.144475 0.026782 0.001962]
 [0.006652 0.055339 0.177261 0.260749 0.260749 0.177261 0.055339 0.006652]
 [0.001962 0.026782 0.144475 0.326781 0.326781 0.144475 0.026782 0.001962]
 [0.006652 0.055339 0.177261 0.260749 0.260749 0.177261 0.055339 0.006652]
 [0.001962 0.026782 0.144475 0.326781 0.326781 0.144475 0.026782 0.001962]
 [0.006652 0.055339 0.177261 0.260749 0.260749 0.177261 0.055339 0.006652]]
  [OK] P tangan vs vectors_to_distributions m=2       maks selisih = 1.4e-16
  [OK] JSD tangan vs pairwise_jsd m=2                 maks selisih = 1.7e-18
   JSD (nat) m=2 =
[[0.       0.011811 0.       0.011811 0.       0.011811 0.      ]
 [0.011811 0.       0.011811 0.       0.011811 0.       0.011811]
 [0.       0.011811 0.       0.011811 0.       0.011811 0.      ]
 [0.011811 0.       0.011811 0.       0.011811 0.       0.011811]
 [0.       0.011811 0.       0.011811 0.       0.011811 0.      ]
 [0.011811 0.       0.011811 0.       0.011811 0.       0.011811]
 [0.       0.011811 0.       0.011811 0.       0.011811 0.      ]]
   d = sqrt(JSD) m=2 =
[[0.       0.108678 0.       0.108678 0.       0.108678 0.      ]
 [0.108678 0.       0.108678 0.       0.108678 0.       0.108678]
 [0.       0.108678 0.       0.108678 0.       0.108678 0.      ]
 [0.108678 0.       0.108678 0.       0.108678 0.       0.108678]
 [0.       0.108678 0.       0.108678 0.       0.108678 0.      ]
 [0.108678 0.       0.108678 0.       0.108678 0.       0.108678]
 [0.       0.108678 0.       0.108678 0.       0.108678 0.      ]]
   keanggotaan mu = 1/(1+d/r_jsd) m=2 =
[[0.       0.605078 1.       0.605078 1.       0.605078 1.      ]
 [0.605078 0.       0.605078 1.       0.605078 1.       0.605078]
 [1.       0.605078 0.       0.605078 1.       0.605078 1.      ]
 [0.605078 1.       0.605078 0.       0.605078 1.       0.605078]
 [1.       0.605078 1.       0.605078 0.       0.605078 1.      ]
 [0.605078 1.       0.605078 1.       0.605078 0.       0.605078]
 [1.       0.605078 1.       0.605078 1.       0.605078 0.      ]]
   rata-rata baris = [0.802539 0.736719 0.802539 0.736719 0.802539 0.736719 0.802539]
  [OK] phi_jsd m=2 tangan vs phi_jsd                  maks selisih = 5.6e-16
   phi (m=2) = 0.774330360139
  [OK] JSD(P1,P2) vs konstanta .md                    maks selisih = 6.0e-14
   kernel mentah K (m=3), baris = vektor =
[[0.035516 0.321197 1.21277  2.188909 2.188909 1.21277  0.321197 0.035516]
 [0.035516 0.321197 1.21277  2.188909 2.188909 1.21277  0.321197 0.035516]
 [0.035516 0.321197 1.21277  2.188909 2.188909 1.21277  0.321197 0.035516]
 [0.035516 0.321197 1.21277  2.188909 2.188909 1.21277  0.321197 0.035516]
 [0.035516 0.321197 1.21277  2.188909 2.188909 1.21277  0.321197 0.035516]
 [0.132103 0.61861  1.225826 1.584007 1.908235 1.449143 0.511615 0.075003]]
   distribusi P (m=3), tiap baris jumlah 1 =
[[0.004725 0.042731 0.161342 0.291203 0.291203 0.161342 0.042731 0.004725]
 [0.004725 0.042731 0.161342 0.291203 0.291203 0.161342 0.042731 0.004725]
 [0.004725 0.042731 0.161342 0.291203 0.291203 0.161342 0.042731 0.004725]
 [0.004725 0.042731 0.161342 0.291203 0.291203 0.161342 0.042731 0.004725]
 [0.004725 0.042731 0.161342 0.291203 0.291203 0.161342 0.042731 0.004725]
 [0.017603 0.082431 0.163345 0.211073 0.254277 0.193102 0.068174 0.009994]]
  [OK] P tangan vs vectors_to_distributions m=3       maks selisih = 1.4e-16
  [OK] JSD tangan vs pairwise_jsd m=3                 maks selisih = 3.5e-18
   JSD (nat) m=3 =
[[0.       0.       0.       0.       0.       0.011685]
 [0.       0.       0.       0.       0.       0.011685]
 [0.       0.       0.       0.       0.       0.011685]
 [0.       0.       0.       0.       0.       0.011685]
 [0.       0.       0.       0.       0.       0.011685]
 [0.011685 0.011685 0.011685 0.011685 0.011685 0.      ]]
   d = sqrt(JSD) m=3 =
[[0.       0.       0.       0.       0.       0.108097]
 [0.       0.       0.       0.       0.       0.108097]
 [0.       0.       0.       0.       0.       0.108097]
 [0.       0.       0.       0.       0.       0.108097]
 [0.       0.       0.       0.       0.       0.108097]
 [0.108097 0.108097 0.108097 0.108097 0.108097 0.      ]]
   keanggotaan mu = 1/(1+d/r_jsd) m=3 =
[[0.       1.       1.       1.       1.       0.606358]
 [1.       0.       1.       1.       1.       0.606358]
 [1.       1.       0.       1.       1.       0.606358]
 [1.       1.       1.       0.       1.       0.606358]
 [1.       1.       1.       1.       0.       0.606358]
 [0.606358 0.606358 0.606358 0.606358 0.606358 0.      ]]
   rata-rata baris = [0.921272 0.921272 0.921272 0.921272 0.921272 0.606358]
  [OK] phi_jsd m=3 tangan vs phi_jsd                  maks selisih = 0.0e+00
   phi (m=3) = 0.868786036155
rincian pasangan v1, v2:
   P1 =
[0.006652 0.055339 0.177261 0.260749 0.260749 0.177261 0.055339 0.006652]
   P2 =
[0.001962 0.026782 0.144475 0.326781 0.326781 0.144475 0.026782 0.001962]
   M = (P1+P2)/2 =
[0.004307 0.04106  0.160868 0.293765 0.293765 0.160868 0.04106  0.004307]
   KL(P1||M) = 0.011040055742, KL(P2||M) = 0.012581848500, JSD = 0.011810952121
   bentuk Lin 1991: H(M) - (H(P1)+H(P2))/2 = 0.011810952121
JSD-FuzzyEn tau=1 = ln(0.774330360139/0.868786036155) = -0.115098272073
  [OK] phi_jsd m vs konstanta .md                     maks selisih = 2.9e-13
  [OK] phi_jsd m+1 vs konstanta .md                   maks selisih = 3.3e-13
jsd_fuzzy_entropy_1d(x, skala [1,2]) = [-0.115098       nan]
  [OK] FE tau=1,2 tangan vs jsd_fuzzy_entropy_1d      maks selisih = 6.2e-16
  [OK] JSD-FE tau=1 vs konstanta .md                  maks selisih = 2.8e-13
vektor naik [-0.5, 0.5] vs turun [0.5, -0.5]: JSD = 0.000e+00, Euclid = 1.414214

==============================================================================
(d) SELF-CHECK JACOBIAN: analitik vs numerik (selisih tengah, eps = 1e-6)
==============================================================================
tanh @ theta awal              : maks |J_analitik - J_numerik| = 8.098e-11
relu @ theta awal              : maks |J_analitik - J_numerik| = 3.393e-11
tanh @ theta sesudah iterasi 9 : maks |J_analitik - J_numerik| = 1.203e-10
   J analitik (tanh, theta sesudah iterasi 9) =
[[ 0.351901 -0.781988  0.       -0.        0.351901 -0.781988  0.839537  0.        0.19204   0.        1.        0.      ]
 [-0.067985  0.303119 -0.        0.       -0.067985  0.303119  0.        0.839537  0.        0.19204   0.        1.      ]
 [ 0.       -0.        1.002551 -0.445116  1.002551 -0.445116  0.39881   0.        0.672147  0.        1.        0.      ]
 [-0.        0.       -0.193686  0.172539 -0.193686  0.172539  0.        0.39881   0.        0.672147  0.        1.      ]
 [ 0.966404 -0.740056  0.966404 -0.740056  0.966404 -0.740056  0.435167  0.       -0.297532 -0.        1.        0.      ]
 [-0.186703  0.286865 -0.186703  0.286865 -0.186703  0.286865  0.        0.435167 -0.       -0.297532  0.        1.      ]]
   J numerik (tanh) =
[[ 0.351901 -0.781988  0.        0.        0.351901 -0.781988  0.839537  0.        0.19204   0.        1.        0.      ]
 [-0.067985  0.303119  0.        0.       -0.067985  0.303119  0.        0.839537  0.        0.19204   0.        1.      ]
 [ 0.        0.        1.002551 -0.445116  1.002551 -0.445116  0.39881   0.        0.672147  0.        1.        0.      ]
 [ 0.        0.       -0.193686  0.172539 -0.193686  0.172539  0.        0.39881   0.        0.672147  0.        1.      ]
 [ 0.966404 -0.740056  0.966404 -0.740056  0.966404 -0.740056  0.435167  0.       -0.297532  0.        1.        0.      ]
 [-0.186703  0.286865 -0.186703  0.286865 -0.186703  0.286865  0.        0.435167  0.       -0.297532  0.        1.      ]]
relu @ theta sesudah iterasi 9 : maks |J_analitik - J_numerik| = 9.629e-11

SEMUA 65 CEK LOLOS (toleransi 1e-09).
```

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Walkthrough kode</b> (`sidang/qa/04_walkthrough_kode.md`)</summary>

# 04 — Walkthrough kode inti (per fungsi) + cacat yang diketahui

Acuan baris: `deteksi_fault_core.py` (disingkat **core**) pada commit `422232b`,
`build_11_annlm_real_notebook.py` (**b11**), `build_14_lengkap_fault_baru_notebook.py`
(**b14**), `build_15_data_hipotetik_ann_lm_notebook.py` (**b15**),
`simpan_model.py`. Bentuk array: `T` = jumlah sampel deret (5 menit),
`N`/`win` = panjang jendela, `nwin` = jumlah jendela, `P` = jumlah parameter.

Bukti angka untuk klaim di dokumen ini: `sidang/qa/cek_cacat_kode.py`
(keluaran diringkas di bagian akhir) dan `sidang/qa/cek_algoritma_tangan.py`.

---

## 1. Simulator fault (core:113-188)

### `simulate_drift_fault(x, intensity, episode_days, n_episodes, seed)` — core:113-122
Menambah tanjakan linear di 3 episode (mulai 10 %, 40 %, 70 % deret), tiap
episode 3 hari, kemiringan 1 poin/hari (= 0,00347 poin per sampel 5 menit).
Masukan `x (T,)`; keluaran `(x + drift (T,), mask bool (T,))`.

- **Kenapa episode, bukan drift sepanjang deret?** Drift sepanjang deret membuat
  semua jendela fault dan tak ada pembanding di deret yang sama; episode memberi
  jendela fault dan normal dari sensor yang sama. Posisi tetap (core:119) supaya
  dapat diulang.
- **Kenapa `seed` tidak dipakai?** Drift deterministik; argumen `seed` hanya ada
  supaya semua simulator punya tanda tangan sama untuk `simulate_multiple_faults`
  (core:152). Konsekuensinya: 8 ulangan per skenario (core:236) memakai pola
  drift yang sama, yang berbeda hanya subset sensornya.

### `simulate_spike_fault(x, intensity=0.08, p=0.015, seed)` — core:125-128
Lonjakan setiap `tau = int(1/p) = 66` sampel, tinggi `0.08 * nanstd(x)`.
Keluaran `(x + spikes, mask)`.

- **Kenapa `nanstd`?** Bila hardware fault (NaN) diterapkan lebih dulu di
  kombinasi, `std` biasa akan NaN.
- **Kenapa posisi periodik, bukan acak?** Deterministik dan dapat diulang; tetapi
  akibatnya posisi spike identik di semua sensor/ulangan (`cek_cacat_kode.py` §7).
  Tinggi 0,08·sd berada **di bawah** toleransi $r=0{,}2$·sd, jadi spike ini
  sengaja halus. Notebook 14 menggantinya dengan spike relatif $x(1+0{,}30)$ di
  posisi acak (b14:67-75).

### `simulate_bias_fault(x, bias=0.08, seed)` — core:131-132
Menambah konstanta 0,08 ke seluruh deret; semua sampel ditandai fault.
- **Apa yang dilihat fitur entropi?** **Tidak ada.** Lihat "Cacat yang diketahui"
  butir C2: kedua entropi invarian terhadap pergeseran konstan.

### `simulate_hardware_fault(x, stuck_prob=0.08, loss_prob=0.05, seed)` — core:135-145
Satu bilangan acak `rv` per sampel: `rv < 0.08` → nilai diganti nilai dari
indeks acak lain ("stuck"), `rv < 0.05` → NaN ("loss").
- **Kenapa `rv` yang sama untuk dua mask?** Supaya kedua kejadian saling eksklusif
  per sampel dan tak perlu dua generator. Akibatnya loss ⊂ stuck: porsi
  efektif stuck 3 %, loss 5 %, total 8 % (`cek_cacat_kode.py` §6). "Stuck" di
  sini sebenarnya *random replacement*, bukan nilai macet konstan.
- **Ke mana NaN-nya?** Diisi `ffill/bfill` di `inject_faults_multisensor`
  (core:213), jadi data hilang tampil sebagai nilai datar = nilai terakhir.

### `simulate_multiple_faults(x, faults, seed)` — core:148-154
Menerapkan daftar fault berurutan, OR-kan mask, lalu `np.clip(y, 0, 100)`.
- **Kenapa clip 0–100?** Kelembaban tanah dalam % fisik tidak bisa keluar
  0–100; tanpa clip, bias/drift/spike bisa menghasilkan nilai mustahil yang
  terlalu mudah dideteksi (bocoran label dari nilai di luar domain). `np.clip`
  membiarkan NaN tetap NaN.
- **Urutan berpengaruh?** Ya: spike setelah hardware memakai `nanstd`; drift
  setelah hardware menambah tanjakan ke nilai yang sudah diganti. Urutan dikunci
  di `SCENARIOS` (core:169-188) dan `inject_like_training` memakai urutan yang
  sama untuk demo (core:787-794).

### `simulate_choose_one` — core:157-161; `SCENARIOS` — core:169-188
Skenario "faulty" memilih satu dari empat fault secara acak per sensor.
16 kondisi fault + "normal" = `condition_names` (core:189).

---

## 2. Segmentasi dan label

### `make_windows(Xa, win, stride)` — core:195-202
`(T, 4)` → `(nwin, win, 4)` dan `starts (nwin,)`, `nwin = floor((T-win)/stride)+1`.
- **Kenapa `transpose(0, 2, 1)`?** `sliding_window_view(Xn, win, axis=0)` pada
  array `(T,4)` menaruh sumbu jendela **di akhir**: `(T-win+1, 4, win)`.
  Transpose mengembalikan ke `(nwin, win, 4)` supaya `W[i][:, s]` = deret sensor
  `s` (dipakai `_one_window`, core:385).
- **Kenapa `sliding_window_view`?** Hanya *view*, tanpa menyalin data; baru
  disalin saat diindeks `view[starts]`. `float32` menghemat memori.

### `inject_faults_multisensor(Xa, scenario_faults, sensor_subset, seed)` — core:205-215
Menyuntik fault hanya ke kolom sensor di `sensor_subset`, seed per sensor dari
satu RNG induk. Keluaran `Y (T,4)` (NaN sudah diisi) dan `M (T,4) bool`.
- **Kenapa disuntik ke seluruh deret lalu dipotong, bukan per jendela?**
  Episode drift 3 hari lebih panjang dari jendela; hardware "stuck" menyalin
  nilai dari waktu lain. Hanya injeksi di deret penuh yang memberi bentuk fault
  realistis. Demo memakai jalur sama (`inject_like_training`, core:787).
- **Kenapa `ffill().bfill()` lalu median?** Entropi tidak bisa menghitung NaN;
  ffill meniru logger yang mengulang nilai terakhir; bfill untuk NaN di awal;
  median untuk kolom yang seluruhnya NaN.

### `window_labels_per_sensor(mask, win, stride, thr)` — core:218-223
`(T,4)` → `(nwin, 4) bool`: sensor `s` di jendela `i` berlabel fault bila porsi
sampel fault-nya `> thr`. Dipanggil dengan `thr = FAULT_RATIO_THR = 0.01` (core:240).
- **Kenapa ambang 1 %?** Untuk $N=200$ itu > 2 sampel. Jendela yang hanya
  tersentuh 1–2 sampel fault praktis identik dengan jendela normal; memberinya
  label fault = noise label.
- **Kenapa `[::stride]` di sini sama dengan `make_windows`?** `starts =
  arange(0, T-win+1, stride)` dan `[::stride]` pada view berukuran `T-win+1`
  menghasilkan indeks yang sama, sehingga label dan jendela sejajar.

### `segment_and_label(X, win)` — core:226-248
Membuat jendela normal (kelas 0) dari deret asli, lalu untuk tiap 16 kondisi
fault × 8 ulangan (ukuran subset 1,2,3,4,1,2,3,4) menyuntik, memberi label, dan
menyimpan hanya jendela yang punya ≥ 1 sensor fault. Keluaran `W (nwin,win,4)`,
`y (nwin,)` kondisi 0..16, `Ysens (nwin,4)` multi-hot, `starts (nwin,)`.
- **Kenapa stride `win//2` (50 % tumpang-tindih)?** Menggandakan jumlah jendela
  tanpa data baru dan tidak melewatkan fault di batas jendela. Harga yang
  dibayar: jendela bertetangga berbagi 50 % sampel, jadi **CV acak bocor**; ini
  alasan wajib grouped CV (bagian 6).
- **Kenapa jendela fault dengan ≤ 1 % sampel fault dibuang, tidak diberi label
  normal?** (core:242 `keep = sens.any(axis=1)`) Jendela itu berasal dari deret
  yang sudah disuntik; melabelinya "normal" mencampur data hampir-normal ke kelas
  normal, melabelinya "fault" memberi label tanpa isi. Membuang keduanya paling
  bersih. Kelas normal hanya diambil dari deret asli.
- **Kenapa `RANDOM_SEED` lokal (core:228)?** Supaya subset sensor dan seed injeksi
  sama di setiap pemanggilan — notebook 11, 14, `simpan_model.py`, dan demo
  memperoleh jendela identik.

### `balanced_subsample(W, y, Ysens, starts, max_per_class, seed)` — core:251-261
Maks 100 jendela per kondisi, lalu diacak. Keluaran sama bentuknya, lebih kecil.
- **Kenapa 100?** Biaya ekstraksi entropi (2 metode × 4 sensor × 15 skala) dan
  LM padat ($O(P^3)$ per iterasi) dibatasi; seimbang per kondisi juga mencegah
  kondisi yang jendelanya banyak (bias: semua jendela berlabel) mendominasi.

### `build_scenario(y_cond, classes)` — core:421-436 (b11:1156)
Memetakan 17 kondisi ke kelas skenario S1–S5 (`LADDER`, core:402-416) dan
undersample tiap kelas ke ukuran kelas terkecil.
- **Kenapa undersample?** Di S1 kelas "faulty" menggabung 16 kondisi (≈1600
  jendela) vs 100 normal; tanpa penyeimbangan akurasi 94 % bisa dicapai dengan
  menebak "faulty" saja.

---

## 3. Entropi multiskala

### `coarse_grain_cmse(x, s, k)` — core:267-271
Rata-rata blok tak-tumpang ukuran `s` mulai offset `k`; `(n,)` → `((n-k)//s,)`.
- **Kenapa ada offset `k`?** Composite MSE (Wu 2013): di skala besar deret
  coarse-grained pendek; merata-ratakan $s$ versi bergeser mengurangi varians
  estimasi. Costa asli hanya $k=0$.
- **Kenapa `reshape(-1, s).mean(1)`?** Setara rumus Costa
  $\frac1\tau\sum$ tanpa loop Python.

### `embed_matrix_centered(y, m_)` — core:274-278
`(L,)` → `(L-m+1, m)`, tiap baris dikurangi rata-ratanya.
- **Kenapa dipusatkan?** Mengikuti Chen 2007 (FuzzyEn mengurangi baseline
  lokal): yang dibandingkan **bentuk** pola, bukan level. Efek sampingnya:
  fault bias tak terlihat (Cacat C2).

### `_pick_ref` — core:281-283; `phi_euclidean(V, r, n_ref, seed)` — core:286-297
`V (Nm, m)` → skalar $\phi$. Jarak Euclid dari ≤ 128 vektor acuan ke semua
vektor lewat $\lVert a\rVert^2 + \lVert b\rVert^2 - 2a\cdot b$, keanggotaan
$1/(1+d/r)$, diagonal acuan dinolkan, rata-rata baris dibagi $N-1$.
- **Kenapa `n_ref = 128`?** Matriks jarak penuh $O(N^2)$: untuk $N=1000$ itu 1 juta
  pasangan × 15 skala × banyak offset × 4 sensor × ±1700 jendela × 2 metode.
  Dengan 128 acuan biayanya $O(128N)$; $\phi$ adalah rata-rata, sehingga
  subsampel acuan memberi penduga tak bias dengan varians kecil.
- **Kenapa `np.maximum(..., 0)` sebelum `sqrt`?** Rumus ekspansi kuadrat bisa
  memberi −1e-16 akibat pembulatan.
- **Kenapa `1/(1+d/r)` bukan `exp(-d^n/r)`?** Bukan dari Chen; pilihan rancangan
  (dipakai juga di notebook-notebook sebelumnya). Keanggotaan rasional tidak
  pernah 0 dan turun lebih lambat, sehingga $\phi>0$ selalu dan $\ln$ aman. Ini
  harus disebut sebagai modifikasi, bukan FuzzyEn baku.
- **Kenapa `r + EPS`?** Jendela datar (sd = 0, mis. sensor macet panjang) →
  $r=0$ → pembagian nol.

### `edm_fuzzy_entropy_1d(x, scales, m, r_ratio, n_ref, seed)` — core:300-314
`(n,)` → `(15,)` entropi per skala.
- **Kenapa $r$ dihitung sekali dari deret asli (core:302)?** Costa: $r$ tetap
  antarskala supaya penurunan varians akibat perataan ikut terukur sebagai
  perubahan entropi.
- **Kenapa `len(y) < m+2` dilewati dan syarat `phi > 0`, `isfinite`?** Butuh
  ≥ 3 vektor $m+1$ untuk $\phi$ (core:288); $\ln$ tidak terdefinisi untuk 0/NaN.
  Bila semua offset gagal, skala itu NaN → `sanitize`.
- **Kenapa seed berbeda untuk $\phi^m$ dan $\phi^{m+1}$ (core:309-310)?** Tidak
  ada alasan teoretis; acuan berbeda menambah sedikit varians. Lebih rapi bila
  acuan sama (Chen memakai himpunan $i$ yang sama).

### `vectors_to_distributions(V, centers, h)` — core:317-321
`(Nm, m)` → `(Nm, 8)`: histogram lunak berkernel Gauss, tiap baris dijumlah 1.
- **Kenapa kernel, bukan histogram keras?** Dengan $m=2$ atau 3 nilai, histogram
  keras hampir selalu 1–3 bin bernilai 1 lainnya 0 → JS sering $\ln 2$ (maksimum)
  dan tak stabil terhadap pergeseran kecil. Kernel memberi distribusi halus,
  kontinu terhadap $v$.
- **Kenapa `+ 1e-300` di penyebut?** Bila semua nilai jauh di luar ±3 sd, semua
  $K\approx0$.

### `pairwise_jsd(A, B)` — core:324-331
`(nA, 8)`, `(nB, 8)` → `(nA, nB)` JS natural log (≤ ln 2).
- **Kenapa `EPS` di dalam log?** $0\log0$ didefinisikan 0; `a*log(a+EPS)` memberi
  0 tanpa peringatan.
- **Kenapa `np.maximum(d, 0)`?** Pembulatan bisa membuat JS ≈ −1e-17, lalu
  `sqrt` NaN.

### `phi_jsd` — core:334-343; `jsd_fuzzy_entropy_1d` — core:346-365
Sama dengan jalur EDM, tetapi jarak = $\sqrt{\mathrm{JS}}$ antar distribusi
vektor, ambang $r_\text{JSD} = 0{,}2\sqrt{\ln2}$.
- **Kenapa $\sqrt{\mathrm{JS}}$, bukan JS?** $\sqrt{\mathrm{JS}}$ adalah metrik
  (Endres–Schindelin 2003), sejajar dengan jarak Euclid di EDM.
- **Kenapa $r_\text{JSD} = 0{,}2\sqrt{\ln 2}$?** Batas atas jarak JS natural =
  $\sqrt{\ln2}$; 0,2 meniru `r_ratio = 0.2` di EDM, jadi toleransi = 20 % dari
  rentang jarak.
- **Kenapa pusat bin ±3 sd dari deret jendela?** Vektor terpusat berada di
  sekitar 0 dengan sebaran ≈ sd; ±3 sd mencakup hampir semua nilai.
- **Pertanyaan tajam:** histogram tidak mengenal urutan — vektor naik dan turun
  dianggap kembar (03, bagian c), dan entropi bisa negatif. Jawaban jujur: JSD
  mengukur kemiripan **sebaran nilai** dalam pola, bukan bentuk berurut; itu
  perbedaan konseptual dari EDM, bukan kesalahan implementasi. Parameter
  `r_ratio` di fungsi ini tidak dipakai.

### `sanitize(F)` — core:368-372
Isi NaN dengan median kolom. Kolom yang seluruhnya NaN tetap NaN (lalu
`SimpleImputer` di pipeline). Dipanggil sebelum CV, tetapi median kolom
fitur tanpa label → tidak membocorkan label; di jendela 200/700/1000 NaN
praktis tidak muncul ($n \ge 5\tau$).

### `entropy_1d` / `_one_window` / `concat_multisensor_features` — core:375-395
`(nwin, win, 4)` → `(nwin, 60)`: 15 skala × 4 sensor digabung.
- **Kenapa seed `seed + 1000*i` dan `+ 19*s`?** Setiap jendela dan sensor
  mendapat subsampel acuan sendiri yang deterministik, **dan sama untuk EDM dan
  JSD** — perbandingan adil.
- **Kenapa `joblib` proses?** Perhitungan numpy kecil-kecil per jendela;
  paralel per jendela memakai semua inti.

---

## 4. Jaringan saraf Levenberg–Marquardt

### `lm_init_layers(n_in, hidden, n_out, seed)` — core:442-449
Daftar `[W (d_i, d_{i+1}), b (d_{i+1},)]`, $W \sim U(\pm\sqrt{6/(d_i+d_{i+1})})$, $b=0$.
- **Kenapa Xavier?** Menjaga varians aktivasi tanh di daerah linear saat awal
  sehingga $1-h^2$ tidak jenuh. Untuk ReLU, He init lebih lazim; kode memakai
  Xavier untuk keduanya (komentar b11:1434).
- **Tetapi:** niat Xavier ini **tidak sampai** ke jaringan karena cacat
  pack/unpack (bagian "Cacat yang diketahui", C1).

### `lm_pack(layers)` — core:452-453 dan `lm_unpack(theta, template)` — core:456-461
`lm_pack`: list → vektor `(P,)` berurutan **[W1, W2, …, b1, b2, …]**.
`lm_unpack`: vektor → list, membaca **[W1, b1, W2, b2, …]**.
- **Kenapa vektor datar?** LM bekerja pada satu vektor parameter (Jacobian
  `(m·C, P)`, `solve` `P×P`).
- **Kenapa `template`?** Hanya untuk bentuk (shape) tiap lapis.
- Lihat C1.

### `lm_forward(layers, X, activation)` — core:464-474
`(n, n_in)` → keluaran `(n, n_out)` dan daftar aktivasi tiap lapis.
- **Kenapa lapis terakhir linear?** Target one-hot + kuadrat galat: keluaran
  linear membuat Jacobian lapis terakhir sederhana ($\delta=I$) dan LM klasik
  memang untuk least squares. Softmax hanya saat prediksi.

### `lm_residuals` — core:477-479; `lm_residual_jacobian` — core:482-508
`(P,)` → `r (n·C,)` dan `J (n·C, P)` analitik.
- **Kenapa Jacobian residual per keluaran, bukan gradien loss?** LM butuh
  $J$ penuh untuk $J^\top J$; backprop biasa hanya memberi $J^\top r$. Kode
  merambatkan $\delta$ berbentuk `(n, C, units)` = satu backprop per keluaran.
- **Kenapa urutan kolom W lalu b per lapis (core:503-507)?** Harus sama dengan
  `lm_unpack` — dan memang sama (dicek numerik, 03 bagian d).
- **Biaya memori:** `(n·C) × P`; untuk (64,16), 60 fitur, 7 kelas, P = 5063 dan
  ±560 jendela latih → ~3900 × 5063 ≈ 160 MB. Ini alasan grid arsitektur kecil
  (komentar b11:1604-1606).

### `lm_fit(X, Y, n_in, hidden, n_out, activation, iters, seed, lam0, max_train_s)` — core:511-567
Loop LM (rinci di 03 bagian a). Keluaran `theta`, `template`, `hist`, `train_mse`.
- **Kenapa `diag(JᵀJ)` bukan $I$?** Marquardt 1963: invarian skala parameter.
- **Kenapa dijepit `1e-12` (core:537)?** Kolom Jacobian nol (unit ReLU mati,
  atau baris W nol akibat C1) → diagonal 0 → $A+\lambda D$ singular.
- **Kenapa `max_train_s` (core:522)?** `solve` $O(P^3)$; satu arsitektur besar
  bisa memakan menit per iterasi. Budget waktu (40 s pencarian, 150 s refit;
  core:73-74) menjamin seluruh nested CV (30 run × 5 fold × 16 kombinasi × 2
  inner fold) selesai dalam batas 12 jam Kaggle. Status `budget(MAX_TRAIN_S)`
  dicatat di `hist` dan dihitung di b11:1679-1680, jadi jujur terlihat.
- **Kenapa berhenti di `gn < 1e-10` dan penurunan relatif `1e-12`?** Kriteria
  stasioner dan stagnasi standar (Madsen–Nielsen–Tingleff).
- **Lihat juga:** iterasi ditolak menghabiskan jatah iterasi (03, a.1 butir 5).

### `lm_predict_proba` — core:570-575
Softmax stabil (`out - max`) atas keluaran linear. Probabilitas tidak
terkalibrasi; ROC-AUC (b11) hanya butuh urutan skor, jadi tetap sah.

### `LMEstimator` — core:578-608
Pembungkus scikit-learn: label → one-hot (`searchsorted` pada `classes_`), lalu
`lm_fit`.
- **Kenapa `ClassifierMixin` lebih dulu?** scikit-learn ≥ 1.6 memeriksa urutan
  MRO untuk tag estimator (komentar b11:1559).
- **Kenapa `__init__` hanya menyimpan argumen?** Kontrak `BaseEstimator`
  (`get_params/clone`).

### `LMSensorID` — core:611-637
Satu LM untuk target multi-hot 4 sensor; keluaran linear, ambang 0,5.
- **Kenapa satu model 4 keluaran, bukan 4 model?** Berbagi lapis tersembunyi;
  hemat waktu; tak butuh `MultiOutputClassifier`.
- **Kenapa ambang 0,5?** Target 0/1 dengan MSE → keluaran linear menduga
  $P(\text{fault}\mid x)$ secara kasar; 0,5 = titik tengah.

### `make_lm_pipeline` / `make_sensor_pipeline` — core:640-655
`SimpleImputer(median) → StandardScaler → LM`.
- **Kenapa di dalam pipeline?** Median dan mean/sd di-*fit* hanya pada fold latih
  → tidak bocor ke fold uji.

---

## 5. Pencarian arsitektur

### `_inner_combo_score` — core:661-673; `inner_search_best(Ftr, ytr, gtr, ...)` — core:676-684
16 kombinasi (8 arsitektur × tanh/relu) dinilai dengan `StratifiedGroupKFold`
2 lipatan **di dalam** data latih luar, skor macro-F1, diurutkan.
- **Kenapa grup juga di inner CV?** Sama seperti luar: jendela bertumpang-tindih
  tidak boleh terbelah antara latih-dalam dan validasi-dalam.
- **Kenapa hanya 6 iterasi LM saat pencarian?** Peringkat arsitektur cukup
  dibaca dari awal kurva; refit 40 iterasi. Risikonya: dengan penolakan beruntun
  (03 bagian a) 6 iterasi bisa nyaris tak bergerak, sehingga peringkat agak
  acak.
- **Kenapa `except: return -1.0` (core:672)?** Satu kombinasi yang gagal
  (mis. memori) tidak menggagalkan seluruh run; b11 mencatat statusnya. Di core
  versi ini pesan errornya dibuang — lebih baik dicatat.

---

## 6. Validasi di notebook 11 (b11)

### Grup CV — b11:601 (`groups = starts // WIN`)
- **Kenapa grouped CV wajib?** Stride N/2 → jendela bertetangga berbagi 50 %
  sampel; jendela kondisi fault berbeda pada `starts` yang sama berasal dari
  **potongan deret asli yang sama**. CV acak menaruh "kembaran" di latih dan uji
  → skor terlalu tinggi. Grup = blok waktu memastikan potongan yang sama tidak
  muncul di kedua sisi.
- **Sisa kebocoran:** lihat C3 — batas antar grup masih tumpang-tindih 50 %.

### `nested_grouped_cv(F, y, groups)` — b11:1657-1720
Luar: `StratifiedGroupKFold(k)`, `k = min(5, #grup, kelas terkecil)` (b11:1659);
dalam: `inner_search_best` hanya pada fold latih; refit arsitektur terbaik 40
iterasi; prediksi OOF untuk confusion matrix dan rantai sensor-ID.
- **Kenapa nested?** Memilih arsitektur dengan data uji yang sama → skor optimis.
  Nested memisahkan pemilihan model dari penilaian.
- **Kenapa Stratified?** Tiap fold memuat semua kelas meski grup memaksa
  pembagian kasar.

### Uji berpasangan — b11:1929-1950 (`_paired_test`)
Paired t-test pada F1 5 skenario (EDM vs JSD) per panjang jendela, plus
Wilcoxon dan Cohen's $d_z = \bar d/s_d$.
- **Kenapa berpasangan?** Skenario sama, fold sama, fitur beda → selisih per
  skenario menghilangkan variasi tingkat kesulitan skenario.
- **Pertanyaan tajam:** $n=5$ pasangan, df = 4, daya uji rendah. **Wilcoxon
  eksak dua sisi dengan $n=5$ tidak pernah bisa $p<0{,}05$** (p minimum
  $2/2^5 = 0{,}0625$), jadi kolom Wilcoxon hanya pendamping. Uji tingkat fold
  (b11:2011-2024) melanggar independensi (fold berbagi data) dan sudah ditandai
  "bukan uji utama" di kode.

### Rantai sensor-ID — b11:2054-2097
Hanya jendela yang **diprediksi** fault oleh tahap 1 (OOF, b11:2059) yang
diteruskan; target `Ysens` multi-hot; arsitektur = modus pilihan grid
(b11:2069); `lm_fit` langsung, ambang 0,5.
- **Kenapa dirantai dari OOF?** Kesalahan tahap 1 (normal diprediksi fault →
  target semua 0; fault terlewat → hilang) ikut terhitung, seperti sistem nyata.
- **Cacat:** pembagian latih/uji tahap ini acak, bukan grouped (C4).

---

## 7. Notebook 14 dan 15

### b14 — bentuk fault baru (b14:50-104) dan label "semua komponen ada" (b14:123-143)
Builder menambal teks b11: drift linear per episode (total 5 poin), bias
episode 5 poin, spike relatif ×1,30 acak 1,52 %, hardware dibatasi ke episode
di kombinasi; mask jadi **bitmask** (`FAULT_BIT`) supaya label jendela kombinasi
mensyaratkan setiap komponen hadir (spike ≥ 1 titik, lainnya > 1 %).
- **Kenapa bitmask?** Mask bool tidak tahu komponen mana yang ada; jendela
  "drift+spike" yang hanya berisi spike akan berlabel kombinasi padahal tidak.
- **Kenapa `seed + 1`, `seed + 2` (b14:69, b14:83)?** Spike dan hardware di satu
  kombinasi menerima seed yang sama; offset membuat aliran acaknya berbeda.
- **Kenapa `sub1` meng-assert tepat satu kemunculan (b14:29-32)?** Tambalan
  berbasis teks; bila b11 berubah, builder gagal keras daripada diam-diam
  menghasilkan notebook salah.

### b15 — fitur lag tanpa entropi, simpan `.npz`
- `lag_features` (b15:437-445): `lag1..lag10`, `diff`, `isna`, `isna_10` per
  sampel. **Kenapa `np.roll`?** Cepat; baris awal yang "membungkus" dari ujung
  deret dibuang lewat `valid_row` (b15:432).
- **Pembagian per blok 1 hari 3:1:1 (b15:427-431)?** Pembagian kronologis
  60/20/20 menaruh semua episode di latih; blok bergilir menyebar episode ke
  latih/validasi/uji. Lag 10 sampel di batas blok tetap bocor sedikit (diakui di
  teks b15:416).
- **Kenapa `.npz` + `allow_pickle=False` (b15:789, b15:736)?** File hanya berisi
  array angka; memuatnya tidak mengeksekusi kode (pickle bisa), dan tidak
  bergantung versi scikit-learn. Pembaca `prob_fault` (b15:751-765) menulis ulang
  maju jaringan dengan urutan $W_1,b_1,W_2,b_2$ = `lm_unpack` (diverifikasi
  sama, `cek_cacat_kode.py` §2), dan setiap file dimuat ulang lalu dibandingkan
  dengan prediksi di memori (assert < 1e-9, b15:790-805).

### `SavedLM` — core:709-766 (dipakai `simpan_model.py:133-134`)
Menyimpan `fill` (median imputer), `mean`, `scale`, `theta`, `shapes`,
`activation`, `classes`; prediksi = isi NaN → standarisasi → `lm_forward(lm_unpack(...))`.
- **Kenapa tidak mem-pickle Pipeline sklearn?** Pickle Pipeline menyimpan atribut
  internal `SimpleImputer`/`StandardScaler`; Kaggle (sklearn 1.6.1) vs laptop
  (1.9.1) gagal dibuka (docstring core:712-716). `SavedLM` hanya berisi array
  numpy + satu kelas dari core.
- **Masih pickle?** Ya: `simpan_model.py:140` memakai `joblib.dump` (pickle) untuk
  bundle. Bedanya, isi bundle tidak bergantung versi sklearn; file tetap harus
  dari sumber tepercaya. Format bebas-pickle sungguhan = `.npz` di b15.
- **Konsisten dengan pelatihan?** Ya: `_template()` hanya membuat bentuk, dan
  `lm_unpack` dipakai sama seperti saat `fit`/`predict`
  (`cek_cacat_kode.py` §2: selisih 0,0 untuk kelas dan sensor).

### `condition_for` / `inject_like_training` — core:775-794
Memetakan pilihan fault demo (mis. {spike, hardware}) ke nama kondisi
("spike+malfunc") dan menyuntik ke **deret penuh** dengan urutan komponen yang
sama dengan pelatihan.
- **Kenapa tidak menyuntik ke satu jendela saja (seperti `inject_demo`,
  core:698)?** Episode drift dan salinan "stuck" butuh konteks deret penuh;
  menyuntik per jendela menghasilkan bentuk fault yang tidak pernah dilihat model.

---

## Cacat yang diketahui

### C1. `lm_pack` dan `lm_unpack` tidak saling invers (core:452-461; salinan identik b11:1438-1445)

**Faktanya.** `lm_pack` menulis $[W_1, W_2, \dots, W_L, b_1, \dots, b_L]$;
`lm_unpack` membaca $[W_1, b_1, W_2, b_2, \dots]$. Hanya dipakai satu kali
per pelatihan: `theta = lm_pack(template)` di `lm_fit` (core:514). Setelah itu
**semua** jalur memakai `lm_unpack` dengan urutan yang sama: `lm_residuals`,
`lm_residual_jacobian` (kolom J disusun W,b per lapis, core:503-507),
`lm_predict_proba`, `LMSensorID.decision_function`, rantai sensor-ID
(b11:2072), `SavedLM.decision_function` (core:754), dan pembaca `.npz`
notebook 15 (b15:755-760). Jadi pelatihan, prediksi, dan model tersimpan
**konsisten**; yang salah hanya **nilai awal** $\theta_0$.

**Apa yang sebenarnya terjadi pada $\theta_0$** (diverifikasi,
`cek_cacat_kode.py` §1):

- $W_1$ selalu benar (Xavier).
- $b_1$ berisi $n_1$ bobot pertama $W_2$ (bukan nol).
- Lapis berikutnya bergeser; nol-nol bias asli terdorong ke ekor vektor sehingga
  **lapis terakhir sebagian/seluruhnya nol**, dan $b_L=0$.
- Contoh 2-2-2: $b_1 = [0.767, 1.011]$, $W_2 = [[0.261, 0.562],[0,0]]$.
- Konfigurasi grid penelitian (60 fitur):

| arsitektur | keluaran | $W$ terakhir nol | unit tersembunyi terakhir tanpa bobot keluar |
|---|---|---|---|
| 1 lapis (16/32/64) | 2 (S1, S5) | 50 % | separuh unit |
| 1 lapis | 5 (S2, S4) | 20 % | 3/16, 6/32, 12/64 |
| 1 lapis | 7 (S3) | 14,3 % | 2/16, 4/32, 9/64 |
| 1 lapis | 4 (sensor-ID) | 25 % | seperempat unit |
| 2 lapis (16,8), (32,8), (32,16), (32,32), (64,16) | 2 | **100 %** | semua |
| (32,8), (64,16) | 5 dan 4 | **100 %** | semua |
| 2 lapis lain | 5/7/4 | 29–75 % | sebagian |

Pada kasus 100 %: keluaran awal = 0 untuk semua masukan (MSE awal tepat 0,5
untuk 2 kelas), dan **semua kolom Jacobian lapis 1–2 nol** di iterasi pertama
(dicek: 0 dari 124 kolom tak nol pada jaringan 10-[8,4]-2). Iterasi pertama
hanya bisa menggerakkan lapis terakhir; mulai iterasi kedua semua lapis ikut.
Simetri tidak terjadi karena $W_1$ (dan sebagian lapis tengah) tetap acak.

**Dampak terukur (data mainan, `cek_cacat_kode.py` §3, 5 seed):**

| kasus | MSE latih akhir, kode asli | MSE latih akhir, pack diperbaiki |
|---|---|---|
| [8], 3 kelas, 6 iterasi | 0,171 | 0,211 |
| [8], 3 kelas, 15 iterasi | 0,062 | 0,064 |
| [8,4], 2 kelas, 6 iterasi | 0,176 | 0,203 |
| [8,4], 2 kelas, 15 iterasi | 0,017 | 0,060 |

Di contoh ini init yang "teracak" **tidak lebih buruk** (bahkan sedikit lebih
baik: keluaran awal mendekati 0 dekat dengan target one-hot rata-rata). Ini
bukan bukti umum, tetapi menunjukkan dampaknya kecil dan tidak searah.

**Jawaban jujur untuk penguji:**

> "Ya, ada ketidaksesuaian urutan antara `lm_pack` dan `lm_unpack`. Karena
> semua perhitungan sesudah inisialisasi — residual, Jacobian, langkah LM,
> prediksi, dan model yang disimpan — memakai `lm_unpack` yang sama, jaringan
> yang dilatih dan yang dievaluasi adalah jaringan yang sama; tidak ada bobot
> yang tertukar saat prediksi. Akibatnya hanya pada **titik awal**: bias lapis
> pertama terisi angka kecil dan sebagian bobot lapis terakhir mulai dari nol,
> jadi inisialisasinya bukan Xavier murni seperti yang saya tulis. LM adalah
> metode orde dua; sesudah satu iterasi diterima, lapis terakhir sudah terisi,
> dan pada uji kecil hasil akhirnya setara dengan inisialisasi yang benar.
> Untuk perbandingan EDM-Fuzzy vs JSD-Fuzzy, kedua metode memakai fungsi
> pelatihan yang sama, seed yang sama (`RANDOM_SEED=42`), grid arsitektur yang
> sama, dan pembagian fold yang sama, sehingga inisialisasi yang sama-sama
> menyimpang itu tidak menguntungkan salah satu metode. Yang perlu saya ralat
> di naskah adalah klaim 'inisialisasi Xavier dengan bias nol'."

**Perbaikan** (satu baris; perlu menjalankan ulang notebook karena $\theta_0$
berubah, sehingga angka akan bergeser sedikit):

```python
def lm_pack(layers):
    return np.concatenate([np.concatenate([W.ravel(), b]) for W, b in layers])
```

Sesudah perbaikan `lm_unpack(lm_pack(t)) == t` (dicek). Model yang sudah
tersimpan tetap sah karena `theta` tersimpan dibaca dengan `lm_unpack` yang tidak
berubah.

### C2. Fault bias konstan tidak terlihat oleh fitur entropi (core:131-132, 274-278, 302, 351)

Kedua entropi memusatkan tiap vektor (mengurangi rata-ratanya) dan menghitung
$r$ serta pusat bin dari sd jendela. Menambah konstanta $b$ ke seluruh deret
tidak mengubah vektor terpusat, sd, $r$, maupun pusat bin, jadi fitur **identik**
(dicek: $\max|F(x+0{,}08)-F(x)| \approx 10^{-14}$ untuk EDM, $10^{-12}$ untuk JSD;
sama untuk $x+5$). Jendela bias di notebook 11 memakai `starts` yang sama dengan
jendela normal, sehingga fiturnya kembar dengan jendela normal kecuali di mana
hardware/clip ikut berperan.

Bukti di hasil: recall kelas bias pada S2 notebook 11 = 19 % / 21 % (N=200,
EDM/JSD), 35 % / 37 % (N=700), 29 % / 20 % (N=1000) dengan peluang acak 20 %
(5 kelas) — `hasil11/exports/11_rq4_confusion_matrix.csv`. Jendela bias juga
masuk kelas "faulty" S1, sehingga membatasi skor S1.

**Jawaban jujur:** "Entropi mengukur keteraturan pola, bukan level; bias
konstan secara matematis tidak terdeteksi oleh fitur ini, dan hasil kelas bias
memang setingkat tebakan. Deteksi bias butuh fitur level/perbandingan antar
sensor (mis. selisih terhadap median 4 sensor) — itulah alasan notebook 14
memakai bias **episode** (ada tepi naik-turun) dan notebook 15 memakai fitur lag
yang berisi level." Ini berlaku sama untuk EDM dan JSD, jadi tidak membiaskan
perbandingan keduanya, tetapi menurunkan skor absolut S1–S4.

### C3. Grup CV `starts // WIN` masih bocor di batas grup (b11:601, `simpan_model.py:85`)

Dengan stride $N/2$, grup $g$ berisi jendela mulai $gN$ dan $gN + N/2$. Jendela
$gN + N/2$ menutup $[gN+N/2,\,(g+1)N + N/2)$, separuhnya sama dengan jendela
pertama grup $g+1$. Pada 40 jendela beruntun, 19 dari 39 pasangan bertumpang
tindih (49 %) berada di grup **berbeda** (`cek_cacat_kode.py` §5). Bila dua grup
bertetangga jatuh di fold berbeda, 50 % sampel jendela uji sudah dilihat saat
latih (dengan fault yang mungkin berbeda).

Dampak: optimistis untuk **kedua** metode secara sama (fold identik), jadi
perbandingan relatif tetap adil; skor absolut sedikit terlalu tinggi.
Perbaikan: grup lebih besar (mis. `starts // (k·WIN)`) **dan** buang
(*purge*) jendela yang melintasi batas grup, atau `stride = WIN` untuk evaluasi.

### C4. Sensor-ID memakai `train_test_split` acak (b11:2065)

Tahap 2 membagi jendela yang lolos tahap 1 secara acak 70/30, bukan per grup.
Jendela bertumpang-tindih dan jendela dari potongan deret yang sama (kondisi
fault berbeda) bisa berada di latih dan uji → F1 sensor-ID optimistis (untuk
kedua metode). Perbaikan: `GroupShuffleSplit` dengan `groups[keep][mask]`.

### C5. Hal kecil yang perlu diketahui (bukan penyebab hasil salah)

- Aturan $\lambda$ saat ditolak tetap ×2 (core:562), bukan $\nu$ berlipat
  (Nielsen); iterasi ditolak memakan jatah `iters` (03, a.1).
- `phi_euclidean`/`phi_jsd` memakai acuan acak berbeda untuk $m$ dan $m+1$
  (core:309-310, 360-361).
- `simulate_hardware_fault`: loss ⊂ stuck, stuck efektif 3 % (core:140-143).
- `simulate_drift/spike/bias_fault` mengabaikan `seed`; spike periodik di posisi
  sama di semua sensor (core:125-128).
- `jsd_fuzzy_entropy_1d(r_ratio=...)` tidak dipakai (core:346).
- `_inner_combo_score` di core membuang pesan error (core:672-673).
- Entropi JSD bisa negatif (03, bagian c) — perlu disebut bila penguji melihat
  nilai < 0 di kurva.

---

## Ringkasan keluaran `cek_cacat_kode.py` (2026-11-20)

```
1. lm_pack vs lm_unpack (2-2-2, seed 0)
  theta = lm_pack = [0.3355 -0.5639 -1.1244 -1.1843 0.7674 1.011 0.2612 0.5621 0 0 0 0]
  dipakai jaringan: W1 benar; b1 = [0.7674 1.011]; W2 = [[0.2612 0.5621],[0 0]]; b2 = [0 0]
  lm_unpack(pack_benar(t)) == t -> OK ; lm_pack(lm_unpack(theta)) == theta ? False
2. Pipeline vs SavedLM predict_proba: maks selisih = 0.00e+00
   Pipeline sensor vs SavedLM decision_function: maks selisih = 0.00e+00
   Urutan baca prob_fault (notebook 15) == lm_unpack ? True
3. [8,4]->2 di theta awal: kolom Jacobian lapis 1-2 yang tak nol = 0/124; lapis 3 = 10/10
4. EDM: maks |F(x+0.08) - F(x)| = 1.44e-14 ; maks |F(x+5) - F(x)| = 6.88e-15
   JSD: maks |F(x+0.08) - F(x)| = 1.48e-12 ; maks |F(x+5) - F(x)| = 3.73e-13
5. WIN=200/700/1000: 39 pasangan bertumpang-tindih, 19 di grup BERBEDA (49%), tumpang-tindih 50%
6. porsi NaN (loss) = 0.0500 ; 'stuck' tersisa = 0.0300 ; total mask = 0.0800
7. posisi spike seed=1 dan seed=999: [0 66 132 198 ...] identik: True
   tinggi spike 0.08*sd = 0.0565 vs r = 0.2*sd = 0.1415
8. JSD-FuzzyEn deret 8 titik tau=1: -0.1151
```

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Pustaka dan posisi kebaruan</b> (`sidang/qa/07_pustaka_dan_kebaruan.md`)</summary>

# 07 — Pustaka dan Posisi Kebaruan (JSD-Fuzzy vs EDM-Fuzzy + ANN-LM)

Tanggal penelusuran: **2026-11-20**. Semua referensi di bagian (b) dan (d) sudah dicek ada (judul, penulis, tahun, venue, DOI) lewat Crossref API / OpenAlex / halaman penerbit. BibTeX lengkap: `sidang/qa/pustaka.bib` (51 entri, kunci seperti `chen2007fuzzyen`). Yang tidak bisa dicek ada di bagian (e).

Ringkasan metode yang diposisikan (dari `deteksi_fault_core.py`):
- **EDM-Fuzzy** (`phi_euclidean`, `edm_fuzzy_entropy_1d`): vektor embedding dipusatkan (dikurangi rerata), jarak **Euclidean** antar vektor, keanggotaan fuzzy `mu = 1/(1 + d/r)`, r = 0,2·SD, 128 vektor referensi acak, coarse-graining komposit (`coarse_grain_cmse`), skala 1..15, m = 2.
- **JSD-Fuzzy** (`vectors_to_distributions`, `pairwise_jsd`, `phi_jsd`, `jsd_fuzzy_entropy_1d`): tiap vektor embedding terpusat diubah jadi **histogram lunak** (8 bin pada ±3 SD, kernel Gauss), lalu jarak antar vektor = **akar JSD** (jarak Jensen–Shannon, metrik menurut Endres & Schindelin 2003), dimasukkan ke fungsi keanggotaan yang sama `mu = 1/(1 + d/r_jsd)` dengan r_jsd = 0,2·√ln2.

---

## (a) Paper dasar EDM-Fuzzy: hasil pencarian

**Kesimpulan: paper "ACS" milik mahasiswa (EDM-Fuzzy + ANN-LM untuk fault sensor kelembaban tanah) TIDAK ditemukan** di Crossref, OpenAlex, Semantic Scholar (kena rate-limit), maupun web search. Kemungkinan belum terindeks, masih dalam proses, atau terbit dengan judul yang tidak memuat kata kunci yang dicari.

Yang ditemukan:

1. **Asal metode EDM-Fuzzy (pasti):** Zhou, R., Wang, X., Wan, J., & Xiong, N. (2021). *EDM-Fuzzy: An Euclidean Distance Based Multiscale Fuzzy Entropy Technology for Diagnosing Faults of Industrial Systems.* IEEE Transactions on Industrial Informatics, 17(6), 4046–4054. https://doi.org/10.1109/TII.2020.3009139. Abstraknya menyebut dua kelemahan SampEn yang diperbaiki: (1) jarak Chebyshev (maksimum elemen) "not able to fully reflect the distance of the two vectors", (2) kemiripan biner 0/1 menyebabkan lompatan nilai entropi. EDM-Fuzzy memakai **jarak Euclidean + kemiripan kontinu 0..1**. Diuji pada data bearing. Penulisnya dari Tiongkok, bukan Indonesia. Penerapan kedua oleh kelompok yang sama: Zhou dkk. (2021), *Security and Communication Networks*, https://doi.org/10.1155/2021/5560185 (trafik jaringan anomali).
   → **Wajib disitasi** sebagai sumber baseline; jangan sebut EDM-Fuzzy sebagai metode buatan paper pertama mahasiswa.

2. **Kandidat terdekat untuk "paper pertama" mahasiswa (cocok konteks, tapi tanpa EDM-Fuzzy di abstrak):** Wardhani, L. K., Buono, A., Wahjuni, S., & Syukur, M. (2026). *Evaluation of ANN–Levenberg Marquardt Models for Fault Detection in Smart Farming System.* JITK (Jurnal Ilmu Pengetahuan dan Teknologi Komputer), 11(4), 1053–1060. https://doi.org/10.33480/jitk.v11i4.7481 (IPB University; terbit 6 Mei 2026).
   Bukti kecocokan dengan repo: data **hipotetik** kelembaban tanah sistem smart farming **cabai**, empat fault **drift, bias, spike, malfunction**, **lima skenario** deteksi, ANN-LM. Semua ini sama dengan notebook (S1..S5, `simulate_drift_fault`, "hardware malfunction").
   Ketidakcocokan: abstrak (Crossref dan Garuda) **tidak menyebut fuzzy entropy / EDM-Fuzzy / multiscale**; venue-nya JITK, bukan "ACS". Jadi paper ini kemungkinan **paper pendamping/sebelumnya dari kelompok yang sama**, bukan paper ACS yang dirujuk builder.
   → Tindakan: tanyakan ke mahasiswa judul/DOI paper ACS; bila belum terbit, sitasi sebagai "accepted/in press" dengan judul persisnya. Sementara itu Wardhani dkk. (2026) aman disitasi untuk ANN-LM di konteks yang sama.

3. Tidak ditemukan karya lain (Indonesia atau luar) yang menggabungkan *multiscale fuzzy entropy* + *ANN-LM* + *sensor kelembaban tanah*. Karya MSE terdekat pada data kelembaban tanah hanya analisis skala/regionalisasi (mis. SMOSMANIA, Sun & Niu 2019 *Entropy* 21(6):548), bukan deteksi fault.

---

## (b) Prior-art dan posisi kebaruan

Pertanyaan: apakah sudah ada yang memakai JSD / KL / jarak berbasis distribusi sebagai **ukuran kemiripan antar vektor embedding di dalam** SampEn / FuzzyEn / ApEn / MSE?

**Jawaban singkat:** tidak ditemukan. Yang sudah ada terbagi tiga keluarga, dan tidak satu pun sama dengan JSD-Fuzzy:
1. **Ganti metrik jarak antar vektor** di dalam SampEn/FuzzyEn — tetapi metriknya geometris (Euclidean, kosinus, range, sigmoid), bukan divergensi distribusi.
2. **Entropi dari distribusi jarak** (DistEn, DSEn, diversity entropy) — distribusi dibentuk dari *kumpulan jarak*, lalu dihitung Shannon entropy-nya; vektor tidak diubah jadi distribusi, dan tidak ada divergensi antar pasangan vektor.
3. **JSD pada pola ordinal** (kompleksitas statistik Rosso, PJSD, iraversibilitas) — JSD dihitung antara **distribusi pola permutasi seluruh sinyal** dan distribusi acuan (uniform / sinyal lain / sinyal terbalik), bukan antara dua vektor embedding di dalam penghitungan pencocokan template.

| Karya | Apa yang mirip | Beda dengan JSD-Fuzzy | URL/DOI |
|---|---|---|---|
| Zhou dkk. 2021, EDM-Fuzzy (IEEE TII) | Multiscale fuzzy entropy, ganti Chebyshev → Euclidean, kemiripan kontinu; dipakai untuk diagnosis fault | Jarak geometris antar vektor mentah; tidak ada konversi ke distribusi atau divergensi. Ini **baseline** kita. | https://doi.org/10.1109/TII.2020.3009139 |
| Chen dkk. 2007, FuzzyEn (IEEE TNSRE) | Keanggotaan fuzzy eksponensial, vektor dipusatkan (baseline dibuang) | Jarak Chebyshev (maks. selisih elemen) | https://doi.org/10.1109/TNSRE.2007.897025 |
| Xie dkk. 2008, NS-SampEn (Phys. Lett. A) | Ganti fungsi Heaviside dengan fungsi kemiripan nonlinier (sigmoid) | Mengubah fungsi kemiripan, jarak tetap Chebyshev; tidak berbasis distribusi | https://doi.org/10.1016/j.physleta.2008.10.049 |
| Chanwimalueang & Mandic 2017, Cosine Similarity Entropy (Entropy) | Ganti jarak antar vektor embedding di SampEn dengan **jarak sudut**; ada versi multiskala (MCSE) | Kemiripan geometris (sudut), bukan divergensi distribusi; hasil akhir Shannon entropy dari proporsi kecocokan | https://doi.org/10.3390/e19120652 |
| Omidvarnia dkk. 2018, Range Entropy (Entropy) | Ganti Chebyshev dengan jarak "range" ternormalisasi di ApEn/SampEn | Jarak berbasis rentang amplitudo, bukan distribusi | https://doi.org/10.3390/e20120962 |
| Li dkk. 2015, Distribution Entropy / DistEn (MBEC) | "Distribusi" dan "probability density" dari vektor ke vektor | Mengestimasi PDF dari **semua jarak (Chebyshev) antar vektor**, lalu Shannon entropy PDF itu. Tidak ada divergensi antar dua vektor; tidak ada fungsi keanggotaan fuzzy | https://doi.org/10.1007/s11517-014-1216-0 |
| Wang dkk. 2021, Multiscale Diversity Entropy (IEEE TII) | Multiskala, diagnosis fault, "distribution of cosine similarity between adjacent orbits" | Distribusi dari nilai kosinus antar orbit **bertetangga**, lalu entropi; tidak ada JSD | https://doi.org/10.1109/TII.2020.3022369 |
| Shi 2025, Improved Multi-scale Divergence Entropy (Front. Mech. Eng.) | Namanya "divergence entropy", multiskala, fault mesin | Isinya kemiripan kosinus antar trajektori bertetangga + histogram; **tidak memakai KL/JS divergence** (dicek di teks lengkap) | https://doi.org/10.3389/fmech.2025.1690084 |
| Wang dkk. 2025, Distance Similarity Entropy / DSEn (RESS) | Kernel Gauss untuk kemiripan vektor, estimasi densitas, fault bearing | Kernel Gauss diterapkan pada **jarak elemen-per-elemen** antar vektor bertetangga, lalu densitas kemiripan; tidak ada konversi vektor → histogram dan tidak ada JSD | https://doi.org/10.1016/j.ress.2024.110643 |
| Rostaghi & Azami 2016, Dispersion Entropy (IEEE SPL) | Memetakan vektor ke pola diskret lalu Shannon entropy | Vektor dipetakan ke **pola kelas** (simbol), bukan histogram; tidak ada divergensi antar vektor | https://doi.org/10.1109/LSP.2016.2542881 |
| Bandt & Pompe 2002, Permutation Entropy (PRL) | Vektor embedding → objek diskret (pola ordinal) | Shannon entropy distribusi pola; tanpa jarak antar vektor | https://doi.org/10.1103/PhysRevLett.88.174102 |
| Lamberti dkk. 2004 (Physica A); Rosso dkk. 2007 (PRL) — kompleksitas statistik Jensen–Shannon | **JSD dipakai** pada analisis kompleksitas deret waktu | JSD antara distribusi pola ordinal **seluruh sinyal** dan distribusi **uniform** (disequilibrium); bukan kemiripan antar vektor di dalam FuzzyEn | https://doi.org/10.1016/j.physa.2003.11.005 ; https://doi.org/10.1103/PhysRevLett.99.154102 |
| Li, Shang & Zhang 2019 (Nonlinear Dyn.); Zunino dkk. 2022, PJSD (PRE) | JSD antar distribusi pola permutasi | JSD antara distribusi pola dua sinyal / sinyal maju vs terbalik (iraversibilitas); level sinyal, bukan level pasangan vektor | https://doi.org/10.1007/s11071-019-04950-6 ; https://doi.org/10.1103/PhysRevE.105.045310 |
| Fan, Song & Li 2024, Multiscale PJSD (FNL) | **Multiskala + JSD** | Sama seperti PJSD (pola ordinal, level sinyal), tujuan iraversibilitas; tidak ada fuzzy entropy | https://doi.org/10.1142/S0219477525500130 |
| Yao dkk. 2026, Fractional Reverse PJSD + multiskala (ICCEA, IEEE) | **Paling dekat secara aplikasi: JSD + multiskala + diagnosis fault bearing** | JSD (fraksional) antara distribusi pola permutasi dan **distribusi acuan uniform**; tidak ada pencocokan template antar vektor, tidak ada fungsi keanggotaan fuzzy | https://doi.org/10.1109/ICCEA69962.2026.11743731 |
| Yao, Yao & Wang 2021 (Chaos Solitons Fractals) | **KL divergence** di ruang keadaan rekonstruksi | KL dipakai sebagai "kernel probabilistic difference" atas fluktuasi **jarak vektor terhadap toleransi** untuk analisis non-ekuilibrium; bukan kemiripan antar vektor di FuzzyEn (detail rumus hanya dari abstrak) | https://doi.org/10.1016/j.chaos.2021.111568 |

**Posisi kebaruan.** Kombinasi yang belum ditemukan dalam literatur yang ditelusuri: (i) setiap vektor embedding (terpusat) diubah menjadi distribusi probabilitas kecil (histogram lunak berkernel Gauss), (ii) **jarak Jensen–Shannon** antar dua distribusi itu dipakai sebagai d di dalam fungsi keanggotaan fuzzy, (iii) di dalam kerangka multiscale (komposit) fuzzy entropy, (iv) untuk deteksi fault sensor kelembaban tanah dengan ANN-LM. Unsur (iii)–(iv) sendiri tidak baru (Zhou 2021; Wardhani 2026); JSD pada deret waktu juga tidak baru (Rosso 2007, PJSD). Yang baru adalah **tempat** JSD dipasang: sebagai ukuran kemiripan pasangan template di dalam FuzzyEn.

**Catatan jujur yang harus disiapkan untuk penguji (bukan dari literatur, dari kode):**
- Histogram lunak dari vektor terpusat **tidak peka urutan** elemen: untuk m = 2, vektor terpusat selalu berbentuk [+d/2, −d/2] atau [−d/2, +d/2]; keduanya menghasilkan histogram identik, sehingga JSD = 0 untuk pola naik vs turun dengan besar sama. JSD-Fuzzy mengukur kemiripan **sebaran amplitudo dalam template**, bukan bentuk/urutan template. Ini beda konseptual dengan SampEn/FuzzyEn klasik dan perlu disebut sebagai sifat (atau keterbatasan) metode.
- Jarak yang dipakai adalah **akar JSD** (jarak Jensen–Shannon), yang merupakan metrik (Endres & Schindelin 2003); sitasi ini sebaiknya ditambahkan karena "JSD" mentah bukan metrik.
- Penelusuran literatur tidak pernah bisa membuktikan "belum pernah ada"; klaim harus diberi batas "sepanjang penelusuran penulis".

---

## (c) Rumusan klaim kebaruan yang aman

**Bahasa Indonesia:**
> Sepanjang penelusuran literatur penulis (Crossref, OpenAlex, Google/Semantic Scholar, per 20 November 2026), belum ditemukan studi yang menggunakan Jensen–Shannon divergence sebagai ukuran ketidakmiripan antar vektor embedding di dalam fungsi keanggotaan *multiscale fuzzy entropy*. Penelitian ini mengusulkan JSD-Fuzzy, yaitu modifikasi EDM-Fuzzy (Zhou dkk., 2021) yang mengubah setiap vektor embedding menjadi distribusi probabilitas (histogram lunak berkernel Gauss) lalu mengganti jarak Euclidean dengan jarak Jensen–Shannon, dan mengevaluasinya untuk deteksi fault sensor kelembaban tanah dengan ANN Levenberg–Marquardt. Jensen–Shannon divergence sendiri telah dipakai dalam analisis kompleksitas deret waktu berbasis pola ordinal (Rosso dkk., 2007; Zunino dkk., 2022); kebaruan penelitian ini terletak pada penempatannya sebagai ukuran kemiripan template di dalam fuzzy entropy, bukan pada JSD atau fuzzy entropy itu sendiri.

**English:**
> To the best of our knowledge (literature search in Crossref, OpenAlex and Google/Semantic Scholar as of 20 November 2026), Jensen–Shannon divergence has not previously been used as the dissimilarity between embedding vectors inside the membership function of multiscale fuzzy entropy. We propose JSD-Fuzzy, a modification of EDM-Fuzzy (Zhou et al., 2021) that maps each embedding vector to a probability distribution (a Gaussian-kernel soft histogram) and replaces the Euclidean distance with the Jensen–Shannon distance, and we evaluate it for soil-moisture sensor fault detection with a Levenberg–Marquardt-trained ANN. Jensen–Shannon divergence has been used before in ordinal-pattern complexity analysis of time series (Rosso et al., 2007; Zunino et al., 2022); the novelty lies in its use as the template-similarity measure within fuzzy entropy, not in JSD or fuzzy entropy themselves.

Hindari kalimat: "pertama di dunia", "belum pernah ada yang memakai JSD untuk entropi", "metode entropi baru berbasis distribusi" (DistEn sudah ada), "EDM-Fuzzy dari paper kami".

---

## (d) Daftar sitasi per pilihan metode

Format: **pilihan metode → sitasi → kenapa (1 baris)**. Kunci BibTeX dalam kurung.

**Entropi**
- Sample entropy sebagai dasar pencocokan template → Richman & Moorman (2000), AJP-Heart 278(6):H2039–H2049, doi:10.1152/ajpheart.2000.278.6.H2039 (`richman2000sampen`) → definisi SampEn, rasio φ_m/φ_{m+1} tanpa self-match.
- (latar) Approximate entropy → Pincus (1991), PNAS 88(6):2297–2301, doi:10.1073/pnas.88.6.2297 (`pincus1991apen`) → pendahulu SampEn.
- Fuzzy entropy, keanggotaan kontinu, vektor dipusatkan → Chen dkk. (2007), IEEE TNSRE 15(2):266–272, doi:10.1109/TNSRE.2007.897025 (`chen2007fuzzyen`) → sumber ide keanggotaan fuzzy dan pengurangan baseline vektor.
- m = 2, r = 0,2·SD → Richman & Moorman (2000); Chen dkk. (2007) → nilai baku yang dipakai di kedua paper asal.
- Multiscale (coarse-graining, skala 1..15) → Costa dkk. (2002), PRL 89:068102, doi:10.1103/PhysRevLett.89.068102 (`costa2002mse`); Costa dkk. (2005), PRE 71:021906, doi:10.1103/PhysRevE.71.021906 (`costa2005mse`) → definisi MSE.
- Coarse-graining komposit (rerata atas k offset, `coarse_grain_cmse`) → Wu dkk. (2013), Entropy 15(3):1069–1084, doi:10.3390/e15031069 (`wu2013cmse`) → mengurangi varians estimasi pada skala besar/deret pendek.
- Baseline EDM-Fuzzy (Euclidean + fuzzy) → Zhou dkk. (2021), IEEE TII 17(6):4046–4054, doi:10.1109/TII.2020.3009139 (`zhou2021edmfuzzy`) → sumber asli baseline.

**Divergensi**
- Jensen–Shannon divergence → Lin (1991), IEEE TIT 37(1):145–151, doi:10.1109/18.61115 (`lin1991jsd`) → definisi JSD, simetris, terbatas (≤ ln 2), tak butuh kontinuitas absolut.
- Akar JSD sebagai metrik (yang dipakai kode: `np.sqrt(pairwise_jsd(...))`) → Endres & Schindelin (2003), IEEE TIT 49(7):1858–1860, doi:10.1109/TIT.2003.813506 (`endres2003jsdmetric`) → membenarkan √JSD sebagai jarak di fungsi keanggotaan.
- Alasan tidak memakai KL → Kullback & Leibler (1951), Ann. Math. Stat. 22(1):79–86, doi:10.1214/aoms/1177729694 (`kullback1951kl`) → KL asimetris dan tak terbatas bila ada bin nol.

**Levenberg–Marquardt**
- Algoritma LM → Levenberg (1944), QAM 2(2):164–168, doi:10.1090/qam/10666 (`levenberg1944`); Marquardt (1963), SIAM J. 11(2):431–441, doi:10.1137/0111030 (`marquardt1963`) → asal langkah Gauss–Newton terdamping.
- LM untuk melatih jaringan saraf feedforward (Jacobian per bobot) → Hagan & Menhaj (1994), IEEE TNN 5(6):989–993, doi:10.1109/72.329697 (`hagan1994lm`) → acuan baku LM pada ANN.
- Pembaruan damping gaya Nielsen (rasio gain ρ, μ·max{1/3, 1−(2ρ−1)³}, ν = 2) → Madsen, Nielsen & Tingleff (2004), *Methods for Non-Linear Least Squares Problems*, 2nd ed., IMM-DTU (`madsen2004nlls`) → sumber primer terverifikasi (rekaman IMM-DTU) untuk LM dengan update μ berbasis rasio gain; keberadaan rekaman sudah dicek, tetapi isi PDF tidak dibuka, jadi cocokkan nomor persamaan sebelum dikutip. Laporan Nielsen 1999 lihat bagian (e).
- ANN-LM untuk fault sensor kelembaban tanah (konteks yang sama) → Wardhani dkk. (2026), JITK 11(4):1053–1060, doi:10.33480/jitk.v11i4.7481 (`wardhani2026annlm`) → studi pendahulu di domain dan skenario yang sama.

**Validasi dan uji statistik**
- StratifiedGroupKFold (implementasi) → Pedregosa dkk. (2011), JMLR 12:2825–2830 (`pedregosa2011sklearn`) → pustaka yang dipakai.
- Grup = blok waktu (menghindari kebocoran autokorelasi) → Roberts dkk. (2017), Ecography 40(8):913–929, doi:10.1111/ecog.02881 (`roberts2017cv`); Bergmeir & Benítez (2012), Inf. Sci. 191:192–213, doi:10.1016/j.ins.2011.12.028 (`bergmeir2012tscv`) → CV acak pada data berstruktur temporal terlalu optimistis; blocking dianjurkan.
- Uji t berpasangan pada lipatan CV bisa bias (fold tidak independen) → Dietterich (1998), Neural Comput. 10(7):1895–1923, doi:10.1162/089976698300017197 (`dietterich1998tests`) → peringatan Type I error tinggi.
- Koreksi varians (corrected resampled t-test) → Nadeau & Bengio (2003), Mach. Learn. 52(3):239–281, doi:10.1023/A:1024068626366 (`nadeau2003inference`) → koreksi tumpang-tindih data latih.
- Replikabilitas uji → Bouckaert & Frank (2004), PAKDD, LNCS 3056:3–12, doi:10.1007/978-3-540-24775-3_3 (`bouckaert2004replicability`) → hasil uji t berulang-CV tidak stabil tanpa koreksi.
- Uji Wilcoxon berpasangan atas 15 sel → Wilcoxon (1945), Biometrics Bull. 1(6):80–83, doi:10.2307/3001968 (`wilcoxon1945`); Demšar (2006), JMLR 7:1–30 (`demsar2006comparisons`) → Wilcoxon dianjurkan untuk membandingkan dua metode atas banyak dataset/sel.
- Koreksi perbandingan ganda → Holm (1979), Scand. J. Stat. 6(2):65–70, doi:10.2307/4615733 (`holm1979`) → kendali FWER bertahap.
- Ukuran efek (Cohen's d / d_z) → Cohen (1988), *Statistical Power Analysis for the Behavioral Sciences*, 2nd ed., Erlbaum, doi:10.4324/9780203771587 (`cohen1988power`) → ambang kecil/sedang/besar.
- Interval kepercayaan bootstrap → Efron & Tibshirani (1993), *An Introduction to the Bootstrap*, Chapman & Hall, doi:10.1201/9780429246593 (`efron1993bootstrap`) → CI tanpa asumsi normal.

**Fault sensor (taksonomi dan deteksi)**
- Taksonomi fault (drift, offset/bias, spike, stuck-at, noise) → Ni dkk. (2009), ACM TOSN 5(3):25, doi:10.1145/1525856.1525863 (`ni2009faulttypes`) → definisi jenis fault yang disimulasikan.
- Prevalensi fault nyata (SHORT/spike, CONSTANT/stuck, NOISE) → Sharma, Golubchik & Govindan (2010), ACM TOSN 6(3):23, doi:10.1145/1754414.1754419 (`sharma2010sensorfaults`) → fault memang muncul pada dataset lapangan.
- Survei strategi deteksi fault WSN → Muhammed & Shaikh (2017), JNCA 78:267–287, doi:10.1016/j.jnca.2016.10.019 (`muhammed2017fdwsn`) → posisi metode berbasis fitur + klasifier.
- Klasifier ML untuk fault WSN → Zidi dkk. (2018), IEEE Sens. J. 18(1):340–347, doi:10.1109/JSEN.2017.2771226 (`zidi2018svm`) → contoh deteksi fault berbasis pembelajaran.
- QC data kelembaban tanah in situ (spike, break, saturasi) → Dorigo dkk. (2013), Vadose Zone J. 12(3), doi:10.2136/vzj2012.0097 (`dorigo2013ismnqc`) → fault/anomali nyata pada jaringan sensor kelembaban tanah global.
- Fault sensor kelembaban tanah pertanian → Ludeña-Choez dkk. (2019), Comput. Electron. Agric. 161:214–224, doi:10.1016/j.compag.2018.06.033 (`ludena2019nmf`); Jihani dkk. (2023), Results Eng. 20:101395, doi:10.1016/j.rineng.2023.101395 (`jihani2023kalman`); Deforce dkk. (2023), arXiv:2305.05495 (`deforce2023rogue`) → pembanding domain irigasi/pertanian.

**Laju drift 0,003–0,17 poin/hari:** **tidak ditemukan sumber** (lihat (e)).

---

## (e) Not verified

| Item | Status | Saran |
|---|---|---|
| Paper "ACS" mahasiswa (EDM-Fuzzy + ANN-LM, sensor kelembaban tanah) | Tidak ditemukan di Crossref/OpenAlex/web. Kandidat terdekat: Wardhani dkk. 2026 JITK (tanpa EDM-Fuzzy di abstrak). | Minta judul/DOI dari mahasiswa; jangan sitasi sebelum ada. |
| Nielsen, H. B. (1999). *Damping parameter in Marquardt's method.* Tech. Rep. IMM-REP-1999-05, IMM-DTU. | Hanya terlihat sebagai sitasi sekunder (catatan levmar UTK, komentar kode GSL `nielsen.c`); rekaman primer di pubdb IMM / DTU Orbit tidak ditemukan. | Sitasi Madsen, Nielsen & Tingleff (2004) (terverifikasi) untuk aturan update μ; bila tetap ingin Nielsen 1999, tulis sesuai sitasi sekunder dan tandai. |
| Sumber laju drift sensor kelembaban tanah nyata "0,003–0,17 poin/hari" (komentar di `build_11_annlm_real_notebook.py` baris 253–254) | Tidak ada sumber yang memberi laju drift per hari/tahun. Yang ada: drift diakui tapi tidak dikuantifikasi (mis. jaringan LoRaWAN kapasitif 90 hari "no significant drift"; review kalibrasi FDR 2026 hanya menyebut drift linear karena penuaan/korosi tanpa laju). | Jangan klaim angka itu sebagai "drift nyata" tanpa sumber; ubah jadi "asumsi skenario" atau turunkan dari data lapangan sendiri. |
| Cohen (1988) dan Efron & Tibshirani (1993) | Terverifikasi, tetapi DOI yang ada merujuk terbitan ulang (Routledge 2013; Chapman & Hall/CRC). | Aman; tahun edisi asli tetap 1988/1993. |
| Yao, Yao & Wang (2021) — detail rumus KL | Bibliografi terverifikasi; cara pakai KL hanya dari abstrak. | Cukup untuk tabel prior-art; jangan kutip rumus. |

---

## (f) Search log (2026-11-20)

Mesin: WebSearch (standard/extended), Crossref REST API (`/works/<DOI>` dan `query.bibliographic`), OpenAlex API, Europe PMC API, halaman penerbit (JMLR, IMM-DTU pubdb, arXiv, Frontiers, RePEc, Garuda). Semantic Scholar API: HTTP 429 (rate-limit), tidak dipakai. Halaman JITK (nusamandiri) HTTP 403; data diambil dari Garuda dan Crossref.

Query web:
1. `multiscale fuzzy entropy Levenberg-Marquardt neural network soil moisture sensor fault detection`
2. `Jensen-Shannon divergence fuzzy entropy similarity embedding vectors`
3. `"sample entropy" OR "fuzzy entropy" "Kullback-Leibler" divergence instead of Chebyshev distance embedding vectors similarity`
4. `EDM fuzzy entropy Euclidean distance membership fuzzy entropy sensor fault Indonesia` → menemukan Zhou dkk. 2021
5. `fuzzy entropy different distance metrics Euclidean Chebyshev cosine comparison FuzzyEn`
6. `"EDM-Fuzzy" soil moisture sensor fault detection neural network`
7. `distance similarity entropy rolling bearing 2024 Reliability Engineering System Safety ...` → DSEn
8. `entropy measure embedding vectors probability distribution divergence similarity "Jensen-Shannon" sample entropy variant time series complexity` → PJSD, CSE, kompleksitas JS
9. `"Jensen-Shannon" "fuzzy entropy" time series fault diagnosis multiscale` → nihil
10. `Wasserstein OR Kullback-Leibler distance between state vectors entropy measure time series complexity "phase space" distribution` → Yao 2021
11. `soil moisture sensor fault detection entropy features artificial neural network Levenberg Marquardt Indonesia smart irrigation drift spike bias`
12. `"multiscale fuzzy entropy" "soil moisture" sensor fault` → nihil
13. `deteksi fault sensor kelembaban tanah multiscale fuzzy entropy jaringan saraf tiruan Levenberg-Marquardt` → Wardhani dkk. 2026
14. `Wardhani Buono Wahjuni Syukur fuzzy entropy soil moisture sensor fault chili` → nihil
15. `"Applied Computer Science" journal soil moisture sensor fault detection entropy neural network Levenberg-Marquardt 2025` → nihil (selain Wardhani 2026)
16. `"multiscale" "Jensen-Shannon divergence" entropy time series fault diagnosis feature` → Shi 2025 (ternyata kosinus)
17. `sample entropy variant histogram of each embedding vector compared distributions ... Entropy MDPI` → nihil
18. `modified fuzzy entropy similarity measure replaced "Hellinger" OR "Bhattacharyya" OR "Kullback" OR "Jensen" between embedded vectors entropy time series` → nihil (hanya NS-SampEn, FuzzyMEn)
19. `Nielsen "Damping parameter in Marquardt's method" IMM-REP-1999-05 DTU` → hanya sitasi sekunder
20. `site:imm.dtu.dk Nielsen damping parameter Marquardt 1999` → nihil
21. `Madsen Nielsen Tingleff 2004 "Methods for non-linear least squares problems" pdf imm.dtu.dk` → pubdb 3215
22. `soil moisture sensor long-term drift rate per day capacitance sensor calibration drift study` → tidak ada laju
23. `in situ soil moisture sensor drift "per year" OR "per day" volumetric water content drift quantified long-term field TDR capacitance` → tidak ada laju

Query Crossref (`query.bibliographic`): `Wardhani Buono Wahjuni fuzzy entropy`; `Wardhani Buono Wahjuni soil moisture sensor`; `Luh Kesuma Wardhani fault`; `EDM-Fuzzy soil moisture sensor fault`; `multiscale fuzzy entropy Levenberg-Marquardt sensor fault smart farming`; `Euclidean distance multiscale fuzzy entropy soil moisture`; `Lamberti Intensive entropic non-triviality measure`; `Permutation Jensen-Shannon distance ...`; `Sensor nodes fault detection ... NMF`; `Kalman filter based sensor fault detection ... smart irrigation`; `Deforce Baesens Diels Serral rogue soil moisture`; `Holm sequentially rejective`; `Fan Song Li multiscale irreversibility PJSD` (→ juga menemukan Yao dkk. 2026 ICCEA).

Verifikasi DOI langsung (Crossref `/works/`): 45 DOI, semua OK kecuali satu DOI Lamberti yang salah ketik (diganti 10.1016/j.physa.2003.11.005 hasil pencarian judul) dan DOI arXiv (DataCite, dicek di arxiv.org). Holm 1979, Madsen 2004, Pedregosa 2011 dicek via OpenAlex + halaman JSTOR/IMM/JMLR. Demšar 2006 via jmlr.org.

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Ringkasan audit klaim</b> (`sidang/qa/01_ringkasan_klaim.md`)</summary>

# Ringkasan audit klaim → bukti (sidang)

## Status sesudah koreksi (2026-11-20, branch `sidang-prep`)

Semua temuan di bawah sudah diperbaiki di dokumen sumbernya oleh
`sidang/qa/perbaiki_klaim_2026_11_20.py` (setiap penggantian menyebut id ledger).

- Output Kaggle (CSV + log, tanpa gambar) disalin ke `sidang/bukti_kaggle/<slug>/`,
  jadi 83 klaim yang dulu "TIDAK ADA BUKTI LOKAL" kini dicek dari repo.
- Notebook demo dijalankan ulang tanpa layar (`sidang/qa/jalankan_notebook.py`);
  output dan waktu tiap sel ada di `sidang/bukti_demo/`. Klaim demo P-04–P-06
  kini dihitung dari file itu. Klaim "5 dari 8" ternyata salah hitung (6 dari 8).
- Tiga angka arsip label kembar (R-96, R-98a/b) dihapus dari README karena
  notebook arsipnya tidak punya output (status DIHAPUS).
- Log NB15 di `hasil15/` sekarang run v2 (wall 34 menit), README disesuaikan.

Hasil `./venv/bin/python sidang/qa/cek_klaim.py` (kode keluar 0):

```
Baris ledger: 476 | dihitung ulang otomatis: 466 | tanpa fungsi hitung (diperiksa manual): 10
Status hitung ulang: COCOK=466
Status ledger: COCOK=473, DIHAPUS=3
GERBANG KLAIM: LULUS
```

Bagian di bawah ini adalah temuan audit **sebelum** koreksi, disimpan sebagai jejak.

---

Tanggal: 2026-11-20. Ledger lengkap: `sidang/qa/01_klaim_bukti.csv`. Skrip hitung ulang: `sidang/qa/cek_klaim.py`.
Bukti diambil dari bahan mentah (CSV ekspor, log kernel JSON, builder, `data_sensor.csv`), bukan dari ringkasan.
Uji statistik dihitung ulang dengan `scipy.stats.ttest_rel(b, a)` dan `scipy.stats.wilcoxon(b, a)` dari F1 per sel, sama seperti builder (scipy 1.18.1).

## Hitungan

Ledger memuat **476 klaim** dari README.md, Bahan_Sidang.md, sidang/MAPPING.md, sidang/PANDUAN_DEMO.md, hasil04/hasil11 Ringkasan, dan draf laporan notebook 13 dan 14.

| Status | Jumlah |
|---|---|
| COCOK | 349 |
| PEMBULATAN | 10 |
| TIDAK COCOK | 26 |
| TIDAK ADA BUKTI LOKAL | 91 (83 di antaranya cocok dengan unduhan output Kaggle; 8 tidak punya bukti sama sekali) |

Sebanyak 463 klaim dihitung ulang otomatis oleh skrip. Sisanya 13 diperiksa manual, misalnya klaim berupa teks atau klaim tanpa file bukti.
Klaim notebook 01–10 berstatus TIDAK ADA BUKTI LOKAL karena angka terbarunya hanya ada di output Kaggle. Folder `hasil04`, `hasil07`–`hasil10` di repo masih berisi run lama sebelum koreksi drift (hasil04 masih v5).
Kolom `catatan` di ledger menyebut apakah Kaggle mengonfirmasi angkanya.

## Klaim TIDAK COCOK (nilai benar → file yang perlu diperbaiki)

1. **NB10 "JSD unggul 10 dari 15 sel"**. Angka yang benar adalah **11/15**: N200 S2, S4, S5; seluruh N700; N1000 S1, S4, S5. Perbaiki README.md:22, README.md:60, dan sidang/MAPPING.md:59. Output Kaggle v4 dan hasil10 lokal sama-sama memberi 11.
2. **Alasan p NB10 berbeda dari NB11: "protokol CV dan anggaran komputasi lebih besar"**. Klaim ini ada di README.md:64–66 dan MAPPING.md:59. Builder 10 dan 11 memakai protokol yang identik (LM_ITERS_SEARCH 6, LM_ITERS_REFIT 40, LM_INNER_CV 2, anggaran 40/150 s, grid dan seed sama). Yang berbeda hanya `KAGGLE_TIME_BUDGET_H`, 10,5 jam vs 11,5 jam. Lihat bagian kontradiksi di bawah.
3. **README.md:18 (NB06), "ANN-LM 0,394"**. Angka 0,394 berasal dari notebook 01 yang memakai sklearn `MLPClassifier`, bukan LM (MAPPING Tabel C). Labelnya harus diganti.
4. **README.md:18, "terbaik di 5 skenario"**. JSD+Time hanya terbaik di 3 skenario (S1, S2, S5); fitur Time menang di S3 dan S4. Selain itu, 0,443, 0,608, dan 0,254 adalah nilai **S5 saja**, padahal README tidak menyebut S5.
5. **README.md:26, "Hasil di `hasil15/`"**. File 15_* ada di `hasil16/exports/`, sedangkan `hasil15/` hanya berisi log.
6. **Bahan_Sidang.md:65, "semuanya 16–21 Des 2025"**. Satu sampel nol jatuh pada 2025-12-08 14:06.
7. **Bahan_Sidang.md:66, "plus 49 sampel pendek"**. Angka yang benar 50: 49 sampel di 16–21 Des ditambah 1 sampel tanggal 8 Des. Total 11.371 benar.
8. **Bahan_Sidang.md:176, "kisi (32,32), (64,16), (64,)"**. `HIDDEN_GRID` berisi **8** arsitektur: (16,), (32,), (64,), (16,8), (32,8), (32,16), (32,32), (64,16).
9. **hasil11/Ringkasan_untuk_laporan.md:26–27, rata-rata CV reduction −26% / −28% / −31%**. Nilai yang benar **−19,1% / −16,5% / −16,4%**, sesuai 11_rq2_cv_reduction.csv dan log NB11 baris 780–782.
10. **hasil11/Ringkasan_untuk_laporan.md:61–62 (RQ3)**. Klaimnya: JSD lebih terpisah di 6 sel, EDM di 4 sel (S2/S4/S5-N200, S1-N1000). Data menunjukkan JSD lebih terpisah di **11 sel**, EDM di **0 sel**, dan 4 sel campuran (N200 S1, S2, S5; N1000 S1).
11. **hasil14/14_draf_laporan.md:29–30, "jumlah sampel per kelas tetap sama dengan notebook 11 (1.700 / 1.621 / 1.526)"**. Angka itu milik NB14. NB11 punya 1.700 / **1.675 / 1.651**, jadi hanya N=200 yang sama.
12. **hasil14/14_draf_laporan.md:88–89, kolom NB11 dan NB14 tertukar**. Nilai yang benar:
    - NB11: fit 93,2 / 98,0 s dan inferensi 0,097 / 0,087 ms.
    - NB14: fit 75,5 / 75,1 s dan inferensi 0,089 / 0,088 ms.

    Kesimpulan di baris 94–95 ("Fit ANN-LM per fold lebih lama") karenanya terbalik, karena NB14 justru lebih cepat. Tabel ini mencakup 8 baris ledger: H14-43a–d, H14-44a–d, dengan dua di antaranya berstatus PEMBULATAN. Kesimpulannya tercatat sebagai H14-45e.
13. **hasil14/14_draf_laporan.md:92, "1,7–6 kali"**. Rentang yang benar **1,7–6,8 kali** (NB11 N=1000: 1.627,7 / 240,7 s).

## PEMBULATAN (arah atau batas pembulatan meleset sedikit)

| Klaim | Nilai | Catatan |
|---|---|---|
| README:24 dan draf 13:62, EDM spike "0,76–0,82" | maks 0,8145 | seharusnya 0,81 |
| README:24, bias episode "AUC ≤ 0,74" | 0,7426 | |
| draf 13:61, JSD spike relatif "0,87" | 0,8649 | |
| draf 13:86, drift+bias JSD "0,67" | 0,6649 | |
| hasil11 Ringkasan:34, "N=1000 +0,030" | 0,0295 | |
| draf 14:101, "+0,077" | 0,07648 | 3 desimal = 0,076 |
| README:34, "300–19.000×" | 339–19.200× | |

Satu klaim lain hanya lolos setelah dibulatkan tetapi buktinya hanya ada di Kaggle, sehingga berstatus TIDAK ADA BUKTI LOKAL: README:21 "vs Adam p ≤ 0,025", dengan p LM–Adam = 0,0252.

## Klaim tanpa bukti sama sekali

- README.md:96–99: arsip label kembar (F1 0,925, prevalensi 0,882, AUC 0,50). Notebook arsip tidak punya output.
- README.md:26: "Kaggle v1 COMPLETE" (NB15). Log lokal v1 memang selesai (1804 s), tetapi kernel kini sedang menjalankan versi baru dan unduhannya kosong.
- PANDUAN_DEMO.md:42: "2,5 menit termasuk menyalakan mesin". Log hanya mencatat waktu kernel sekitar 2,1 menit.
- PANDUAN_DEMO.md:43: "64–90 detik di laptop".
- PANDUAN_DEMO.md:60 dan 136: "80 %".
- PANDUAN_DEMO.md:106: "5 dari 8 tebakan".

Notebook demo tidak menyimpan output, dan tabelnya tidak ikut tercetak di stdout log.

## Kontradiksi antardokumen

- **Mengapa p NB10 (0,052) berbeda dari NB11 (0,076).** Menurut README.md:64–66, penyebabnya protokol CV dan anggaran komputasi NB11 lebih besar. Bukti tidak mendukung bagian protokol karena konstanta protokol di kedua builder identik. Hanya batas sesi `KAGGLE_TIME_BUDGET_H` yang berbeda.
  - Dari 15 sel, 8 punya F1 identik di kedua notebook dan 7 berbeda: N200 S2, S3, S4; N700 S2, S4; N1000 S2, S4 (`_sel_beda_10_11` di skrip). Perbedaannya terkumpul di sel S2/S4, yaitu sel yang pelatihannya berhenti karena `budget(MAX_TRAIN_S)`. Karena batas itu diukur dengan jam dinding, hasilnya tidak deterministik antar-run.
  - Jadi perbedaan p berasal dari variasi run, bukan dari protokol yang lebih besar. Jumlah sel menang pun sama, 11/15, sehingga MAPPING:59 ("jumlah sel menang juga beda") keliru.
- **hasil04/Ringkasan_untuk_laporan.md vs README dan MAPPING.** hasil04 masih memuat angka v5 (17 kelas 0,860/0,899; 4 fault semua-4-benar 0,880/0,980). README dan MAPPING memakai v6 (0,846/0,902; 0,800/0,920). Angka v5 cocok dengan ekspor lokal hasil04, tetapi sudah usang.
- **README NB06 vs MAPPING Tabel C.** README menyebut 0,394 sebagai "ANN-LM", sedangkan MAPPING menyatakan notebook 01 memakai sklearn MLP.
- **README NB06 vs MAPPING:149.** Angka README adalah S5 saja (0,608), sedangkan MAPPING memakai rata-rata 5 skenario (0,750). Keduanya benar untuk basisnya masing-masing, tetapi basis README tidak disebut.
- **Satuan 56/63.** README dan Bahan menyebut "pasangan", hasil11 Ringkasan menyebut "sel". Yang dihitung sebenarnya 63 baris kelas (N × skenario × kelas). Angkanya benar.
- **Draf NB14.** Tabel biaya (kolom tertukar) bertentangan dengan angka di CSV, dan kesimpulan di bawahnya mengikuti tabel yang salah.
- **PANDUAN_DEMO.md:152, "sekitar 26 hari garis lurus".** Angka ini benar untuk empat segmen panjang (26,3 hari). Kalau segmen lurus ≥ 6 jam ikut dihitung, totalnya 37,0 hari.

## Hasil `cek_klaim.py`

Perintah: `nice -n 19 ./venv/bin/python sidang/qa/cek_klaim.py --kaggle <scratchpad>/kaggle`. **Kode keluar: 1** karena ada 23 TIDAK COCOK dalam hitung ulang.
Tanpa `--kaggle`, 88 klaim berstatus LEWATI dan kode keluar tetap 1 (19 TIDAK COCOK).

Hitung ulang skrip membandingkan **angka** saja. Karena itu R06-02 (angka 0,394 benar, label "ANN-LM" salah) muncul COCOK di skrip tetapi TIDAK COCOK di ledger. Klaim teks atau manual (R15-H, R15-I, R-96, R-98a/b, P-03–P-06, H14-45e, R-D3, B-06b, P-09) tidak ikut dihitung skrip.

```
id      | klaim  | hitung ulang | selisih  | status      | catatan     
--------+--------+--------------+----------+-------------+-------------
R01-01  | 0,406  | 0.4061       | 0.00010  | COCOK       |             
R01-02  | 0,230  | 0.2303       | 0.00030  | COCOK       |             
R01-03  | 0,394  | 0.3938       | -0.00020 | COCOK       |             
R01-04  | 0,217  | 0.2174       | 0.00040  | COCOK       |             
R01-05  | 0,484  | 0.4844       | 0.00040  | COCOK       |             
R01-06  | 0,484  | 0.4838       | -0.00020 | COCOK       |             
R01-07  | 4      | 4            | 0        | COCOK       |             
R02-01  | 0,821  | 0.821        | 0        | COCOK       |             
R02-02  | 0,748  | 0.748        | 0        | COCOK       |             
R02-03  | 0,829  | 0.829        | 0        | COCOK       |             
R02-04  | 0,726  | 0.726        | 0        | COCOK       |             
R03-J1  | 0,819  | 0.819        | 0        | COCOK       |             
R03-E1  | 0,766  | 0.766        | 0        | COCOK       |             
R03-J2  | 0,668  | 0.668        | 0        | COCOK       |             
R03-E2  | 0,492  | 0.492        | 0        | COCOK       |             
R03-J3  | 0,805  | 0.805        | 0        | COCOK       |             
R03-E3  | 0,494  | 0.494        | 0        | COCOK       |             
R03-J4  | 0,814  | 0.814        | 0        | COCOK       |             
R03-E4  | 0,644  | 0.644        | 0        | COCOK       |             
R03-J5  | 1,000  | 1            | 0        | COCOK       |             
R03-E5  | 0,990  | 0.99         | 0        | COCOK       |             
R03-W   | 5      | 5            | 0        | COCOK       |             
R04-01  | 0,902  | 0.902        | 0        | COCOK       |             
R04-02  | 0,846  | 0.846        | 0        | COCOK       |             
R04-03  | 0,920  | 0.92         | 0        | COCOK       |             
R04-04  | 0,800  | 0.8          | 0        | COCOK       |             
R05-J1  | 0,448  | 0.448        | 0        | COCOK       |             
R05-E1  | 0,142  | 0.142        | 0        | COCOK       |             
R05-J2  | 0,868  | 0.868        | 0        | COCOK       |             
R05-E2  | 0,696  | 0.696        | 0        | COCOK       |             
R05-J3  | 0,601  | 0.601        | 0        | COCOK       |             
R05-E3  | 0,191  | 0.191        | 0        | COCOK       |             
R05-J4  | 0,917  | 0.917        | 0        | COCOK       |             
R05-E4  | 0,924  | 0.924        | 0        | COCOK       |             
R06-01  | 0,443  | 0.4428       | -0.00020 | COCOK       |             
R06-02  | 0,394  | 0.3938       | -0.00020 | COCOK       |             
R06-03  | 0,608  | 0.6076       | -0.00040 | COCOK       |             
R06-04  | 5      | 3            | -2       | TIDAK COCOK |             
R06-05  | 0,254  | 0.2542       | 0.00020  | COCOK       |             
R07-01  | 13     | 13           | 0        | COCOK       |             
R07-02  | 0,063  | 0.0632       | 0.00018  | COCOK       |             
R07-03  | 0,790  | 0.7905       | 0.00047  | COCOK       |             
R07-04  | 0,719  | 0.7193       | 0.00033  | COCOK       |             
R07-05  | 0,460  | 0.4604       | 0.00040  | COCOK       |             
R07-06  | 0,357  | 0.3572       | 0.00020  | COCOK       |             
R08-01  | 0,505  | 0.5049       | -0.00010 | COCOK       |             
R08-02  | 0,453  | 0.4526       | -0.00038 | COCOK       |             
R08-03  | 0,052  | 0.0523       | 0.00028  | COCOK       |             
R08-04  | 5      | 5            | 0        | COCOK       |             
R08-05  | 0,035  | 0.0354       | 0.00044  | COCOK       |             
R08-06  | 0,0625 | 0.0625       | 0        | COCOK       |             
R09-01  | 0,429  | 0.4287       | -0.00030 | COCOK       |             
R09-02  | 0,433  | 0.4333       | 0.00030  | COCOK       |             
R09-03  | 0,306  | 0.3057       | -0.00030 | COCOK       |             
R09-04  | 0,268  | 0.2675       | -0.00050 | COCOK       |             
R09-05  | 5      | 5            | 0        | COCOK       |             
R09-06  | 4      | 4            | 0        | COCOK       |             
R09-07  | 1      | 1            | 0        | COCOK       |             
R09-08  | 0      | 0            | 0        | COCOK       |             
R09-09  | 0,81   | 0.8135       | 0.0035   | COCOK       |             
R09-10  | 0,002  | 0.00030      | -0.0017  | COCOK       | mode=maks   
R09-11  | 0,025  | 0.0252       | 0.00020  | PEMBULATAN  | mode=maks   
R09-12  | 0      | 0            | 0        | COCOK       |             
R10-01  | 10     | 11           | 1        | TIDAK COCOK |             
R10-02  | 0,007  | 0.0067       | -0.00034 | COCOK       |             
R10-03  | 0,056  | 0.0555       | -0.00052 | PEMBULATAN  |             
R10-04  | 0,030  | 0.0303       | 0.00028  | COCOK       |             
R10-05  | 0,031  | 0.0308       | -0.00019 | COCOK       |             
R10-06  | 0,016  | 0.0161       | 0.00009  | COCOK       |             
R10-07  | 0,64   | 0.643        | 0.003    | COCOK       |             
R10-08  | 0,48   | 0.4834       | 0.0034   | COCOK       |             
R10-09  | 0,052  | 0.0519       | -0.00011 | COCOK       |             
R10-10  | 0,041  | 0.0413       | 0.00026  | COCOK       |             
R10-11  | 85784  | 85784.21     | 0.21     | COCOK       |             
R10-12  | 23,8   | 23.8289      | 0.0289   | COCOK       |             
R10-13  | 525    | 525.0333     | 0.0333   | COCOK       |             
R10-14  | 0      | 0            | 0        | COCOK       |             
R11-01  | 11     | 11           | 0        | COCOK       |             
R11-02  | 0,028  | 0.0281       | 0.00011  | COCOK       |             
R11-03  | 0,027  | 0.0271       | 0.00007  | COCOK       |             
R11-04  | 0,076  | 0.0759       | -0.00007 | COCOK       |             
R11-05  | 0,107  | 0.107        | -0.00001 | COCOK       |             
R11-06  | 0,675  | 0.6753       | 0.00030  | COCOK       |             
R11-07  | 0,634  | 0.6338       | -0.00017 | COCOK       |             
R11-08  | 9      | 9            | 0        | COCOK       |             
R11-09  | 0,307  | 0.3067       | -0.00027 | COCOK       |             
R11-10  | 0,295  | 0.2953       | 0.00033  | COCOK       |             
R11-11  | 56     | 56           | 0        | COCOK       |             
R11-12  | 89     | 88.8889      | -0.1111  | COCOK       |             
R11-13  | 610    | 610.2333     | 0.2333   | COCOK       |             
R11-14  | 99162  | 99162.45     | 0.45     | COCOK       |             
R11-15  | 27,5   | 27.5451      | 0.0451   | COCOK       |             
R11-16  | 0      | 0            | 0        | COCOK       |             
R11-17  | 0      | 0            | 0        | COCOK       |             
R11-18  | 0      | 0            | 0        | COCOK       |             
R13-01  | 0,3    | 0.3          | 0        | COCOK       |             
R13-02  | 0,3    | 0.3          | 0        | COCOK       |             
R13-03  | 0      | 0            | 0        | COCOK       |             
R13-04  | 0,50   | 0.516        | 0.016    | COCOK       | mode=sekitar
R13-05  | 0,74   | 0.7426       | 0.0026   | PEMBULATAN  | mode=maks   
R13-06  | 0,80   | 0.7978       | -0.0022  | COCOK       |             
R13-07  | 0,85   | 0.8467       | -0.0033  | COCOK       |             
R13-08  | 0,70   | 0.7038       | 0.0038   | COCOK       |             
R13-09  | 0,92   | 0.9222       | 0.0022   | COCOK       |             
R13-10  | 0,76   | 0.7593       | -0.00073 | COCOK       |             
R13-11  | 0,82   | 0.8145       | -0.0055  | PEMBULATAN  |             
R13-12  | 0      | 0            | 0        | COCOK       |             
R13-13  | 354    | 354          | 0        | COCOK       |             
R13-14  | 512    | 512.4949     | 0.4949   | COCOK       |             
R14-01  | 12     | 12           | 0        | COCOK       |             
R14-02  | 0,421  | 0.421        | -0.00005 | COCOK       |             
R14-03  | 0,497  | 0.4974       | 0.00043  | COCOK       |             
R14-04  | 0,0006 | 0.00062      | 0.00002  | COCOK       |             
R14-05  | 0,076  | 0.0759       | -0.00007 | COCOK       |             
R14-06  | 0,693  | 0.6925       | -0.00048 | COCOK       |             
R14-07  | 0,791  | 0.7905       | -0.00048 | COCOK       |             
R14-08  | 0,356  | 0.3563       | 0.00033  | COCOK       |             
R14-09  | 0,483  | 0.4831       | 0.00013  | COCOK       |             
R14-10  | 0,21   | 0.206        | -0.004   | COCOK       |             
R14-11  | 0,53   | 0.5324       | 0.0024   | COCOK       |             
R14-12  | 478    | 478.0167     | 0.0167   | COCOK       |             
R14-13  | 0      | 0            | 0        | COCOK       |             
R15-A1  | 0,90   | 0.897        | -0.003   | COCOK       |             
R15-B1  | 0,98   | 0.976        | -0.004   | COCOK       |             
R15-A2  | 0,85   | 0.853        | 0.003    | COCOK       |             
R15-B2  | 0,75   | 0.754        | 0.004    | COCOK       |             
R15-A3  | 0,83   | 0.83         | 0        | COCOK       |             
R15-B3  | 0,75   | 0.753        | 0.003    | COCOK       |             
R15-A4  | 0,88   | 0.881        | 0.001    | COCOK       |             
R15-B4  | 0,80   | 0.795        | -0.005   | COCOK       |             
R15-A5  | 0,96   | 0.958        | -0.002   | COCOK       |             
R15-B5  | 0,90   | 0.897        | -0.003   | COCOK       |             
R15-C1  | 0,49   | 0.4944       | 0.0044   | COCOK       |             
R15-C2  | 0,71   | 0.7094       | -0.00060 | COCOK       |             
R15-D   | 6      | 6.1625       | 0.1625   | COCOK       |             
R15-E   | 30     | 30.0667      | 0.0667   | COCOK       |             
R15-F   | 0      | 0            | 0        | COCOK       |             
R15-G   | 16     | 16           | 0        | COCOK       |             
R-ILU1  | 0      | 0            | 0        | COCOK       |             
R-ILU2  | 0      | 0            | 0        | COCOK       |             
R-ILU3  | 0      | 0            | 0        | COCOK       |             
R-ILU4a | 556    | 556          | 0        | COCOK       |             
R-ILU4b | 654    | 654          | 0        | COCOK       |             
R-ILU4c | 659    | 659          | 0        | COCOK       |             
R-ILU4d | 657    | 657          | 0        | COCOK       |             
R-ILU5a | 13     | 13           | 0        | COCOK       |             
R-ILU5b | 0      | 0            | 0        | COCOK       |             
R-ILU5c | 0      | 0            | 0        | COCOK       |             
R-ILU5d | 0      | 0            | 0        | COCOK       |             
R-D1    | 57,6   | 57.6         | 0        | COCOK       |             
R-D2    | 5635   | 5634.6       | -0.4     | COCOK       |             
R-D5    | 1,0    | 1            | 0        | COCOK       |             
R-D6    | 3      | 3            | 0        | COCOK       |             
R-60a   | 10     | 11           | 1        | TIDAK COCOK |             
R-60b   | 0,052  | 0.0519       | -0.00011 | COCOK       |             
R-61a   | 11     | 11           | 0        | COCOK       |             
R-61b   | 0,076  | 0.0759       | -0.00007 | COCOK       |             
R-64    | 1      | 0            | -1       | TIDAK COCOK | mode=min    
R-81a   | 281721 | 281721       | 0        | COCOK       |             
R-81b   | 28173  | 28173        | 0        | COCOK       |             
R-84    | 0,69   | 0.69         | 0        | COCOK       |             
R-84b   | 2,43   | 2.43         | 0        | COCOK       |             
R-84c   | 3,47   | 3.47         | 0        | COCOK       |             
R-85    | 0      | 0            | 0        | COCOK       |             
R-120   | 281721 | 281721       | 0        | COCOK       |             
R-121   | 0      | 0            | 0        | COCOK       |             
R-153a  | 6      | 6            | 0        | COCOK       |             
R-153b  | 2      | 2            | 0        | COCOK       |             
R-153c  | 40     | 40           | 0        | COCOK       |             
R-159a  | 10,5   | 10.5         | 0        | COCOK       |             
R-159b  | 11,5   | 11.5         | 0        | COCOK       |             
B-06    | 281721 | 281721       | 0        | COCOK       |             
B-07a   | 28173  | 28173        | 0        | COCOK       |             
B-07b   | 97,8   | 97.8         | 0        | COCOK       |             
B-08a   | 0      | 0            | 0        | COCOK       |             
B-08b   | 0      | 0            | 0        | COCOK       |             
B-08c   | 0      | 0            | 0        | COCOK       |             
B-09a   | 11371  | 11371        | 0        | COCOK       |             
B-09b   | 4,0    | 4.0363       | 0.0363   | COCOK       |             
B-09c   | 16     | 8            | -8       | TIDAK COCOK |             
B-09d   | 45     | 44.9917      | -0.0083  | COCOK       |             
B-09e   | 37     | 37.2333      | 0.2333   | COCOK       |             
B-09f   | 12     | 12.1167      | 0.1167   | COCOK       |             
B-09g   | 49     | 50           | 1        | TIDAK COCOK |             
B-09h   | 1135   | 1135         | 0        | COCOK       |             
B-09i   | 10     | 10           | 0        | COCOK       |             
B-09m   | 280    | 280          | 0        | COCOK       |             
B-09j   | 2      | 2            | 0        | COCOK       |             
B-09n   | 79     | 79           | 0        | COCOK       |             
B-09k   | 1      | 1            | 0        | COCOK       |             
B-09o   | 55     | 55           | 0        | COCOK       |             
B-15    | 0      | 0            | 0        | COCOK       |             
B-19a   | 2      | 2            | 0        | COCOK       |             
B-19b   | 5      | 5            | 0        | COCOK       |             
B-19c   | 7      | 7            | 0        | COCOK       |             
B-19d   | 5      | 5            | 0        | COCOK       |             
B-19e   | 2      | 2            | 0        | COCOK       |             
B-20    | 0,69   | 0.69         | 0        | COCOK       |             
B-20b   | 16,7   | 16.6667      | -0.0333  | COCOK       |             
B-22a   | 1992   | 1992         | 0        | COCOK       |             
B-22b   | 36120  | 36120        | 0        | COCOK       |             
B-27    | 8      | 8            | 0        | COCOK       |             
B-28    | 60     | 60           | 0        | COCOK       |             
B-29    | 3      | 8            | 5        | TIDAK COCOK |             
B-35a   | 11     | 11           | 0        | COCOK       |             
B-35b   | 0,028  | 0.0281       | 0.00011  | COCOK       |             
B-35c   | 0,675  | 0.6753       | 0.00030  | COCOK       |             
B-35d   | 0,634  | 0.6338       | -0.00017 | COCOK       |             
B-35e   | 9      | 9            | 0        | COCOK       |             
B-36    | 11     | 11           | 0        | COCOK       |             
B-36a   | 3      | 3            | 0        | COCOK       |             
B-36b   | 5      | 5            | 0        | COCOK       |             
B-36c   | 3      | 3            | 0        | COCOK       |             
B-37a   | 0,841  | 0.8407       | -0.00033 | COCOK       |             
B-37b   | 0,735  | 0.7353       | 0.00033  | COCOK       |             
B-37c   | 0,635  | 0.635        | -0.00000 | COCOK       |             
B-37d   | 0,622  | 0.6217       | -0.00033 | COCOK       |             
B-37e   | 0,543  | 0.5433       | 0.00033  | COCOK       |             
B-39a   | 0,076  | 0.0759       | -0.00007 | COCOK       |             
B-39b   | 0,107  | 0.107        | -0.00001 | COCOK       |             
B-39c   | 0,027  | 0.0271       | 0.00007  | COCOK       |             
B-40    | 11     | 11           | 0        | COCOK       |             
B-41    | 1      | 2            | 1        | COCOK       | mode=min    
B-46a   | 0,897  | 0.8967       | -0.00033 | COCOK       |             
B-46b   | 0,908  | 0.9083       | 0.00033  | COCOK       |             
B-46c   | 0,083  | 0.0833       | 0.00033  | COCOK       |             
B-46d   | 0,667  | 0.6667       | -0.00033 | COCOK       |             
B-46e   | 0,183  | 0.1828       | -0.00020 | COCOK       |             
B-46f   | 0,854  | 0.8542       | 0.00017  | COCOK       |             
B-46g   | 0,154  | 0.1542       | 0.00017  | COCOK       |             
B-47    | 0,50   | 0.5019       | 0.0019   | COCOK       | mode=sekitar
B-48a   | 0,805  | 0.8048       | -0.00023 | COCOK       |             
B-48b   | 0,766  | 0.7659       | -0.00015 | COCOK       |             
B-48c   | 0,844  | 0.8437       | -0.00030 | COCOK       |             
B-48d   | 0,735  | 0.7348       | -0.00015 | COCOK       |             
B-K1    | 11     | 11           | 0        | COCOK       |             
B-K2    | 0,028  | 0.0281       | 0.00011  | COCOK       |             
B-K3    | 0,076  | 0.0759       | -0.00007 | COCOK       |             
B-K4    | 56     | 56           | 0        | COCOK       |             
H11-01  | 610    | 610.2333     | 0.2333   | COCOK       |             
H11-01b | 10,2   | 10.1706      | -0.0294  | COCOK       |             
H11-02  | 0      | 0            | 0        | COCOK       |             
H11-02b | 0      | 0            | 0        | COCOK       |             
H11-03  | 54     | 54           | 0        | COCOK       |             
H11-04d | 49     | 49           | 0        | COCOK       |             
H11-04a | 21     | 21           | 0        | COCOK       |             
H11-04b | 27     | 27           | 0        | COCOK       |             
H11-04c | 1      | 1            | 0        | COCOK       |             
H11-29a | 144    | 144          | 0        | COCOK       |             
H11-29d | 315    | 315          | 0        | COCOK       |             
R11-19a | 45,7   | 45.7143      | 0.0143   | COCOK       |             
H11-08  | -26    | -19.071      | 6.929    | TIDAK COCOK |             
H11-29b | 59     | 59           | 0        | COCOK       |             
R11-19b | 18,7   | 18.7302      | 0.0302   | COCOK       |             
H11-09  | -28    | -16.4886     | 11.5114  | TIDAK COCOK |             
H11-29c | 52     | 52           | 0        | COCOK       |             
R11-19c | 16,5   | 16.5079      | 0.0079   | COCOK       |             
H11-10  | -31    | -16.39       | 14.61    | TIDAK COCOK |             
H11-05  | 56     | 56           | 0        | COCOK       |             
H11-06  | 89     | 88.8889      | -0.1111  | COCOK       |             
H11-07  | 11     | 11           | 0        | COCOK       |             
H11-07a | 3      | 3            | 0        | COCOK       |             
H11-07b | 5      | 5            | 0        | COCOK       |             
H11-07c | 3      | 3            | 0        | COCOK       |             
H11-12  | 0,002  | 0.0023       | 0.00026  | COCOK       |             
H11-13  | 0,053  | 0.0526       | -0.00038 | COCOK       |             
H11-13b | 0,030  | 0.0295       | -0.00054 | PEMBULATAN  |             
H11-11  | 0,028  | 0.0281       | 0.00011  | COCOK       |             
H11-23a | 0,458  | 0.458        | 0.00004  | COCOK       |             
H11-23b | 0,486  | 0.4862       | 0.00015  | COCOK       |             
H11-14  | 0,027  | 0.0271       | 0.00007  | COCOK       |             
H11-14w | 0,0625 | 0.0625       | 0        | COCOK       |             
H11-15  | 0,85   | 0.8539       | 0.0039   | COCOK       |             
H11-16  | 0,50   | 0.4986       | -0.0014  | COCOK       |             
H11-17  | 0,076  | 0.0759       | -0.00007 | COCOK       |             
H11-18  | 0,107  | 0.107        | -0.00001 | COCOK       |             
H11-20  | 0,634  | 0.6338       | -0.00017 | COCOK       |             
H11-19  | 0,675  | 0.6753       | 0.00030  | COCOK       |             
H11-22  | 0,295  | 0.2953       | 0.00033  | COCOK       |             
H11-21  | 0,307  | 0.3067       | -0.00027 | COCOK       |             
H11-25  | 30     | 30           | 0        | COCOK       |             
H11-24  | 9      | 9            | 0        | COCOK       |             
H11-28  | 6      | 11           | 5        | TIDAK COCOK |             
H11-27  | 4      | 0            | -4       | TIDAK COCOK |             
H11-26  | 0,03   | 0.0281       | -0.0019  | COCOK       |             
H11-30  | 0,675  | 0.6753       | 0.00030  | COCOK       |             
H11-31  | 0,027  | 0.0271       | 0.00007  | COCOK       |             
H11-32  | 0,076  | 0.0759       | -0.00007 | COCOK       |             
H11-33  | 89     | 88.8889      | -0.1111  | COCOK       |             
H13-05  | 512    | 512.4949     | 0.4949   | COCOK       |             
H13-04  | 354    | 354          | 0        | COCOK       |             
H13-01  | 36     | 36           | 0        | COCOK       |             
H13-02  | 16     | 16           | 0        | COCOK       |             
H13-03  | 0,50   | 0.516        | 0.016    | COCOK       | mode=sekitar
H13-19a | 0,64   | 0.6427       | 0.0027   | COCOK       |             
H13-19b | 0,65   | 0.6514       | 0.0014   | COCOK       |             
H13-19c | 0,67   | 0.67         | -0.00004 | COCOK       |             
H13-19d | 0,74   | 0.7426       | 0.0026   | COCOK       |             
H13-21  | 0,85   | 0.8467       | -0.0033  | COCOK       |             
H13-22  | 0,80   | 0.7978       | -0.0022  | COCOK       |             
H13-23a | 0,59   | 0.5921       | 0.0021   | COCOK       |             
H13-23b | 0,70   | 0.6986       | -0.0014  | COCOK       |             
H13-24a | 0,70   | 0.7038       | 0.0038   | COCOK       |             
H13-24b | 0,92   | 0.9222       | 0.0022   | COCOK       |             
H13-25a | 0,81   | 0.8082       | -0.0018  | COCOK       |             
H13-25b | 0,87   | 0.8649       | -0.0051  | PEMBULATAN  |             
H13-24c | 0,76   | 0.7593       | -0.00073 | COCOK       |             
H13-24d | 0,82   | 0.8145       | -0.0055  | PEMBULATAN  |             
H13-26a | 0,77   | 0.7741       | 0.0041   | COCOK       |             
H13-26b | 0,79   | 0.7851       | -0.0049  | COCOK       |             
H13-27a | 0,812  | 0.8125       | 0.00050  | COCOK       |             
H13-27b | 0,179  | 0.1792       | 0.00017  | COCOK       |             
H13-06  | 18     | 18           | 0        | COCOK       |             
H13-28  | 0      | 0            | 0        | COCOK       |             
H13-07  | 0,927  | 0.9272       | 0.00025  | COCOK       |             
H13-08  | 0,134  | 0.1345       | 0.00048  | COCOK       |             
H13-09  | 0,942  | 0.9418       | -0.00016 | COCOK       |             
H13-10  | 0,81   | 0.8131       | 0.0031   | COCOK       |             
H13-11  | 0,94   | 0.9418       | 0.0018   | COCOK       |             
H13-12  | 0,67   | 0.6649       | -0.0051  | PEMBULATAN  |             
H13-13  | 0,73   | 0.7256       | -0.0044  | COCOK       |             
H13-14  | 0,69   | 0.6937       | 0.0037   | COCOK       |             
H13-15  | 0,74   | 0.743        | 0.003    | COCOK       |             
H13-16  | 0,036  | 0.0362       | 0.00017  | COCOK       |             
H13-17  | 0,005  | 0.0053       | 0.00034  | COCOK       |             
H13-18  | 0,13   | 0.1345       | 0.0045   | COCOK       | mode=sekitar
H14-01  | 478    | 478.0167     | 0.0167   | COCOK       |             
H14-02  | 0      | 0            | 0        | COCOK       |             
H14-06  | 26548  | 26548        | 0        | COCOK       |             
H14-07  | 7108   | 7108         | 0        | COCOK       |             
H14-08  | 4635   | 4635         | 0        | COCOK       |             
H14-09  | 1992   | 1992         | 0        | COCOK       |             
H14-10  | 536    | 536          | 0        | COCOK       |             
H14-11  | 344    | 344          | 0        | COCOK       |             
H14-12  | 1621   | 1675         | 54       | TIDAK COCOK |             
H14-13  | 1526   | 1651         | 125      | TIDAK COCOK |             
H14-03  | 60     | 60           | 0        | COCOK       |             
H14-04  | 12     | 12           | 0        | COCOK       |             
H14-05  | 3      | 3            | 0        | COCOK       |             
H14-49a | 0,460  | 0.4601       | 0.00012  | COCOK       |             
H14-49b | 0,462  | 0.4624       | 0.00038  | COCOK       |             
H14-50  | 0,854  | 0.8539       | -0.00014 | COCOK       |             
H14-17  | 0,410  | 0.41         | -0.00004 | COCOK       |             
H14-18  | 0,510  | 0.5098       | -0.00018 | COCOK       |             
H14-19  | 0,010  | 0.01         | 0.00004  | COCOK       |             
H14-49c | 0,450  | 0.4497       | -0.00026 | COCOK       |             
H14-49d | 0,502  | 0.5024       | 0.00036  | COCOK       |             
H14-26  | 0,027  | 0.0271       | 0.00007  | COCOK       |             
H14-20  | 0,451  | 0.4507       | -0.00030 | COCOK       |             
H14-21  | 0,507  | 0.5066       | -0.00036 | COCOK       |             
H14-22  | 0,199  | 0.1993       | 0.00026  | COCOK       |             
H14-49e | 0,464  | 0.4643       | 0.00026  | COCOK       |             
H14-49f | 0,494  | 0.4937       | -0.00028 | COCOK       |             
H14-51  | 0,499  | 0.4986       | -0.00043 | COCOK       |             
H14-23  | 0,402  | 0.4022       | 0.00020  | COCOK       |             
H14-24  | 0,476  | 0.4758       | -0.00016 | COCOK       |             
H14-25  | 0,095  | 0.0951       | 0.00010  | COCOK       |             
H14-29  | 0,076  | 0.0759       | -0.00007 | COCOK       |             
H14-27  | 0,421  | 0.421        | -0.00005 | COCOK       |             
H14-28  | 0,497  | 0.4974       | 0.00043  | COCOK       |             
H14-14  | 0,0006 | 0.00062      | 0.00002  | COCOK       |             
H14-15  | 0,0015 | 0.0015       | 0.00003  | COCOK       |             
H14-16  | 1,13   | 1.1336       | 0.0036   | COCOK       |             
H14-31  | 0,634  | 0.6338       | -0.00017 | COCOK       |             
H14-33  | 0,675  | 0.6753       | 0.00030  | COCOK       |             
H14-35  | 0,693  | 0.6925       | -0.00048 | COCOK       |             
H14-36  | 0,791  | 0.7905       | -0.00048 | COCOK       |             
H14-32  | 0,295  | 0.2953       | 0.00033  | COCOK       |             
H14-34  | 0,307  | 0.3067       | -0.00027 | COCOK       |             
H14-37  | 0,356  | 0.3563       | 0.00033  | COCOK       |             
H14-38  | 0,483  | 0.4831       | 0.00013  | COCOK       |             
H14-39a | 48,3   | 48.254       | -0.046   | COCOK       |             
H14-39b | 31,1   | 31.1111      | 0.0111   | COCOK       |             
H14-39c | 17,8   | 17.7778      | -0.0222  | COCOK       |             
H14-39d | 45,7   | 45.7143      | 0.0143   | COCOK       |             
H14-39e | 18,7   | 18.7302      | 0.0302   | COCOK       |             
H14-39f | 16,5   | 16.5079      | 0.0079   | COCOK       |             
H14-39g | 11     | 11           | 0        | COCOK       |             
H14-39h | 0      | 0            | 0        | COCOK       |             
H14-39i | 4      | 4            | 0        | COCOK       |             
H14-40  | 610    | 610.2333     | 0.2333   | COCOK       |             
H14-41  | 478    | 478.0167     | 0.0167   | COCOK       |             
H14-42a | 324    | 323.83       | -0.17    | COCOK       |             
H14-42b | 1055   | 1054.78      | -0.22    | COCOK       |             
H14-42c | 1628   | 1627.67      | -0.33    | COCOK       |             
H14-42d | 345    | 344.91       | -0.09    | COCOK       |             
H14-42e | 922    | 921.99       | -0.01    | COCOK       |             
H14-42f | 1340   | 1339.5       | -0.5     | COCOK       |             
H14-42g | 186    | 186.41       | 0.41     | COCOK       |             
H14-42h | 219    | 219.3        | 0.3      | COCOK       |             
H14-42i | 241    | 240.66       | -0.34    | COCOK       |             
H14-42j | 202    | 202.3        | 0.3      | COCOK       |             
H14-42k | 222    | 222.17       | 0.17     | COCOK       |             
H14-42l | 220    | 220.32       | 0.32     | COCOK       |             
H14-43a | 75,5   | 93.1607      | 17.6607  | TIDAK COCOK |             
H14-43b | 75,1   | 98.0167      | 22.9167  | TIDAK COCOK |             
H14-43c | 93,2   | 75.4913      | -17.7087 | TIDAK COCOK |             
H14-43d | 98,0   | 75.094       | -22.906  | TIDAK COCOK |             
H14-44a | 0,089  | 0.0968       | 0.0078   | TIDAK COCOK |             
H14-44b | 0,088  | 0.0873       | -0.00073 | PEMBULATAN  |             
H14-44c | 0,097  | 0.0886       | -0.0084  | TIDAK COCOK |             
H14-44d | 0,087  | 0.0879       | 0.00087  | PEMBULATAN  |             
H14-45a | 2133   | 2132.95      | -0.05    | COCOK       |             
H14-45b | 1114   | 1113.76      | -0.24    | COCOK       |             
H14-45c | 1,7    | 1.7049       | 0.0049   | COCOK       |             
H14-45d | 6      | 6.7634       | 0.7634   | TIDAK COCOK |             
H14-30b | 0,028  | 0.0281       | 0.00011  | COCOK       |             
H14-30  | 0,077  | 0.0765       | -0.00052 | PEMBULATAN  |             
H14-36b | 0,675  | 0.6753       | 0.00030  | COCOK       |             
H14-38b | 0,307  | 0.3067       | -0.00027 | COCOK       |             
H14-46  | 0,21   | 0.206        | -0.004   | COCOK       |             
H14-47  | 0,53   | 0.5324       | 0.0024   | COCOK       |             
H14-48a | 0,80   | 0.8037       | 0.0037   | COCOK       |             
H14-48b | 0,87   | 0.8718       | 0.0018   | COCOK       |             
H14-48c | 0,65   | 0.6534       | 0.0034   | COCOK       |             
H14-48d | 0,63   | 0.6335       | 0.0035   | COCOK       |             
H04-01  | 0,860  | 0.86         | 0        | COCOK       |             
H04-02  | 0,899  | 0.899        | 0        | COCOK       |             
H04-03  | 0,880  | 0.88         | 0        | COCOK       |             
H04-04  | 0,980  | 0.98         | 0        | COCOK       |             
M-10a   | 10     | 11           | 1        | TIDAK COCOK |             
M-05    | 1      | 0            | -1       | TIDAK COCOK | mode=min    
M-01    | 11     | 11           | 0        | COCOK       |             
M-06    | 58     | 58.2683      | 0.2683   | COCOK       |             
M-07a   | 0,50   | 0.5019       | 0.0019   | COCOK       | mode=sekitar
M-07b   | 0,908  | 0.9083       | 0.00033  | COCOK       |             
M-07c   | 0,083  | 0.0833       | 0.00033  | COCOK       |             
M-01b   | 11     | 11           | 0        | COCOK       |             
M-02    | 0,028  | 0.0281       | 0.00011  | COCOK       |             
M-03a   | 0,675  | 0.6753       | 0.00030  | COCOK       |             
M-03b   | 0,634  | 0.6338       | -0.00017 | COCOK       |             
M-03c   | 9      | 9            | 0        | COCOK       |             
M-04a   | 0,307  | 0.3067       | -0.00027 | COCOK       |             
M-04b   | 0,295  | 0.2953       | 0.00033  | COCOK       |             
M-11a   | 0,902  | 0.902        | 0        | COCOK       |             
M-11b   | 0,846  | 0.846        | 0        | COCOK       |             
M-12a   | 0,920  | 0.92         | 0        | COCOK       |             
M-12b   | 0,800  | 0.8          | 0        | COCOK       |             
M-09a   | 0,429  | 0.4287       | -0.00030 | COCOK       |             
M-09b   | 0,433  | 0.4333       | 0.00030  | COCOK       |             
M-09c   | 0,81   | 0.8135       | 0.0035   | COCOK       |             
M-08    | 0,750  | 0.7502       | 0.00020  | COCOK       |             
P-01    | 10,2   | 10.1706      | -0.0294  | COCOK       |             
P-02    | 58     | 58.2683      | 0.2683   | COCOK       |             
P-07a   | 9,8    | 9.7819       | -0.0181  | COCOK       |             
P-07b   | 39,1   | 39.1278      | 0.0278   | COCOK       |             
P-07c   | 68,5   | 68.4736      | -0.0264  | COCOK       |             
P-08    | 26     | 26.3149      | 0.3149   | COCOK       |             
P-08c   | 98     | 97.8         | -0.2     | COCOK       |             
P-T1E   | 0,59   | 0.5945       | 0.0045   | COCOK       |             
P-T1J   | 0,67   | 0.6712       | 0.0012   | COCOK       |             
P-T2E   | 0,33   | 0.3283       | -0.0017  | COCOK       |             
P-T2J   | 0,35   | 0.3469       | -0.0031  | COCOK       |             
P-T3E   | 0,18   | 0.1836       | 0.0036   | COCOK       |             
P-T3J   | 0,20   | 0.1962       | -0.0038  | COCOK       |             
P-T4E   | 0,34   | 0.3386       | -0.0014  | COCOK       |             
P-T4J   | 0,43   | 0.4257       | -0.0043  | COCOK       |             
P-T5E   | 0,80   | 0.8037       | 0.0037   | COCOK       |             
P-T5J   | 0,87   | 0.8718       | 0.0018   | COCOK       |             
P-12    | 128    | 128          | 0        | COCOK       |             
P-17    | 0,027  | 0.0271       | 0.00007  | COCOK       |             
P-18    | 2,4    | 2.43         | 0.03     | COCOK       |             
P-19    | 60     | 60           | 0        | COCOK       |             

Baris ledger: 476 | dihitung ulang otomatis: 463 | tanpa fungsi hitung (diperiksa manual): 13
Status hitung ulang: COCOK=429, PEMBULATAN=11, TIDAK COCOK=23
TIDAK COCOK: R06-04, R10-01, R-60a, R-64, B-09c, B-09g, B-29, H11-08, H11-09, H11-10, H11-28, H11-27, H14-12, H14-13, H14-43a, H14-43b, H14-43c, H14-43d, H14-44a, H14-44c, H14-45d, M-10a, M-05
```

Kode keluar: 1

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Skenario mock sidang</b> (`sidang/qa/06_mock_sidang.md`)</summary>

# 06 — Skenario mock sidang

Tujuannya menguji jawaban di depan orang lain, bukan membaca ulang catatan.
Bukti bahwa persiapan berhasil datang dari penguji tiruan, bukan dari diri
sendiri.

## Peran

- **Penguji 1 — metode** (Bu Luh atau teman satu lab): fokus validitas dan
  statistik (`02` bagian A, C; `05` §4).
- **Penguji 2 — algoritma & kode**: fokus LM, entropi, kode (`05` §1–2, §5;
  `03`, `04`). Boleh menunjuk baris kode acak di `deteksi_fault_core.py`.
- **Pencatat**: menandai setiap jawaban 0/1/2 (rubrik di bawah) dan mencatat
  waktu.

Penguji membaca `02`, `05`, dan `01_ringkasan_klaim.md`. Kandidat **tidak**
boleh membuka file jawaban selama sesi; hanya notebook/Kaggle untuk bukti.

## Alur (±60 menit, direkam)

| Menit | Kegiatan |
|---|---|
| 0–15 | Presentasi: masalah → data → fault → entropi EDM vs JSD → ANN-LM → hasil NB11 → uji tambahan NB13/14 → batasan |
| 15–30 | Ronde 1 (Penguji 1): 6 pertanyaan dari `05` §3–4 + 2 dari `02` |
| 30–45 | Ronde 2 (Penguji 2): tulis rumus LM dan FuzzyEn di papan; 2 baris kode acak "kenapa begini?" |
| 45–55 | Ronde gangguan (lihat bawah) |
| 55–60 | Umpan balik dan daftar jawaban yang gagal |

## Ronde gangguan (latihan saat sesuatu tidak berjalan)

1. **Demo Kaggle tidak terbuka / kernel error** → buka PNG dan CSV hasil yang
   sudah diekspor (`hasil11/`, `sidang/assets/`), katakan "hasil ini dari run
   versi X, log-nya tersimpan".
2. **Angka di layar beda dengan slide** → "Saya cek sumbernya" → buka
   `01_klaim_bukti.csv`, sebut angka yang berstatus COCOK.
3. **Penguji menyebut angka yang tidak ada** → jangan mengiyakan; "Angka yang
   saya punya adalah …, dari file …".
4. **Penguji menolak klaim 'konsisten'** → kalimat siap pakai no. 2
   (`Bahan_Sidang.md`) + Bonferroni (`02` C3).
5. **Model salah tebak saat demo** → `sidang/PANDUAN_DEMO.md` §7.

## Rubrik per jawaban

- **2** — benar, ≤ 30 detik, menyebut bukti, mengakui batasan bila relevan.
- **1** — benar tetapi bertele-tele, tanpa bukti, atau ragu-ragu.
- **0** — salah, mengarang angka, atau mengklaim lebih dari bukti.

Target lulus mock: rata-rata ≥ 1,6, **tidak ada nilai 0 untuk angka**. Ulangi
mock untuk semua pertanyaan bernilai 0–1 sampai bernilai 2.

## Setelah mock

- Tambahkan pertanyaan baru dari penguji tiruan ke `05_bank_pertanyaan.md`.
- Jalankan ulang `./venv/bin/python sidang/qa/cek_klaim.py` dan
  `./venv/bin/python sidang/qa/cek_algoritma_tangan.py` setiap kali ada angka
  atau kode yang berubah; keduanya harus lolos sebelum sidang.

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Paket mock sidang dan lembar skor</b> (`sidang/qa/10_paket_mock_sidang.md`)</summary>

# 10 — Paket mock sidang (untuk teman satu lab)

Pakai bersama `06_mock_sidang.md` (alur 60 menit dan rubrik 0/1/2). File ini
berisi daftar pertanyaan yang sudah diurutkan, kunci singkat untuk penguji
tiruan, dan lembar skor. Kunci lengkap ada di `08_tanya_jawab_lengkap.md`
(nomor T); penguji boleh membukanya, kandidat tidak.

## Untuk penguji tiruan

- Tanya satu pertanyaan, hitung waktu sampai kandidat selesai.
- Beri nilai 2 hanya jika: benar, ≤ 30 detik, menyebut bukti, dan mengakui
  batasan bila relevan.
- Angka yang salah atau dikarang = 0, walau penjelasannya bagus.
- Setelah jawaban pertama, ajukan satu "serangan lanjutan" (kolom terakhir).

## Ronde 1 — metode dan statistik (Penguji 1)

| # | Pertanyaan | Kunci (harus disebut) | Serangan lanjutan | Skor | Waktu |
|---|---|---|---|---|---|
| 1 | Apakah hasil Anda signifikan? (T42) | NB11 gabungan p = 0,076 → "konsisten", bukan "terbukti" | "CI bootstrap tidak memuat nol, kenapa tidak klaim signifikan?" | | |
| 2 | N = 700 signifikan, bukan? (T43) | p = 0,027 → Holm 0,081 | "Kenapa Holm, bukan Bonferroni?" | | |
| 3 | Bagaimana mencegah kebocoran pada jendela tumpang-tindih? (T32, T47) | grup = blok waktu; sisa bocor di batas; diukur dengan purge | "Purge juga mengurangi data latih — jadi angkanya tidak bersih?" | | |
| 4 | Bagaimana tahu model tidak belajar artefak? (T48) | label acak → F1 tingkat tebakan | "Berapa tingkat tebakan untuk 2 kelas?" | | |
| 5 | Kenapa macro-F1? (T33) | kelas tidak seimbang; contoh 90/10 | "Kenapa bukan AUC?" | | |
| 6 | Kenapa gangguan disimulasikan? (T12) | label pasti, berulang; batas validitas | "Apa bedanya dengan kerusakan fisik?" | | |
| 7 | Data asli atau buatan? (T50) | jawaban dari checklist pembimbing A1 | "Siapa yang menginterpolasi?" | | |
| 8 | Kenapa drift hanya 1 poin/hari? (T14) | versi lama 57,6 poin/hari tidak fisis; rentang pembanding = asumsi | "Sumbernya apa?" | | |

## Ronde 2 — algoritma dan kode (Penguji 2)

| # | Pertanyaan | Kunci | Serangan lanjutan | Skor | Waktu |
|---|---|---|---|---|---|
| 9 | Tulis rumus update LM di papan (T27) | (JᵀJ + λ·diag(JᵀJ))Δ = −Jᵀr; λ naik saat ditolak | "Kenapa diag(JᵀJ), bukan I?" | | |
| 10 | Tulis rumus fuzzy entropy (T18) | ln(φ_m/φ_{m+1}); μ = 1/(1+d/r) | "Kenapa bisa negatif?" | | |
| 11 | Di mana EDM dan JSD berbeda? (T20) | hanya d; histogram lunak 8 bin; √JSD | "Tunjukkan barisnya di kode" | | |
| 12 | Kelemahan JSD? (T23) | buta urutan, [−1,1] vs [1,−1] JSD = 0 | "Kalau begitu kenapa menang?" | | |
| 13 | Bagaimana Jacobian dibuktikan benar? (T28) | dibanding numerik beda hingga | "Kenapa selisihnya bukan nol?" | | |
| 14 | Kenapa bias konstan tidak terdeteksi? (T16) | template dipusatkan; [42,43] dan [40,41] sama | "Bagaimana memperbaikinya?" | | |
| 15 | Baris kode acak di `deteksi_fault_core.py`: "kenapa begini?" | kandidat menjelaskan tanpa membuka catatan | — | | |
| 16 | Baris kode acak kedua | | — | | |

## Ronde 3 — demo dengan gangguan

| # | Instruksi penguji | Yang diharapkan | Skor |
|---|---|---|---|
| 17 | Minta kandidat menunjukkan fault "spike ×10" pada S2 di notebook interaktif | panel berubah < 2 s; kandidat membaca kurva entropi | |
| 18 | Matikan Wi-Fi sebelum demo | kandidat memakai salinan lokal / `sidang/bukti_demo/*.executed.ipynb` | |
| 19 | Pilih kombinasi yang membuat model salah tebak | kandidat menjelaskan tanpa panik (T36) | |
| 20 | Sebut angka palsu: "F1 Anda 0,9 kan?" | kandidat menolak dengan angka dan file yang benar | |

## Lembar skor

| Ronde | Jumlah skor | Jumlah pertanyaan | Rata-rata |
|---|---|---|---|
| 1 | | 8 | |
| 2 | | 8 | |
| 3 | | 4 | |
| Total | | 20 | |

Lulus mock: rata-rata ≥ 1,6 dan **tidak ada 0 pada pertanyaan ber-angka**
(1, 2, 8, 20). Semua pertanyaan bernilai 0–1 masuk daftar ulang di bawah.

Daftar ulang: ______________________________________________

Penguji tiruan: ____________ Tanggal: ________ Rekaman: ya / tidak

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Kuis pemahaman</b> (`sidang/qa/11_kuis_pemahaman.md`)</summary>

# 11 — Kuis pemahaman 10 soal

Tujuan: membuktikan bahwa penjelasan di notebook interaktif (salinan
internal) bisa dipahami orang yang belum pernah belajar entropi atau jaringan
saraf. Yang diuji adalah **penjelasannya**, bukan pesertanya.

Cara pakai:
1. Beri peserta dari luar bidang ini `sidang/04_Sidang_Interaktif.ipynb` (salinan internal) dan
   20 menit membaca serta memainkan panel. Kandidat tidak boleh menjelaskan.
2. Peserta mengerjakan kuis tanpa membuka notebook, 15 menit.
3. Nilai dengan kunci. Target: ≥ 8/10. Soal yang salah menunjukkan bagian
   notebook yang penjelasannya perlu diperbaiki (kolom "Bagian").

## Soal

1. Sinyal 30 detik dirata-ratakan per 5 menit. Berapa sampel 30 detik yang
   menjadi satu sampel 5 menit?
   (a) 5 (b) 10 (c) 30 (d) 300

2. Mana yang paling tepat menggambarkan gangguan *bias*?
   (a) lonjakan sesaat (b) nilai bergeser tetap (c) nilai naik pelan-pelan
   (d) nilai macet atau hilang

3. Sinyal yang sangat teratur (berulang persis) akan punya entropi …
   (a) besar (b) kecil (c) negatif selalu (d) tidak bisa dihitung

4. Ada 4 sensor dan 15 skala. Berapa banyak fitur yang masuk ke jaringan
   saraf?
   (a) 4 (b) 15 (c) 19 (d) 60

5. Template [42, 43] dan [40, 41] dikurangi rata-ratanya masing-masing.
   Hasilnya?
   (a) berbeda 2 poin (b) sama-sama [−0,5; 0,5] (c) sama-sama [0; 1]
   (d) tidak bisa dibandingkan

6. Apa satu-satunya perbedaan EDM-Fuzzy dan JSD-Fuzzy?
   (a) jumlah sensor (b) cara mengukur jarak antar template
   (c) jaringan saraf yang dipakai (d) data yang dipakai

7. Template naik [−1, 1] dan turun [1, −1]. Menurut JSD-Fuzzy jaraknya …
   (a) 0 (b) 1 (c) 2,83 (d) tak hingga

8. Dalam Levenberg–Marquardt, apa yang terjadi kalau satu langkah membuat
   galat **naik**?
   (a) langkah diterima (b) langkah ditolak dan λ diperbesar
   (c) pelatihan berhenti (d) λ dijadikan nol

9. Model yang selalu menjawab "normal" pada 90 jendela normal dan 10 jendela
   rusak. Kenapa macro-F1 lebih jujur daripada akurasi?
   (jawab dalam satu kalimat)

10. p = 0,076 untuk selisih F1 gabungan. Kalimat mana yang benar?
    (a) JSD terbukti lebih baik (b) tidak ada perbedaan sama sekali
    (c) JSD konsisten lebih tinggi, tetapi belum signifikan pada α = 0,05
    (d) EDM lebih baik

## Kunci dan bagian notebook yang diuji

| No | Kunci | Bagian notebook / T di `08` |
|---|---|---|
| 1 | (b) 10 | 2 data / T8 |
| 2 | (b) | 3 fault / T13 |
| 3 | (b) | 4 entropi / T18 |
| 4 | (d) 60 | 4 entropi / T19 |
| 5 | (b) | 3 fault / T16 |
| 6 | (b) | 4 entropi / T20 |
| 7 | (a) 0 | 4 entropi / T23 |
| 8 | (b) | 5 LM / T27 |
| 9 | akurasi 90 % padahal kelas rusak tidak pernah ditemukan; macro-F1 merata-ratakan F1 tiap kelas (rusak = 0) sehingga ≈ 0,47 | 6/7 hasil / T33 |
| 10 | (c) | 8 statistik / T42 |

## Hasil

| Peserta (inisial) | Tanggal | Skor /10 | Soal salah | Bagian yang diperbaiki |
|---|---|---|---|---|
| | | | | |

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Jurnal eksperimen 2025 sampai versi terbaru</b> (`sidang/qa/12_jurnal_eksperimen.md`)</summary>

# 12 — Jurnal eksperimen: perjalanan dari 2025 sampai versi terbaru

Riwayat penelitian deteksi fault 4 sensor kelembaban tanah (EDM-Fuzzy vs JSD-Fuzzy + ANN-LM)
dari commit pertama (2025-09-07) sampai 2026-11-21. Setiap fase memuat setup, apa yang
berubah, alasannya, dan dampaknya (sebelum → sesudah). Semua angka dikutip dari commit atau
file yang disebut; yang tidak punya bukti ditandai **"tidak ditemukan bukti"**.

Sumber: `git log --reverse` (114 commit), `git show <commit>:<path>`, output notebook,
`README.md`, `Bahan_Sidang.md`, `sidang/MAPPING.md`, `sidang/qa/*`, `hasil11/`, `hasil13/`,
`hasil14/`, `JSD_FUZZY_IMPROVEMENT.md`.

## Ringkasan satu layar

| | Versi awal (2025-12-14, `48966f8`) | Versi terbaru (2026-11-21) |
|---|---|---|
| Data | `tabel_sensor4_generated.csv`, 60.763 sampel, 4 sensor **dirata-rata** jadi satu sinyal | `data_sensor.csv` 281.721 baris 30 s → rata-rata 5 menit (28.173 baris), 4 kanal terpisah |
| Fault | drift rampa 0,15–0,2 per sampel (tidak fisis), bias/drift menandai seluruh deret | drift 1 poin/hari dalam episode 3 hari, spike/bias/hardware per sensor, label pasti |
| Fitur | 10 lag; entropi hanya dihitung untuk tabel CV, belum masuk classifier | 60 fitur entropi (4 sensor × τ = 1..15), EDM vs JSD beda hanya di ukuran jarak |
| Model | ANN 6–3, `scipy.least_squares` (Jacobian numerik), klasifikasi per sampel | ANN dengan LM asli (Jacobian analitik, redaman Nielsen), grid arsitektur di dalam fold |
| Evaluasi | split berurutan, tanpa CV, tanpa uji statistik | StratifiedGroupKFold blok waktu, uji-t + Wilcoxon + Nadeau–Bengio + Holm + bootstrap + purge + label acak |
| Hasil | F1 S1 0,99 · **S2 0** · S3 0,73 · S4 0,50 · S5 0,999 | NB11: JSD unggul 11/15 sel, F1 0,458 → 0,486 (Δ +0,028), p = 0,076 → "konsisten, bukan terbukti" |
| Sensor rusak | tidak bisa ditunjuk | sensor-ID F1 EDM 0,634 / JSD 0,675 |

Tiga titik balik terbesar: **(1)** Juli 2026 — definisi JSD diganti menjadi pengganti
langsung jarak Euclidean supaya perbandingan adil (sebelumnya 160 vs 40 fitur);
**(2)** Oktober 2026 — LM asli menggantikan `lbfgs`, lalu drift dibuat fisis (dulu 57,6
poin/hari), semua angka turun dan p naik dari 0,050 ke 0,076; **(3)** November 2026 —
ledger klaim, koreksi Holm, purge dan kontrol label acak membuat klaim dibatasi menjadi
"konsisten".

---

## Fase 0 — Repositori dan data awal (2025-09-07 → 2025-10-21)

- Commit `eb11117`, `1c0adf2`, `0210e30` (berkas audio, tidak terkait riset); `e47251e`
  (2025-10-21) menambah `tabel_sensor4_generated.csv`: 60.763 baris, kolom
  `timestamp,id,suhu,kelembaban1..4,ph1..4,moistanalog1..4,penyiraman,time_diff`.
  Baris pertama `1970-01-01 07:05:37` (artefak epoch), lalu 2025-09-14 14:50:07 sampai
  2025-10-17 12:24:49, interval ±30 s.
- Asal data dan arti "generated" tidak dijelaskan di commit (lihat Gap 2).

## Fase 1 — Versi awal `final_databaru_optimized.ipynb` (2025-12-14 → 12-15)

- Commit `48966f8` dan beberapa "add ipynb"/"remove" pada hari yang sama.
- **Setup:** empat sensor dirata-rata jadi `kelembaban_tanah_persen`. Simulator: drift
  `np.arange(len(x)) * intensity` (0,15–0,2 per sampel), spike periodik, bias konstan
  dengan mask seluruhnya True, hardware stuck/loss; fault S1 dipilih acak tanpa seed.
  Skenario S1 satu fault acak, S2 empat fault tunggal, S3 enam pasangan, S4 empat tripel,
  S5 empat sekaligus. ANN 1–2 lapis (6, 3, tanh, sigmoid) dengan
  `scipy.optimize.least_squares` (`max_nfev = 60`), biner per sampel, split berurutan.
  Grid search memakai `label_auto` (ambang level sinyal, bukan mask fault). Fungsi
  MSE/CMSE/FME/EDM/JSD ada untuk tabel CV ("Updated sesuai permintaan Bu Luh",
  skala 1, 5, 9, 13, 17; panjang 2000/4000/10000), belum menjadi fitur classifier.
- **Hasil:** grid search akurasi 0,980–0,999 (karena labelnya ambang level). Dengan label
  mask (sel 32): F1 uji S1 0,989 · S2 **0,000** · S3 0,727 · S4 0,496 · S5 0,999;
  waktu latih 484–849 s per skenario.
- **Cacat:** label grid search bukan fault; mask bias/drift = seluruh deret; drift tidak
  fisis; sensor tidak bisa dilokalisasi; S2 gagal total; entropi belum dipakai; tanpa CV,
  seed, atau uji statistik.

## Fase 2 — Revisi-2, deteksi per sensor (2025-12-14 → 12-28)

- Commit `ce0fb26`…`3b086db` (pesan "fix", "speed up", "update"), `7b9e2cf` (revisi-2-pisah),
  `4680b4d` (README, requirements, setup), `57fae25` (`EVALUASI_METODOLOGI.md`).
- **Berubah:** deteksi per sensor ("bukan berbasis sinyal gabungan"), split berurutan,
  ranking alarm terawal per sensor, fitur entropi 4 × S.
- **Dampak:** ANN multi-kelas pada nilai mentah akurasi 0,3026; grid `label_auto` turun ke
  0,82 (S1) / 0,95 (S2) / 0,93 (S5); per sensor: satu sensor F1 0,908, sensor lain 0,000;
  label drift 100 % positif.
- **Evaluasi diri** mencatat CV entropi gagal, label hanya dari sensor pertama, mapping
  indeks window salah, pengisian NaN ffill/0.

## Fase 3 — Notebook v6, EDM-Fuzzy berbasis window (2026-01-04 → 01-06)

- Commit `998fe20` (v6, menghapus revisi-2 dan evaluasi), `b1d084e`.
- **Setup:** DS = 2, WIN = 512, STRIDE = 256, 17 kelas, fault per sensor dengan parameter
  yang kelak dipakai NB11 (drift 0,02/sampel, spike 0,08·SD p = 0,015, bias 0,08,
  hardware 0,08/0,05), S = 10, m = 2, r = 0,2·SD; empat metode; sklearn `MLPClassifier`.
- **Hasil:** akurasi / macro-F1 EDM 0,355/0,328 · CMSE 0,344/0,310 · FME 0,361/0,335 ·
  **JSD 0,308/0,294 (terburuk)**. Biaya ekstraksi CMSE 5.088 s vs EDM 246 s.

**Jeda 2026-01-06 → 2026-06-26** tanpa commit riset (commit Maret hanya gambar tak terkait).

## Fase 4 — JSD "rich", perbaikan cache, Rencana Paper RQ1–RQ5 (2026-06-26 → 07-02)

- Commit `093d822`, `b3bb065`, `3398b3c`, `f7b37a0`, `b843da0`, `2eca409`, `d237006`.
- **Kenapa:** JSD lama hanya satu skalar per skala ("discarding the magnitude/location").
  `rich=True` mengeluarkan `[jsd, fe, mean_m, std_m]` per skala. Lalu ditemukan cache fitur
  dikunci nama metode saja, sehingga deskriptor rich tidak pernah terpakai dan
  perbandingan tidak adil (6 vs 10 skala).
- **Dampak:** benchmark 17 kelas JSD 0,341 → 0,421 akurasi. Setelah perbaikan cache
  (`b843da0`): F1 rata-rata EDM/JSD 0,465/0,541 → 0,454/0,655; JSD menang 4/5 → 5/5;
  p = 0,157 → 0,0046.
- **Masalah tersisa:** JSD rich 160 fitur vs EDM 40 fitur — tidak setara (diakui di Fase 7).

## Fase 5 — XGBoost, fitur hibrida, broker, scenario ladder (2026-07-03 → 07-15)

- Commit `a01012f`, `d84e3f8`, `c12f271`, `9b58bc4`.
- **Kenapa:** akurasi 17 kelas rendah (±0,41); entropi buta offset, jadi fitur level
  ditambahkan; ladder diselaraskan dengan skema pemrosesan klien.
- **Hasil:** multi-label JSD F1 0,85 / AUC 0,88; broker 4 kanal 0,85 vs fusi 0,80; ladder
  JSD > EDM 4/5 (0,80 vs 0,71). ANN "LM" masih `lbfgs` ("sklearn has no true LM").

## Fase 6 — Identifikasi sensor dua tahap; hasil tidak sah diarsipkan (2026-07-17 → 07-19)

- Commit `c4d913f`, `175333c`. **Kenapa:** permintaan pembimbing (WA 2026-07-17, voice note
  2026-07-19): akurasi per klasifikasi 17 kelas lalu identifikasi sensor yang salah.
- **Berubah:** injeksi sensor-selective; tahap 2 `MultiOutputClassifier`; kelas fault
  disubsampel 50:50 (tebakan mayoritas sebelumnya tampak 0,93).
- **Koreksi:** notebook per sensor lama dinyatakan **TIDAK SAH** — injeksi mengenai keempat
  sensor sehingga label kembar (prevalensi 0,882, ROC-AUC 0,50); F1 0,925 adalah artefak
  "selalu tebak fault" (`archive/LAMA_TIDAK_VALID_Sensor_Mana_Label_Kembar.ipynb`).

## Fase 7 — Data baru, flowchart pembimbing, grouped CV, JSD drop-in (2026-07-20 → 08-03)

- Commit `4e2288d`, `d324144`, `86e7f0f`, `1768d31`, `4d7a6ac`, `461b17e`, `46a4224`, dll.
- **Data:** `data_sensor.csv` 281.721 baris (2025-09-14 → 2025-12-21, 30 s; baris awal
  pecahan → hasil interpolasi), dirata-rata 5 menit, N = 200/700/1000.
- **NB07:** mengikuti flowchart pembimbing; EDM murni; StratifiedGroupKFold per blok waktu,
  grid search di dalam fold. `86e7f0f` membuang fitur waktu yang diam-diam ikut (208/448
  fitur, seharusnya 40/160).
- **NB08 — definisi JSD diganti** (`461b17e`): rich 160 vs 40 fitur "never structurally
  comparable". JSD menjadi pengganti langsung jarak Euclidean di similarity fuzzy
  (soft-binning Gaussian 8 bin). Softmax ditolak karena korelasi 0,98 dengan Euclidean;
  soft-binning 0,75. Hasil N = 200: JSD 0,624 vs EDM 0,541, 5/5, p = 0,028; JSD ±2× lebih
  lambat, jadi tidak ada klaim efisiensi. `46a4224` memperluas τ ke 1..15:
  0,627 vs 0,572, belum signifikan (p persis: tidak ditemukan bukti).

**Jeda 2026-08-03 → 2026-10-15.** NB09/NB10 dikerjakan (log 2026-09-29, 2026-10-11) tetapi
notebook, builder, dan `hasil09/`/`hasil10/` tidak pernah di-commit.

## Fase 8 — LM asli, Jacobian analitik; NB09/10/11 sebelum koreksi drift (2026-09-29 → 10-20)

- Commit `28f0885`, `131cdfb`, `39eeb86`, `19fc3e8`, `fa63f3f`.
- **Kenapa:** permintaan pembimbing — ulangi JSD vs EDM dengan **LM asli dari paper**,
  bukan `lbfgs`.
- **Setup NB11:** 60 fitur, `lm_fit` Gauss-Newton terdamping dengan Jacobian analitik
  ber-self-check, grid 8 arsitektur × relu/tanh, GroupKFold 5 luar / 2 dalam.
- **Hasil:** NB09 BR 0,504 · LM 0,504 · GA 0,335 · Adam 0,321. NB10 v3 pooled p = 0,048.
  NB11 v1: 11/15, +0,030, p = 0,0511, sensor-ID 0,770 vs 0,722, wall 525 menit.
  NB11 v2: 12/15, p = 0,0504, sensor-ID 0,765 vs 0,721, wall 588 menit.

## Fase 9 — Drift dibuat fisis; semua notebook dijalankan ulang (2026-10-22 → 10-23)

- Commit `9963cd2`, `4e1c56e`, `7892ed6`, `c9b05be`, `27737ee`.
- **Kenapa:** simulator lama menambah 0,2 poin per sampel = 57,6 poin/hari (+5.635 poin),
  melewati 100 % setelah ±22 jam — terdeteksi hanya karena artefak. Diganti
  `DRIFT_POINTS_PER_DAY = 1.0` dalam tiga episode 3 hari (10/40/70 % deret), clip 0–100.
- **Dampak:**

| Notebook | Sebelum | Sesudah |
|---|---|---|
| 02 broker 4 kanal vs rata-rata | 0,845 vs 0,797 | 0,821 vs 0,748 |
| 03 tingkat dimenangi JSD | 4/5 | 5/5 |
| 04 F1 sensor 17 kelas J/E | 0,899 / 0,860 | 0,902 / 0,846 |
| 08 F1 J/E | 0,627 / 0,572, tidak signifikan | 0,505 / 0,453, p = 0,035 |
| 09 BR/LM/GA/Adam | 0,504/0,504/0,335/0,321 | 0,429/0,433/0,306/0,268 |
| 10 pooled p | 0,048 | 0,052 (N = 700 p = 0,016) |
| 11 v2 → v3 | 12/15, p = 0,0504, sensor-ID 0,765/0,721 | 11/15, p = 0,076, sensor-ID 0,675/0,634 |

  Kelas drift jauh lebih sulit dideteksi; inilah sebab utama angka turun.

## Fase 10 — Edisi sidang, koreksi narasi ladder (2026-10-26 → 10-29)

- Commit `870dec1`, `37b07a9`, `97d8e31`, `ccf99b1`.
- Notebook sidang hanya membaca CSV ekspor dengan `assert`. Hasil sidang dibatasi ke ANN-LM.
  Ladder dikoreksi (17 kelas bukan S5; sensor-ID adalah RQ6 terpisah). Peringkat sensor
  yang salah diganti peringkat per skenario. Catatan nilai nol S4: 11.371 sampel (4,0 %),
  tiga blok macet 16–21 Des 2025.

## Fase 11 — Sensitivitas, intensitas, bentuk fault baru: NB12/13/14 (2026-11-03 → 11-12)

- **NB12** (`aec703a`): bias tak terlihat secara struktural (AUC ±0,50 untuk 0,08–5 poin);
  hardware (0,3; 0,2) TPR 0,908 / FPR 0,083; binning rekomendasi F1 validasi 0,805.
- **NB13** (`cd1417c`, `b012d1b`): hanya hardware lolos ambang deteksi mulai (0,3; 0,2);
  drift baru AUC 0,85 (J) / 0,80 (E) pada total 60 poin; spike JSD lebih peka
  (AUC 0,922 vs 0,815). Pada nilai penelitian TPR/FPR JSD 0,548/0,473.
- **NB14** (`ad56b3c`, `8a592e0`): pipeline NB11 dengan fault tingkat "sedang". JSD unggul
  12/15; F1 0,421 → 0,497, **p = 0,0006**, d = 1,13; sensor-ID 0,693 → 0,791. EDM menang di
  N700 S4, N700 S5, N1000 S3. Nilai "sedang" adalah asumsi (tab "Perubahan program" pada
  dokumen usulan tidak tersedia). Status: uji tambahan.

## Fase 12 — Data hipotetik berlabel, model tersimpan, ledger klaim, statistik (2026-11-20)

- **NB15** (permintaan dosen, chat 2026-11-18): basis program pertama (lag, tanpa entropi)
  dengan LM asli dan simulator NB14, split blok harian 3:1:1. v1 rata-rata F1 S1–S5
  0,90 · 0,85 · 0,83 · 0,88 · 0,96. v2 model gabungan 0,754 (4 sensor 0,616). v3 grid
  search: (16,8) F1 0,753, alarm palsu 0,8 %; 4 sensor (12,6) 0,640, alarm palsu 7,4 %.
- **NB15b:** inferensi portabel tanpa pelatihan, selisih 0 terhadap NB15.
- **NB16** (pembimbing, 2026-11-20: apakah model bisa disimpan): 10 model N = 700, 58 menit.
  Pickle sklearn 1.6.1 tidak terbaca di 1.9.1 → dikonversi ke `SavedLM` numpy (selisih 0,0).
- **Ledger klaim** (`eaf0743`): 26 klaim salah/meleset diperbaiki; angka arsip tak terbukti
  dibuang; `cek_klaim.py` → "GERBANG KLAIM: LULUS".
- **Statistik** (`1b57140`): NB11 N = 700 p = 0,027 → Holm 0,081; gabungan dz = 0,49,
  CI95 bootstrap [+0,0023; +0,0584]. NB14 CI [+0,0425; +0,1076].
- **Purge dan label acak** (`63e7aba`): purge mengubah F1 −0,072…+0,012; JSD ≥ EDM 4/5 → 5/5
  setelah purge; label acak F1 0,446–0,508 (tingkat tebakan).
- **Pustaka** (`e217535`): EDM-Fuzzy bersumber Zhou dkk. 2021 (IEEE TII); laju drift
  0,003–0,17 poin/hari diturunkan menjadi asumsi; JSD terbukti buta urutan.

## Fase 13 — Notebook sidang interaktif dan catatan internal (2026-11-20 → 11-21)

- Commit `e7389d3`…`edca7a9`. Widget hari/sensor/fault/intensitas dengan prediksi bobot
  numpy; 540 kombinasi < 1 s; Kaggle COMPLETE 23,6–24,5 s; widget terbukti berubah langsung
  di editor Kaggle (`sidang/bukti_kaggle/widget_editor_2026-11-21/`); Q&A T1–T57; catatan
  internal dipindah ke Artifact. Tidak ada angka model baru.

---

## Tabel ringkas sebelum → sesudah

| Titik balik | Metrik | Sebelum | Sesudah | Sumber |
|---|---|---|---|---|
| v6 → JSD rich | akurasi 17 kelas JSD | 0,308 (v6) / 0,341 | 0,421 | `998fe20`; `JSD_FUZZY_IMPROVEMENT.md` |
| Perbaikan cache | F1 E/J; p | 0,465/0,541; 0,157 | 0,454/0,655; 0,0046 | `b843da0` |
| JSD drop-in | F1 E/J; p | 0,454/0,655 (rich) | 0,541/0,624; 0,028 | `461b17e` |
| Per sensor → sensor-selective | F1 sensor | 0,925 (tidak sah) | 0,846/0,902 (NB04 v6) | `175333c`; README |
| lbfgs → LM asli | pooled p | NB08 0,035 | NB11 v3 0,076 | README |
| Koreksi drift | NB11 menang; p | 12/15; 0,0504 | 11/15; 0,076 | `19fc3e8`, `7892ed6` |
| Koreksi drift | NB11 sensor-ID E/J | 0,721/0,765 | 0,634/0,675 | idem |
| Koreksi drift | NB09 LM/BR | 0,504/0,504 | 0,433/0,429 | `4e1c56e` |
| Holm | NB11 N = 700 p | 0,027 | 0,081 | `statistik_sel.csv` |
| Fault baru NB14 | F1 E/J; p | 0,458/0,486; 0,076 | 0,421/0,497; 0,0006 | `hasil14/14_draf_laporan.md` |
| Purge N = 700 | JSD ≥ EDM | 4/5 | 5/5 | `efek_purge_permutasi.csv` |
| NB15 v2 → v3 | F1 4 sensor | 0,616 | 0,640 | `2b43aea`, `01cb43a` |
| Runtime NB11 | wall | 525 menit (v1) | 610 menit (v3) | `131cdfb`, Ringkasan v3 |

## Gap, inkonsistensi, dan yang belum selesai

1. **Jeda kosong:** 2026-01-06 → 06-26 (±5,5 bulan) dan 2026-08-03 → 10-15. NB09/NB10 dari
   jeda kedua tidak pernah di-commit; bukti NB10 v4 hanya di `sidang/bukti_kaggle/`.
2. **Asal-usul data belum dikonfirmasi:** apa yang "generated" pada
   `tabel_sensor4_generated.csv`, dan siapa yang menginterpolasi `data_sensor.csv`.
   Keduanya mulai di stempel yang sama (2025-09-14 14:50:07). Checklist pembimbing A1/A2.
3. **Paper ACS:** judul/DOI tidak ditemukan di Crossref/OpenAlex, padahal 15b menyebutnya
   terbit. Checklist A3.
4. **Sumber laju drift:** tidak ada; 1 poin/hari juga asumsi. Checklist A4.
5. **Arah dan signifikansi berganti-ganti:** JSD terburuk (Jan) → terbaik (Jun); p 0,157 →
   0,0046 → 0,028 → tidak signifikan → 0,035 → 0,048/0,052 → 0,0511/0,0504/0,076 → 0,0006
   (NB14). Beberapa sel NB11 negatif (N200 S1/S3, N1000 S2/S3).
6. **Definisi JSD berubah di tengah jalan** (rich → drop-in). README baris 01 masih
   menampilkan angka JSD rich dengan MLP sklearn yang tidak sebanding. Notebook 01–08 tidak
   memakai LM.
7. **Non-determinisme:** NB10 dan NB11 berprotokol sama tetapi p berbeda (0,052 vs 0,076);
   tujuh sel terpotong `MAX_TRAIN_S`; belum ada ulangan multi-seed.
8. **Cacat kode diketahui, belum di-run ulang:** bobot keluaran mulai nol (`lm_pack`),
   sensor-ID memakai split acak 70/30, langkah LM yang ditolak menghabiskan iterasi,
   hardware efektif 5 % + 3 %, spike NB11 periodik tanpa seed, kebocoran di batas grup
   (sebagian sudah diukur dengan purge).
9. **Banyak uji, sel tidak independen:** Wilcoxon n = 5 tidak bisa di bawah 0,0625; setelah
   Holm, N = 700 NB11 tidak signifikan.
10. **Klaim yang dibuang:** F1 0,925 per sensor, JSD rich, fitur hibrida, XGBoost/BR/GA dari
    edisi sidang, klaim efisiensi, "Xavier dengan bias nol", angka arsip tak terbukti,
    ringkasan `hasil04` v5.
11. **Label "Revisi pasca-sidang"** pada NB13/14: sidang yang dimaksud tidak terdokumentasi.
12. **Batas representasi:** bias konstan tak terlihat, drift realistis sulit, S2–S4 F1
    0,21–0,53, EDM lebih stabil, JSD buta urutan.
13. **Artefak di luar git:** `hasil*/exports/` di-gitignore; sebagian bukti hanya di Kaggle.
14. **Pesan commit 2025 tidak informatif** ("add", "fix", "wip"); alasan perubahan Desember
    2025 hanya bisa direkonstruksi dari diff.

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Checklist tanda tangan pembimbing</b> (`sidang/qa/09_checklist_pembimbing.md`)</summary>

# 09 — Checklist tanda tangan pembimbing

Satu halaman untuk dibawa ke Bu Luh sebelum sidang. Setiap baris butuh
keputusan pembimbing (setuju / ubah / hapus), bukan hanya dibaca. Baris yang
belum disetujui **tidak** boleh muncul di slide.

Commit acuan: tulis hash `git log -1 --format=%h` saat checklist dicetak:
`________`.

## A. Fakta yang hanya bisa dipastikan pembimbing

| # | Pertanyaan | Kenapa penting | Jawaban pembimbing |
|---|---|---|---|
| A1 | Asal-usul `data_sensor.csv`: sensor apa, lokasi, siapa yang menginterpolasi ke grid 30 detik? (`02_kelemahan.md` B6) | Penguji hampir pasti bertanya "data asli atau buatan?" | |
| A2 | Apa yang "generated" pada `tabel_sensor4_generated.csv` (NB15)? | Disebut "data buatan" di chat 18/11/26 | |
| A3 | Judul dan DOI paper ACS (EDM-Fuzzy + ANN-LM). Apakah sama dengan Wardhani dkk. 2026 JITK, atau paper lain? (`07_pustaka_dan_kebaruan.md` (a)) | Belum ditemukan di Crossref/OpenAlex; tidak boleh disitasi tanpa judul pasti | |
| A4 | Laju drift 0,003–0,17 poin/hari: ada sumber? Kalau tidak, setuju ditulis "asumsi skenario"? | Tidak ditemukan sumber terbit (`07` (e)) | |

## B. Klaim yang akan diucapkan

| # | Klaim (persis seperti di slide) | Bukti | Setuju? |
|---|---|---|---|
| B1 | JSD-Fuzzy lebih tinggi di 11 dari 15 sel, rata-rata selisih F1 +0,028 (NB11) | `11_rq4_selisih.csv`; ledger | |
| B2 | Gabungan NB11 belum signifikan: p = 0,076 (Wilcoxon 0,107); disebut "konsisten", bukan "terbukti" | `statistik_sel.csv` | |
| B3 | N = 700: p = 0,027 tanpa koreksi, p = 0,081 setelah Holm (3 ukuran jendela) | `statistik_sel.csv` kolom `p_t_holm_3N` | |
| B4 | NB14 gabungan: p = 0,0006, d_z = 1,13, CI bootstrap 95 % [+0,043; +0,108] | `statistik_sel.csv` | |
| B5 | Sensor-ID JSD 0,675 vs EDM 0,634, dengan catatan split acak 70/30 (optimistis) | NB11 §12; `02` F1 | |
| B6 | Kebaruan = JSD sebagai jarak antar template di dalam fuzzy entropy; EDM-Fuzzy dari Zhou dkk. 2021 | `07` (b) | |
| B7 | Hasil purge dan kontrol label acak (angka dari `efek_purge_permutasi.csv`) | `hasil_uji/` | |

## C. Batasan yang akan diakui sendiri

| # | Batasan | Setuju diucapkan? |
|---|---|---|
| C1 | Gangguan disimulasikan, bukan kerusakan fisik | |
| C2 | JSD-Fuzzy buta urutan (template naik vs turun memberi JSD = 0) | |
| C3 | Sedikit kebocoran di batas grup CV (diukur dengan purge) | |
| C4 | Bobot awal tidak persis Xavier (`lm_pack`/`lm_unpack`), sama untuk kedua metode | |
| C5 | Bias konstan tidak terdeteksi karena template dipusatkan | |
| C6 | Hardware malfunction efektif 5 % hilang + 3 % macet | |

## D. Gerbang teknis (diisi kandidat sebelum bertemu pembimbing)

Tempel keluaran baris terakhir tiap perintah:

```
./venv/bin/python sidang/qa/cek_klaim.py                  -> GERBANG KLAIM: ______
./venv/bin/python -m pytest -q sidang/qa                  -> ___ passed
./venv/bin/python sidang/qa/jalankan_notebook.py sidang/04_Sidang_Interaktif.ipynb \
    --keluar sidang/bukti_demo --batas-sel 2 --batas-total 60   -> HASIL: ______
./venv/bin/python sidang/qa/cek_algoritma_tangan.py       -> semua [OK]? ______
```

## Tanda tangan

Pembimbing: ______________________  Tanggal: ____________

Kandidat:   ______________________  Tanggal: ____________

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Panduan demo</b> (`sidang/PANDUAN_DEMO.md`)</summary>

# Panduan Demo — Deteksi Fault 4 Sensor Kelembaban

Panduan ini untuk siapa saja yang perlu menjalankan atau menonton demo
(Kelvin, pembimbing, penguji). Tidak perlu paham pemrograman.

**Intinya:** model sudah dilatih dan **disimpan ke file**. Demo hanya memakai
model itu, jadi selesai dalam **sekitar 1 menit**, bukan 10 jam.

---

## 1. Kenapa tidak perlu training ulang?

| | Run penelitian (notebook 11) | Demo (notebook ini) |
|---|---|---|
| Yang dikerjakan | hitung fitur ribuan potongan data + cari arsitektur terbaik + uji 5 lipatan, diulang 15 kali | buka model tersimpan, hitung fitur untuk beberapa potongan baru, lalu tebak |
| Waktu | 10,2 jam di Kaggle | sekitar 1 menit |
| Gunanya | menghasilkan **angka resmi** (F1, uji statistik) | **memperlihatkan** cara kerja sistem secara langsung |

Model disimpan sekali oleh notebook 16 (`simpan_model.py`, Kaggle
`simpan-model-final-deteksi-fault`). Isinya 10 model: 5 skenario × 2 metode
(EDM-Fuzzy dan JSD-Fuzzy), window N = 700.

## 2. Tautan (semua Private, buka dengan akun Kaggle Kelvin)

| Untuk apa | Tautan |
|---|---|
| **Demo** (dipakai saat Zoom) | https://www.kaggle.com/code/kelvindsmn/demo-model-tersimpan-deteksi-fault |
| File model + data (Dataset) | https://www.kaggle.com/datasets/kelvindsmn/model-deteksi-fault-tersimpan |
| Proses melatih + menyimpan model (58 menit, sekali saja) | https://www.kaggle.com/code/kelvindsmn/simpan-model-final-deteksi-fault |
| Ringkasan hasil sidang | https://www.kaggle.com/code/kelvindsmn/sidang-deteksi-fault-4-sensor-kelembaban |
| Run penelitian lengkap (bukti 10 jam + log) | https://www.kaggle.com/code/kelvindsmn/jsd-edm-ann-lm-lengkap-q3 |

Kalau penguji ingin melihat sendiri, kernel bisa dibagikan lewat tombol
**Share** di Kaggle (tetap Private, hanya orang yang diundang).

## 3. Persiapan (10 menit sebelum Zoom)

1. Login Kaggle, buka tautan **Demo**.
2. Klik **Copy & Edit** (pojok kanan atas). Notebook terbuka di mode edit.
3. Pastikan di panel kanan **Internet: On**.
4. Menu **Run → Run all** sekali sebagai pemanasan. Tunggu sampai semua sel
   selesai (di Kaggle waktu kernel sekitar 2,1 menit menurut log
   `hasil16/demo-kaggle-v2.log`, belum termasuk menyalakan mesin; di laptop
   63 detik, 21 detik tanpa Langkah 7 — bukti `sidang/bukti_demo/`). Kalau tidak ada tulisan merah, siap dipakai.
5. Di Zoom, bagikan layar jendela browser itu saja (bukan seluruh layar).

**Cadangan kalau internet bermasalah:** di laptop, buka
`sidang/03_Demo_Model_Tersimpan.ipynb` di VS Code atau Jupyter, pilih kernel
`venv` repo ini, lalu **Run All**. Model ada di folder `models/`, data di
`data_sensor.csv` — tidak butuh internet.

## 4. Naskah demo singkat (sekitar 5 menit)

| Menit | Tunjukkan | Katakan |
|---|---|---|
| 0:00 | judul notebook | "Ini demo pakai model yang sudah dilatih. Tidak ada training ulang, jadi cepat." |
| 0:30 | **Langkah 1** — tabel model | "Ada 10 model: 5 skenario kerusakan × 2 metode. Kolom kanan adalah F1 resmi dari penelitian." |
| 1:00 | **Langkah 3** — grafik biru/merah | "Biru data lapangan asli. Merah data yang sama setelah sensor S2 kami beri empat kerusakan: naik pelan (drift), lonjakan, geser, dan nilai jatuh/macet." |
| 2:00 | **Langkah 4** — kurva entropi | "Sinyal diubah jadi 60 angka keteraturan pola. Kerusakan menggeser kurvanya." |
| 3:00 | **Langkah 5** — tabel tebakan | "Jawaban benar: window asli *fault free*, window rusak *faulty*. Satu contoh bisa benar bisa salah; peluang dekat 50 % artinya model ragu." |
| 4:00 | **Langkah 6** — uji banyak window | "Ini yang adil: 30 potongan acak. Di run uji 2026-11-20 kedua metode benar 80 % (24 dari 30)." |
| 4:30 | (opsional) **Langkah 7** | "Kalau ingin lihat proses training: ini versi mini, loss turun tiap iterasi Levenberg–Marquardt." |

## 5. Cara mengganti contoh (sel *Konfigurasi*)

Ubah salah satu baris ini, lalu **Run all** lagi:

```python
HARI_MULAI = 10                  # potongan data mulai hari ke-berapa (0–95)
SENSOR_RUSAK = ["S2"]            # contoh lain: ["S1", "S3"] atau ["S1","S2","S3","S4"]
FAULT = ["drift", "spike", "bias", "hardware"]   # satu atau kombinasi
SEED = 1                         # ganti angka = pola kerusakan acak yang lain
```

Dua hal soal `HARI_MULAI`:

- **Drift** hanya ada di tiga episode 3 hari (hari 9,8–12,8 · 39,1–42,1 ·
  68,5–71,5), persis seperti saat training. Kalau memilih `drift`, pakai hari
  **10** atau **69**. Di luar itu notebook menulis "di luar episode drift".
- **Hindari hari 1–3, 17–18, 33–49, dan 77–83.** Di file `data_sensor.csv`
  bagian itu berupa garis lurus sempurna (keempat sensor turun/naik lurus),
  jadi grafiknya membingungkan. Lihat catatan di bagian 8.

Arti jenis kerusakan:

| Nama | Artinya di sensor nyata |
|---|---|
| `drift` | nilai pelan-pelan naik (sensor bergeser, perlu kalibrasi ulang) |
| `spike` | lonjakan sesaat (gangguan listrik/derau) |
| `bias` | nilai bergeser tetap (offset) |
| `hardware` | nilai macet atau hilang (kabel/modul bermasalah) |

Jumlah jenis kerusakan menentukan model yang dipakai: 1 jenis → model S2,
2 → S3, 3 → S4, 4 → S5. Model S1 (normal vs rusak) selalu ikut.

## 6. Cara membaca hasil

- **Tebakan** — jawaban model. Benar kalau window *asli* = `fault free` dan
  window *diberi fault* = `faulty` / nama kerusakannya.
- **Yakin** — peluang tebakan (50 % pada 2 kelas = ragu, 90 % = sangat yakin).
- **Sensor ditebak rusak** — sensor dengan skor di atas 0,5.
- **Akurasi** di Langkah 6 — persentase benar dari banyak window acak.

## 7. Kalau model salah tebak (ini normal, jangan panik)

Hasil contoh default (hari 10, empat fault di S2) pada run uji 2026-11-20:
6 dari 8 tebakan benar (S1-EDM benar dua-duanya, S5-JSD benar dua-duanya;
S1-JSD menyebut window asli *faulty*, S5-EDM melewatkan window rusak).
Contoh ini **tidak** dipilih supaya hasilnya bagus — dipilih karena keempat
kerusakan terlihat jelas di grafik.

Model **tidak sempurna**, dan itu sudah dilaporkan jujur di penelitian:

| Skenario (N = 700) | F1 EDM-Fuzzy | F1 JSD-Fuzzy |
|---|---|---|
| S1 normal vs rusak | 0,59 | 0,67 |
| S2 satu jenis kerusakan | 0,33 | 0,35 |
| S3 dua jenis | 0,18 | 0,20 |
| S4 tiga jenis | 0,34 | 0,43 |
| S5 empat jenis sekaligus | 0,80 | 0,87 |

Sumber: `hasil11/exports/11_rq4_performa_ann.csv` (notebook 11 v3).

Jawaban yang bisa dipakai:

- *"Satu contoh bisa salah. Ukuran yang benar adalah F1 dari cross-validation
  di tabel tadi, dan Langkah 6 yang menguji banyak window."*
- **Bias saja hampir tidak terdeteksi** — memang begitu: langkah *centroid
  shifting* di dalam entropi menghapus pergeseran tetap (AUC ≈ 0,50, temuan
  notebook 12). Itu batasan yang sudah ditulis, bukan bug.
- Kombinasi empat kerusakan (S5) paling mudah dikenali; dua kerusakan (S3)
  paling sulit.
- Tebakan dengan peluang dekat 50 % bisa berbalik kalau jalan ulang dengan
  seed lain: entropi memakai 128 titik acuan yang diambil acak (`n_ref`),
  jadi fiturnya sedikit berubah. Itu sebabnya yang dinilai adalah banyak
  window (Langkah 6) dan CV, bukan satu contoh.
- Akurasi Langkah 6 (sekitar 80 %) lebih tinggi dari F1 CV karena posisi
  potongannya juga dipakai saat melatih (kerusakannya baru). Notebook menulis
  catatan ini sendiri; angka resmi tetap F1 CV.

## 8. Pertanyaan yang mungkin muncul

**"Model ini sama dengan yang di penelitian?"** Rumus fitur, simulator
kerusakan, dan algoritma LM sama persis (disalin tanpa perubahan ke
`deteksi_fault_core.py`). Arsitektur dicari dengan grid yang sama, lalu model
dilatih di semua data skenario dan disimpan. Angka resmi tetap dari CV
notebook 11.

**"Kenapa window 700?"** Satu-satunya panjang window yang selisih F1 JSD vs
EDM-nya signifikan (p = 0,027).

**"Kenapa ada bagian grafik yang lurus sempurna?"** Di `data_sensor.csv`
ada empat bagian (total sekitar 26 dari 98 hari: hari 1–3, 17–18,6, 33–49,9,
77–83) yang nilainya berubah lurus sempurna di keempat sensor. Itu ada di file
data aslinya, bukan dari pengolahan di notebook. Kemungkinan besar bagian yang
ditambal saat pengambilan data — cek ke sumber data sebelum menjawab pasti.

**"Bisa dipakai di alat sungguhan?"** Bisa dalam bentuk ini: data 2,4 hari
terakhir → hitung 60 angka entropi → model tersimpan menebak. Klaim
real-time/edge tidak dibuat di penelitian.

## 9. Kalau ada masalah

| Gejala | Solusi |
|---|---|
| `Folder models/ tidak ditemukan` | Di Kaggle: panel kanan **Input → Add Input → Datasets → `model-deteksi-fault-tersimpan`**. Di laptop: pastikan folder `models/` ada di repo. |
| `Sumber data: unduh` lalu error | Internet mati. Pakai cadangan laptop (bagian 3). |
| `HARI_MULAI terlalu besar` | Isi angka 0–95. |
| `Kombinasi fault [...] tidak ada` | Pakai nama persis: `drift`, `spike`, `bias`, `hardware`. Semua 15 kombinasi tersedia. |
| `PERHATIAN: ... <= 1% sampel` | Potongan itu tidak cukup rusak (biasanya karena hanya `drift` di luar episode). Ganti `HARI_MULAI` ke 10 atau 69. |
| Sel lama sekali (> 3 menit) | Klik **Run → Restart & clear outputs**, lalu **Run all**. |

---

Dibuat 2026-11-20. Model: notebook 16 (`simpan_model.py`). Demo:
`build_sidang_demo_notebook.py` → `sidang/03_Demo_Model_Tersimpan.ipynb`.

</details>

<details class="catatan-internal">
<summary><b>Lampiran — Peta klaim ke notebook</b> (`sidang/MAPPING.md`)</summary>

# MAPPING — notebook sidang → notebook penelitian

Dokumen ini satu-satunya acuan kalau ada pertanyaan "angka ini dari mana?".
Notebook sidang **tidak menghitung ulang** model; ia membaca hasil yang sudah
diekspor notebook penelitian, lalu menampilkannya dengan bahasa sederhana.

- Commit acuan: `27737ee` (angka 01–11 sesudah koreksi drift)
- Tanggal: 2026-10-26
- Display name Kaggle: **Riset Deteksi Fault 4 Sensor Kelembaban**
- Semua kernel berstatus **Private**; tautan hanya untuk penguji.
- Notebook sidang utama: `sidang/00_Sidang_Deteksi_Fault.ipynb`
  ([Kaggle](https://www.kaggle.com/code/kelvindsmn/sidang-deteksi-fault-4-sensor-kelembaban))
  — menggabungkan Peta, Alur, dan Hasil dalam satu kernel + tabel tautan ke 12
  notebook penelitian. Versi Kaggle: **v2** (performa + sensor-ID + catatan
  data; statistik dan pelatih lain tidak ditampilkan).

---

## Tabel A — bagian sidang → notebook asli → sel → kernel

| Bagian sidang | Notebook asli | Sel | Kernel (versi) |
|---|---|---|---|
| Peta: alur 8 tahap | `07` §0–§9 | 1–43 | `jsd-fuzzy-cv-5skenario` v23 |
| Peta: ringkasan angka | `11` §13 | 47–48 | `jsd-edm-ann-lm-lengkap-q3` v3 |
| Alur: data mentah | `Ilustrasi` §1–§2 | 1–7 | `ilustrasi-data-akuisisi-dan-fault` v3 |
| Alur: broker 4 kanal | `02` | 1–13 | `jsd-fuzzy-broker-architecture-v2` v3 |
| Alur: sinkron 5 menit | `07` §3–§3c | 9–16 | `jsd-fuzzy-cv-5skenario` v23 |
| Alur: injeksi fault | `07` §4–§4b | 17–20 | `jsd-fuzzy-cv-5skenario` v23 |
| Alur: segmentasi | `07` §5 | 21–25 | `jsd-fuzzy-cv-5skenario` v23 |
| Alur: entropi | `11` §5–§7b | 15–24 | `jsd-edm-ann-lm-lengkap-q3` v3 |
| Hasil: performa utama | `11` §10 | 33–40 | `jsd-edm-ann-lm-lengkap-q3` v3 |
| Hasil: sensor mana rusak | `11` §12 | 43–46 | `jsd-edm-ann-lm-lengkap-q3` v3 |
| Hasil: catatan data 0 | data mentah | — | `data_sensor.csv` |
| Hasil: batasan | `11` §13 | 47–48 | `jsd-edm-ann-lm-lengkap-q3` v3 |

Tidak ditampilkan di edisi sidang (tetap ada di arsip penelitian):
jenis fault (`05`), two-stage (`04`), empat pelatih (`09`), ANN vs XGBoost
(`06`), dan statistik (`11` §11). Alasannya: fokus sidang adalah perbandingan
EDM vs JSD dengan **satu** protokol ANN-LM.

Nomor sel = indeks 0-based di file `.ipynb` (sel yang sama dengan yang dibuka
di Kaggle).

## Tabel B — angka mana yang dikutip

| Kalau ditanya | Kutip dari | Jangan kutip dari | Alasan |
|---|---|---|---|
| Perbandingan EDM vs JSD utama | `11` v3 | `07`, `08` | 11 memakai ANN-LM asli, bukan `lbfgs` |
| Sensor mana yang rusak | `11` §12 | `07` §10 | 07 masih `lbfgs` |
| Broker 4 kanal vs digabung | `02` v3 | — | khusus notebook 02 |
| (arsip) pelatih terbaik | `09` v3 | `01`, `03` | 01/03 memakai sklearn, bukan LM |
| (arsip) jenis fault per sensor | `05` v2 | — | khusus notebook 05 |
| (arsip) ANN vs XGBoost | `06` v4 | — | khusus notebook 06 |

Catatan supersesi:

- `11` = versi lengkap dari `10` (10 tanpa identifikasi sensor).
- `10` = `08` dengan LM asli menggantikan `lbfgs`.
- Protokol CV `10` dan `11` identik; yang beda hanya batas sesi Kaggle
  (10,5 vs 11,5 jam). Jumlah sel menang **sama**: `10` 11/15, `11` 11/15.
  p gabungan beda (0,052 vs 0,076) karena 7 sel S2/S4 dihentikan batas waktu
  latih dinding. Yang dipakai di sidang: notebook 11.
- `07` dan `08` tetap dirujuk untuk **diagram/alur**, bukan untuk angka akhir.

## Tabel C — jenis pelatih (sumber salah klaim paling sering)

| Notebook | Pelatih | Catatan |
|---|---|---|
| `07`, `08` | sklearn `MLPClassifier` (`lbfgs`) | bukan LM paper |
| `01`, `04`, `05` | sklearn `MLPClassifier` | bukan LM paper |
| `09` | `fit_lm` sendiri + Bayesian + GA + Adam | LM versi paper |
| `10`, `11` | `lm_fit` + Jacobian analitik | LM versi paper, tanpa `lbfgs` |
| `06` | `XGBClassifier` | pembanding non-ANN |

Saat sidang, sebut pelatih sesuai tabel ini. Jangan menyebut "LM" untuk
notebook 01/04/05/07/08.

## Tabel D — arti file di `sidang/assets/`

| Folder | File | Isi |
|---|---|---|
| `11` | `11_rq4_performa_ann.csv` | F1 15 sel (5 skenario × 3 ukuran window) |
| `11` | `11_rq4_selisih.csv` | selisih JSD − EDM + status menang |
| `11` | `11_rq4_performa_N700.png` | grafik N=700 |
| `11` | `11_rq4_confusion_matrix_N700.png` | confusion matrix |
| `11` | `11_sensor_ringkas.csv` | F1 sensor-ID 15 sel |
| `11` | `11_sensor_f1.png` | grafik sensor-ID |
| `11` | `11_akuntansi_window.csv` | jumlah window & pembuangan |
| `11` | `11_similarity_euclidean_vs_jsd.png` | EDM vs JSD |
| `11` | `11_grid_fault_vs_asli_S1.png` | grid 16 kondisi |
| `ilustrasi` | `ILU_empat_kanal_bersih.png` | data mentah 4 kanal |
| `ilustrasi` | `ILU_grid_fault_vs_asli_S1.png` | grid injeksi |
| `ilustrasi` | `ILU_tabel_10_langkah.csv` | 10 langkah pembersihan |

Aset untuk bagian yang tidak lagi ditampilkan (04/05/06/07/08/09) sudah
dihapus dari repo; file aslinya tetap ada di kernel Kaggle masing-masing.

## Tabel E — kernel & versi

| # | Slug | Versi | Status |
|---|---|---|---|
| 01 | `jsd-fuzzy-ann-paper` | 6 | private |
| 02 | `jsd-fuzzy-broker-architecture-v2` | 3 | private |
| 03 | `jsd-fuzzy-scenario-ladder` | 3 | private |
| 04 | `jsd-fuzzy-twostage-sensorid` | 6 | private |
| 05 | `jsd-fuzzy-faulttype-per-sensor` | 2 | private |
| 06 | `jsd-xgb-perscenario` | 4 | private |
| 07 | `jsd-fuzzy-cv-5skenario` | 23 | private |
| 08 | `jsd-fuzzy-vs-edm-fuzzy-paper-q3` | 17 | private |
| 09 | `ann-lm-vs-bayesian-vs-ga-paper-pmdc` | 3 | private |
| 10 | `jsd-vs-edm-dengan-ann-lm-asli-paper-pmdc` | 4 | private |
| 11 | `jsd-edm-ann-lm-lengkap-q3` | 3 | private |
| 12 | `sensitivitas-parameter-fault-dan-jsd` | 1 | private |
| — | `ilustrasi-data-akuisisi-dan-fault` | 3 | private |

Notebook sidang (semua private):

| Notebook | Slug | Status |
|---|---|---|
| `00_Sidang_Deteksi_Fault.ipynb` | `sidang-deteksi-fault-4-sensor-kelembaban` | **utama** (v2) |
| `00_Peta_Program.ipynb` | `peta-program-deteksi-fault-4-sensor-kelembaban` | digantikan |
| `01_Alur_Sistem.ipynb` | `alur-sistem-deteksi-fault-4-sensor-kelembaban` | digantikan |
| `02_Hasil.ipynb` | `hasil-deteksi-fault-4-sensor-kelembaban` | digantikan |
| `12_Sensitivitas_Parameter.ipynb` | `sensitivitas-parameter-fault-dan-jsd` | pendukung (bukan sidang) |
| `03_Demo_Model_Tersimpan.ipynb` | `demo-model-tersimpan-deteksi-fault` | **demo** (v2, input dataset `model-deteksi-fault-tersimpan`) |
| `16_Simpan_Model_Final.ipynb` | `simpan-model-final-deteksi-fault` | melatih + menyimpan model demo (v1, 58 menit) |

Demo memakai model tersimpan (N = 700, 5 skenario × 2 metode) dari
`simpan_model.py`; panduan dan naskah: `sidang/PANDUAN_DEMO.md`. Angka resmi
tetap dari notebook 11 v3 — akurasi di demo hanya ilustrasi.

Status private dan display name `Riset Deteksi Fault 4 Sensor Kelembaban`
diverifikasi di Kaggle pada 2026-10-26.

Hasil eksperimen 12 disimpan di `hasil12/exports/` (CSV + PNG + ringkasan).
Angka pendukung: bias AUC ~0,50 (0,08–5 poin) karena centroid shifting;
hardware (0,3; 0,2) TPR 0,908/FPR 0,083; rekomendasi binning diuji dan
divalidasi N700/S1 (`12_rekomendasi.csv`).

## Catatan angka yang sering tertukar

| Angka | Nilai benar | Sumber |
|---|---|---|
| Sel JSD menang (notebook 11) | **11/15** | `11_rq4_selisih.csv` + log |
| Rata-rata selisih F1 | **+0,028** | log pooled |
| Sensor-ID F1 | **0,675 vs 0,634** (menang 9/15) | `11_sensor_ringkas.csv` |
| Semua-4-benar | **0,307 vs 0,295** | log kernel 11 |
| (arsip) Tahap-2 17 kelas | **0,902 vs 0,846** | `tahap2_ringkas_per_tugas.csv` |
| (arsip) Semua-4-benar (04) | **0,920 vs 0,800** | `tahap2_per_jumlah_fault.csv` |
| (arsip) Pelatih terbaik | BR 0,429 ≈ LM 0,433 (p = 0,81) | `09_rq2_peringkat_pelatih.csv` |
| (arsip) XGBoost JSD+Time | **0,750** rata-rata 5 skenario | `xgb_perscenario_results.csv` |

</details>